# RoboLab 复现与学习记录（RTX 4090）

**目标**：在高保真仿真中复现 NVIDIA RoboLab，重点观察通用机器人策略在视觉识别、空间关系推理、多步骤/过程化操作三个维度上的泛化能力。

**硬件定位**：RTX 4090 24GB VRAM 可以做安装验证、单任务 smoke run、小批量子集评测；完整 RoboLab-120 或较高 `num_envs` 并行需要更谨慎，官方推荐 48GB+ VRAM。

**本 notebook 的定位**：

- 先做可记录、可复跑、可交付的复现实验台账。
- 默认不执行重型安装和仿真命令，避免在非 Ubuntu/非 4090 环境误跑。
- 搬到 Ubuntu 22.04+ / RTX 4090 机器后，把配置 cell 里的执行开关改为 `True`，按阶段运行。
- 所有命令输出、指标汇总、图表和学习记录统一写到 `robolab_repro_artifacts/`。

**来源边界**：本 notebook 只围绕论文、官网、GitHub README/docs 中公开的信息组织复现流程；具体模型权重、策略服务端和许可证按实际使用的模型另行记录。

## 0. 官方来源与当前事实核对

复现前先固定来源，避免照着旧命令走偏。

- 论文：`arXiv:2604.09860v3`，RoboLab-120 包含 120 个任务，覆盖 visual / procedural / relational 三个能力轴。
- 官网：强调 agentic scene/task generation、结果 dashboard、RoboLab benchmark 和 sensitivity analysis。
- GitHub README 当前说明：需要 `uv` 与系统 `ffmpeg`；`uv sync` 会自动安装 Isaac Sim 5.0 和 Isaac Lab 2.2.0。
- GitHub README 当前说明：依赖要求为 Ubuntu 22.04+、Python 3.11、NVIDIA RTX GPU；官方建议 48GB+ VRAM，4090 需要先降并行度。
- GitHub HEAD（准备 notebook 时）：`7d45d74904eade3b578a8eb1f2f9f89bc3d40326`。

**执行策略修正**：主路线不先手动 clone/build Isaac Sim。RoboLab 现在的主安装路径是 clone RoboLab 后用 `uv sync` 拉起 Isaac Sim/Lab 依赖。只有做 Isaac Sim 内核开发或官方安装失败时，才单独走 Isaac Sim 源码构建路线。

In [ ]:
# ===== 1. 导入标准库：这些库负责路径、时间、JSON、系统信息和命令执行 =====
from pathlib import Path  # 用 Path 统一处理 Windows/Linux 路径，避免手写字符串拼路径。
import datetime as _dt  # 给日志和学习记录生成带日期/时区的时间戳。
import json  # 读写 repro_status、远端 summary、命令日志等结构化证据。
import os  # 读取环境变量，例如 ROBOLAB_WORK_ROOT / ROBOLAB_DIR。
import platform  # 判断当前是不是 Linux，避免在 Windows 上误启动 Isaac Sim。
import shlex  # 在展示 shell 命令时安全地转义路径和参数。
import subprocess  # 真正执行 git、uv、pytest、RoboLab 脚本等外部命令。
import sys  # 输出当前 Python 版本，便于排查 Python 3.11 要求。
import time  # 计算命令运行耗时。

# ===== 2. 导入可选分析库：没有安装也不让 notebook 中断 =====
try:
    import pandas as pd  # 用来把 episode_results.jsonl 汇总成表格。
except Exception:
    pd = None  # pandas 不存在时，后面的分析 cell 会退化为打印 JSON。

try:
    import matplotlib.pyplot as plt  # 用来画成功率柱状图。
except Exception:
    plt = None  # matplotlib 不存在时跳过绘图，不影响复现记录主体。

try:
    from IPython.display import Markdown, display  # 在 Jupyter 里渲染 Markdown 表格。
except Exception:
    Markdown = None  # 非 Jupyter 环境下没有 Markdown 对象时使用 print。
    display = print  # 让脚本模式也能看到输出。

# ===== 3. 定义 notebook 的工作目录和证据目录 =====
NOTEBOOK_ROOT = Path.cwd()  # 当前执行 notebook 的目录；建议就是本交付物目录。
ARTIFACT_DIR = NOTEBOOK_ROOT / "robolab_repro_artifacts"  # 所有本地运行证据统一放这里。
WORK_ROOT = Path(os.environ.get("ROBOLAB_WORK_ROOT", NOTEBOOK_ROOT / "robolab_workspace")).expanduser()
# WORK_ROOT 是 RoboLab 仓库的父目录；可通过环境变量改到大磁盘。
ROBOLAB_DIR = Path(os.environ.get("ROBOLAB_DIR", WORK_ROOT / "RoboLab")).expanduser()
# ROBOLAB_DIR 是 RoboLab 仓库目录；如果你已经 clone 好，可用环境变量指向它。

# ===== 4. 判断当前平台，决定哪些 cell 默认可以执行 =====
IS_LINUX = platform.system().lower() == "linux"  # Isaac Sim/RoboLab 主流程需要 Ubuntu/Linux。

# ===== 5. 执行开关：默认只允许轻量 preflight，重型步骤必须手动打开 =====
EXECUTE_PREFLIGHT = IS_LINUX  # Linux 上允许跑系统检查；Windows 上默认 dry-run。
EXECUTE_INSTALL = False  # 是否真的 clone/uv sync；默认 False 防止误装大依赖。
EXECUTE_TESTS = False  # 是否真的跑 pytest；Isaac import 很重，所以默认 False。
EXECUTE_NO_POLICY_SMOKE = False  # 是否跑无策略仿真 smoke；需要 GPU/Isaac 环境。
EXECUTE_POLICY_SMOKE = False  # 是否跑 Pi0/Pi05 单任务策略；需要 OpenPI server ready。
EXECUTE_SUBSET_EVAL = False  # 是否跑小子集策略评测；耗时更长，默认关闭。

# ===== 6. 4090 保守参数：先保证能跑通，再逐步提高并行度 =====
NUM_ENVS_4090_SMOKE = 1  # 4090 首次策略 smoke 只开 1 个环境，降低 OOM 风险。
NUM_ENVS_4090_CAUTIOUS = 2  # 小子集评测的保守并行度；确认显存后再提高。
NUM_RUNS = 1  # 每个任务先跑 1 次，只验证链路，不做统计结论。
POLICY_NAME = "pi05"  # RoboLab policy runner 使用的策略名。
SMOKE_TASK = "BananaInBowlTask"  # 第一条闭环任务，简单、低压力、便于排错。
# SUBSET_TASKS 覆盖 pick/place、语言组合、空间关系、重定向、堆叠五类能力。
SUBSET_TASKS = [
    "BananaInBowlTask",  # 语义识别 + 基础 pick/place。
    "RubiksCubeAndBananaTask",  # 多目标 conjunction：同时处理 cube 和 banana。
    "RubiksCubeLeftOfBowlTask",  # 空间关系：判断 left of bowl。
    "ReorientWhiteMugsTask",  # 过程/重定向：把白色杯子转正。
    "Stack3RubiksCubeTask",  # 多步骤堆叠：观察长 horizon 失败模式。
]

# ===== 7. Isaac/Kit 运行环境变量 =====
OMNI_ENV = {"OMNI_KIT_ACCEPT_EULA": "Y"}  # 非交互运行时显式接受 EULA，避免卡在提示界面。
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)  # 确保证据目录存在。

# ===== 8. 打印当前配置，方便截图/日志留档 =====
print("Notebook root:", NOTEBOOK_ROOT)  # notebook 当前执行位置。
print("Artifact dir :", ARTIFACT_DIR)  # 本地证据输出目录。
print("Work root    :", WORK_ROOT)  # RoboLab 工作区父目录。
print("RoboLab dir  :", ROBOLAB_DIR)  # RoboLab 仓库目录。
print("Platform     :", platform.platform())  # OS/内核等平台信息。
print("Python       :", sys.version.replace("\n", " "))  # 当前 Python 解释器版本。

In [ ]:
COMMAND_LOG = ARTIFACT_DIR / "command_log.jsonl"

def _now():
    # 生成带时区的时间戳，方便本地日志和远端日志对齐。
    return _dt.datetime.now().astimezone().isoformat(timespec="seconds")

def _append_jsonl(path: Path, record: dict):
    # JSONL 适合持续追加；dry-run 和真实执行的命令都会留下证据。
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def run(cmd, *, cwd=None, env=None, execute=True, check=False, timeout=None, label=None):
    # 统一的命令执行入口：开关关闭时只记录计划，开关打开时执行并记录证据。
    cwd_path = Path(cwd).expanduser() if cwd else NOTEBOOK_ROOT
    record = {
        "time": _now(),
        "label": label,
        "cwd": str(cwd_path),
        "cmd": cmd if isinstance(cmd, str) else list(cmd),
        "execute": bool(execute),
    }
    printable = cmd if isinstance(cmd, str) else " ".join(shlex.quote(str(x)) for x in cmd)
    print(f"\n$ cd {cwd_path}")
    print(f"$ {printable}")
    if not execute:
        # dry-run 也写入 command_log.jsonl，后续能审计“原计划要跑什么”。
        record.update({"status": "dry_run", "returncode": None, "stdout": "", "stderr": ""})
        _append_jsonl(COMMAND_LOG, record)
        print("[dry-run] command recorded only")
        return record

    # 合并单条命令需要的环境变量，例如 Isaac/Kit 的 EULA 确认变量。
    run_env = os.environ.copy()
    if env:
        run_env.update(env)
    start = time.time()
    proc = subprocess.run(
        cmd,
        cwd=str(cwd_path),
        env=run_env,
        shell=isinstance(cmd, str),
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    elapsed = round(time.time() - start, 3)
    record.update(
        {
            "status": "completed",
            "returncode": proc.returncode,
            "elapsed_s": elapsed,
            "stdout": proc.stdout[-20000:],
            "stderr": proc.stderr[-20000:],
        }
    )
    _append_jsonl(COMMAND_LOG, record)
    # notebook 里只打印尾部，避免输出过长；更完整的尾部保存在 JSONL 里。
    if proc.stdout:
        print(proc.stdout[-4000:])
    if proc.stderr:
        print(proc.stderr[-4000:])
    print("returncode:", proc.returncode, "elapsed_s:", elapsed)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed: {printable}")
    return record

def write_status(name: str, data: dict):
    # 状态 JSON 是最终 checklist 的机器可读依据，不只靠人眼看日志。
    path = ARTIFACT_DIR / f"{name}.json"
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print("wrote", path)
    return path

## 0.00 精讲问题来源与核心内容索引

下面这节来自本目录的 [EXPLAIN_SOURCE_QUESTION_INDEX.md](./EXPLAIN_SOURCE_QUESTION_INDEX.md)。它补的是所有精讲共同缺的一层：每个精讲到底回答哪个问题，问题来自论文哪一节、哪张图、哪个 appendix 或哪段源码，来源里的核心内容是什么，以及复现时应该落到哪些代码/证据文件。

# 精讲问题来源与核心内容索引

> **【绿色标识｜核心结论】** 这张索引不是新的论文精讲，而是所有精讲的“出处地图”。它回答：每个精讲为什么要讲、问题来自论文哪一节/哪张图/哪个 appendix/哪段代码、原始内容的核心是什么、我们在复现里应该怎么用。  
> **【蓝色标识｜主要来源】** 论文 [RoboLab arXiv HTML](https://arxiv.org/html/2604.09860v3)、项目页 [NVIDIA RoboLab](https://research.nvidia.com/labs/srl/projects/robolab/)、官方代码 [NVlabs/RoboLab](https://github.com/NVlabs/RoboLab)、本地远端 4090 复现证据 `remote_logs/` / `remote_outputs/`。  
> **【橙色标识｜边界】** 下面的“来源核心内容”是可溯源摘要，不是论文或源码逐字摘录。要核查完整上下文，应点开对应论文节、appendix 或 GitHub 文件。

## 先看这张总表

| 精讲 | 要回答的问题 | 核心来源 | 来源里的核心内容 | 复现/代码落点 |
|---|---|---|---|---|
| 00 全局总览 | RoboLab 到底是 benchmark、生成框架还是复现工程？ | Paper Abstract / Introduction / III RoboLab / IV Experiments；GitHub README | RoboLab 用高保真仿真评测真实数据训练的通用策略，强调任务泛化、细粒度分析和可扩展场景/任务生成 | `README.md`、`robolab/eval/`、`robolab/scene_gen/`、本 notebook 全局路线 |
| 01 real-to-sim 评估 | 论文说的真实场景到模拟评估，RoboLab 实际怎么做？ | Paper II-B Real-to-sim Evaluation；III-A Scene/Task Generation | 不是逐场景视频重建，而是用高质量资产、程序化布局、物理检查和扰动矩阵快速构造可评测仿真场景 | `assets/`、`scene_gen/`、Isaac Sim/Isaac Lab、variation runners |
| 02 场景/任务/环境生成 | 从场景到任务再到 env 的三步如何落到代码？ | Paper III-A；Figure workflow；GitHub `robolab/scene_gen`、`robolab/core/environments` | 先定位和定向对象形成 scene，再用语言目标定义 task，最后绑定 robot、policy、camera、lighting、background 生成 environment | `runtime.py`、task registry、scene configs、policy runner |
| 03 扩展任务生成 | LLM 生成任务代码怎样验证和修复？ | Paper III-A 2；Appendix D；`skills/robolab-taskgen` | LLM 生成 task code 后做语法、资产、容器尺寸和物理可行性验证，失败后把错误打回 prompt 修复 | taskgen skill、`conditionals`、`load_task_from_file`、asset validation |
| 04 能力轴/难度 | visual、procedural、relational 和难度分数怎么定义？ | Paper III-B Benchmark Design；Figure 4；Appendix A | 任务可多标签，不是单一能力；难度由子任务长度和最高需求级别共同决定 | task metadata、`subtask_utils.py`、`constants.py` |
| 05 SPARC | 轨迹平滑度为什么要看频谱弧长？ | Paper III-C Trajectory Metrics；SPARC reference；analysis metrics | 成功率不说明动作质量，SPARC 用速度频谱复杂度衡量末端运动平滑性 | HDF5 trajectory、`analysis/read_results.py`、trajectory metrics |
| 06 MNPE | 扰动敏感性为什么要做 posterior，而不是只看平均成功率？ | Paper III-D；Appendix B；`analysis/sensitivity_analysis/posterior_inference.py` | 把 lighting/camera/object pose 等扰动作为参数，学习成功/失败条件下参数后验分布，定位高风险区域 | variation CSV、posterior inference、sensitivity plots |
| 07 Baseline 方法 | Appendix C-C 的 baseline 是什么，为什么不是策略 baseline？ | Paper Appendix C-C Baseline Method | baseline 是 scene generation 对照：LLM 选物体 + 网格布局 + jitter + settle；缺少谓词、solver 和反馈修复 | grid+jitter toy baseline、scene generation comparison |
| 08 实验总览 | 论文实验不是单一跑分，它的证据链是什么？ | Paper IV Experiments；III-C Metrics；Appendix C-D / D | 组合验证 RoboLab-120、细粒度分析、扰动敏感性、真实相关性、场景生成质量和任务生成质量 | `runner.py`、`results.py`、dashboard、analysis scripts |
| 09 DTGE | Task generation evaluation 具体评什么？ | Paper Appendix D Details on Task Generation Evaluation | 用 LLM-as-judge 和静态分析评估 instruction 与 code success condition 的关系、对象、量词、清晰度和可行性 | AST 抽取、judge prompt、task generation metrics |
| 10 Prompt 设计 | 为什么 scene prompt 要写这么多规则？ | Paper Appendix C Stage I；Figure prompt examples | prompt 把真实场景原则、坐标系、placement types、JSON-only、对象目录和尺寸限制注入给 LLM | prompt schema、catalog injection、JSON validation |
| 11 Spatial/Physical/Feedback | 2D 空间求解、3D 物理放置和失败反馈如何串起来？ | Paper Appendix C-B；Algorithm 1；Figure 17；Algorithm 2；`spatial_solver.py` / `physical_solver.py` | Spatial 先解 base 2D 位姿，Physical 再解 `place-on` / `place-in`，失败反馈回到 LLM 修复 | `spatial_solver.py`、`physical_solver.py`、`feedback_system.py` |
| 12 Gaussian/前沿 | 本文用了哪些 Gaussian 思路，和 2026 NVIDIA 前沿有什么关系？ | Paper real-to-sim discussion / MNPE KDE；NVIDIA NuRec、3DGUT、Lyra、Isaac Sim sources | RoboLab 本文不是主打 3DGS 重建；Gaussian 更多出现在相邻 real-to-sim 路线和 MNPE KDE，前沿路线可作为后续扩展 | NuRec/3DGUT/Lyra links、Isaac Sim、future reading |
| 13 剩余核心内容 | 还有哪些评测侧问题没有在前面讲透？ | Paper III-C / IV-B / IV-D / Appendix A / Limitations | success 与 score、语言变体、复杂度 sweep、event tracking、真实相关性和统计置信共同构成论文级证据 | `episode_results.jsonl`、event log、dashboard、analysis |
| 13b 证据链深挖 | 单条 rollout 怎么变成论文级结论？ | Paper metrics + event tracking + dashboard/result schema | 视频、人眼判断、HDF5、JSONL、event log、dashboard 角色不同，不能混用 | HDF5、event JSON、`results.py`、dashboard loader |
| 14 runtime 主链 | 真实策略评测代码从哪里开始，证据怎么落盘？ | GitHub `robolab/eval/runner.py`、`episode.py`、`base_client.py`、Pi05 client、`summarize.py` | runner 选任务和输出目录，episode 逐步执行 policy，client 请求动作，summary 写结果 | `runner.py`、`episode.py`、policy client、summary |
| 14b runtime 深挖 | 多 env、action chunk、WorldState、EventTracker 这些状态边界怎么理解？ | GitHub runtime/eval/world/logging source files | 多 env 要按 env_id 隔离 chunk 和 episode；WorldState 支撑谓词；EventTracker 记录稀疏失败事件；HDF5/JSONL 是证据源 | `WorldState`、`EventTracker`、RecorderManager、dashboard |
| 15 审稿人视角 | 如果作为审稿人，这篇论文强在哪里、弱在哪里？ | Paper full text；Limitations；GitHub install/runtime evidence | 贡献是 benchmark+生成+诊断工具链；风险在仿真真实性、资产依赖、统计样本和生成任务验收 | reviewer rubric、未来路线、复现边界 |
| 16 推荐阅读 | 读完 RoboLab 后该补哪些 2026-first 相关工作？ | RoboLab related work；官方项目页；OpenPI、Isaac、Lightwheel、RDT、RoboCasa365 等来源 | 把后续阅读按 policy、benchmark、asset、sim-to-real、world model、real data 等路线组织 | source-linked reading map |

## 按论文结构反查精讲

| 论文位置 | 原问题 | 对应精讲 |
|---|---|---|
| Abstract / Introduction | 为什么需要新的仿真 benchmark？现有 benchmark 为什么会饱和或域重叠？ | 00、01、08、15 |
| II-B Real-to-sim Evaluation | 为什么不逐场景重建真实视频？RoboLab 的 scale 优势在哪里？ | 01、12 |
| III-A Scene and Task Generation | 场景、任务、环境如何生成和验证？ | 02、03、10、11 |
| III-B Benchmark Design | 三条能力轴、任务属性、子任务和难度如何定义？ | 04 |
| III-C Metrics | 为什么不能只看成功率？score、语言变体、trajectory metrics 如何补充？ | 05、08、13、13b |
| III-D Sensitivity | 如何找出哪些扰动最影响策略？ | 06、08 |
| IV Experiments | 论文实验具体证明了什么？ | 08、13、15 |
| IV-D Real-World Verification | 仿真评测如何与真实世界排名关联？ | 08、13、15 |
| Appendix A | 统计显著性、复杂度、score gap 和异常 horizon 怎么解释？ | 13、13b |
| Appendix B | MNPE 变量、prior、posterior 和 importance correction 怎么理解？ | 06 |
| Appendix C | scene generation prompt、solver、baseline 和 scene quality experiment 怎么实现？ | 07、10、11、08 |
| Appendix D | task generation judge 怎么评估自动任务？ | 03、09 |
| GitHub eval/runtime | 策略评测代码怎么运行，结果怎么保存？ | 14、14b |

## 按复现问题反查来源

| 你在复现中问的问题 | 应该先看 | 为什么 |
|---|---|---|
| “为什么视频看起来成功但 `success=False`？” | 13、13b、14b | 视频只是人眼证据，最终判定来自 predicate/subtask score/event log |
| “为什么不先全跑 RoboLab-120？” | 08、13、15 | 全量需要固定任务、策略、episode、资产和分析口径；4090 先做小矩阵更稳 |
| “OpenPI Pi05 和 RoboChallenge Pi 怎么对比？” | 14、16 | 先统一 policy client、任务、输出和 action/observation schema |
| “为什么下载慢或资产不齐会影响结果？” | 00、02、11、14 | scene asset、USD、physics 和 recorder 都是评测证据链的一部分 |
| “怎么判断策略弱在视觉、空间还是程序操作？” | 04、08、13 | 要按 task attributes、difficulty、subtask 和 event reason 分组 |
| “为什么 prompt 要写得这么复杂？” | 10、11 | prompt 是生成 typed predicates 的约束入口，solver/feedback 依赖它 |
| “为什么要讲 Gaussian/NuRec 等前沿？” | 01、12、16 | 它们不是本文主流程，但决定未来 real-to-sim 和高保真资产路线 |

## 本索引如何维护

新增或修改精讲时，应同步补三件事：

1. 在本文件加入“问题 -> 来源 -> 核心内容 -> 代码/复现落点”。
2. 在 `source_manifest.json` 的 `used_for` 里记录对应来源用途。
3. 在 notebook 里保留一个轻量验证 cell，检查精讲是否至少有问题、来源和落点三类信息。

这样后续不会只剩“讲解内容”，而是能追溯到论文、代码和复现证据。

In [ ]:
# ===== 精讲问题来源索引轻量验证 =====
# 这个 cell 不验证论文结论本身，只验证所有精讲是否有可追溯入口：
# question / source / source_content / implementation_anchor 四类信息必须齐全。

source_question_index = [
    {"id": "00", "file": "EXPLAIN_00_global_overview.md", "question": "RoboLab 的全局定位是什么？", "source": "Paper Abstract/Introduction/III/IV + GitHub README", "source_content": "benchmark platform, RoboLab-120, scene/task generation, policy analysis", "implementation_anchor": ["robolab/eval", "robolab/scene_gen", "notebook evidence"]},
    {"id": "01", "file": "EXPLAIN_01_real_to_sim_eval.md", "question": "real-to-sim evaluation 在 RoboLab 中如何落地？", "source": "Paper II-B + III-A", "source_content": "scale scene/task generation instead of per-scene video reconstruction", "implementation_anchor": ["assets", "scene_gen", "variation runners"]},
    {"id": "02", "file": "EXPLAIN_02_scene_task_env_generation.md", "question": "scene/task/env 三步如何生成？", "source": "Paper III-A", "source_content": "place objects, define task instruction, instantiate env with robot/policy/variations", "implementation_anchor": ["runtime.py", "task registry", "scene configs"]},
    {"id": "03", "file": "EXPLAIN_03_task_generation_validation.md", "question": "LLM task code 如何验证和修复？", "source": "Paper III-A2 + Appendix D + taskgen skill", "source_content": "syntax/resource/feasibility validation and repair prompt loop", "implementation_anchor": ["skills/robolab-taskgen", "conditionals", "asset validation"]},
    {"id": "04", "file": "EXPLAIN_04_competency_axes_difficulty.md", "question": "能力轴和难度分数怎么定义？", "source": "Paper III-B + Figure 4", "source_content": "visual/procedural/relational axes, multi-label attributes, subtask difficulty", "implementation_anchor": ["task metadata", "subtask_utils.py", "constants.py"]},
    {"id": "05", "file": "EXPLAIN_05_sparc_trajectory_metric.md", "question": "SPARC 为什么衡量轨迹平滑度？", "source": "Paper III-C Trajectory Metrics", "source_content": "spectral arc length of end-effector speed profile", "implementation_anchor": ["HDF5 trajectory", "analysis/read_results.py"]},
    {"id": "06", "file": "EXPLAIN_06_mnpe_sensitivity_analysis.md", "question": "MNPE 如何解释扰动敏感性？", "source": "Paper III-D + Appendix B", "source_content": "learn posterior over variation parameters conditioned on rollout outcome", "implementation_anchor": ["analysis/sensitivity_analysis/posterior_inference.py", "variation CSV"]},
    {"id": "07", "file": "EXPLAIN_07_baseline_method.md", "question": "Appendix C-C baseline 到底是什么？", "source": "Paper Appendix C-C", "source_content": "LLM object selection plus grid/jitter one-pass scene baseline", "implementation_anchor": ["grid baseline toy check", "scene generation comparison"]},
    {"id": "08", "file": "EXPLAIN_08_paper_experiments.md", "question": "论文实验的证据链是什么？", "source": "Paper IV + III-C + Appendix C-D/D", "source_content": "policy benchmark, granular analysis, sensitivity, real-world verification, generation quality", "implementation_anchor": ["runner.py", "results.py", "dashboard", "analysis scripts"]},
    {"id": "09", "file": "EXPLAIN_09_dtge.md", "question": "DTGE 具体评什么？", "source": "Paper Appendix D", "source_content": "LLM-as-judge over instruction/code alignment, relation/object/quantifier/clarity/feasibility", "implementation_anchor": ["AST extraction", "judge schema"]},
    {"id": "10", "file": "EXPLAIN_10_prompt_design.md", "question": "scene prompt 为什么这样写？", "source": "Paper Appendix C Stage I + prompt figures", "source_content": "real-world scene principles, JSON-only contract, object catalog, placement types, size limits", "implementation_anchor": ["prompt schema", "catalog injection", "JSON validation"]},
    {"id": "11", "file": "EXPLAIN_11_spatial_physical_solver_feedback.md", "question": "Spatial/Physical/Feedback 怎么串起来？", "source": "Paper Appendix C-B + Algorithm 1/Figure 17/Algorithm 2 + source files", "source_content": "2D base pose solving, 3D place-on/place-in, feedback to LLM repair", "implementation_anchor": ["spatial_solver.py", "physical_solver.py", "feedback_system.py"]},
    {"id": "12", "file": "EXPLAIN_12_gaussian_sim_methods.md", "question": "本文和 Gaussian/2026 前沿路线有什么关系？", "source": "Paper real-to-sim discussion + MNPE KDE + NVIDIA sources", "source_content": "RoboLab is not primarily per-scene Gaussian reconstruction; frontier methods are adjacent extensions", "implementation_anchor": ["NuRec/3DGUT/Lyra links", "Isaac Sim route"]},
    {"id": "13", "file": "EXPLAIN_13_remaining_core_topics.md", "question": "评测侧还有哪些核心内容没讲透？", "source": "Paper III-C/IV-B/IV-D/Appendix A/Limitations", "source_content": "success-score gap, language variants, complexity sweeps, event tracking, confidence, real-world correlation", "implementation_anchor": ["episode_results.jsonl", "event log", "dashboard"]},
    {"id": "13b", "file": "EXPLAIN_13_deep_evaluation_evidence_chain.md", "question": "单条 rollout 如何变成论文级结论？", "source": "Paper metrics + event/result schema", "source_content": "video, HDF5, JSONL, event log, dashboard have different evidence roles", "implementation_anchor": ["HDF5", "event JSON", "results.py", "dashboard loader"]},
    {"id": "14", "file": "EXPLAIN_14_core_code_runtime_chain.md", "question": "真实策略评测 runtime 主链是什么？", "source": "GitHub eval runner/episode/base_client/pi0_family/summarize", "source_content": "task selection, episode loop, policy client, summary/result writing", "implementation_anchor": ["runner.py", "episode.py", "base_client.py", "summarize.py"]},
    {"id": "14b", "file": "EXPLAIN_14_deep_runtime_code_chain.md", "question": "多 env/action chunk/WorldState/EventTracker 状态边界是什么？", "source": "GitHub eval/world/logging/dashboard source files", "source_content": "env_id isolation, action chunk cache, predicate state, sparse failure events, artifact loading", "implementation_anchor": ["WorldState", "EventTracker", "RecorderManager", "dashboard"]},
    {"id": "15", "file": "EXPLAIN_15_reviewer_synthesis.md", "question": "审稿人视角怎么看这篇论文？", "source": "Paper full text + Limitations + reproduction evidence", "source_content": "strengths, weaknesses, reproducibility risk, future innovation routes", "implementation_anchor": ["review rubric", "subset protocol", "hardware boundary"]},
    {"id": "16", "file": "EXPLAIN_16_recommended_reading.md", "question": "读完 RoboLab 后该补哪些来源？", "source": "RoboLab related work + official project/source links", "source_content": "2026-first reading map across policies, benchmarks, assets, real data, world models", "implementation_anchor": ["source-linked reading map"]},
]

required_keys = {"id", "file", "question", "source", "source_content", "implementation_anchor"}
tests = [
    ("index_markdown_exists", (NOTEBOOK_ROOT / "EXPLAIN_SOURCE_QUESTION_INDEX.md").exists()),
    ("covers_all_explain_files", len(source_question_index) >= 19),
    ("every_entry_has_required_keys", all(required_keys.issubset(entry) for entry in source_question_index)),
    ("every_entry_file_exists", all((NOTEBOOK_ROOT / entry["file"]).exists() for entry in source_question_index)),
    ("every_entry_has_nonempty_source_content", all(entry["source_content"] for entry in source_question_index)),
    ("contains_paper_sections_and_code_sources", any("Appendix C" in entry["source"] for entry in source_question_index) and any("GitHub" in entry["source"] for entry in source_question_index)),
    ("contains_reproduction_artifact_anchors", any("HDF5" in " ".join(entry["implementation_anchor"]) for entry in source_question_index)),
]

report = {
    "entries": source_question_index,
    "tests": [{"name": name, "passed": bool(ok)} for name, ok in tests],
    "all_passed": all(ok for _, ok in tests),
    "boundary": "This validates source/question coverage for the lecture index; it does not re-verify paper claims or execute RoboLab.",
}

print(json.dumps(report, ensure_ascii=False, indent=2))
for name, ok in tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")
assert report["all_passed"], tests
write_status("explain_source_question_index_tests", report)

## 0.0 论文精讲 0：RoboLab 全局总览

下面这节来自本目录的 [EXPLAIN_00_global_overview.md](./EXPLAIN_00_global_overview.md)。它现在是所有精讲的全局入口：先说明如何配合 `EXPLAIN_SOURCE_QUESTION_INDEX.md` 阅读，再把论文动机、系统架构、四条主线、任务标注、策略接入、证据口径、4090 边界、全精讲路线图和后续对比路线讲清楚。

# 精讲 0：RoboLab 全局总览，先把整件事讲透

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>颜色标识</strong>：绿色表示核心结论，蓝色表示源码/输入输出路径，橙色表示边界、风险和容易误解的点。
</div>

## 先说结论

RoboLab 不是“一个能跑 pick-and-place 的仿真 demo”，而是一套用高保真仿真评估真实世界通用机器人策略的 benchmark 框架。它真正要解决的问题不是“机器人能不能在一个固定场景里拿香蕉”，而是：

```text
真实世界数据训练出来的通用策略
  在高保真、可控、可扩展的仿真任务里
  到底能不能泛化到新物体、新语言、新空间关系、新操作流程和新视觉扰动？
```

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>核心结论</strong>：RoboLab 的主线是“把真实策略放进可控仿真考场”。场景、任务、机器人、策略、相机、光照、背景和指标是分层解耦的。这样同一个任务可以换策略，同一个策略可以跑不同任务，同一个任务还可以系统化改变相机/光照/材质来分析鲁棒性。
</div>

一句话记住：

```text
RoboLab = 高保真场景 + 语言任务 + 任务谓词 + Isaac Lab 环境 + VLA policy 接口 + 多维指标 + 敏感性分析
```

## 0. 这版精讲0应该怎么读

现在精讲 1-16 已经基本补齐，所以精讲 0 的职责不再只是“先讲个背景”。它应该成为整个学习包的入口。

建议阅读顺序是：

```text
先看 EXPLAIN_SOURCE_QUESTION_INDEX
  -> 知道每个精讲的问题来源、论文位置、源码位置
再看精讲0
  -> 建立系统全局心智模型
按问题进入专项精讲
  -> scene/task/env、TaskGen、metrics、solver、runtime、review、reading
最后回到复现实证
  -> video/HDF5/JSON/event log/episode_results 是否支撑结论
```

| 如果你现在的问题是 | 先看哪里 | 原因 |
|---|---|---|
| 不知道每个精讲为什么存在 | `EXPLAIN_SOURCE_QUESTION_INDEX.md` | 它列出原问题、来源、核心内容和代码落点 |
| 想快速理解 RoboLab 是什么 | 本精讲 0 | 它把论文问题、系统架构和复现边界串起来 |
| 想知道代码怎么跑起来 | 精讲 14 / 14b | 它讲 `runner -> episode -> client -> recorder -> summarize` |
| 想知道论文实验怎么读 | 精讲 8 / 13 / 13b | 它讲 success、score、event、trajectory、dashboard 的证据链 |
| 想知道场景/任务怎么生成 | 精讲 2 / 3 / 10 / 11 | 它们讲 scene/task/env、TaskGen、prompt、spatial/physical solver |
| 想做后续对比或创新 | 精讲 15 / 16 | 审稿视角和 2026-first 推荐阅读 |

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>这版精讲0的核心变化</strong>：它不是替代后面的 16 个精讲，而是告诉你后面的精讲分别在回答哪类问题，以及复现实验里应该用哪些证据去验证这些问题。
</div>

## 1. 论文为什么要做 RoboLab

论文的背景是：机器人 foundation policy 已经越来越强，但评测很容易出现两个问题。

| 问题 | 说人话解释 | RoboLab 的应对 |
|---|---|---|
| 训练域和评测域太像 | 模型可能只是熟悉 benchmark，而不是具备真正泛化能力 | 用真实数据训练的策略，在高保真仿真里做新任务评测 |
| 只看 success 太粗 | 成功/失败不能解释错在哪里、动作顺不顺、对相机/光照是否脆弱 | 输出 success、score、subtask/event、SPARC、speed、path length、MNPE 敏感性 |
| 任务太静态 | 固定任务集容易被刷熟，不能长期保持挑战 | 支持人工和 LLM 辅助生成场景/任务 |
| 真实世界评测太贵 | 真实机器人跑 120 个任务、多个策略、多个扰动成本很高 | 用 Isaac Sim / Isaac Lab 构造可重复、可批量运行的仿真评测 |

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>不要把它理解成 sim2real 训练框架</strong>：RoboLab 论文重点是评估真实世界策略，不是主要拿仿真数据去训练策略。它关心的是“真实策略在仿真考场里的表现是否能揭示泛化问题”。
</div>

## 2. 全局数据流

从最上层看，一次 RoboLab 评测大概是下面这条链：

```text
论文/任务设计
  -> 资产库：USD/SimReady objects, fixtures, robots, backgrounds
  -> 场景：某个 USD scene，里面摆好对象和支撑关系
  -> 任务：Task Python 类，写语言指令、成功条件、subtasks
  -> 注册：auto_env_registrations 把 task 变成 gym env id
  -> 环境：create_env / RobolabEnv / Isaac Lab managers
  -> 策略：Pi0/Pi05/GR00T/PaliGemma/ReKep 等接口适配
  -> episode：仿真 step、policy action、event tracking、recorder
  -> 输出：video, HDF5, event log, episode_results.jsonl
  -> 分析：success/score/axis/difficulty/SPARC/MNPE
```

对复现来说，最重要的是每一层都要有证据：

| 层 | 输入 | 输出 | 证据文件 |
|---|---|---|---|
| 安装层 | RoboLab repo、`uv sync`、Isaac Sim/Lab 依赖 | Python 环境可导入 | `uv_freeze.txt`、安装日志 |
| 资产层 | LFS 场景/物体/机器人/背景 | 可加载的 USD assets | 资产目录大小、缺失资产报错 |
| 任务层 | `Task` 类、scene、instruction、subtasks | task metadata / env id | `task_metadata.json`、task py 文件 |
| 环境层 | task name、robot、camera、variation | Isaac Lab env | `env_cfg.json`、Isaac 日志 |
| 策略层 | observations、language prompt | action chunk | OpenPI server log、policy timing |
| episode 层 | actions + simulation steps | success/score/events/video/HDF5 | `episode_results.jsonl`、`log_0_env0.json`、`run_0.hdf5`、mp4 |
| 分析层 | 多任务 episode records | 表格、图、posterior | summary csv/json、notebook 图表 |

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>源码/输入输出路径</strong>：复现时不要只看终端有没有报错。真正的闭环证据是 <code>output/&lt;run_id&gt;/.../episode_results.jsonl</code>、任务 event log、HDF5、视频，以及 notebook 里同步出来的 <code>remote_outputs/</code> 和 <code>robolab_repro_artifacts/</code>。
</div>

## 2b. 全局架构可以压成四条主线

如果把 RoboLab 压成最少的系统图，它不是一条线，而是四条并行主线汇合到评测结果：

```text
生成线：scene prompt -> predicates -> spatial/physical solve -> valid scene
任务线：task template -> Python task -> conditionals/subtasks -> score/success
运行线：env -> observation -> policy client/server -> action chunk -> env.step
证据线：recorder -> HDF5/video/event log -> episode_results -> dashboard/analysis
```

这四条线分别对应后面精讲：

| 主线 | 负责什么 | 对应精讲 |
|---|---|---|
| 生成线 | 让场景真实、可放置、可物理 settle | 01、02、07、10、11、12 |
| 任务线 | 让语言目标、成功条件、难度和能力标签一致 | 03、04、09 |
| 运行线 | 让真实策略能在仿真里闭环执行 | 14、14b |
| 证据线 | 让单条 rollout 能变成可分析结果 | 05、06、08、13、13b |
| 评价线 | 从论文贡献、风险和未来路线重新看整个系统 | 15、16 |

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>最容易出错的地方</strong>：四条线任何一条断了，最终结果都不能直接解释成“模型能力强/弱”。例如资产缺失是生成线问题，任务谓词错是任务线问题，server timeout 是运行线问题，dashboard 没读到 HDF5 是证据线问题。
</div>

## 3. 代码目录按职责看

RoboLab 仓库可以先按职责分成十块，不要一上来平铺读所有文件：

| 目录/文件 | 负责什么 | 该怎么读 |
|---|---|---|
| `robolab/constants.py` | 包根目录、资产目录、输出目录、能力权重、难度阈值 | 先看路径和全局常量 |
| `assets/` | scene/object/fixture/robot/background 等 USD 资产 | 跑任务报缺资产时回到这里 |
| `robolab/tasks/benchmark/*.py` | 每个 benchmark 任务的定义 | 读 language、scene、termination、subtasks |
| `robolab/tasks/_metadata/` | 任务索引和 metadata | 按任务名看能力轴、难度、subtask 数 |
| `robolab/core/task/` | 条件谓词、subtask、task 基类 | 读 success/score 是怎么计算的 |
| `robolab/core/environments/` | 从任务名创建 Isaac Lab env，并管理 episode 生命周期 | 读 `create_env`、`RobolabEnv.step`、reset/freeze |
| `robolab/registrations/` | 把任务注册成 gym env | 读 auto registration 如何绑定 robot/camera/background |
| `robolab/variations/` | 相机、灯光、背景等扰动配置 | 读 sensitivity / robustness 实验 |
| `policies/pi0_family/` | Pi0/Pi05 policy runner 和 client | 读 observation/action 如何接 OpenPI |
| `analysis/sensitivity_analysis/` | MNPE/NPE 后验分析 | 读 CSV -> `theta/x` -> posterior |

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>阅读顺序</strong>：先读 <code>BananaInBowlTask</code>，再读 <code>conditionals.py</code> / <code>subtask.py</code>，再读 <code>runtime.py:create_env</code>，最后读 <code>policies/pi0_family/run.py</code>。这样能从一个具体任务把整条链串起来。
</div>

## 4. 一个任务到底“标注”了什么

RoboLab 的“标注”不是传统视觉数据集那种 bounding box 或 segmentation mask。一个 RoboLab task 的标注更像“可执行任务规范”。

以 `BananaInBowlTask` 为例，一个任务至少包含：

| 字段 | 说人话 | 作用 |
|---|---|---|
| `scene` | 用哪个 USD 场景 | 决定桌面、碗、香蕉等对象在哪里 |
| `contact_object_list` | 哪些对象需要 contact sensor | 抓取、碰撞、掉落等事件依赖它 |
| `instruction` | 给策略看的语言任务 | policy 输入的一部分 |
| `terminations` | 成功/失败/超时条件 | 决定 episode 什么时候结束 |
| `attributes` | 颜色、语义、空间、堆叠、重定向等标签 | 后续按能力轴统计 |
| `subtasks` | 把任务拆成可评分步骤 | 用来算 partial score 和错误分析 |

所以：

```text
Task = 场景 + 语言 + 对象 + 成功条件 + 子任务评分 + 能力标签
```

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>关键区别</strong>：RoboLab 不是先录一段视频再人工标注“哪一帧成功”。它是先把任务目标写成代码里的谓词和 subtask，episode 运行时自动判断事件、成功条件和分数。
</div>

## 5. 场景、任务、环境三者不要混

这三个词非常容易混，必须拆开：

| 概念 | 是什么 | 例子 |
|---|---|---|
| Scene | 世界里有什么对象，摆在哪里 | `banana_bowl.usda` |
| Task | 要完成什么目标，怎么判成功 | “put the banana in the bowl” + `object_in_container` |
| Environment | 把 scene/task/robot/camera/policy/variation 装配成能 step 的仿真 | `gym.make(task_env, cfg=env_cfg)` |

论文说的三步是：

```text
1. 定位和定向物体，创建场景
2. 把目标状态写成语言任务
3. 选择机器人、策略、摄像头、光照、背景等变化，实例化环境
```

源码里对应为：

```text
scene/import_scene
  -> Task class
  -> registration / parse_env_cfg / create_env
  -> RobolabEnv
```

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>不要用“能打开场景”替代“任务复现成功”</strong>：USD 能加载，只说明资产和 stage 基本可用；任务成功还需要 contact sensor、termination、subtask、policy action、recorder 都正常。
</div>

## 6. 策略接入层：Pi05 是怎么进来的

RoboLab 自己不把 OpenPI 权重硬编码进任务里。它让任务和策略解耦：

```text
RoboLab env observation
  -> Pi0/Pi05 client 打包成 OpenPI 请求
  -> OpenPI server 根据图像/关节/夹爪/语言输出 action chunk
  -> client 拆 action
  -> env.step(action)
```

对 Pi05 来说，我们这次关心的几类输入是：

| 输入 | 来源 | 作用 |
|---|---|---|
| 外部相机图 | RoboLab camera render | 看桌面和目标物 |
| 腕部相机图 | wrist camera render | 看末端附近细节 |
| 关节位置 | robot state | 让 policy 知道机械臂状态 |
| 夹爪状态 | robot state | 控制抓取/释放 |
| 语言指令 | Task instruction | 告诉 policy 当前目标 |

输出是 action chunk，也就是一段连续动作。client 会按 `open_loop_horizon` 执行一小段，再重新请求下一段动作。

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>怎么判断策略真的接上了</strong>：只看 Isaac 启动不够。要看到 OpenPI server restored checkpoint、8000 端口监听、policy inference timing、episode_results 里有真实 policy 字段和动作导致的 success/score。
</div>

## 7. 4090 上我们已经复现到哪一步

这部分必须说实话，不能把 smoke 当完整 benchmark。

已经完成：

| 项目 | 状态 | 含义 |
|---|---|---|
| RoboLab 环境安装 | 已完成 `uv sync` | Isaac Sim 5.0 / Isaac Lab 2.2 / RoboLab 可导入 |
| 资产补齐 | 已按任务补齐核心 scenes/robots/fixtures | 足够跑 Banana 和一批 subset，但不是完整 LFS 全量 |
| no-policy smoke | 累计 21 个任务初始化和日志导出 | 证明环境链路可启动，不代表策略成功率 |
| Pi05 checkpoint | 已下载并校验 | OpenPI 权重文件可加载 |
| Pi05 server | 已监听 8000 | 策略服务端可用 |
| 单任务闭环 | BananaInBowlTask 成功 | 证明 Pi05 -> RoboLab -> Isaac -> 视频/HDF5/JSON 通了 |
| 复杂任务抽样 | 3 个任务，1 成功 2 失败 | 初步看到长时序/集合/重定向更难 |

还没完成：

| 项目 | 为什么还不能说完成 |
|---|---|
| 完整 RoboLab-120 | 需要更多资产、更多 GPU 时间和严格结果汇总 |
| 正式 MNPE | 需要同一任务/策略在多组扰动参数下的大量 rollout CSV |
| RoboChallenge pi / ReKep 对比 | 需要统一任务、接口、相机、动作空间、成功条件后才能公平比较 |
| 论文级表格复现 | 小样本 smoke 不能和论文完整 benchmark 直接比较 |

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>结果口径</strong>：环境成功、策略接上、任务成功、论文复现是四个层级。当前已经有真实 Pi05 单任务成功闭环和复杂任务抽样，但还不是完整 RoboLab-120 论文表格复现。
</div>

## 8. 为什么不能一上来跑 120 个任务

RoboLab-120 全量评测不是一条命令就能可信完成，至少有四个约束：

1. **资产完整性**：很多任务依赖不同 object/material/background LFS 文件，缺一个就会在 Isaac 初始化时报错。
2. **4090 显存**：24GB 可以跑单任务和小子集，但复杂任务、高清相机、视频/HDF5、policy server 同时占资源时很容易接近上限。
3. **时间成本**：完整 benchmark 是几十 GPU 小时量级，失败重跑也要成本。
4. **统计口径**：中途失败要区分环境失败、资产失败、策略失败、timeout，不能只把所有非 0 return code 混成失败率。

正确路线是：

```text
单任务闭环
  -> 能力轴小子集
  -> 补齐资产失败项
  -> 每类任务多 episode
  -> 输出统一 CSV/JSON
  -> 再扩展 RoboLab-120
```

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>4090 策略</strong>：先 <code>num_envs=1</code> 保守跑通，确认显存、视频、HDF5、episode_results 稳定后再加并行度。不要为了看起来快，直接把并行度拉爆导致半路 OOM。
</div>

## 9. 全部精讲如何互相连接

现在完整学习包已经不是 1-6 的单点拆解，而是一组从论文问题到代码复现的路线图。

| 层级 | 精讲 | 解决的问题 | 在全局里的位置 |
|---|---|---|---|
| 来源入口 | 来源索引、精讲0 | 每个问题从哪里来，整篇论文/代码怎么串 | 全局导航 |
| 场景与任务生成 | 01、02、03、07、10、11、12 | 场景从哪里来、任务怎么生成、prompt/solver/baseline/Gaussian 路线如何理解 | 生成可评测环境 |
| Benchmark 设计 | 04、08、09 | 能力轴、难度、实验矩阵、任务生成质量如何评估 | 定义考题和读表方式 |
| 指标与证据 | 05、06、13、13b | SPARC、MNPE、success/score、event、dashboard 如何共同解释策略表现 | 从 rollout 到结论 |
| 运行时代码 | 14、14b | `runner`、`episode`、policy client、WorldState、recorder、summary 如何工作 | 复现执行链 |
| 评审与延展 | 15、16 | 论文贡献/风险/未来路线，以及后续推荐阅读 | 判断价值和下一步 |

如果只看其中一个，会容易误解：

| 只看某一块 | 容易误解成 | 实际应该放回全局理解 |
|---|---|---|
| 只看场景 | RoboLab 是资产库 | 场景只是任务评测的物理载体 |
| 只看任务类 | RoboLab 是 hard-coded task list | Task 是目标状态规范，可人工/LLM 扩展 |
| 只看 success | 模型强弱只有二值成功率 | 还要看 subtask score、错误类型、轨迹质量、敏感性 |
| 只看 Pi05 成功 demo | 已复现论文 | 单任务成功只是闭环证据，不是 RoboLab-120 |
| 只看 MNPE | 可以不用跑任务直接分析 | MNPE 必须依赖大量扰动 rollout 数据 |

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>读法建议</strong>：以后如果要继续加精讲，不要只加正文解释。先在 <code>EXPLAIN_SOURCE_QUESTION_INDEX.md</code> 里补“问题来源”，再在本精讲0里判断它属于生成线、任务线、运行线、证据线还是评价线。
</div>

## 10. 一次完整复现应该长什么样

我建议把“完整复现”分成五级，而不是只说“跑没跑成”。

| 等级 | 名称 | 通过标准 |
|---|---|---|
| L0 | 安装级 | RoboLab / Isaac Sim / Isaac Lab / OpenPI 可导入 |
| L1 | 环境级 | no-policy smoke 能创建 env 并导出 episode log |
| L2 | 策略级 | Pi05 server 接上，policy runner 能执行真实动作 |
| L3 | 单任务闭环 | 单任务成功，有 video/HDF5/event/episode_results |
| L4 | 小子集评测 | 多任务覆盖 visual/procedural/relational，有成功/失败分析 |
| L5 | 论文级复现 | RoboLab-120 或明确子集，多策略/多扰动/统计汇总/可复查证据 |

当前学习包已经到 L3，并有 L4 的初步抽样证据；还没到 L5。

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>这也是 notebook 的结构</strong>：前面记录安装和证据，中间讲源码与论文机制，后面放运行结果、指标、可视化、学习日志和最终 gate。这样每一步都有可复查文件，而不是只靠终端输出。
</div>

## 11. RoboChallenge pi、OpenPI pi05、ReKep 怎么放在同一张图里

这几个东西不要混成“都是一个模型”。

| 名称 | 更像什么 | 什么时候用 |
|---|---|---|
| RoboLab | 评测框架和 benchmark | 需要统一任务、场景、指标、视频/HDF5 输出时 |
| OpenPI pi05 | VLA policy checkpoint / server | 需要在 RoboLab 里跑真实 learned policy 时 |
| RoboChallenge 的 pi | 另一个项目语境里的策略/接口资产 | 做对比前要先适配 observation/action 和成功条件 |
| ReKep | 偏基于关键点/约束/规划的机器人方法 | 做“学习策略 vs 结构化方法”对比时 |

公平对比的前提：

```text
同一任务
同一场景或可说明差异的场景
同一机器人/动作空间或明确适配层
同一相机/观测输入
同一成功条件和 subtask score
同一 episode 数和随机种子策略
同一输出格式
```

否则结果只能叫“跑通案例”，不能叫 benchmark 对比。

## 12. 最后用一张总图记住

```text
为什么评测
  -> 真实策略泛化难，只看 success 不够

评测什么
  -> RoboLab-120：visual / procedural / relational
  -> simple / moderate / complex

在哪里评测
  -> Isaac Sim + Isaac Lab + USD/SimReady assets

怎么定义任务
  -> Task = scene + instruction + termination + subtask + attributes

怎么跑策略
  -> RoboLab env observation -> policy client/server -> action chunk -> env.step

怎么留证据
  -> episode_results.jsonl + event log + HDF5 + video + notebook artifacts

怎么分析
  -> success / score / subtask / error reason / SPARC / MNPE

当前状态
  -> Pi05 单任务成功闭环已完成，复杂任务抽样已完成，完整 RoboLab-120 尚未执行
```

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>一句话总括</strong>：RoboLab 的价值不是“做了一个更漂亮的仿真场景”，而是把任务定义、策略接口、可控扰动、细粒度指标和可回放证据组织成一套评估通用机器人策略泛化能力的实验系统。
</div>

In [ ]:
# ===== 精讲0：全局总览结构轻量验证 =====
# 这个 cell 验证精讲0是否承担“全局入口”职责：
# 需要覆盖来源索引、四条主线、复现层级、证据文件和后续阅读路线。

global_overview_map = {
    "entry_points": ["EXPLAIN_SOURCE_QUESTION_INDEX.md", "EXPLAIN_00_global_overview.md"],
    "main_threads": {
        "generation": ["scene prompt", "predicates", "spatial/physical solve", "valid scene"],
        "task": ["task template", "Python task", "conditionals/subtasks", "score/success"],
        "runtime": ["env", "observation", "policy client/server", "action chunk", "env.step"],
        "evidence": ["recorder", "HDF5", "video", "event log", "episode_results", "dashboard"],
        "review": ["reviewer synthesis", "recommended reading", "future route"],
    },
    "reproduction_levels": ["L0_install", "L1_environment", "L2_policy", "L3_single_task", "L4_subset", "L5_paper_level"],
    "artifact_contract": ["episode_results.jsonl", "log_0_env0.json", "run_0.hdf5", "mp4", "remote_outputs", "robolab_repro_artifacts"],
    "lecture_groups": {
        "scene_task_generation": ["01", "02", "03", "07", "10", "11", "12"],
        "benchmark_design": ["04", "08", "09"],
        "metrics_evidence": ["05", "06", "13", "13b"],
        "runtime_code": ["14", "14b"],
        "review_extension": ["15", "16"],
    },
}

overview_text = (NOTEBOOK_ROOT / "EXPLAIN_00_global_overview.md").read_text(encoding="utf-8")
tests = [
    ("mentions_source_question_index", "EXPLAIN_SOURCE_QUESTION_INDEX" in overview_text),
    ("mentions_four_main_threads", all(term in overview_text for term in ["生成线", "任务线", "运行线", "证据线"])),
    ("mentions_runtime_policy_chain", "policy client/server" in json.dumps(global_overview_map, ensure_ascii=False)),
    ("mentions_artifact_contract", all(term in overview_text for term in ["episode_results.jsonl", "HDF5", "event log"])),
    ("mentions_reproduction_levels", all(level in overview_text for level in ["L0", "L1", "L2", "L3", "L4", "L5"])),
    ("lecture_groups_cover_all_later_lectures", sum(len(v) for v in global_overview_map["lecture_groups"].values()) >= 18),
    ("entry_points_include_source_index_and_overview", set(global_overview_map["entry_points"]) == {"EXPLAIN_SOURCE_QUESTION_INDEX.md", "EXPLAIN_00_global_overview.md"}),
]

report = {
    "global_overview_map": global_overview_map,
    "tests": [{"name": name, "passed": bool(ok)} for name, ok in tests],
    "all_passed": all(ok for _, ok in tests),
    "boundary": "This validates the overview structure only; it does not execute RoboLab or verify paper claims.",
}

print(json.dumps(report, ensure_ascii=False, indent=2))
for name, ok in tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")
assert report["all_passed"], tests
write_status("global_overview_structure_tests", report)

## 0.1 2026-06-19 远端 RTX 4090 实测进展

            本节记录一次已完成的远端 4090 复现推进，证据已同步到本目录的 `remote_logs/`。

            已确认：

            - 远端机器：Ubuntu 22.04.4 LTS，RTX 4090 24GB，NVIDIA driver 580.159.03。
            - `uv sync` 已完成，安装到 Python 3.11 环境；`robolab==0.1.0`、`isaacsim==5.0.0.0`、`isaaclab==2.2.0`、`torch==2.7.0+cu128` 均可导入。
            - 当前 GitHub HEAD：`7d45d74904eade3b578a8eb1f2f9f89bc3d40326`。
- Required RoboLab-120 LFS assets and backgrounds are checked out; `missing_count=0` was verified before the clean full run.
- The test10 preflight run `robolab120_pi05_assetsfixed_test10_20260620_162403` completed 10/10 tasks with `run_returncode=0` and `verify_returncode=0`.
- `CURRENT_ROBOLAB120_STATUS.md`: live RTX 4090 / Pi05 / RoboLab-120 checkpoint, with manifest progress, test10 preflight, error rows, GPU state, and next analysis steps. Embedded in the notebook.
            - `BananaInBowlTask` 的 headless smoke 已完成 2 step 并导出 episode log。这里的 `success: False` 是空动作/no-policy 运行的预期结果，不代表策略评测失败。
            - 已扩展到累计 21 个 no-policy 初始化 smoke，覆盖语义、颜色、空间关系、顺序组合、重定向、堆叠等任务属性；这些仍是环境初始化与日志导出验证，不是策略成功率。
            - README 中的 `uv run pytest tests/` 在当前 HEAD 返回 4，因为仓库没有 `tests/` 路径；这应记录为 README 与仓库当前文件面的不一致。
            - `policies/pi0_family/run.py` 是 Pi0/Pi05 评测入口；OpenPI Pi05 `pi05_droid_jointpos` checkpoint 已完成下载与 26 个对象大小校验，policy server 已监听 8000。
            - 已完成真实 Pi05 policy 单任务复现：`BananaInBowlTask` 1 episode，`success=True`，`score=1.0`，`episode_step=178`，平均 policy inference `84.2 ms`。

In [ ]:
import re
from textwrap import dedent

# 这些 JSON summary 来自远端 RTX 4090 实测结果，已同步到本地 remote_logs。
# notebook 直接读取证据文件，避免依赖聊天记录或过期口头描述。
REMOTE_LOG_DIR = NOTEBOOK_ROOT / "remote_logs"
REMOTE_SUMMARY_PATH = REMOTE_LOG_DIR / "remote_4090_repro_summary.json"
REMOTE_SUBSET3_SUMMARY_PATH = REMOTE_LOG_DIR / "remote_4090_subset3_summary.json"
REMOTE_POLICY_SUBSET21_SUMMARY_PATH = REMOTE_LOG_DIR / "remote_4090_policy_subset21_summary.json"
REMOTE_POLICY_SUBSET19_SUMMARY_PATH = REMOTE_LOG_DIR / "remote_4090_policy_subset19_summary.json"
REMOTE_POLICY_SUBSET10_SUMMARY_PATH = REMOTE_LOG_DIR / "remote_4090_policy_subset10_summary.json"
# 优先使用最新累计 summary；旧 summary 只作为本地文件不完整时的回退。
REMOTE_POLICY_SUMMARY_CANDIDATES = [
    REMOTE_POLICY_SUBSET21_SUMMARY_PATH,
    REMOTE_POLICY_SUBSET19_SUMMARY_PATH,
    REMOTE_POLICY_SUBSET10_SUMMARY_PATH,
]
REMOTE_POLICY_SUMMARY_PATH = next(
    (path for path in REMOTE_POLICY_SUMMARY_CANDIDATES if path.exists()),
    REMOTE_POLICY_SUBSET10_SUMMARY_PATH,
)
REMOTE_PI05_POLICY_SMOKE_DIR = REMOTE_LOG_DIR / "pi05_policy_smoke_20260620_005711"

if REMOTE_SUMMARY_PATH.exists():
    # 第一份 summary：安装/导入验证 + BananaInBowlTask 无策略 smoke。
    remote_summary = json.loads(REMOTE_SUMMARY_PATH.read_text(encoding="utf-8"))
    smoke = remote_summary.get("smoke", {})
    install = remote_summary.get("installation", {})
    assets = remote_summary.get("assets", {})
    host = remote_summary.get("host", {})
    # Isaac 日志里可能有 ANSI 颜色控制码，写入 Markdown 表格前先清理。
    clean_success = re.sub(r"\x1b\[[0-9;]*m", "", smoke.get("success_line", ""))
    report = f'''
    ### 远端 4090 证据摘要

    | 项目 | 结果 |
    |---|---|
    | Host | `{host.get('hostname')}` |
    | GPU | `{host.get('gpu')}` |
    | Repo HEAD | `{remote_summary.get('repo_head')}` |
    | uv sync | `{install.get('uv_sync_status')}` |
    | pytest tests/ | return code `{install.get('pytest_rc')}`，当前 HEAD 无 `tests/` 路径 |
    | Scenes assets | `{assets.get('scenes_status')}` / {assets.get('scenes_du')} |
    | Core assets | fixtures `{assets.get('core_assets_status')}` / robots {assets.get('robots_du')} |
    | Smoke task | `{smoke.get('task')}` |
    | Smoke setup/export | setup `{smoke.get('completed_setup')}`，episode export `{smoke.get('exported_episode')}`，traceback `{smoke.get('traceback_present')}` |
    | Smoke result line | `{clean_success}` |

    证据文件目录：`{REMOTE_LOG_DIR}`
    '''
    display(Markdown(dedent(report).strip()) if Markdown else dedent(report).strip())
else:
    print("remote evidence not found:", REMOTE_SUMMARY_PATH)

if REMOTE_SUBSET3_SUMMARY_PATH.exists():
    # 第二份 summary：在不接策略服务的情况下验证额外 3 个任务可初始化和导出日志。
    subset_summary = json.loads(REMOTE_SUBSET3_SUMMARY_PATH.read_text(encoding="utf-8"))
    task_rows = "\n".join(
        f"| `{task}` | env_cfg `{info.get('env_cfg_exists')}` | episode log `{info.get('episode_log_exists')}` |"
        for task, info in subset_summary.get("outputs", {}).items()
    )
    subset_report = f'''
    ### 三任务 no-policy subset smoke

    | 项目 | 结果 |
    |---|---|
    | Command | `{subset_summary.get('command')}` |
    | Return code | `{subset_summary.get('status_file_rc')}` |
    | Traceback | `{subset_summary.get('traceback_present')}` |
    | Completed setup count | `{subset_summary.get('completed_setup_count')}` |
    | Episodes exported count | `{subset_summary.get('episodes_exported_count')}` |
    | Boundary | {subset_summary.get('policy_boundary')} |

    | Task | Output |
    |---|---|
    {task_rows}

    注意：`output/run_empty_env/episode_results.jsonl` 是累计文件，因此 summary 里会同时包含此前的 `BananaInBowlTask` no-policy smoke。
    '''
    display(Markdown(dedent(subset_report).strip()) if Markdown else dedent(subset_report).strip())
else:
    print("subset evidence not found:", REMOTE_SUBSET3_SUMMARY_PATH)

if REMOTE_POLICY_SUMMARY_PATH.exists():
    # 第三份 summary：累计 no-policy 任务 + OpenPI/Pi05 准备状态；优先读取最新的 subset21。
    policy_summary = json.loads(REMOTE_POLICY_SUMMARY_PATH.read_text(encoding="utf-8"))
    no_policy = policy_summary.get("no_policy_smoke", {})
    openpi = policy_summary.get("openpi_policy_readiness", {})
    # 按任务名建立 metadata 索引，用于给 episode 结果补充属性和难度。
    meta_by_task = {
        row.get("task_name"): row
        for row in no_policy.get("task_metadata", [])
        if row.get("task_name")
    }

    def _clean_table_text(value):
        # Markdown 表格遇到竖线或换行会错列，所以展示前做最小清理。
        return str(value or "").replace("|", "/").replace("\n", " ")

    rows = []
    for rec in no_policy.get("records", []):
        # episode_results.jsonl 是累计文件，因此逐条渲染，避免误以为只包含本轮任务。
        task = rec.get("env_name")
        meta = meta_by_task.get(task, {})
        rows.append(
            "| `{}` | {} | {} | `{}` | {} |".format(
                task,
                _clean_table_text(meta.get("attributes") or "-"),
                _clean_table_text(meta.get("difficulty_label") or "-"),
                rec.get("success"),
                _clean_table_text(rec.get("instruction")),
            )
        )
    task_rows = "\n".join(rows)
    # 候选任务只有在 episode_results.jsonl 真实新增记录且没有 Python Traceback 时才计入累计数。
    successful_probe_tasks = ", ".join(
        f"`{task}`" for task in no_policy.get("candidate_probe_successful_tasks", [])
    ) or "-"
    failed_probe_rows = []
    for item in no_policy.get("candidate_probe_failed_tasks", []):
        failed_probe_rows.append(
            "| `{}` | `{}` | {} |".format(
                item.get("task"),
                item.get("records_added"),
                _clean_table_text(item.get("main_error") or "-"),
            )
        )
    failed_probe_table = "\n".join(failed_probe_rows) or "| - | - | - |"
    evidence_name = Path(policy_summary.get("evidence_package", "")).name
    evidence_path = f"remote_logs/{evidence_name}" if evidence_name else "remote_logs/<missing evidence>"
    policy_record_count = no_policy.get("num_records_cumulative")
    # 只有 OpenPI server 监听 8000 后，才算可以开始真实 policy eval。
    openpi_state = "ready" if openpi.get("server_port_8000_ready") else "downloading / not ready"
    policy_report = f'''
    ### {policy_record_count} 任务 no-policy smoke 与 OpenPI/Pi05 准备状态

    | 项目 | 结果 |
    |---|---|
    | no-policy subset10 return code | `{no_policy.get('subset10_status')}` |
    | no-policy subset_more return code | `{no_policy.get('subset_more_status')}` |
    | no-policy cumulative records | `{no_policy.get('num_records_cumulative')}` |
    | counted candidate probes | {successful_probe_tasks} |
    | OpenPI repo HEAD | `{openpi.get('openpi_head')}` |
    | OpenPI client install | `{openpi.get('openpi_client_install_status')}` |
    | OpenPI uv sync | `{openpi.get('openpi_uv_sync_status')}` |
    | OpenPI import/JAX verify | `{openpi.get('openpi_verify_status')}` |
    | Pi05 checkpoint | `{openpi.get('checkpoint_uri')}` |
    | Pi05 download progress | `{openpi.get('last_progress', {}).get('downloaded')}` / `{openpi.get('last_progress', {}).get('total')}` |
    | Pi05 server port 8000 | `{openpi_state}` |
    | Evidence package | `{evidence_path}` |

    | Task | Attributes | Difficulty | no-policy success | Instruction |
    |---|---|---|---|---|
    {task_rows}

    #### 未计入累计的候选任务

    | Task | records added | 主要失败原因 |
    |---|---:|---|
    {failed_probe_table}

    边界：上表的 `success=False` 来自空动作/no-policy run，只证明场景、任务、日志导出链路可跑；未计入任务主要卡在对象 payload/contact reporter 不完整。真实 Pi05 policy score 已在下一节单独读取与展示。
    '''
    display(Markdown(dedent(policy_report).strip()) if Markdown else dedent(policy_report).strip())
else:
    print("policy readiness evidence not found:", REMOTE_POLICY_SUMMARY_PATH)

In [ ]:
# ===== 真实 Pi05 policy smoke 结果 =====
# 这一段读取 4090 上真实跑过的 OpenPI Pi05 + RoboLab 结果，不再是 no-policy 空动作 smoke。
PI05_POLICY_SMOKE_SUMMARY_PATH = ARTIFACT_DIR / "pi05_policy_smoke_summary.json"

if REMOTE_PI05_POLICY_SMOKE_DIR.exists():
    # 1. checkpoint_verify 证明 11.58GiB pi05_droid_jointpos 权重文件完整。
    verify_path = REMOTE_PI05_POLICY_SMOKE_DIR / "pi05_droid_jointpos_download_verify.json"
    verify = json.loads(verify_path.read_text(encoding="utf-8")) if verify_path.exists() else {}

    # 2. server_log 证明 OpenPI websocket server 已加载本地 checkpoint 并监听 8000。
    server_log_path = REMOTE_PI05_POLICY_SMOKE_DIR / "openpi_pi05_server_local.log"
    server_log = server_log_path.read_text(encoding="utf-8", errors="replace") if server_log_path.exists() else ""
    server_ready = "server listening on 0.0.0.0:8000" in server_log
    checkpoint_restored = "Finished restoring checkpoint" in server_log

    # 3. smoke_status 为 RoboLab policy runner 的退出码；0 表示脚本正常完成。
    status_path = REMOTE_PI05_POLICY_SMOKE_DIR / "robolab_pi05_banana_smoke.status"
    smoke_status = status_path.read_text(encoding="utf-8").strip() if status_path.exists() else "missing"

    # 4. episode_results.jsonl 是最关键结果：这里才有真实 policy 的 success/score/timing。
    episode_files = sorted(REMOTE_PI05_POLICY_SMOKE_DIR.glob("pi05_banana_smoke_*/episode_results.jsonl"))
    result_rows = []
    for episode_file in episode_files:
        for line in episode_file.read_text(encoding="utf-8").splitlines():
            if line.strip():
                row = json.loads(line)
                row["_source"] = str(episode_file.relative_to(REMOTE_PI05_POLICY_SMOKE_DIR))
                result_rows.append(row)

    # 5. task 内部 event log 解释为什么 episode 在 178 step 结束。
    event_log_files = sorted(REMOTE_PI05_POLICY_SMOKE_DIR.glob("pi05_banana_smoke_*/BananaInBowlTask/log_0_env0.json"))
    event_rows = []
    for event_file in event_log_files:
        event_data = json.loads(event_file.read_text(encoding="utf-8"))
        for event in event_data.get("events", []):
            event_rows.append(
                {
                    "step": event.get("step"),
                    "name": event.get("name"),
                    "score": event.get("score"),
                    "info": event.get("info"),
                }
            )

    summary = {
        "checkpoint_verify_ok": verify.get("ok"),
        "checkpoint_objects": verify.get("objects"),
        "checkpoint_total_gib": verify.get("total_gib"),
        "server_ready": server_ready,
        "checkpoint_restored": checkpoint_restored,
        "smoke_status": smoke_status,
        "num_episode_records": len(result_rows),
        "episode_records": result_rows,
        "event_rows": event_rows,
        "evidence_dir": str(REMOTE_PI05_POLICY_SMOKE_DIR),
    }
    PI05_POLICY_SMOKE_SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    if result_rows:
        first = result_rows[0]
        timing = first.get("timing", {})
        metrics = first.get("metrics", {})
        report = f'''
        ### Pi05 真实策略 smoke：BananaInBowlTask

        | 项目 | 结果 |
        |---|---|
        | Checkpoint verify | `{verify.get('ok')}`，objects `{verify.get('objects')}`，total `{verify.get('total_gib', 0):.3f}` GiB |
        | Checkpoint restored by OpenPI | `{checkpoint_restored}` |
        | OpenPI websocket server | `{server_ready}` |
        | RoboLab runner exit code | `{smoke_status}` |
        | Task | `{first.get('task_name')}` |
        | Policy | `{first.get('policy')}` |
        | Instruction | {first.get('instruction')} |
        | Success | `{first.get('success')}` |
        | Score | `{first.get('score')}` |
        | Episode step | `{first.get('episode_step')}` |
        | Sim duration | `{first.get('duration')}` s |
        | Policy inference avg | `{timing.get('policy_inference_avg_ms')}` ms |
        | Env step avg | `{timing.get('env_step_avg_ms')}` ms |
        | Reason | {first.get('reason')} |
        | Evidence | `{REMOTE_PI05_POLICY_SMOKE_DIR}` |

        结论：这条记录已经越过“环境 smoke”边界，属于真实 VLA/OpenPI policy 通过 websocket 驱动 RoboLab 的单任务复现证据。
        它仍然只是 1 个任务、1 个 episode，不能外推为 RoboLab-120 完整论文结果。
        '''
        display(Markdown(dedent(report).strip()) if Markdown else dedent(report).strip())

        if pd:
            display(pd.DataFrame(result_rows)[[
                "task_name", "policy", "success", "score", "episode_step", "duration", "reason"
            ]])
            display(pd.DataFrame(event_rows))
        else:
            display(result_rows)
            display(event_rows)

        print("wrote", PI05_POLICY_SMOKE_SUMMARY_PATH)
    else:
        print("Pi05 policy smoke evidence exists, but no episode_results.jsonl record was parsed.")
else:
    print("Pi05 policy smoke evidence not found:", REMOTE_PI05_POLICY_SMOKE_DIR)

## 0.2 论文核心机制：这篇论文到底要测什么

论文不是只提出一个“能跑仿真”的项目，而是提出一个用高保真仿真分析通用机器人策略的 benchmark。
读代码前先抓住五个核心问题：

| 论文概念 | 含义 | 复现时对应要看什么 |
|---|---|---|
| 高保真评测环境 | 让真实世界训练的策略在逼真的 Isaac/Omniverse 场景里被评测，避免只在训练同域里刷高分 | USD 场景、对象 payload、材质/相机/灯光、Isaac Lab env 创建是否稳定 |
| RoboLab-120 | 120 个任务，不是一个 pick-place demo；任务按视觉、过程化、关系三类能力轴拆分 | `task_metadata.json`、每个 `robolab/tasks/benchmark/*.py` 的 `attributes` 和 `subtasks` |
| 任务无关于机器人/策略 | 任务先定义“目标状态”，运行时再绑定 robot、policy、camera、variation | `create_env(...)` 里把任务名解析成 env config，再给 policy runner 使用 |
| 细粒度指标 | 不只看最终 success，还看 normalized/subtask score、语言变化、错误事件、轨迹质量 | `Subtask`、event log、`episode_results.jsonl`、trajectory metrics |
| 敏感性分析 | 通过光照、相机、背景、物体位姿扰动观察策略鲁棒性 | `policies/pi0_family/run_*_variation.py` 和 `create_env(..., events=...)` |

所以当前 notebook 里的 no-policy smoke 只能说明“环境和任务链路可启动、可导出日志”，还不能回答论文里的策略泛化问题。
真正对齐论文，需要等 Pi05/OpenPI server ready 后跑 policy episodes，再用同一套 task metadata、subtask/event、trajectory metric 做分析。

参考源：arXiv 论文 `https://arxiv.org/html/2604.09860`，尤其是 Abstract、III RoboLab、III-B Benchmark Design、III-C Metrics、III-D Sensitivity Analysis。

## 0.3 核心源码地图：论文概念如何落到代码

下面这一节是读 RoboLab 源码的主线。不要从 100 多个 task 文件平铺看起，先按“入口 -> 环境 -> 任务定义 -> 条件/评分 -> 导出 -> policy”这条链路读。

In [ ]:
# ===== 论文概念到源码文件的阅读地图 =====
# 这个表不是运行 RoboLab，而是把“论文里讲的机制”映射到“应该读哪些核心代码”。
core_code_map = [
    {
        "论文机制": "RoboLab-120 任务集合与能力轴",
        "核心文件": "robolab/tasks/_metadata/task_metadata.json",
        "关键字段/函数": "task_name, scene, instruction, attributes, num_subtasks, difficulty_label",
        "怎么理解": "这是任务索引表；论文里的 visual/procedural/relational 和 difficulty 在这里落成可统计字段。",
    },
    {
        "论文机制": "单个任务如何定义目标状态",
        "核心文件": "robolab/tasks/benchmark/banana_in_bowl_task.py",
        "关键字段/函数": "contact_object_list, scene, terminations, instruction, attributes, subtasks",
        "怎么理解": "一个 task 文件不是策略代码，而是目标状态声明：用哪个场景、哪些对象需要 contact sensor、成功条件是什么、拆成哪些 subtask。",
    },
    {
        "论文机制": "从任务名创建仿真环境",
        "核心文件": "robolab/core/environments/runtime.py",
        "关键字段/函数": "create_env, parse_env_cfg, resolve_instruction, merge_events_cfg, gym.make",
        "怎么理解": "这里把 task name 变成 Isaac Lab env；语言版本、扰动事件、policy 名称都在这个阶段绑定进去。",
    },
    {
        "论文机制": "评测 episode 的生命周期",
        "核心文件": "robolab/core/environments/env.py",
        "关键字段/函数": "RobolabEnv.step, _reset_idx, all_terminated, reset_eval_state",
        "怎么理解": "评测时环境不应自动重置并丢掉结果；RoboLab 把已终止 env 冻结，记录成功/失败和终止 step，再导出 episode。",
    },
    {
        "论文机制": "subtask/event 评分",
        "核心文件": "robolab/core/task/conditionals.py + robolab/core/task/subtask.py",
        "关键字段/函数": "pick_and_place, object_in_container, object_grabbed, Subtask(score/logical/K)",
        "怎么理解": "论文说不只看二值 success；这里把任务拆成条件序列，支持 all/any/choose 和子分数。",
    },
    {
        "论文机制": "no-policy 环境 smoke",
        "核心文件": "examples/run_empty.py",
        "关键字段/函数": "auto_register_droid_envs, get_envs, create_env, run_empty_episode, update_experiment_results",
        "怎么理解": "这是当前 4090 已跑通的主入口：它验证环境能创建和导出日志，但没有调用 VLA policy。",
    },
    {
        "论文机制": "真实 policy 评测入口",
        "核心文件": "policies/pi0_family/run.py + robolab/eval/runner.py",
        "关键字段/函数": "add_common_eval_args, run_evaluation, client_factory, run_episode, summarize_run",
        "怎么理解": "Pi0/Pi05 入口只负责接 policy server；通用循环在 runner.py 里，按 task/run/env_id 写 episode 结果。",
    },
    {
        "论文机制": "Pi05 观测打包与动作解包",
        "核心文件": "policies/pi0_family/client.py",
        "关键字段/函数": "_extract_observation, _pack_request, _query_server, _unpack_response, open_loop_horizon",
        "怎么理解": "这里把 RoboLab 的相机图像、关节状态、夹爪状态和语言指令转成 OpenPI server 请求，再把 action chunk 还给环境。",
    },
    {
        "论文机制": "结果汇总和错误分析",
        "核心文件": "robolab/core/logging/results.py",
        "关键字段/函数": "init_experiment, update_experiment_results, summarize_experiment_results, summarize_error_reasons",
        "怎么理解": "episode_results.jsonl 的写入和汇总在这里；后续按任务、能力轴、错误原因统计都依赖它。",
    },
    {
        "论文机制": "轨迹质量指标",
        "核心文件": "robolab/core/metrics/trajectory_metrics.py",
        "关键字段/函数": "compute_sparc, compute_ee_path_length, compute_*_isj",
        "怎么理解": "论文里的 motion quality/trajectory metric 在这里落地；没有 policy action 时这些指标没有论文意义。",
    },
]

if pd:
    display(pd.DataFrame(core_code_map))
else:
    display(core_code_map)

# 写出机器可读的源码阅读地图，作为 notebook 已补足核心代码讲解的证据。
core_map_path = ARTIFACT_DIR / "core_code_reading_map.json"
core_map_path.write_text(json.dumps(core_code_map, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote", core_map_path)

## 0.4 一次 no-policy smoke 的真实调用链

当前已经在 4090 上跑通的 21 个任务，走的是下面这条链路：

```text
examples/run_empty.py
  -> AppLauncher 启动 Isaac/Kit
  -> auto_register_droid_envs() 注册 RoboLab benchmark tasks
  -> get_envs(task=...) 把任务名筛出来
  -> create_env(task_env, device, num_envs, use_fabric)
  -> parse_env_cfg + gym.make(...) 创建 Isaac Lab 环境
  -> run_empty_episode(...) 执行空动作 steps
  -> get_all_env_events(...) 抓取 event/subtask 记录
  -> end_episode(...) 导出 HDF5 / logs
  -> update_experiment_results(...) 追加 episode_results.jsonl
```

这条链路证明三件事：

- task 注册和 env config 能解析；
- Isaac stage、robot、scene、contact sensors 至少在已计入任务上能初始化；
- episode 结果和日志能落盘。

它没有证明两件事：

- policy server 已经产生动作；
- 模型在视觉、关系、过程化任务上有成功率。

## 0.5 核心代码逐段讲解

### 入口层：`examples/run_empty.py`

这个文件是环境烟测入口。它先用 `AppLauncher` 启动 Isaac Sim/Isaac Lab，再调用 `auto_register_droid_envs()` 注册任务。
如果传了 `--task`，`get_envs(task=args_cli.task)` 只选指定任务；没有传 task/tag 时会取默认任务集合。
对每个任务，它调用 `create_env(...)` 得到 `env` 和 `env_cfg`，再用 `run_empty_episode(...)` 执行空动作。
最后它把每个 episode 的 `success`、`instruction`、可选 `score/reason` 写入 `episode_results.jsonl`。

记忆方式：`run_empty.py = 启动 Isaac + 找任务 + 建环境 + 空动作 + 写结果`。

### 环境创建层：`robolab/core/environments/runtime.py`

`create_env` 是任务名进入 Isaac 的关键门口。
它先清空 stage，再调用 `parse_env_cfg(scene, ...)` 解析 task/env config。
如果任务有多个语言版本，`resolve_instruction(...)` 会根据 `instruction_type` 选择 default/vague/specific。
如果要做 camera/light/background 扰动，`merge_events_cfg(...)` 会把 variation event 合并到 env config。
最后 `gym.make(scene, cfg=env_cfg).unwrapped` 创建真正的 Isaac Lab env，并把 `env_cfg.json` 写到输出目录。

记忆方式：`create_env = task name -> env_cfg -> Isaac Lab env`。

### 任务定义层：`robolab/tasks/benchmark/banana_in_bowl_task.py`

一个任务类通常包含五块：

- `contact_object_list`：哪些对象需要 contact sensor；
- `scene = import_scene(...)`：加载哪个 USD 场景；
- `terminations`：成功/超时等终止条件；
- `instruction`：同一任务的多种语言说法；
- `subtasks`：用于 graded score 和错误分析的子任务结构。

以 BananaInBowl 为例，最终成功条件是 `object_in_container(object="banana", container="bowl", ...)`。
subtask 使用 `pick_and_place(object=["banana"], container="bowl", score=1.0)`，也就是“先抓到 banana，再确认 banana 进 bowl”。

记忆方式：`Task = 场景 + 指令 + 成功条件 + 子任务评分`。

### 条件/评分层：`conditionals.py` 与 `subtask.py`

论文强调不能只看二值 success，所以 RoboLab 把任务拆成 subtask 和 event。
`pick_and_place(...)` 是 composite condition，会展开成每个目标对象的一组条件，例如 `object_grabbed` 和 `object_in_container`。
`Subtask` 保存条件组、`logical` 模式和 `score`：`all` 表示全部完成，`any` 表示任一完成，`choose` 表示完成 K 个即可。
这就是 normalized/subtask score 能算出来的原因。

记忆方式：`conditionals = 判断事实；Subtask = 给事实排序和打分`。

### 评测 env 层：`RobolabEnv`

RoboLab 继承 Isaac Lab 的 `ManagerBasedRLEnv`，但评测逻辑和训练环境不同。
在 `_reset_idx` 里，已终止 env 不会立刻自动 reset，而是标记为 frozen，记录 success/truncated 和终止 step，并触发 recorder 导出。
`step` 会把 frozen env 的 action 置零，避免已结束 episode 被后续动作污染。

记忆方式：`RobolabEnv = 为评测冻结终局、保护记录`。

### policy 层：`policies/pi0_family/run.py` 与 `client.py`

`run.py` 负责解析 `--policy pi05`、`--remote-host`、`--remote-port`、`--num-envs`、`--enable-subtask` 等参数。
真正的通用评测循环在 `robolab/eval/runner.py`：创建 env，创建 policy client，调用 `run_episode`，最后 `summarize_run`。
`Pi0DroidJointposClient` 把 RoboLab observation 转成 OpenPI 请求：外部相机图、腕部相机图、关节位置、夹爪位置和语言 prompt。
server 返回 action chunk 后，client 按 `open_loop_horizon` 执行一段动作，再请求下一段。

记忆方式：`policy runner = RoboLab 环境循环；client = 观测/动作协议适配器`。

### 指标层：`results.py` 与 `trajectory_metrics.py`

`results.py` 负责写入和汇总 `episode_results.jsonl`，还能统计错误原因，例如 wrong object、object bumped、target dropped。
`trajectory_metrics.py` 计算 EE path length、SPARC、ISJ 等运动质量指标。
这些指标只有在真实 policy 产生动作后才有意义；no-policy 的空动作 smoke 不应拿来和论文里的 policy 表格比较。

记忆方式：`results = 成败和错误原因；metrics = 动作轨迹质量`。

## 0.6 交流过程中的核心问题与决策记录

这一节不是聊天流水账，而是复现过程中反复校准过的“判断口径”。后面看结果时要按这些边界理解。

| 问题 | 结论 | 对复现实验的影响 |
|---|---|---|
| 4090 能不能跑 RoboLab？ | 能跑，但 24GB VRAM 只能保守跑单任务或小子集。复杂任务运行时显存接近 `22.5GB / 24GB`。 | 首轮固定 `--num-envs 1`，不要直接拉高并行数。 |
| 为什么下载和运行慢？ | 慢点主要来自三类：Git LFS/资产、OpenPI checkpoint、Isaac Sim 首次加载与视频/HDF5 写盘。 | “看起来卡住”不等于失败，要看日志、文件增长、GPU 显存和 episode JSON。 |
| 能不能加速下载？ | 可以按任务补资产、保留缓存、复用 OpenPI server、避免重复拉 LFS；但完整资产和模型仍然是大文件。 | 当前采用“先跑通必要资产，再逐步扩展任务”的路线。 |
| `pi` 原来装过，和这次下载的 pi05 有什么区别？ | RoboChallenge 里的 pi 是另一套 benchmark/接口语境下的策略或环境；这次用的是 OpenPI `pi05_droid_jointpos` checkpoint，经 RoboLab `policies/pi0_family/run.py` 调用。 | 公平对比前必须统一 observation/action 协议、任务、相机、成功条件，不能直接把两个仓库的结果混算。 |
| 视频保存在哪里？ | 远端在 RoboLab `output/<run_id>/`，本地同步到 `remote_outputs/<run_id>/`。每个任务一般有 policy camera 主视频和 viewport 视频。 | 报告里必须给出远端原始路径、本地同步路径和 `ffprobe` 验证信息。 |
| 为什么有些任务没复现成功？ | 要区分两种失败：环境/资产失败和策略失败。`BlockStackingSpecifiedOrderTask` 是 contact sensor/asset 初始化失败；复杂三任务中的两个失败是策略跑满步数未完成。 | 环境失败不计入策略成功率；策略失败才用于能力分析。 |
| 120 个任务要不要一次性全跑？ | 不建议先盲跑完整 RoboLab-120。官方量级约几十 GPU 小时，且当前 checkout 还有任务资产完整性问题。 | 先按能力轴抽样，确认资产、成功条件和视频导出稳定，再扩展到 120。 |
| 代码加中文注释会不会影响运行？ | 注释不影响逻辑，但直接改官方仓库会增加 merge/复现噪声。 | 更稳妥做法是保留原 repo，同时维护 `roboplay_robolab_cn` 注释/讲解版或 notebook 精讲版。 |
| 所有“看进度”的窗口应该在哪？ | 终端、Cursor、VNC 都应在 4090 上看；本机只做同步和整理。 | 后续不要再启动本机 terminal 作为可视化进度窗口。 |
| GitHub 仓库能不能推？ | `dictmap/roboplay` 的 SSH key 已在 4090 上测试通过。 | 后续可把注释版代码和 notebook 推到该仓库，但推送前要确认不含大模型权重和无关输出。 |

记忆方式：先分清四件事：`环境是否可启动`、`策略是否接上`、`任务是否成功`、`证据是否可回放`。这四件事缺一不可，但不能互相替代。

## 0.7 论文精讲：真实场景到模拟场景的评估

下面这节来自本目录的 [EXPLAIN_01_real_to_sim_eval.md](./EXPLAIN_01_real_to_sim_eval.md)，已经把论文里“真实场景到模拟场景评估”的说法翻译成代码实现链路。重点看输入、输出和边界：RoboLab 主流程不是逐场景重建真实视频，而是用资产库、USD 场景、程序化布局、任务谓词和扰动矩阵快速生成可交互、可评分的仿真评估场景。

# 精讲 1：真实场景到模拟场景的评估，代码怎么实现

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>颜色标识</strong>：绿色表示核心结论，蓝色表示源码/输入输出路径，橙色表示边界、风险和容易误解的点。
</div>

## 先说结论

论文这段不是说 RoboLab 自己在主流程里做“真实视频 -> 3DGS/NeRF -> 数字孪生场景”的逐场景重建。RoboLab 的核心思路更务实：

> 不逐个真实场景做昂贵三维重建，而是用已有的高质量 USD/SimReady 资产、HDR 背景、物理参数、程序化场景布局、任务谓词和扰动矩阵，快速生成一批“足够逼真、可交互、可自动评分”的仿真评测场景。

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>核心结论</strong>：RoboLab 的 real-to-sim 思路不是“从真实视频逐场景重建”，而是“用高质量资产库 + 程序化布局 + 物理检查 + 任务谓词”快速生成可评测场景。
</div>

所以代码实现的重点不是 `video_to_3dgs.py` 这类重建脚本，而是下面这条 pipeline：

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>注意边界</strong>：如果在仓库里找不到 <code>video_to_3dgs.py</code> 这类脚本，不是复现漏掉了主流程；论文这里强调的是更快的资产化、程序化评估路线。
</div>

```mermaid
flowchart TD
    A["真实世界经验/任务描述"] --> B["对象资产库 object_catalog.json"]
    B --> C["场景生成: 选择对象 + 空间/物理 predicates"]
    C --> D["SpatialSolver / PhysicalSolver 解出物体位姿"]
    D --> E["生成或读取 USD/USDA 场景"]
    E --> F["Task 文件: 指令 + 成功条件 + subtask"]
    F --> G["registration: 接机器人、相机、动作、灯光、背景"]
    G --> H["Isaac Lab 环境"]
    H --> I["策略评测: 图像 + proprio + instruction -> action"]
    I --> J["结果: success rate / video / HDF5 / event log"]
```

## 1. 资产库：用真实物体的 USD 资产替代逐场景重建

核心路径：

```text
assets/objects/
assets/objects/object_catalog.json
assets/scenes/
assets/backgrounds/
assets/materials/
```

代码入口：

```text
robolab/constants.py
```

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>源码入口</strong>：先看 <code>robolab/constants.py</code> 和 <code>assets/objects/object_catalog.json</code>。前者定义目录，后者定义可生成、可放置、可验证的对象资产。
</div>

关键逻辑：

- `ASSET_DIR = <repo>/assets`
- `OBJECT_DIR = assets/objects`
- `SCENE_DIR = assets/scenes`
- `BACKGROUND_ASSET_DIR = assets/backgrounds`
- `OBJECT_CATALOG_PATH = assets/objects/object_catalog.json`

`object_catalog.json` 是这套方法的基础。每个对象不只是一个名字，还包含：

| 字段 | 作用 |
|---|---|
| `name` | 任务和谓词里使用的对象名 |
| `usd_path` | 真实加载的 USD 资产 |
| `class` / `description` | 语义类别和自然语言描述，用于选物体 |
| `dims` | 几何尺寸，用于放置和碰撞检测 |
| `mass` / friction / restitution | 物理参数，用于可交互仿真 |

例如我们跑的 `BananaInBowlTask` 里：

- `banana` 来自 `assets/objects/ycb/banana.usd`
- `bowl` 来自 `assets/objects/ycb/bowl.usd`
- 两者都有尺寸、质量、摩擦等属性

说人话：RoboLab 不是每次从真实视频里重新重建香蕉和碗，而是复用一个可物理交互的香蕉 USD 和碗 USD，再把它们摆到新的任务场景里。

## 2. 场景文件：USD/USDA 里保存“桌子、物体、材质、位姿”

核心路径：

```text
assets/scenes/banana_bowl.usda
assets/scenes/base_empty.usda
```

`banana_bowl.usda` 里可以看到对象是通过 USD payload 引入的：

```text
def "bowl" (
    prepend payload = @../objects/ycb/bowl.usd@
)
...
double3 xformOp:translate = (0.5611, 0.1526, 0.0303)

def "banana" (
    prepend payload = @../objects/ycb/banana.usd@
)
...
double3 xformOp:translate = (0.5097, -0.1158, 0.0212)
```

也就是说，一个场景本质上是：

- 基础桌面、地面、物理世界配置
- 若干对象 USD payload
- 每个对象的位姿、旋转、缩放
- 材质和纹理

这就是“几分钟生成场景”的关键：生成/修改一个 USDA 文件比训练一个 3DGS 快得多。

## 3. 从“真实场景经验”到“可评估仿真”的最小闭环

这里不再展开 `SpatialSolver`、`PhysicalSolver`、`Task` 字段和 env registration 的细节，那些是精讲2的核心内容。本节只保留 real-to-sim 评估需要的骨架：

| 阶段 | 输入 | 输出 | 为什么对 real-to-sim 评估重要 |
|---|---|---|---|
| 资产化 | 真实物体类别、尺寸、材质经验 | `object_catalog.json` + USD 资产 | 让“真实物体”变成可复用、可物理交互的模拟对象 |
| 场景化 | 物体组合、空间关系、容器/堆叠关系 | `.usda` 场景 | 用低成本方式生成大量可控场景，而不是逐场景重建 |
| 任务化 | 目标状态和语言指令 | `Task` 类、success predicate、subtask | 把“看起来像”升级成“能不能完成任务” |
| 环境化 | 机器人、相机、动作空间、光照、背景 | Isaac Lab/Gym env | 让同一任务可被策略批量执行和复现实验 |
| 评估化 | policy、num_envs、num_runs、扰动矩阵 | success/video/HDF5/log | 输出可比较的成功率、耗时、失败类型和视频证据 |

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>章节分工</strong>：本节只讲 real-to-sim 评估链路为什么成立；<code>scene_gen</code>、<code>import_scene</code>、<code>Task</code>、registration 的代码细节见 [EXPLAIN_02_scene_task_env_generation.md](./EXPLAIN_02_scene_task_env_generation.md)；自动生成 Task 和验证闭环见 [EXPLAIN_03_task_generation_validation.md](./EXPLAIN_03_task_generation_validation.md)。
</div>

### 3.1 这里的“逼真”不是单纯像素相似

RoboLab 对逼真的要求更工程化：场景需要足够支持策略评测，而不是逐像素复刻某个真实厨房视频。换句话说，评估关心的是：

- 视觉上是否有足够真实的物体、纹理、背景和相机视角，让 VLA 模型产生合理感知输入。
- 几何上是否有正确的尺寸、相对位置、容器开口、堆叠关系。
- 物理上是否能抓取、接触、放置、释放，而不是只有静态渲染。
- 任务上是否能自动判断成功/失败，而不是靠人工看视频。

这也是为什么 `object_catalog.json`、`.usda`、success predicate、subtask log、video/HDF5 这些东西比单纯渲染截图更重要。

### 3.2 控制变量比单一场景重建更有评测价值

逐场景 3DGS/NeRF 的优势是像某一个真实场景，但代价是慢、难扩展、难自动评分。RoboLab 选择另一条路线：

```text
同一个任务目标
  × 多个物体组合
  × 多个空间关系
  × 多个背景/光照/相机扰动
  × 多个策略模型
  -> 得到可横向比较的 success rate 和失败模式
```

这对评测 VLA 泛化能力更直接：我们可以问“模型是不是只在默认背景能成功”“是不是颜色换一下就失败”“是不是多物体目标会显著掉分”，而不只是问“这个渲染像不像某段真实视频”。

## 4. 评估输出：不是只看画面像不像，而是看策略能不能完成任务

完整评估入口：

```text
policies/pi0_family/run.py
robolab/eval/runner.py
robolab/eval/episode.py
robolab/eval/summarize.py
```

运行时输入：

```text
task/env name
policy client
num_envs
num_runs
instruction_type
video_mode
```

输出：

| 文件 | 用途 |
|---|---|
| `episode_results.jsonl` | 每个 episode 的成功/失败、步数、耗时、指标 |
| `run_0.hdf5` | 轨迹、动作、物体状态、bbox、末端位姿 |
| `log_0_env0.json` | 每个 env 的事件日志 |
| `.mp4` | sensor / viewport 视频 |
| `env_cfg.json` | 本轮环境配置证据 |

我们已经跑通的单任务复现就是这个机制：

```text
BananaInBowlTask
policy = pi05
success = true
episode_step = 198
video = Pick_up_the_banana_and_place_it_in_the_bowl_0.mp4
viewport video = Pick_up_the_banana_and_place_it_in_the_bowl_0_viewport.mp4
```

## 5. 这段论文对应代码的一句话总结

论文说“相比逐场景 3D 重建，RoboLab 几分钟生成大规模逼真场景和任务”，代码上对应的不是某一个神奇重建脚本，而是一套评估工程：

1. **复用资产**：用 `object_catalog.json` 和 USD 资产把真实物体经验变成可复用模拟对象。
2. **控制变量**：用场景、任务、背景、光照、相机组合出可控扰动矩阵。
3. **自动评分**：用 success predicate 和 subtask log 判断任务完成，而不是人工看视频。
4. **证据闭环**：用 `episode_results.jsonl`、HDF5、event log、视频记录每次策略运行。
5. **横向比较**：同一套任务可替换 policy、背景、相机和难度，用成功率和失败模式比较泛化能力。

## 6. 要记住的边界

RoboLab 主线不是：

```text
真实视频 -> 3DGS/NeRF 重建 -> 单个数字孪生场景
```

RoboLab 主线是：

```text
真实物体/场景经验 -> 可复用 USD 资产 -> 程序化/LLM 辅助生成场景 -> 任务定义 -> 批量策略评测
```

所以它牺牲了一点“逐场景像素级一致性”，换来：

- 更低生成成本；
- 更容易扩展到 100+ 任务；
- 能自动打分；
- 能系统改变背景、光照、相机、物体组合；
- 更适合比较 VLA 策略的泛化能力。

## 0.8 论文精讲：场景、任务和环境生成

下面这节来自本目录的 [EXPLAIN_02_scene_task_env_generation.md](./EXPLAIN_02_scene_task_env_generation.md)，对应论文里的三步：先定位/定向物体生成场景，再把目标状态写成语言任务，最后选择机器人、策略、摄像头、光照、背景来实例化环境。

# 精讲 2：场景、任务和环境生成，代码怎么实现

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>颜色标识</strong>：绿色表示核心结论，蓝色表示源码/输入输出路径，橙色表示边界、风险和容易误解的点。
</div>

## 先说结论

论文这段可以拆成三层：

1. **创建场景**：先决定工作空间里有哪些物体、物体放在哪、朝向是什么，最后落成一个 USD/USDA 场景文件。
2. **定义任务**：再把“目标状态”写成语言指令和成功条件，例如“魔方在碗左边”“所有红色物体在灰色盒子里”。
3. **实例化环境**：最后选择机器人、动作空间、策略观测相机、viewport 相机、灯光、背景，再注册成 Gym/Isaac Lab 环境，供 policy runner 执行。

说人话：**场景是舞台，任务是目标，环境是把舞台、目标、机器人和摄像机装配成一次可运行实验。**

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>核心结论</strong>：这一节要记住三件事：场景负责“有什么、在哪里”，任务负责“要达到什么目标”，环境负责“把机器人、相机、动作和评测输出装配起来”。
</div>

```mermaid
flowchart TD
    A["对象资产库 object_catalog.json"] --> B["场景谓词 JSON"]
    B --> C["SpatialSolver / PhysicalSolver"]
    C --> D["USD/USDA 场景文件"]
    D --> E["Task 类: scene + instruction + terminations + subtasks"]
    E --> F["registration: robot + actions + observations + cameras + lighting + background"]
    F --> G["Gym env id / EnvCfg"]
    G --> H["create_env -> RobolabEnv"]
    H --> I["policy runner: image/proprio/instruction -> action"]
    I --> J["episode_results.jsonl / HDF5 / videos"]
```

注意一个边界：RoboLab benchmark 运行时通常直接读取已经生成好的 `assets/scenes/*.usda`。`robolab/scene_gen/llm_scene_gen/` 是“如何生成新场景”的代码能力；评测时不一定每次重新生成场景。

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>注意边界</strong>：复现 benchmark 时，主流程通常是读取现成 <code>.usda</code> 场景；<code>scene_gen</code> 更像扩展和生成新任务/新场景的能力，不等于每次评测都在线生成。
</div>

## 1. 场景生成：把物体定位和定向

核心文件：

```text
robolab/scene_gen/llm_scene_gen/predicates.py
robolab/scene_gen/llm_scene_gen/spatial_solver.py
robolab/scene_gen/llm_scene_gen/physical_solver.py
skills/robolab-scenegen/SKILL.md
assets/objects/object_catalog.json
assets/scenes/base_empty.usda
assets/scenes/*.usda
robolab/core/scenes/utils.py
```

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>源码入口</strong>：场景生成先看 <code>predicates.py</code>、<code>spatial_solver.py</code>、<code>physical_solver.py</code>；任务装配看 <code>robolab/tasks/benchmark/*.py</code>；环境注册看 <code>registrations/</code> 和 <code>core/environments/runtime.py</code>。
</div>

### 输入是什么

场景生成的输入不是一张图片，而是结构化信息：

| 输入 | 例子 | 作用 |
|---|---|---|
| 对象目录 | `object_catalog.json` | 知道有哪些物体、USD 路径、尺寸、质量等 |
| 谓词 JSON | `left-of`, `place-on-base`, `place-in` | 描述物体之间的空间/物理关系 |
| 工作空间边界 | `table_bounds=(0.25, 0.85, -0.45, 0.45)` | 限制物体只能放在桌面安全区域 |
| base scene | `assets/scenes/base_empty.usda` | 提供桌子、地面、基础 fixture |

`predicates.py` 定义了两类谓词：

| 类型 | 代码对象 | 解决什么 |
|---|---|---|
| 空间谓词 | `SpatialPredicate`, `PlaceOnBasePredicate`, `RelativePositionPredicate` | 桌面 2D 位置：x、y、yaw |
| 物理谓词 | `PhysicalPredicate`, `PlaceOnPredicate`, `PlaceInPredicate` | 3D 放置：z、堆叠、容器内布局 |

典型谓词 JSON：

```json
{
  "objects": [
    {"name": "bowl"},
    {"name": "banana"},
    {"name": "rubiks_cube"}
  ],
  "predicates": [
    {"type": "place-on-base", "object": "bowl", "x": 0.55, "y": 0.0},
    {"type": "left-of", "object": "rubiks_cube", "reference": "bowl", "distance": 0.18},
    {"type": "place-on-base", "object": "banana", "x": 0.40, "y": -0.20},
    {"type": "random-rot", "object": "banana"}
  ]
}
```

### 中间过程

代码主线：

```python
from robolab.scene_gen.llm_scene_gen import (
    ObjectState,
    PlaceOnBasePredicate,
    parse_predicates_from_dict,
    SpatialSolver,
    PhysicalSolver,
    FeedbackSystem,
)

# 1. 每个对象先变成 ObjectState。
object_states = {
    "bowl": ObjectState(name="bowl"),
    "banana": ObjectState(name="banana"),
    "rubiks_cube": ObjectState(name="rubiks_cube"),
}

# 2. JSON 谓词变成 typed Predicate，并挂到对应 object 上。
for pred_data in llm_result["predicates"]:
    pred = parse_predicates_from_dict(pred_data)
    object_states[pred.target_object].predicates.append(pred)

# 3. 空间求解器解 x/y/yaw，并做碰撞检查。
spatial = SpatialSolver(table_bounds=(0.25, 0.85, -0.45, 0.45))
success, msg = spatial.solve(object_states, object_dims)

# 4. 物理求解器处理 place-on / place-in / place-anywhere。
physical = PhysicalSolver()
success, msg = physical.solve(
    object_states,
    object_dims,
    object_usd_paths,
    "assets/scenes/base_empty.usda",
)

# 5. 语法反馈检查哪些物体还缺位置或朝向。
feedback = FeedbackSystem.generate_grammar_feedback(object_states)
```

`SpatialSolver.solve()` 的核心逻辑是：

1. 先处理 `place-on-base`，给物体初始桌面坐标。
2. 再迭代处理 `left-of/right-of/front-of/back-of`。
3. 再应用 `facing-*` 和 `random-rot`。
4. 最后检查碰撞和桌面边界；如果物体太密，会放松 margin 重试。

`PhysicalSolver.solve()` 的核心逻辑是：

1. 把 `place-on` 按支撑物分组，处理堆叠。
2. 处理 `place-in`，把多个物体放进容器。
3. 处理 `place-anywhere`，给未指定位置的物体找可放区域。
4. 可选地用物理仿真 settle/validate 稳定性。

### 输出是什么

输出分两步：

1. 求解结果：每个物体的 `x, y, z, yaw, usd_path`。
2. USDA 场景文件：在 `base_empty.usda` 的基础上插入物体 prim。

USDA prim 大致长这样：

```usda
def "banana" (
    prepend payload = @../objects/ycb/banana.usd@
)
{
    quatf xformOp:orient = (1, 0, 0, 0)
    float3 xformOp:scale = (1, 1, 1)
    double3 xformOp:translate = (0.40, -0.20, 0.035)
    uniform token[] xformOpOrder = ["xformOp:translate", "xformOp:orient", "xformOp:scale"]
}
```

已经生成好的场景被任务文件这样读取：

```python
from robolab.core.scenes.utils import import_scene

scene = import_scene("banana_bowl.usda", ["banana", "bowl", "table"])
```

`import_scene()` 会做三件事：

1. 在 `assets/scenes/` 下找到 `banana_bowl.usda`。
2. 解析 USD 里的 rigid body、kinematic body、static body。
3. 动态生成一个 Isaac Lab `SceneConfig`，里面每个物体变成 `RigidObjectCfg` 或 `AssetBaseCfg`。

## 2. 任务定义：把目标状态写成语言指令

核心文件：

```text
robolab/tasks/benchmark/*.py
robolab/core/task/task.py
robolab/core/task/conditionals.py
robolab/core/task/subtask.py
```

一个任务类通常包含五件事：

| 字段 | 例子 | 作用 |
|---|---|---|
| `contact_object_list` | `["banana", "bowl", "table"]` | 哪些物体要启用接触检测 |
| `scene` | `import_scene("banana_bowl.usda", ...)` | 任务使用哪个场景 |
| `terminations` | `success = DoneTerm(func=object_in_container, ...)` | 什么时候成功或超时 |
| `instruction` | default/vague/specific | 给策略模型的语言输入 |
| `subtasks` | `pick_and_place(...)` | 过程化进度和 graded score |

### 示例 A：香蕉放进碗

对应文件：`robolab/tasks/benchmark/banana_in_bowl_task.py`

```python
@configclass
class BananaInBowlTerminations:
    time_out = DoneTerm(func=mdp.time_out, time_out=True)
    success = DoneTerm(
        func=object_in_container,
        params={
            "object": "banana",
            "container": "bowl",
            "gripper_name": "gripper",
            "tolerance": 0.0,
            "require_contact_with": True,
            "require_gripper_detached": True,
        },
    )

@dataclass
class BananaInBowlTask(Task):
    contact_object_list = ["banana", "bowl", "table"]
    scene = import_scene("banana_bowl.usda", contact_object_list)
    terminations = BananaInBowlTerminations
    instruction = {
        "default": "Pick up the banana and place it in the bowl",
        "vague": "Put the fruit in the bowl",
        "specific": "Grasp the yellow banana and place it inside the red bowl on the table",
    }
    episode_length_s = 50
    attributes = ["semantics"]
    subtasks = [
        pick_and_place(object=["banana"], container="bowl", logical="all", score=1.0)
    ]
```

这里的“目标状态”不是自然语言本身，而是 `object_in_container(banana, bowl)` 这个物理/几何谓词。语言只是给 VLA 模型看的 prompt。

### 示例 B：魔方放到碗左边

对应文件：`robolab/tasks/benchmark/rubiks_cube_left_of_bowl.py`

```python
success = DoneTerm(
    func=object_left_of,
    params={
        "object": "rubiks_cube",
        "reference_object": "bowl",
        "frame_of_reference": "robot",
        "mirrored": False,
        "require_gripper_detached": True,
    },
)

instruction = {
    "default": "Put the rubiks cube to the left of the bowl",
    "vague": "Put the cube left of the bowl",
    "specific": "Pick up the rubiks cube and place it on the table to the left side of the bowl",
}
```

这个任务测的是关系推理：模型要识别 bowl，理解 left-of，并把 rubiks cube 放到正确相对位置。

### 示例 C：所有红色物体放进灰色盒子

对应文件：`robolab/tasks/benchmark/red_items_in_bin.py`

```python
success = DoneTerm(
    func=object_in_container,
    params={
        "object": ["mug", "bowl"],
        "container": "grey_bin",
        "logical": "all",
        "require_gripper_detached": True,
    },
)

instruction = {
    "default": "Put all the red things in the grey bin",
    "vague": "Sort items by color",
    "specific": "Identify every red-colored object on the table and place each one into the grey bin",
}
```

这里的难点是集合目标：不是移动一个对象，而是识别所有红色物体，并且 `logical="all"` 要求全部完成。

### 示例 D：3 个魔方堆叠

对应文件：`robolab/tasks/benchmark/rubiks_cube_stacking_task.py`

```python
success = DoneTerm(
    func=stacked,
    params={
        "objects": ["rubiks_cube", "rubiks_cube_1", "rubiks_cube_2"],
        "order": "None",
    },
)

subtasks = [
    Subtask(
        name="stack_any_2_cubes",
        conditions={
            "cube_0_and_1": partial(stacked, objects=["rubiks_cube", "rubiks_cube_1"], order=None),
            "cube_0_and_2": partial(stacked, objects=["rubiks_cube", "rubiks_cube_2"], order=None),
            "cube_1_and_2": partial(stacked, objects=["rubiks_cube_1", "rubiks_cube_2"], order=None),
        },
        logical="any",
        score=0.5,
    ),
    Subtask(
        name="stack_all_3_cubes",
        conditions=partial(stacked, objects=["rubiks_cube", "rubiks_cube_1", "rubiks_cube_2"], order=None),
        score=0.5,
    ),
]
```

这个例子说明 `subtasks` 不是摆设：即使最终没成功，也能知道模型有没有完成“任意两个先堆起来”这种中间进度。

## 3. 环境实例化：选择机器人、策略和场景变化

核心文件：

```text
robolab/registrations/droid/auto_env_registrations_jointpos.py
robolab/core/environments/config.py
robolab/core/environments/factory.py
robolab/core/environments/runtime.py
robolab/robots/droid.py
robolab/variations/camera.py
robolab/variations/lighting.py
robolab/variations/backgrounds.py
robolab/eval/runner.py
policies/pi0_family/run.py
```

### 注册阶段

Pi05 这次走的是 DROID joint-position 注册路线：

```python
from robolab.registrations.droid.auto_env_registrations_jointpos import auto_register_droid_envs

auto_register_droid_envs(
    task_dirs=["benchmark"],
    task=["BananaInBowlTask"],
    randomize_background=False,
)
```

`auto_register_droid_envs()` 做了这些事：

1. 选择相机 preset。默认 `WRIST_LEFT = [OverShoulderLeftCameraCfg, WristCameraCfg]`。
2. 根据相机生成 `image_obs`，再加上 `proprio_obs` 和 `viewport_cam`。
3. 选择机器人：`DroidCfg`，实际是 Franka + Robotiq gripper 的 USD。
4. 选择动作空间：`DroidJointPositionActionCfg`。
5. 选择灯光：默认 `SphereLightCfg`。
6. 选择背景：默认 `HomeOfficeBackgroundCfg`，也可随机背景。
7. 调用 `auto_discover_and_create_cfgs()` 扫描 task 文件并注册 Gym env。

关键组合发生在 `generate_scene_env_cfg()`：

```python
bases = [task_class.scene, robot_cfg, InteractiveSceneCfg]

if camera_cfg is not None:
    bases.append(camera_cfg)
if lighting_cfg is not None:
    bases.append(lighting_cfg)
if background_cfg is not None:
    bases.append(background_cfg)

cfg_cls = type(f"{task_class.__name__}SceneEnvCfg", tuple(bases), {})
```

这就是论文里“选择机器人、摄像头、光照、背景来实例化环境”的代码实现：它不是写死一个大类，而是把多个 config mixin 动态拼成一个 scene env config。

### 运行阶段

真正创建环境发生在 `robolab/core/environments/runtime.py`：

```python
env_cfg = parse_env_cfg(
    scene,
    device=device,
    seed=seed,
    num_envs=num_envs,
    env_spacing=env_spacing,
    use_fabric=use_fabric,
    eye=eye,
    lookat=lookat,
)

env_cfg._instruction_variants = env_cfg.instruction
env_cfg.instruction = resolve_instruction(env_cfg.instruction, instruction_type)

if events is not None:
    env_cfg.events = merge_events_cfg(env_cfg.events, events)

env = gym.make(scene, cfg=env_cfg).unwrapped
```

输入是：

| 输入 | 例子 |
|---|---|
| env name | `BananaInBowlTask` |
| device | `cuda:0` |
| num_envs | `1` |
| instruction type | `default`, `vague`, `specific` |
| optional events | camera pose jitter, reset randomization |
| policy | `pi05` |

输出是：

| 输出 | 说明 |
|---|---|
| `env` | 真正的 `RobolabEnv` / Isaac Lab 环境 |
| `env_cfg` | 完整配置：scene、robot、actions、observations、events、terminations |
| `env_cfg.json` | 每个任务目录下保存一份，用于复现 |
| 运行结果 | `episode_results.jsonl`、HDF5、事件日志、视频 |

### policy runner 怎么接上

Pi05 命令入口是：

```bash
uv run python policies/pi0_family/run.py \
  --policy pi05 \
  --remote-host localhost \
  --remote-port 8000 \
  --task BananaInBowlTask \
  --num-envs 1 \
  --num-runs 1 \
  --video-mode all \
  --headless \
  --device cuda:0
```

`policies/pi0_family/run.py` 的逻辑是：

1. 先启动 Isaac Sim `AppLauncher`。
2. 调用 `auto_register_droid_envs(...)` 注册任务环境。
3. 创建 `Pi0DroidJointposClient` 连接 OpenPI server。
4. 进入 `robolab/eval/runner.py::run_evaluation()`。
5. 对每个 task 调用 `create_env()`。
6. 每个 episode 中执行：observation + instruction -> policy action -> env.step。
7. 写出结果和视频。

## 4. 额外示例

### 示例 1：只换语言指令难度

同一个任务可以用不同语言粒度运行：

```bash
uv run python policies/pi0_family/run.py \
  --policy pi05 \
  --remote-host localhost \
  --remote-port 8000 \
  --task BananaInBowlTask \
  --instruction-type vague \
  --num-envs 1 \
  --num-runs 1 \
  --video-mode all \
  --headless \
  --device cuda:0
```

效果：场景和成功条件不变，只把模型看到的 instruction 从 default 换成 vague。这样可以单独测语言模糊度对策略的影响。

### 示例 2：随机背景，测视觉鲁棒性

```bash
uv run python policies/pi0_family/run.py \
  --policy pi05 \
  --remote-host localhost \
  --remote-port 8000 \
  --task BananaInBowlTask RedItemsInBinTask \
  --randomize-background \
  --background-seed 42 \
  --num-envs 1 \
  --num-runs 1 \
  --video-mode all \
  --headless \
  --device cuda:0
```

效果：每个 task 注册时会抽一个非默认 HDR/EXR 背景，并固定写入该 task 的 `env_cfg`。这适合比较“同一策略在默认背景 vs 随机背景”的成功率变化。

### 示例 3：背景矩阵实验

`auto_env_registrations_bg_variations.py` 会为每个 `(task, background)` 组合注册一个 env：

```python
from robolab.registrations.droid.auto_env_registrations_bg_variations import (
    auto_register_droid_envs_bg_variations,
)

auto_register_droid_envs_bg_variations(
    tasks=["BananaInBowlTask", "RubiksCubeAndBananaTask"],
    backgrounds=["empty_warehouse.hdr", "billiard_hall.hdr"],
)
```

它会形成类似：

```text
BananaInBowlTask_bg_empty_warehouse
BananaInBowlTask_bg_billiard_hall
RubiksCubeAndBananaTask_bg_empty_warehouse
RubiksCubeAndBananaTask_bg_billiard_hall
```

这适合做论文里的 sensitivity analysis，但任务数乘背景数会很快变大。4090 上建议先小矩阵验证。

### 示例 4：自定义任务只需要补哪几块

这部分不再展开完整 Task 文件，避免和精讲3重复。只记住：假设已有 `assets/scenes/my_banana_bowl.usda`，一个最小任务只需要补齐 6 个字段。

| 字段 | 最小内容 | 作用 |
|---|---|---|
| `contact_object_list` | `["banana", "bowl", "table"]` | 告诉环境哪些对象需要接触/状态跟踪 |
| `scene` | `import_scene("my_banana_bowl.usda", contact_object_list)` | 绑定这个任务使用哪个场景 |
| `terminations` | `object_in_container(...)` | 定义怎样算成功 |
| `instruction` | default / vague / specific | 给 VLA policy 的语言输入 |
| `episode_length_s` | 例如 `50` | 控制最长尝试时间 |
| `subtasks` | `pick_and_place(...)` | 记录过程进度和 graded score |

最小骨架可以理解成：

```python
class MyBananaTask(Task):
    scene = import_scene("my_banana_bowl.usda", contact_object_list)
    instruction = {...}
    terminations = MyBananaTerminations
    subtasks = [...]
```

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>分工提醒</strong>：本节只说明任务如何接入环境装配；完整 Task 生成、语法验证、资产验证和修复提示见 [EXPLAIN_03_task_generation_validation.md](./EXPLAIN_03_task_generation_validation.md)。
</div>

关键点：任务文件不需要写机器人和相机。机器人、动作空间、观测、相机、灯光、背景都在注册阶段统一装配。

## 5. 跟论文三句话逐句对应

| 论文表述 | 代码实现 | 输入 | 输出 |
|---|---|---|---|
| 在工作空间中定位和定向物体来创建场景 | `predicates.py` + `SpatialSolver` + `PhysicalSolver` + USDA 生成 | object catalog、谓词 JSON、base scene | solved poses、`*.usda` |
| 将任务定义为场景中目标状态的语言指令 | `tasks/benchmark/*.py` + `conditionals.py` + `subtask.py` | scene 文件、对象名、目标谓词、instruction variants | `Task` 类、成功条件、subtask score |
| 通过选择机器人、策略以及场景特征变化实例化环境 | `auto_env_registrations_jointpos.py` + `config.py` + `runtime.py` + `runner.py` | Task、robot/action/obs/camera/light/background/policy | `RobolabEnv`、`env_cfg.json`、视频、HDF5、JSONL |

一句话记忆：

```text
Scene = objects + poses + USD
Task = scene + instruction + success predicate + subtasks
Env = task + robot + observations + actions + cameras + light + background + policy runner
```

## 0.9 论文精讲：扩展任务生成、验证和自动修复

下面这节来自本目录的 [EXPLAIN_03_task_generation_validation.md](./EXPLAIN_03_task_generation_validation.md)，对应论文里的 TaskGen 闭环：先给 LLM 场景对象、能力轴、任务示例和谓词库，让它生成 Task Python 代码；再做语法、资产、容器尺寸等验证；失败后把原始提示、无效输出和错误信息重新打包给 LLM 修复。

# 精讲 3：扩展任务生成、验证和自动修复，代码怎么实现

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>颜色标识</strong>：绿色表示核心结论，蓝色表示源码/输入输出路径，橙色表示边界、风险和容易误解的点。
</div>

## 先说结论

论文这段讲的是 **TaskGen 的闭环**：

```text
给 LLM 足够上下文
  -> LLM 生成 RoboLab Task Python 代码
  -> 检查 Python 语法
  -> 检查对象/场景/容器/物理可行性
  -> 失败时把原始提示、无效输出、错误信息打包成修复提示
  -> 让 LLM 重写，再重复验证
```

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>核心结论</strong>：TaskGen 不是“让 LLM 一次写完就相信它”，而是“生成代码 -&gt; 自动验证 -&gt; 把错误反馈回 LLM -&gt; 重写”，直到语法、资产和物理可行性都过关。
</div>

当前 RoboLab checkout 里，这个流程不是一个单独的 `run_taskgen.py` 大脚本，而是分散在几类文件里：

| 功能 | 当前实现位置 | 说明 |
|---|---|---|
| LLM 任务生成提示/模板 | `skills/robolab-taskgen/SKILL.md` | 定义让 LLM 生成什么、问用户什么、输出什么 |
| 任务示例 | `skills/robolab-taskgen/references/examples.md` | 简单 pick/place、多物体排序、有序堆叠 |
| 谓词库说明 | `skills/robolab-taskgen/references/conditionals.md` + `robolab/core/task/conditionals.py` | 成功条件、终止条件、subtask 条件 |
| Python 语法/导入验证 | `compile(...)`、`robolab/core/task/task_utils.py::load_task_from_file` | 检查代码能不能作为 Task 类加载 |
| metadata/难度统计 | `robolab/tasks/_utils/generate_task_metadata.py` + `load_task_info.py` | 生成 `task_metadata.json`、CSV、README |
| 场景资源验证 | `robolab/core/scenes/utils.py::scrape_scene/import_scene` | 检查 USD 场景里有哪些对象 |
| 场景/物理反馈 | `robolab/scene_gen/llm_scene_gen/feedback_system.py` | 给 LLM 返回碰撞、越界、不稳定等反馈 |
| 仿真 smoke | `examples/run_empty.py --task <TaskClassName>` | 真的启动 Isaac，检查注册和运行是否通 |

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>源码入口</strong>：TaskGen 先看 <code>skills/robolab-taskgen/SKILL.md</code>、<code>references/examples.md</code>、<code>references/conditionals.md</code>；验证链路再看 <code>task_utils.py::load_task_from_file</code>、<code>core/scenes/utils.py::scrape_scene/import_scene</code> 和 <code>scene_gen/.../feedback_system.py</code>。
</div>

说人话：**LLM 负责编代码，RoboLab 负责把代码变成可运行 Task；验证层负责告诉 LLM 哪个对象不存在、哪个容器装不下、哪个谓词写错了。**

## 1. LLM 输入：TaskGen 给模型什么信息

论文说给 LLM 五类上下文，当前代码/skill 对应如下：

| 论文里的输入 | 当前对应来源 | 用途 |
|---|---|---|
| 场景对象目录和尺寸元数据 | `assets/objects/object_catalog.json`、`assets/scenes/_metadata/scene_metadata.json`、`scrape_scene()` | 知道对象名、尺寸、USD 路径、物理属性 |
| 任务结构示例 | `skills/robolab-taskgen/references/examples.md`、`robolab/tasks/benchmark/*.py` | 让 LLM 模仿 Task 文件格式 |
| 子任务成功/终止谓词库 | `references/conditionals.md`、`conditionals.py` | 把自然语言目标映射成 `object_in_container` 等函数 |
| 能力轴语言模板 | taskgen skill 的 instruction variants、attributes | 生成 default/vague/specific 和 `color/spatial/sorting/stacking` 标签 |
| 难度和物理可行性约束 | `episode_length_s` 规则、`compute_difficulty_score`、容器/堆叠尺寸检查 | 避免生成不可能完成或不可评估的任务 |

`skills/robolab-taskgen/SKILL.md` 对任务文件的定义很明确：

> 一个 task 是 Python 文件，里面有 `Task` dataclass，把 USD 场景绑定到语言指令和 termination criteria。

它给 LLM 的模板核心是：

```python
@configclass
class <TaskName>Terminations:
    time_out = DoneTerm(func=mdp.time_out, time_out=True)
    success = DoneTerm(
        func=<conditional_function>,
        params={<params_dict>},
    )

@dataclass
class <TaskName>Task(Task):
    contact_object_list = [<all_object_names>]
    scene = import_scene("<scene_file>.usda", contact_object_list)
    terminations = <TaskName>Terminations
    instruction = {
        "default": "<clear instruction>",
        "vague": "<ambiguous instruction>",
        "specific": "<detailed instruction>",
    }
    episode_length_s: int = <seconds>
    attributes = [<attribute_tags>]
    subtasks = [<optional_subtasks>]
```

## 2. 生成：从能力轴和语言模板到 Task 代码

自然语言目标首先被路由到一个谓词函数：

| 任务描述 | termination 函数 | subtask 写法 |
|---|---|---|
| “Put X in Y” | `object_in_container` | `pick_and_place` |
| “Put X on Y” | `object_on_top` | `pick_and_place_on_surface` |
| “Move X left/right of Y” | `object_left_of` / `object_right_of` | `Subtask(partial(...))` |
| “Sort X into bins” | `object_groups_in_containers` | 多个 `pick_and_place` |
| “Stack X, Y, Z” | `stacked` | 多个 `Subtask(partial(stacked, ...))` |
| “Take X out of Y” | `object_outside_of` | `Subtask(partial(...))` |
| “Stand X upright” | `object_upright` | `Subtask(partial(...))` |

### 示例：生成一个颜色分类任务

输入信息可以组织成这样：

```json
{
  "scene": "rubiks_cube_banana_bowl_mug_bin.usda",
  "objects": ["mug", "bowl", "grey_bin", "banana", "rubiks_cube", "table"],
  "axis": ["visual.color", "relational.sorting"],
  "instruction_template": "Put all the {color} things in the {container}",
  "slots": {
    "color": "red",
    "container": "grey_bin",
    "target_objects": ["mug", "bowl"]
  },
  "difficulty": "moderate",
  "constraints": {
    "all_target_objects_must_fit_container": true,
    "require_gripper_detached": true
  }
}
```

LLM 应输出类似：

```python
@configclass
class RedItemsInBinTerminations:
    time_out = DoneTerm(func=mdp.time_out, time_out=True)
    success = DoneTerm(
        func=object_in_container,
        params={
            "object": ["mug", "bowl"],
            "container": "grey_bin",
            "logical": "all",
            "require_gripper_detached": True,
        },
    )

@dataclass
class RedItemsInBinTask(Task):
    contact_object_list = ["mug", "bowl", "grey_bin", "table", "banana", "rubiks_cube"]
    scene = import_scene("rubiks_cube_banana_bowl_mug_bin.usda", contact_object_list)
    terminations = RedItemsInBinTerminations
    instruction = {
        "default": "Put all the red things in the grey bin",
        "vague": "Sort items by color",
        "specific": "Identify every red-colored object on the table and place each one into the grey bin",
    }
    episode_length_s: int = 60
    attributes = ["color", "sorting"]
    subtasks = [
        pick_and_place(object=["mug", "bowl"], container="grey_bin", logical="all", score=1.0)
    ]
```

这个生成结果对应我们已经跑过的 `RedItemsInBinTask`。它在复杂任务实测里失败，失败原因不是 Task 代码无效，而是策略没有在 900 步内完成所有红色物体入盒。

## 3. 语法验证：先保证 Python 文件能被加载

最低成本验证是 Python 编译：

```python
compile(generated_code, "generated_task.py", "exec")
```

它能抓：

- 括号没闭合；
- 缩进错误；
- 字符串引号错误；
- 非法 Python 语法。

更接近 RoboLab 的验证是加载 Task 类：

```python
from robolab.core.task.task_utils import load_task_from_file

task_class = load_task_from_file("generated_task.py", allow_multiple=False)
```

`load_task_from_file()` 会：

1. 用 `importlib.util.spec_from_file_location` 导入 Python 文件。
2. 扫描模块里所有类。
3. 找到 `issubclass(attr, Task)` 且不是基类 `Task` 的类。
4. 没找到就报 `No Task subclass found`。

所以它不只是语法检查，还能发现：

- 没写 `Task` 子类；
- class 名写错；
- import 错；
- `@dataclass` 中可变默认值写法导致导入失败。

## 4. 资源验证：对象是否存在、是否禁用、容器是否装得下

论文里说的资源验证包含两类：

1. **对象引用验证**：任务代码里引用的对象必须存在于场景对象集合中，也不能落入禁用集合。
2. **物理可行性验证**：例如容器任务里，内部对象要能放进容器，并留出 margin。

当前 RoboLab 提供了 `scrape_scene()` / `import_scene()` 从 USD 场景中提取对象：

```python
from robolab.core.scenes.utils import scrape_scene

scene_info = scrape_scene("assets/scenes/rubiks_cube_banana_bowl_mug_bin.usda")
available_objects = set(scene_info["dynamic_bodies"] + scene_info["kinematic_bodies"] + scene_info["static_bodies"])
```

`import_scene(scene_path, objects_of_interest)` 会只把 `objects_of_interest` 里的对象转成 `RigidObjectCfg` / `AssetBaseCfg`。如果对象名不在场景里，后面创建 contact sensor 或 env 时就会失败。

论文提到的“禁用集合”和“容器尺寸 margin”在当前 checkout 里没有看到一个统一的公开 `TaskValidator` 类；实际工程上可以在 LLM 输出落盘前补一个轻量验证层：

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>注意边界</strong>：当前公开 checkout 里 TaskGen 验证能力是分散的，不是一个完整公开的 <code>TaskValidator</code> 大类。轻量测试能提前拦截明显错误，但不能替代 Isaac Sim 里的真实注册、接触传感器和物理 smoke。
</div>

```python
def validate_objects(task_objects, scene_objects, disabled_objects):
    errors = []
    missing = sorted(set(task_objects) - set(scene_objects))
    disabled = sorted(set(task_objects) & set(disabled_objects))
    if missing:
        errors.append(f"Objects not found in scene: {missing}")
    if disabled:
        errors.append(f"Objects are disabled for task generation: {disabled}")
    return errors

def validate_container_fit(objects, container, dims, margin=0.02):
    errors = []
    cx, cy, cz = dims[container]
    usable_x = cx - 2 * margin
    usable_y = cy - 2 * margin
    for obj in objects:
        ox, oy, oz = dims[obj]
        # 保守检查：允许物体旋转，取较短边对较短边、较长边对较长边。
        obj_short, obj_long = sorted([ox, oy])
        box_short, box_long = sorted([usable_x, usable_y])
        if obj_short > box_short or obj_long > box_long:
            errors.append(
                f"{obj} ({ox:.2f}x{oy:.2f}) does not fit into {container} "
                f"usable opening ({usable_x:.2f}x{usable_y:.2f}) with margin={margin}"
            )
    return errors
```

这个检查不替代 Isaac Sim 物理验证，但可以提前拦掉明显不可能的任务。

## 5. 修复提示：把错误反馈给 LLM

论文里的修复提示可以写成：

```text
Original prompt Q:
<原始任务生成请求>

Invalid output:
<LLM 上一次生成的 Python 代码>

Validation errors E:
1. SyntaxError: '(' was never closed
2. Objects not found in scene: ['apple']
3. mug does not fit into small_box with margin=0.02

Please revise the RoboLab task. Requirements:
- Keep the same scene unless the error says the scene is invalid.
- Use only objects that exist in the scene and are not disabled.
- For container tasks, choose target objects that physically fit.
- Return one complete Python task file only.
```

这个循环和 `scene_gen` 里的 `FeedbackSystem` 思路一致：失败时不要只说“不行”，要告诉 LLM 具体哪个谓词、哪个对象、哪个尺寸或哪段语法错了。

## 6. 完整闭环伪代码

```python
def taskgen_loop(prompt, scene_objects, object_dims, disabled_objects, max_rounds=3):
    q = build_initial_prompt(
        prompt=prompt,
        scene_objects=scene_objects,
        object_dims=object_dims,
        examples=TASK_EXAMPLES,
        conditionals=CONDITIONAL_LIBRARY,
        axis_templates=CAPABILITY_AXIS_TEMPLATES,
        constraints=PHYSICAL_CONSTRAINTS,
    )

    last_output = None
    last_errors = []

    for round_idx in range(max_rounds):
        generated_code = llm(q)
        errors = []

        errors += validate_python_syntax(generated_code)
        if not errors:
            spec = extract_task_spec(generated_code)
            errors += validate_objects(
                task_objects=spec.contact_object_list,
                scene_objects=scene_objects,
                disabled_objects=disabled_objects,
            )
            errors += validate_conditionals(spec)
            errors += validate_container_fit_from_spec(spec, object_dims)

        if not errors:
            return generated_code

        last_output = generated_code
        last_errors = errors
        q = build_repair_prompt(
            original_prompt=prompt,
            invalid_output=last_output,
            errors=last_errors,
        )

    raise RuntimeError(f"Task generation failed after {max_rounds} rounds: {last_errors}")
```

## 7. 和 RoboLab 当前代码逐步对应

| 论文步骤 | 当前代码/可补验证 |
|---|---|
| 生成任务代码 | `skills/robolab-taskgen/SKILL.md` 的模板 + examples + conditionals |
| 验证代码语法 | `compile(...)`，以及 `task_utils.load_task_from_file()` |
| 验证场景资源选择 | `scrape_scene()` / `import_scene()` + `contact_object_list` 对比 |
| 验证容器尺寸 | 可基于 `object_catalog.json` / scene metadata 的 `dims` 增加 margin 检查 |
| 生成任务 metadata | `generate_task_metadata.py`、`load_task_info.py` |
| 仿真 smoke | `examples/run_empty.py --task <TaskClassName>` |
| 失败反馈修复 | `FeedbackSystem` 思路 + repair prompt |

## 8. 额外测试用例设计

下面这些测试不需要启动 Isaac Sim，适合放在 notebook 里快速跑，用来验证 TaskGen 生成前后的“轻量 gate”：

| 测试 | 输入 | 期望 |
|---|---|---|
| `test_valid_spec_passes` | banana、bowl、table 都在 scene，banana 能放进 bowl | 0 个错误 |
| `test_syntax_error_fails` | 少一个右括号的 Python 代码 | 返回 `SyntaxError` |
| `test_missing_object_fails` | task 引用 `apple`，scene 没有 apple | 返回 missing object |
| `test_disabled_object_fails` | task 引用 `knife`，禁用集合含 knife | 返回 disabled object |
| `test_container_too_small_fails` | `large_box` 放进 `small_bowl` | 返回 does not fit |
| `test_repair_prompt_contains_context` | Q、无效代码、错误 E | 修复提示包含三者 |

这些测试验证的是 TaskGen 的前置质量门，不是最终机器人策略成功率。真正要证明任务可运行，还要跑 `examples/run_empty.py --task <TaskClassName>`。

In [ ]:
# ===== 精讲3：TaskGen 轻量验证测试 =====
# 这组测试不启动 Isaac Sim，只验证论文 TaskGen 闭环里的前置检查逻辑：
# 1. Python 代码语法是否有效；
# 2. 任务引用的对象是否存在于场景中；
# 3. 是否误用了禁用对象；
# 4. 容器任务里目标物体是否能放进容器，并留出 margin；
# 5. 失败时是否能构造包含 Q / invalid output / E 的修复提示。

def validate_python_syntax(code_text: str) -> list[str]:
    '''检查 LLM 生成的任务文件是否至少是合法 Python 代码。'''
    try:
        compile(code_text, "generated_task.py", "exec")
        return []
    except SyntaxError as exc:
        return [f"SyntaxError: {exc.msg} at line {exc.lineno}"]


def validate_objects(task_objects, scene_objects, disabled_objects):
    '''检查任务中引用的对象是否存在于场景中，并且不在禁用集合里。'''
    errors = []
    missing = sorted(set(task_objects) - set(scene_objects))
    disabled = sorted(set(task_objects) & set(disabled_objects))
    if missing:
        errors.append(f"Objects not found in scene: {missing}")
    if disabled:
        errors.append(f"Objects are disabled for task generation: {disabled}")
    return errors


def validate_container_fit(objects, container, dims, margin=0.02):
    '''保守检查容器任务是否物理可行：目标物体平面尺寸要小于容器可用开口。'''
    errors = []
    if container not in dims:
        return [f"Container not found in dims: {container}"]
    cx, cy, _ = dims[container]
    usable_x = cx - 2 * margin
    usable_y = cy - 2 * margin
    for obj in objects:
        if obj not in dims:
            errors.append(f"Object dim missing: {obj}")
            continue
        ox, oy, _ = dims[obj]
        # 允许物体在桌面平面旋转，所以短边对短边、长边对长边。
        obj_short, obj_long = sorted([ox, oy])
        box_short, box_long = sorted([usable_x, usable_y])
        if obj_short > box_short or obj_long > box_long:
            errors.append(
                f"{obj} ({ox:.2f}x{oy:.2f}) does not fit into {container} "
                f"usable opening ({usable_x:.2f}x{usable_y:.2f}) with margin={margin}"
            )
    return errors


def build_repair_prompt(original_prompt, invalid_output, errors):
    '''把验证失败信息整理成给 LLM 的修复提示，对应论文里的 Q + invalid output + E。'''
    return "\n".join(
        [
            "Original prompt Q:",
            original_prompt.strip(),
            "",
            "Invalid output:",
            invalid_output.strip(),
            "",
            "Validation errors E:",
            *[f"{idx + 1}. {err}" for idx, err in enumerate(errors)],
            "",
            "Please revise the RoboLab task. Return one complete Python task file only.",
        ]
    )


VALID_CODE = "from dataclasses import dataclass\n@dataclass\nclass DummyTask:\n    pass\n"
BAD_CODE = "def broken_task(\n    return 1\n"

SCENE_OBJECTS = {
    "banana",
    "bowl",
    "table",
    "grey_bin",
    "mug",
    "rubiks_cube",
    "knife",
    "large_box",
    "small_bowl",
}
DISABLED_OBJECTS = {"knife"}
DIMS = {
    "banana": (0.18, 0.04, 0.04),
    "bowl": (0.24, 0.24, 0.11),
    "grey_bin": (0.36, 0.28, 0.16),
    "mug": (0.08, 0.08, 0.10),
    "large_box": (0.30, 0.22, 0.12),
    "small_bowl": (0.18, 0.18, 0.08),
}

taskgen_tests = []
taskgen_tests.append(
    (
        "valid_spec_passes",
        validate_python_syntax(VALID_CODE) == []
        and validate_objects(["banana", "bowl"], SCENE_OBJECTS, DISABLED_OBJECTS) == []
        and validate_container_fit(["banana"], "bowl", DIMS) == [],
    )
)
taskgen_tests.append(
    ("syntax_error_fails", any("SyntaxError" in err for err in validate_python_syntax(BAD_CODE)))
)
taskgen_tests.append(
    ("missing_object_fails", any("apple" in err for err in validate_objects(["apple"], SCENE_OBJECTS, DISABLED_OBJECTS)))
)
taskgen_tests.append(
    ("disabled_object_fails", any("knife" in err for err in validate_objects(["knife"], SCENE_OBJECTS, DISABLED_OBJECTS)))
)
taskgen_tests.append(
    ("container_too_small_fails", any("does not fit" in err for err in validate_container_fit(["large_box"], "small_bowl", DIMS)))
)

repair_prompt = build_repair_prompt(
    "Generate a RoboLab sorting task.",
    BAD_CODE,
    validate_python_syntax(BAD_CODE),
)
taskgen_tests.append(
    (
        "repair_prompt_contains_context",
        all(
            required in repair_prompt
            for required in ["Original prompt Q", "Invalid output", "Validation errors E", "SyntaxError"]
        ),
    )
)

for test_name, ok in taskgen_tests:
    print(f"{test_name}: {'PASS' if ok else 'FAIL'}")

assert all(ok for _, ok in taskgen_tests), taskgen_tests
write_status(
    "taskgen_lightweight_validation_tests",
    {
        "all_passed": all(ok for _, ok in taskgen_tests),
        "tests": [{"name": name, "passed": ok} for name, ok in taskgen_tests],
        "boundary": "This is a lightweight pre-Isaac validation test; it does not replace full simulation smoke.",
    },
)

## 0.10 论文精讲：能力轴、任务属性、子任务和难度分数

下面这节来自本目录的 [EXPLAIN_04_competency_axes_difficulty.md](./EXPLAIN_04_competency_axes_difficulty.md)，对应论文 III-B Benchmark Design：visual / procedural / relational 三条能力轴、多标签任务属性、subtask 并行事件，以及 `difficulty_score = num_subtasks + max(w)` 的难度分数。

# 精讲 4：能力轴、任务属性、子任务和难度分数，代码怎么实现

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>颜色标识</strong>：绿色表示核心结论，蓝色表示源码/输入输出路径，橙色表示边界、风险和容易误解的点。
</div>

## 先说结论

论文这一段讲的是 RoboLab-120 的“任务组织方式”：它不只是列 120 个任务，然后看总体成功率，而是给每个任务贴上一个或多个属性标签，再把这些标签归到三条能力轴：

| 能力轴 | 关注什么 | 典型属性 |
|---|---|---|
| Visual | 模型能不能识别颜色、语义、大小，并把感知属性用于推理 | `color`, `semantics`, `size` |
| Procedural | 模型能不能执行带动作导向推理的任务，例如可供性、重定向、堆叠、分类 | `affordance`, `reorientation`, `stacking`, `sorting` |
| Relational | 模型能不能理解多对象语言、计数和空间关系 | `conjunction`, `counting`, `spatial` |

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>核心结论</strong>：RoboLab 的任务不是单标签分类。一个任务可以同时是 visual + procedural，例如 <code>RedItemsInBinTask</code> 同时考颜色识别和多对象分类；难度也不是手写主观等级，而是由 <code>num_subtasks + max(skill_weight)</code> 计算出来。
</div>

这部分对应代码主线是：

```text
Task.attributes
  -> BENCHMARK_TASK_CATEGORIES / SKILL_WEIGHTS
  -> Task.subtasks
  -> count_subtasks()
  -> compute_difficulty_score()
  -> task_metadata.json / task_table.csv / README.md
  -> 结果分析按 visual/procedural/relational/simple/moderate/complex 分组
```

## 1. 能力轴：不是互斥分类，而是多标签诊断

论文说任何单个任务都很难“只评估一个属性”。代码里对应的是 `Task.attributes`：

```python
@dataclass
class BananaInBowlTask(Task):
    attributes = ["semantics"]

@dataclass
class RedItemsInBinTask(Task):
    attributes = ["color", "sorting"]

@dataclass
class RubiksCubeLeftOfBowlTask(Task):
    attributes = ["spatial"]

@dataclass
class Stack3RubiksCubeTask(Task):
    attributes = ["stacking"]
```

这些属性再由 `robolab/constants.py` 映射到能力轴：

```python
BENCHMARK_TASK_CATEGORIES = {
    "size": "visual",
    "color": "visual",
    "semantics": "visual",
    "spatial": "relational",
    "conjunction": "relational",
    "counting": "relational",
    "stacking": "procedural",
    "sorting": "procedural",
    "reorientation": "procedural",
    "affordance": "procedural",
}
```

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>源码入口</strong>：属性写在 <code>robolab/tasks/benchmark/*.py</code> 的 <code>Task.attributes</code>；能力轴映射在 <code>robolab/constants.py::BENCHMARK_TASK_CATEGORIES</code>；metadata 汇总在 <code>robolab/tasks/_utils/load_task_info.py</code>。
</div>

说人话：`attributes` 是任务的“诊断标签”。它告诉我们一次失败更可能暴露哪类能力短板：识别错颜色、空间关系理解错、还是动作序列/堆叠能力不足。

## 2. Visual 能力：从“看见对象”到“用属性推理”

Visual 不是只看图像识别，而是看策略能不能把视觉属性接到目标推理上。

| 属性 | 任务在问什么 | 失败时通常说明什么 |
|---|---|---|
| `semantics` | 这是香蕉、碗、杯子还是魔方 | 语义识别或语言 grounding 出错 |
| `color` | 哪些东西是红色/蓝色/指定颜色 | 颜色识别、颜色和对象绑定出错 |
| `size` | 大小是否满足目标 | 尺寸判断、遮挡下几何理解不足 |

例子：

```python
RedItemsInBinTask.attributes = ["color", "sorting"]
instruction = "Put all the red things in the grey bin"
success = object_in_container(object=["mug", "bowl"], container="grey_bin", logical="all")
```

这个任务不是单纯“看到红色”就完了。策略还要：

1. 在图像里找出所有红色目标。
2. 把红色属性绑定到具体对象实例，例如 `mug` 和 `bowl`。
3. 忽略非红色干扰物，例如 banana、rubiks cube。
4. 把目标对象逐个放进同一个容器。

所以它同时考 visual 和 procedural。

## 3. Procedural 能力：不只是知道目标，还要会执行动作链

Procedural 评估“动作导向推理”。它关注模型是否知道怎样利用物体可供性、怎样重定向、怎样堆叠或分类。

| 属性 | 典型目标 | 动作难点 |
|---|---|---|
| `affordance` | 使用对象的可供性，例如可放入、可支撑、可抓取部位 | 不只是识别对象，还要理解对象能怎么被操作 |
| `reorientation` | 把杯子、容器、物体转正或转到特定方向 | 抓取姿态和释放姿态都更难 |
| `stacking` | 把多个物体堆起来 | 接触、稳定性和中间状态要求高 |
| `sorting` | 按颜色/类别/目标容器分配多个对象 | 多轮 pick-place，目标集合容易漏或错 |

以 `Stack3RubiksCubeTask` 为例：

```python
attributes = ["stacking"]
subtasks = [
    Subtask(name="stack_any_2_cubes", logical="any", score=0.5),
    Subtask(name="stack_all_3_cubes", score=0.5),
]
```

第一阶段用 `logical="any"`：任意两个 cube 先堆起来都算阶段进展。第二阶段再要求三个 cube 都堆成塔。这样做的好处是：即使最终失败，也能知道策略是否已经完成了部分过程。

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>核心机制</strong>：success predicate 只判断终点，<code>subtasks</code> 记录过程。RoboLab 用 subtask 让失败有层次：是没抓到、抓错对象、放错位置，还是完成了前半段但长时序崩了。
</div>

## 4. Relational 能力：语言连接词、多对象和空间结构

Relational 评估策略是否理解场景结构和多对象语言。

| 属性 | 典型指令 | 关键难点 |
|---|---|---|
| `conjunction` | “把 X 和 Y 放到 Z” | 需要同时满足多个对象目标 |
| `counting` | “放两个/三个对象” | 需要计数和停止条件 |
| `spatial` | “把 X 放到 Y 左边/右边/前面” | 需要把语言空间关系映射到机器人/场景坐标 |

以 `RubiksCubeLeftOfBowlTask` 为例：

```python
attributes = ["spatial"]
success = object_left_of(
    object="rubiks_cube",
    reference_object="bowl",
    frame_of_reference="robot",
)
```

这里策略不仅要抓起魔方，还要判断“left of bowl”在机器人参考系下是什么方向。空间关系任务经常失败在两个地方：

- 语言理解对了，但放置方向反了。
- 方向对了，但距离/接触/释放状态没有满足成功谓词。

## 5. Subtask：顺序阶段里可以包含并行事件

论文里的例子是：

```text
Put the apple and orange on the plate, then put the banana in the bowl
```

它可以拆成：

```text
Stage 1: PickPlace(apple) 和 PickPlace(orange) 并行属于同一阶段
Stage 2: PickPlace(banana)
```

RoboLab 代码里用 `Subtask` 表达这个结构：

```python
Subtask(
    conditions={
        "apple": [grasp, hover, drop, done],
        "orange": [grasp, hover, drop, done],
    },
    logical="all",
)
```

`Subtask` 里有两个层次：

| 层次 | 代码结构 | 含义 |
|---|---|---|
| 顺序阶段 | `subtasks = [stage1, stage2, ...]` | 前后阶段组成 task horizon |
| 并行组 | `conditions={"apple": ..., "orange": ...}` | 同一阶段内多个对象都要/任选/选 K 个完成 |

`logical` 决定并行组怎么计数：

| `logical` | 计数逻辑 | 例子 |
|---|---|---|
| `all` | 所有对象组都要完成 | apple 和 orange 都要放到 plate |
| `any` | 任意一个对象组完成即可 | 三个 cube 里任意两个先形成一组堆叠 |
| `choose` | 完成 K 个对象组即可 | 从 5 个物体里选 2 个满足目标 |

## 6. 难度分数：任务长度 + 最难能力权重

论文公式可以写成：

```text
difficulty_score = num_subtasks + max(w)
```

其中 `w` 表示任务属性里最难的能力要求。当前代码里的权重在 `robolab/constants.py`：

```python
SKILL_WEIGHTS = {
    "color": 0,
    "semantics": 0,
    "size": 0,
    "conjunction": 0,
    "vague": 0,
    "spatial": 1,
    "counting": 2,
    "sorting": 2,
    "stacking": 2,
    "affordance": 2,
    "reorientation": 3,
}
DIFFICULTY_THRESHOLDS = (2, 4)
```

难度标签：

| 分数 | 标签 |
|---|---|
| `<= 2` | simple |
| `3-4` | moderate |
| `>= 5` | complex |

计算位置在 `robolab/core/task/subtask_utils.py`：

```python
def compute_difficulty_score(num_subtasks, attributes):
    non_diff = [a for a in attributes if a not in ("simple", "moderate", "complex")]
    skill_weight = max((SKILL_WEIGHTS.get(a, 0) for a in non_diff), default=0)
    score = num_subtasks + skill_weight
    ...
```

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>注意边界</strong>：难度分数是任务设计层面的粗粒度估计，不等于某个 policy 的真实失败概率。我们已经实测过 <code>Stack3RubiksCubeTask</code> 成功、<code>RedItemsInBinTask</code> 失败，说明策略表现还会受到物体姿态、抓取质量、模型偏好和 rollout 长度影响。
</div>

## 7. 从 Task 文件到 metadata 表

`load_task_info.py::extract_task_metadata()` 会从 Task 类抽取：

```text
task_name
instruction / instruction_variants
episode_s
scene
contact_objects
attributes
subtasks
num_sequential_stages
num_atomic_conditions
num_subtasks
difficulty_score
difficulty_label
```

其中难度相关字段来自：

```text
count_stages_and_conditions(subtasks)
count_subtasks(subtasks)
compute_difficulty_score(num_subtasks, attributes)
```

`generate_task_metadata.py` 再把这些结果写成：

```text
robolab/tasks/_metadata/task_metadata.json
robolab/tasks/_metadata/task_table.csv
robolab/tasks/README.md
```

这些文件就是后续按能力轴、属性和难度等级做结果分析的基础。

## 8. 用几个任务算一遍

| 任务 | attributes | subtask 计数直觉 | max(w) | score | label |
|---|---|---|---:|---:|---|
| `BananaInBowlTask` | `["semantics"]` | 1 个 pick-place | 0 | 1 | simple |
| `RubiksCubeLeftOfBowlTask` | `["spatial"]` | 1 个空间放置 | 1 | 2 | simple |
| `RedItemsInBinTask` | `["color", "sorting"]` | 2 个对象都要入箱 | 2 | 4 | moderate |
| `Stack3RubiksCubeTask` | `["stacking"]` | 任意两块先堆 + 三块全堆 | 2 | 4 | moderate |
| 复杂重定向长任务 | `["reorientation", ...]` | 多阶段动作链 | 3 | 通常 >=5 | complex |

说人话：`num_subtasks` 衡量“任务有多长”，`max(w)` 衡量“最难推理/灵巧要求是什么”。两者相加，就是 RoboLab 给用户快速扫结果用的 difficulty label。

## 9. 分析结果时该怎么看

不要只看总体 success rate。更有价值的切法是：

1. **按能力轴看**：visual / procedural / relational 哪条轴掉分最大。
2. **按细粒度属性看**：是 `color` 差，还是 `spatial` 差，还是 `stacking` 差。
3. **按难度看**：simple 是否接近饱和，moderate 是否开始掉，complex 是否几乎失败。
4. **按 subtask 看**：失败发生在 grasp、hover、drop、done 哪个阶段。
5. **按场景构成看**：物体数量、干扰物、容器大小、背景/光照是否改变成功率。

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>一句话记忆</strong>：能力轴回答“模型弱在哪类能力”，subtask 回答“失败卡在哪一步”，difficulty 回答“这个任务设计上有多难”，三者合起来才比单个 success rate 更有解释力。
</div>

In [ ]:
# ===== 精讲4：能力轴和难度分数轻量验证 =====
# 这组测试复刻 RoboLab 源码中的核心计算：
# difficulty_score = num_subtasks + max(skill_weight(attributes))
# label: simple <= 2, moderate <= 4, complex > 4

SKILL_WEIGHTS_LIGHT = {
    "color": 0,
    "semantics": 0,
    "size": 0,
    "conjunction": 0,
    "vague": 0,
    "spatial": 1,
    "counting": 2,
    "sorting": 2,
    "stacking": 2,
    "affordance": 2,
    "reorientation": 3,
}
CATEGORY_MAP_LIGHT = {
    "size": "visual",
    "color": "visual",
    "semantics": "visual",
    "spatial": "relational",
    "conjunction": "relational",
    "counting": "relational",
    "stacking": "procedural",
    "sorting": "procedural",
    "reorientation": "procedural",
    "affordance": "procedural",
}

def difficulty_score_light(num_subtasks: int, attributes: list[str]):
    non_difficulty_attrs = [a for a in attributes if a not in {"simple", "moderate", "complex"}]
    hardest_weight = max((SKILL_WEIGHTS_LIGHT.get(attr, 0) for attr in non_difficulty_attrs), default=0)
    score = num_subtasks + hardest_weight
    if score <= 2:
        label = "simple"
    elif score <= 4:
        label = "moderate"
    else:
        label = "complex"
    return score, label

def capability_axes_light(attributes: list[str]):
    return sorted({CATEGORY_MAP_LIGHT[attr] for attr in attributes if attr in CATEGORY_MAP_LIGHT})

# num_subtasks 这里按 subtask_utils.count_subtasks 的直觉复刻：
# logical=all 计全部对象组，logical=any 计 1，logical=choose 计 K。
benchmark_examples = [
    {
        "task": "BananaInBowlTask",
        "attributes": ["semantics"],
        "num_subtasks": 1,
        "expected_axes": ["visual"],
        "expected_score": 1,
        "expected_label": "simple",
    },
    {
        "task": "RubiksCubeLeftOfBowlTask",
        "attributes": ["spatial"],
        "num_subtasks": 1,
        "expected_axes": ["relational"],
        "expected_score": 2,
        "expected_label": "simple",
    },
    {
        "task": "RedItemsInBinTask",
        "attributes": ["color", "sorting"],
        "num_subtasks": 2,
        "expected_axes": ["procedural", "visual"],
        "expected_score": 4,
        "expected_label": "moderate",
    },
    {
        "task": "Stack3RubiksCubeTask",
        "attributes": ["stacking"],
        "num_subtasks": 2,
        "expected_axes": ["procedural"],
        "expected_score": 4,
        "expected_label": "moderate",
    },
    {
        "task": "LongReorientationExample",
        "attributes": ["reorientation"],
        "num_subtasks": 3,
        "expected_axes": ["procedural"],
        "expected_score": 6,
        "expected_label": "complex",
    },
]

competency_tests = []
rows = []
for example in benchmark_examples:
    score, label = difficulty_score_light(example["num_subtasks"], example["attributes"])
    axes = capability_axes_light(example["attributes"])
    ok = (
        score == example["expected_score"]
        and label == example["expected_label"]
        and axes == example["expected_axes"]
    )
    competency_tests.append((example["task"], ok))
    rows.append(
        {
            "task": example["task"],
            "attributes": example["attributes"],
            "axes": axes,
            "num_subtasks": example["num_subtasks"],
            "score": score,
            "label": label,
            "passed": ok,
        }
    )

for row in rows:
    print(
        f"{row['task']}: axes={row['axes']} "
        f"num_subtasks={row['num_subtasks']} score={row['score']} label={row['label']} "
        f"{'PASS' if row['passed'] else 'FAIL'}"
    )

assert all(ok for _, ok in competency_tests), competency_tests
write_status(
    "competency_difficulty_lightweight_tests",
    {
        "all_passed": all(ok for _, ok in competency_tests),
        "rows": rows,
        "boundary": "This validates the metadata/difficulty formula; it does not measure policy success probability.",
    },
)

## 0.11 论文精讲：SPARC 轨迹平滑度指标

下面这节来自本目录的 [EXPLAIN_05_sparc_trajectory_metric.md](./EXPLAIN_05_sparc_trajectory_metric.md)，对应论文 III-C Trajectory Metrics：SPARC 用末端执行器速度谱的弧长衡量动作平滑度，值越接近 0 越平滑，越负表示速度频谱越复杂、动作越抖。

# 精讲 5：SPARC 轨迹平滑度指标，代码怎么实现

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>颜色标识</strong>：绿色表示核心结论，蓝色表示源码/输入输出路径，橙色表示边界、风险和容易误解的点。
</div>

## 先说结论

SPARC 全称是 **Spectral Arc Length**，论文把它放在 `III-C Metrics for Evaluation -> Trajectory Metrics`。它不是任务成功率，也不是 subtask score，而是一个连续轨迹质量指标，用来衡量末端执行器运动是否平滑。

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>核心结论</strong>：RoboLab 用 SPARC 看“策略动作是不是顺”。成功率回答“有没有完成”，score 回答“完成到哪一步”，SPARC 回答“运动过程是不是平滑、有无抖动”。论文里说：越平滑的运动，SPARC 越接近 0；越抖、频率成分越复杂，SPARC 越负。
</div>

在 RoboLab 里，SPARC 的代码链路是：

```text
run_*.hdf5
  -> ee_pose/position 或 ee_pose/linear_velocity
  -> speed = ||velocity||
  -> FFT 得到归一化速度频谱
  -> 只保留 adaptive cutoff 以内的频率
  -> 计算频谱曲线弧长
  -> SPARC = - arc_length
  -> episode_metrics.json 里的 ee_sparc / joint_sparc_mean
```

## 1. SPARC 在论文里解决什么问题

RoboLab 不是只看二值 success，因为两个策略都成功时，动作质量可能差很多：

| 现象 | success 能看出来吗 | SPARC 能提供什么 |
|---|---|---|
| 一路顺滑抓取并放下 | 能看到成功 | SPARC 接近 0，表示频谱简单、运动平滑 |
| 抖动、来回试探，最后也成功 | 只能看到成功 | SPARC 更负，说明运动频谱更复杂 |
| 没完成，但中间动作很稳定 | success 只给失败 | SPARC 可单独反映运动质量 |
| 原地不动或几乎不动 | success 失败 | SPARC 应该谨慎处理，源码用 motion gate 返回 NaN |

论文原意是：success rate 很重要，但不能解释策略的运动效率、平滑度和最优性。SPARC、end-effector speed、path length 这些轨迹指标一起补上了“怎么动”的信息。

## 2. 公式说人话

论文描述的是：先取末端执行器速度曲线，再看这条速度曲线在频域里的形状。

```text
位置 p(t)
  -> 速度 v(t)
  -> 速率 s(t) = ||v(t)||
  -> Fourier magnitude spectrum V(ω)
  -> normalize V(ω)
  -> 在 cutoff 频率内计算频谱曲线弧长
  -> SPARC = -弧长
```

为什么频谱能衡量平滑？

- 平滑运动的速度曲线通常变化慢，频谱集中在低频，频谱曲线比较短。
- 抖动运动会引入高频成分，频谱曲线更曲折、更长。
- RoboLab 返回负弧长，所以弧长越大，SPARC 越负。

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>方向不要读反</strong>：RoboLab 论文文本说“smoother motions yield values closer to zero, jerkier trajectories produce more negative values”。当前 checkout 的 <code>compute_sparc()</code> docstring 前半段有一句“More negative values indicate smoother movements”容易误导；实际代码 <code>sparc = -arc_length</code>、函数后半段注释和论文一致：越接近 0 越平滑，越负越不平滑。
</div>

## 3. 源码入口

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>源码入口</strong>：SPARC 核心计算在 <code>robolab/core/metrics/trajectory_metrics.py::compute_sparc</code>；从 HDF5 读轨迹并写入 episode metrics 的入口在 <code>robolab/core/metrics/compute_metrics.py::compute_episode_metrics</code> 和 <code>compute_experiment_metrics</code>。
</div>

核心函数：

```python
def compute_sparc(
    speed,
    dt,
    padlevel=4,
    fc=10.0,
    amplitude_threshold=0.05,
    min_speed=1e-6,
):
    if len(speed) < 2 or np.max(np.abs(speed)) < min_speed:
        return float("nan")

    nfft = int(2 ** np.ceil(np.log2(N)) * padlevel)
    speed_fft = np.fft.rfft(speed, n=nfft)
    freq = np.fft.rfftfreq(nfft, d=dt)

    magnitude = np.abs(speed_fft)
    magnitude = magnitude / magnitude.max()

    above_threshold = magnitude >= amplitude_threshold
    fc_adaptive = min(freq[last_idx], fc)

    freq_mask = freq <= fc_adaptive
    d_magnitude = np.diff(magnitude_sel)
    d_freq = np.diff(freq_sel)
    arc_length = np.sum(np.sqrt((d_freq / fc_adaptive) ** 2 + d_magnitude**2))

    sparc = -arc_length
```

关键参数：

| 参数 | 默认值 | 作用 |
|---|---:|---|
| `dt` | 由 `env_cfg.json` 或默认 `1/15` | 采样时间间隔 |
| `padlevel` | `4` | FFT 零填充，提高频谱采样密度 |
| `fc` | `10.0 Hz` | 最大 cutoff 频率 |
| `amplitude_threshold` | `0.05` | adaptive cutoff 阈值，只保留有意义频段 |
| `min_speed` | `1e-6` | motion gate，几乎不动时返回 NaN |

## 4. 从 HDF5 到 `ee_sparc`

`compute_metrics.py` 负责把实验输出变成轨迹指标：

```python
data = load_demo_data(run_0.hdf5, demo_key)
ee_position = data["ee_position"]
ee_velocity = data.get("ee_linear_velocity")

if has_ee_velocity:
    metrics["ee_sparc"] = compute_ee_sparc_from_velocity(ee_velocity, dt)
else:
    metrics["ee_sparc"] = compute_ee_sparc_from_position(ee_position, dt)
```

HDF5 输入字段：

| 字段 | 用途 |
|---|---|
| `actions` | 计算 joint tracking RMSE |
| `states/articulation/robot/joint_position` | 关节状态 |
| `states/articulation/robot/joint_velocity` | joint SPARC / joint ISJ |
| `ee_pose/position` | 末端执行器位置、path length、位置差分速度 |
| `ee_pose/linear_velocity` | 如果存在，优先用于 EE SPARC 和 EE ISJ |

输出字段：

```text
ee_sparc
joint_sparc_mean
ee_isj
joint_isj
ee_path_length
joint_rmse_mean
ee_speed_max
ee_speed_mean
```

## 5. SPARC 和其他指标的区别

| 指标 | 看什么 | 什么时候有用 |
|---|---|---|
| success | 是否达成最终目标 | 最基础任务完成率 |
| normalized score | subtask 进度 | 失败但有部分进展时 |
| SPARC | 速度频谱平滑度 | 比较动作是否抖动、是否顺滑 |
| ISJ | jerk 的积分平方 | 对加速度变化更敏感 |
| path length | 末端路径长度 | 是否绕远、是否效率低 |
| speed mean/max | 动作快慢 | 评估是否过快、停滞或动作激进 |

SPARC 不应该单独解释策略好坏。一个策略可能 SPARC 很好但没完成任务，也可能成功但动作很抖。因此 RoboLab 把它和 success、score、path length、speed 一起看。

## 6. 为什么用 adaptive cutoff

频谱尾部可能有噪声。如果固定看很高频，噪声会把 arc length 拉大，导致 SPARC 过度惩罚。因此源码会：

1. 计算速度频谱幅值。
2. 归一化到最大幅值为 1。
3. 找到幅值仍高于 `amplitude_threshold=0.05` 的最高频率。
4. 用 `min(这个频率, fc)` 作为实际 cutoff。

说人话：只计算“频谱里真的有能量”的那段曲线，忽略后面几乎全是噪声的长尾。

## 7. 实际结果里怎么读

在论文表格中，SPARC 和 success/score 一起出现。读法建议：

1. 先看 success：任务是否完成。
2. 再看 score：失败时是否有部分 subtask 进展。
3. 再看 SPARC：动作是否平滑。
4. 再看 path length / speed：动作是否绕远或异常快慢。

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>一句话记忆</strong>：success 看结果，score 看进度，SPARC 看动作质量。SPARC 越接近 0 越平滑，越负说明速度频谱越复杂、动作越抖。
</div>

## 8. 和我们复现结果的关系

我们已经有完整复现输出：

```text
run_0.hdf5
episode_results.jsonl
event log
sensor / viewport mp4
```

理论上可以对 `run_0.hdf5` 再跑：

```python
from robolab.core.metrics import compute_experiment_metrics

compute_experiment_metrics(output_dir, overwrite=True)
```

然后读取 `episode_metrics.json` 中的 `ee_sparc`。这一步不改变仿真结果，只是离线分析轨迹质量。

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>注意边界</strong>：SPARC 依赖采样率 <code>dt</code>、速度是否真实记录、轨迹是否足够长、是否几乎静止。源码里对静止轨迹返回 <code>NaN</code>，后续汇总时应过滤，而不是把 NaN 当成 0 或成功平滑。
</div>

In [ ]:
# ===== 精讲5：SPARC 轻量验证 =====
# 这组测试复刻 RoboLab trajectory_metrics.py::compute_sparc 的核心逻辑：
# 1. 平滑速度曲线的 SPARC 应该更接近 0；
# 2. 抖动速度曲线包含更多高频成分，SPARC 应该更负；
# 3. 静止轨迹通过 motion gate 返回 NaN，避免把“没动”误判成“很平滑”。

import math
import numpy as np

def compute_sparc_light(
    speed,
    dt,
    padlevel=4,
    fc=10.0,
    amplitude_threshold=0.05,
    min_speed=1e-6,
):
    speed = np.asarray(speed, dtype=np.float64)
    if len(speed) < 2 or np.max(np.abs(speed)) < min_speed:
        return float("nan")

    n_samples = len(speed)
    nfft = int(2 ** np.ceil(np.log2(n_samples)) * padlevel)
    speed_fft = np.fft.rfft(speed, n=nfft)
    freq = np.fft.rfftfreq(nfft, d=dt)

    magnitude = np.abs(speed_fft)
    magnitude = magnitude / magnitude.max() if magnitude.max() > 0 else magnitude

    above_threshold = magnitude >= amplitude_threshold
    if np.any(above_threshold):
        last_idx = np.max(np.where(above_threshold)[0])
        fc_adaptive = min(freq[last_idx], fc)
    else:
        fc_adaptive = fc

    if fc_adaptive <= 0:
        return float("nan")

    freq_mask = freq <= fc_adaptive
    freq_sel = freq[freq_mask]
    magnitude_sel = magnitude[freq_mask]
    if len(freq_sel) < 2:
        return 0.0

    d_magnitude = np.diff(magnitude_sel)
    d_freq = np.diff(freq_sel)
    arc_length_elements = np.sqrt((d_freq / fc_adaptive) ** 2 + d_magnitude**2)
    return -float(np.sum(arc_length_elements))

dt = 1.0 / 100.0
t = np.arange(0.0, 2.0, dt)

# 平滑速度：单个低频正弦起伏，频谱集中。
smooth_speed = 0.5 + 0.2 * np.sin(2 * np.pi * 0.5 * t)

# 抖动速度：在同样低频趋势上叠加明显高频振荡，频谱更长更复杂。
jerky_speed = smooth_speed + 0.08 * np.sin(2 * np.pi * 8.0 * t)

# 静止速度：源码应通过 motion gate 返回 NaN。
stationary_speed = np.zeros_like(t)

smooth_sparc = compute_sparc_light(smooth_speed, dt)
jerky_sparc = compute_sparc_light(jerky_speed, dt)
stationary_sparc = compute_sparc_light(stationary_speed, dt)

sparc_tests = [
    ("smooth_is_closer_to_zero", smooth_sparc > jerky_sparc),
    ("jerky_is_more_negative", jerky_sparc < smooth_sparc),
    ("stationary_returns_nan", math.isnan(stationary_sparc)),
]

print(f"smooth_sparc={smooth_sparc:.4f}")
print(f"jerky_sparc={jerky_sparc:.4f}")
print(f"stationary_sparc={stationary_sparc}")
for name, ok in sparc_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert all(ok for _, ok in sparc_tests), sparc_tests
write_status(
    "sparc_lightweight_tests",
    {
        "all_passed": all(ok for _, ok in sparc_tests),
        "smooth_sparc": smooth_sparc,
        "jerky_sparc": jerky_sparc,
        "stationary_sparc_is_nan": math.isnan(stationary_sparc),
        "tests": [{"name": name, "passed": ok} for name, ok in sparc_tests],
        "boundary": "This validates SPARC direction and motion gate on synthetic speed profiles; real runs should compute metrics from HDF5.",
    },
)

## 0.12 论文精讲：MNPE 敏感性分析

下面这节来自本目录的 [EXPLAIN_06_mnpe_sensitivity_analysis.md](./EXPLAIN_06_mnpe_sensitivity_analysis.md)，对应论文 III-D Sensitivity Analysis 和 Appendix B：MNPE 用 rollout 后的扰动评测数据学习 `p(theta | x)`，也就是“给定成功/失败结果，哪些相机、光照、材质、物体位姿参数最可能导致这个结果”。

# 精讲 6：MNPE 敏感性分析，代码怎么实现

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>颜色标识</strong>：绿色表示核心结论，蓝色表示源码/输入输出路径，橙色表示边界、风险和容易误解的点。
</div>

## 先说结论

MNPE 全称是 **Mixed Neural Posterior Estimation**。论文把它放在 `III-D Sensitivity Analysis` 和 `Appendix B Details of MNPE Sensitivity Analysis`。它不是机器人策略，也不是新的任务成功率，而是 rollout 之后的一种诊断工具：用已经采集到的仿真评测数据，反过来估计“什么环境参数最可能对应成功/失败”。

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>核心结论</strong>：普通评测问的是“给定相机、光照、物体位置，策略能不能成功”；MNPE 问的是反向问题：“如果我只看到了成功或失败，哪些相机偏移、光照、桌面材质、物体初始位姿最可能导致这个结果”。所以它适合找策略脆弱点，而不是替代 policy rollout。
</div>

在 RoboLab 里，MNPE 的代码链路可以理解成：

```text
扰动评测脚本
  -> 运行同一任务在多组 camera/background/table/object pose 变化下的 episodes
  -> 汇总成 CSV，每行包含参数 theta 与观测 x
  -> posterior_inference.py 读取 CSV
  -> prepare_data_for_mnpe() 整理参数列和观测列
  -> 有离散参数时用 MNPE，只有连续参数时退回 NPE
  -> append_simulations(theta, x).train()
  -> build_posterior()
  -> 给定 x_o = success=1 或 success=0 采样 p(theta | x_o)
  -> 输出均值、95% credible interval、类别概率和图
```

## 1. MNPE 在论文里解决什么问题

RoboLab 已经能输出 success、score、SPARC、path length 等指标，但这些指标回答的是“结果怎么样”。敏感性分析想继续追问：

| 问题 | 普通 success 表能回答吗 | MNPE 能补什么 |
|---|---|---|
| 相机轻微移动会不会让策略崩 | 只能看到某些设置下成功/失败 | 估计成功样本对应的相机偏移分布 |
| 失败是否集中在某类光照/桌面材质 | 需要人工分组对比 | 输出每个离散类别在 posterior 里的概率 |
| 物体初始位姿离参考位置多远时更容易失败 | 需要自己画分箱曲线 | 输出连续参数的均值和 95% credible interval |
| 策略到底是对某个参数敏感，还是整体都不稳 | 单个均值难判断 | 看 posterior 是窄而集中，还是宽而接近 prior |

论文里的判断逻辑很直接：如果 `p(theta | success=1)` 紧紧集中在参考配置附近，说明策略只有在参数接近默认值时才容易成功，也就是对这个参数敏感；如果 posterior 很宽，说明成功不依赖某个很窄的参数范围，鲁棒性更强。

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>不要把 MNPE 读成因果证明</strong>：它给的是“在已有实验分布下，参数和结果的后验关联”。如果采样数据太少、任务太少、或者某些参数组合根本没跑过，MNPE 只能反映已有数据，不会自动证明真实世界因果。
</div>

## 2. 输入输出说清楚

MNPE 的输入有两类：

| 名称 | 符号 | 在代码里是什么 | 例子 |
|---|---|---|---|
| 参数 | `theta` | `param_columns` 整理出的张量 | `external_cam_initial_pose_distance`、`wrist_cam_initial_pose_distance`、`lighting_intensity`、`table_material` |
| 观测 | `x` | `obs_columns` 整理出的张量 | `success`、`success_rate`、`task_duration`、`score` |

输出是 posterior：

```text
p(theta | x_o)
```

这里的 `x_o` 是查询条件，例如：

```text
x_o = [success=1.0]  -> 什么参数最可能对应成功
x_o = [success=0.0]  -> 什么参数最可能对应失败
x_o = [success=1.0, duration=30.0] -> 什么参数最可能对应成功且耗时约 30 秒
```

RoboLab 脚本最终会输出：

| 输出 | 含义 |
|---|---|
| continuous mean | 连续参数在 posterior 里的平均值 |
| 95% credible interval | 连续参数的后验可信区间 |
| categorical probabilities | 离散类别在 posterior 样本里的占比 |
| pairplot / histogram / bar plot | 帮人看参数间的联合关系和边缘分布 |
| ESS | importance sampling 后的有效样本量，太低说明纠偏不稳 |

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>源码入口</strong>：核心脚本是 <code>analysis/sensitivity_analysis/posterior_inference.py</code>；使用说明在 <code>analysis/sensitivity_analysis/README_posterior_inference.md</code>；产生扰动数据的入口包括 <code>policies/pi0_family/run_camera_pose_variation.py</code>、<code>run_background_variation.py</code>、<code>run_table_variation.py</code>，底层扰动配置在 <code>robolab/variations/camera.py</code>、<code>backgrounds.py</code>、<code>lighting.py</code>。
</div>

## 3. 扰动数据从哪里来

MNPE 不能凭空分析，必须先有一批带扰动参数的 rollout 数据。RoboLab 里常见来源有三类：

```text
run_camera_pose_variation.py
  -> 对 external camera / wrist camera 做 reset-time pose perturbation
  -> 输出每个 task + camera variation 的 episode result

run_background_variation.py
  -> 把同一任务注册成多个 background variation env
  -> 输出不同 HDR/EXR 背景下的 episode result

run_table_variation.py
  -> 运行不同 table material
  -> episode_results.jsonl 里额外记录 table_material
```

这些脚本的共同点是：它们不是只跑一个默认环境，而是系统化扫一组环境变化。后处理时再把结果汇总成 CSV，让每一行长得像这样：

```text
policy, env_name, experiment_name,
external_cam_initial_pose, wrist_cam_initial_pose,
lighting_intensity, table_material,
success, score, task_duration
```

然后 `posterior_inference.py --use-real-data --csv-file ...` 才能把这些列变成 MNPE 的训练数据。

## 4. `prepare_data_for_mnpe()` 做了什么

这一步是整条链路最容易低估的部分。MNPE 本身只认识数值张量，真实 CSV 里却混着字符串、布尔值、类别和位姿字符串，所以脚本先做数据清洗。

源码等价逻辑：

```python
df = read_csv(csv_file)
df = filter_by_policy_task_experiment(df)

theta_columns = continuous_columns + categorical_columns
theta_df = df[theta_columns]
obs_df = df[obs_columns]

theta_df = encode_categories(theta_df)
theta_df = normalize_continuous_to_0_1(theta_df)
obs_df = convert_success_bool_to_0_1(obs_df)

theta = torch.tensor(theta_df.values)
x = torch.tensor(obs_df.values)
prior = create_uniform_prior(...)
```

重点有四个：

1. **过滤实验范围**：可以只看某个 policy、某个 task、某个 experiment，避免把不同问题混在一起。
2. **连续参数放前面，离散参数放后面**：这是脚本对 MNPE 的约束，方便区分 normalizing flow 和 categorical distribution。
3. **连续参数归一化到 `[0, 1]`**：相机距离、物体位姿距离、角度等量纲不同，先归一化能让神经网络更容易训练。
4. **离散参数编码成整数**：例如桌面材质、光照强度档位、背景类别，最终变成 `0, 1, 2...`。

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>怎么记</strong>：<code>theta</code> 是“我动了哪些旋钮”，<code>x</code> 是“策略最后表现如何”。MNPE 学的是“看到表现以后，哪些旋钮位置最可能”。
</div>

## 5. 位姿参数怎么变成一个数

论文里把 camera pose / object pose 当成 SE(3) 位姿。位姿本来是 7 维：位置 `x,y,z` 加四元数 `qw,qx,qy,qz`。为了做敏感性分析，代码会把位姿转成“离参考位姿多远”的连续参数。

说人话：

```text
pose_distance =
  平移距离
  + beta * 旋转 geodesic distance
```

其中：

- 平移距离看相机或物体位置偏了多少米。
- 旋转距离看四元数方向偏了多少弧度。
- `beta` 控制“旋转偏差”在总距离里占多大权重。

README 里给了两个典型相机列：

```text
external_cam_initial_pose -> external_cam_initial_pose_distance
wrist_cam_initial_pose    -> wrist_cam_initial_pose_distance
```

如果只分析物体初始位置，脚本也支持把类似 `banana_initial_pose`、`bowl_initial_pose` 的字符串解析成到原点的欧氏距离。

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>距离是压缩表示</strong>：把 7DoF 位姿压成一个距离，便于训练和可视化，但会丢掉“往左偏”和“往右偏”的方向信息。如果要分析方向性，应该把位姿拆成多个参数，而不是只用距离。
</div>

## 6. 训练和推理的核心代码

RoboLab 脚本的核心训练逻辑可以压缩成下面几步：

```python
if num_categorical > 0:
    inference = MNPE(device=device)
else:
    inference = NPE(device=device)

inference.append_simulations(theta, x)
density_estimator = inference.train(max_num_epochs=max_epochs)
posterior = inference.build_posterior(density_estimator, prior=target_prior)
samples = posterior.sample((num_samples,), x=x_o)
```

这里有一个容易误解的点：

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>论文叫 MNPE，但代码不总是调用 MNPE</strong>：当参数里有桌面材质、光照档位、背景类别这类离散列时，脚本用 <code>sbi.inference.MNPE</code>。如果只有相机距离、物体距离这类连续参数，脚本会用 <code>NPE</code>。这不是绕开论文，而是同一类 posterior estimation 在纯连续场景下的合适实现。
</div>

训练目标是让神经网络学习 `p(theta | x)`。训练完成后，我们不再问“这个 theta 会不会成功”，而是给一个观测 `x_o`：

```text
success = 1.0
```

再从 posterior 里采样很多组 `theta`。这些样本就是“在模型看来，最能解释成功的环境参数分布”。

## 7. Importance sampling 为什么存在

论文强调使用 uniform prior，是为了让分析不要先验偏向某个参数区域。但真实评测数据经常不是均匀采样的：比如 70% 都在默认光照，只有少量极暗光照；或者某个背景跑得很多，另一个背景跑得少。

代码里的 correction 思路是：

```text
目标：想要 uniform prior 下的 posterior
现实：训练数据来自不均匀 empirical proposal
做法：给 posterior samples 乘 importance weight
权重：target prior probability / empirical proposal probability
```

输出里的 ESS，也就是 Effective Sample Size，用来提醒这个纠偏是否可靠：

| ESS 状态 | 解读 |
|---|---|
| 高 | 数据覆盖和目标 prior 比较匹配，重加权较稳 |
| 中 | 可以看趋势，但别过度解释细节 |
| 很低 | 说明你跑的数据太偏，应该补采样，而不是相信漂亮图 |

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>低 ESS 是数据问题，不是画图问题</strong>：如果某些参数区域没有 rollout，后验纠偏只能靠很少的样本硬撑，结论会很脆。
</div>

## 8. 结果怎么读

假设我们查询：

```text
obs_columns = ["success"]
obs_values = [1.0]
```

看到下面几种结果时应该这样理解：

| posterior 形状 | 说人话解释 |
|---|---|
| `external_cam_distance` 均值很小，95% CI 很窄 | 成功大多发生在相机接近默认位置时，策略对外部相机偏移敏感 |
| `wrist_cam_distance` CI 很宽 | 腕部相机偏移对成功影响没那么集中，可能更鲁棒 |
| `table_material=wood` 概率远高于其他材质 | 成功更常和某个桌面材质关联，可能有纹理/反光/对比度影响 |
| success=0 的 posterior 集中在极暗光照 | 失败更可能来自照明不足 |
| success=1 和 success=0 的 posterior 差不多 | 这个参数可能不是主要失败因素，或者数据量不足 |

## 9. 和本次 4090 复现的关系

我们目前已经完成：

- 一条 `Pi05 + BananaInBowlTask` 完整成功闭环。
- 三个复杂任务抽样，其中 1 个成功、2 个失败。
- 视频、HDF5、event log、episode_results.jsonl 已同步。

这还不够支撑正式 MNPE 敏感性分析。原因是 MNPE 需要同一任务/同一策略在很多扰动参数组合下的结果，至少要形成可训练的 `theta, x` 数据表。当前几条复现更适合作为“rollout 链路通了”的证据，而不是 posterior inference 的统计证据。

下一步如果要真的跑 MNPE，正确顺序是：

```text
1. 选一个任务，例如 BananaInBowlTask
2. 固定一个策略，例如 pi05_droid_jointpos
3. 跑 camera pose variation / lighting variation / table variation
4. 汇总每个 episode 的参数列和 success/score/duration
5. 用 posterior_inference.py 查询 success=1 和 success=0
6. 比较 posterior 是否集中在某些参数区域
```

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>最小可行实验</strong>：先不要上 RoboLab-120。先用一个任务、一个策略、两类扰动，每类 20-50 个 episode，确认 CSV 和 posterior 图能跑通，再扩展到多任务。
</div>

## 10. 额外示例：一个轻量 MNPE 思维实验

为了让 notebook 不依赖 `sbi` 也能验证 MNPE 的直觉，我们可以用一个轻量贝叶斯重加权测试：

```text
先均匀采样 camera_offset in [0, 1]
再设定 success probability 随 camera_offset 增大而下降
然后只观察 success=1
最后看 posterior 里的 camera_offset 是否比 prior 更靠近 0
```

如果测试结果显示：

```text
prior camera mean      ≈ 0.50
posterior success mean < 0.50
posterior success CI   更偏向小偏移
good_lighting posterior probability > prior probability
```

这说明“成功”确实把参数分布往更友好的区域拉了过去，也就是 MNPE 在论文里要表达的核心直觉。

如果我们把 success probability 设成和 camera_offset 无关，posterior 应该接近 prior。这对应“策略对这个参数不敏感，或者数据没有显示出这个参数的影响”。

## 11. 本节最重要的边界

- MNPE 是离线分析，不在每一步机器人控制循环里运行。
- MNPE 依赖扰动评测数据，不能用 3-5 条 episode 就下结论。
- posterior 集中说明“成功样本关联到某个参数区域”，不等价于严格因果。
- 论文里的 MNPE 框架支持混合连续/离散参数；RoboLab 代码在纯连续参数时会自动退回 NPE。
- 如果实验数据的采样分布很偏，必须关注 importance sampling 和 ESS。
- 对 4090 来说，MNPE 本身通常不是显存瓶颈；真正耗时耗显存的是前面的 Isaac Sim + VLA rollout 数据采集。

## 12. 一句话总结

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>核心结论</strong>：RoboLab 的 MNPE 敏感性分析，是把“跑完很多带扰动的 episode”之后得到的表格数据，变成 <code>p(环境参数 | 成功或失败)</code>。它的价值是定位策略对相机、光照、材质、物体位姿等因素的脆弱性，而不是再给一个简单平均分。
</div>

In [ ]:
# ===== 精讲6：MNPE 轻量验证 =====
# 这个测试不依赖 sbi，也不假装替代真正的 MNPE。
# 它只验证论文里最核心的 posterior 直觉：
#   prior: camera_offset 在 [0, 1] 上均匀采样；
#   likelihood: camera_offset 越大，success probability 越低；
#   posterior p(theta | success=1) 应该更偏向小 camera_offset。

import numpy as np

rng = np.random.default_rng(260409860)
n = 20000

# theta 的两个维度：一个连续扰动，一个离散光照类别。
camera_offset = rng.uniform(0.0, 1.0, size=n)
lighting_category = rng.integers(0, 3, size=n)  # 0=dim, 1=normal, 2=bright
lighting_bonus = np.array([-1.0, 1.2, 0.0])[lighting_category]

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# 合成一个“策略对相机偏移敏感”的世界：
# 相机越偏离参考位姿，成功概率越低；正常光照成功概率更高。
success_likelihood = sigmoid(2.0 - 5.0 * camera_offset + lighting_bonus)
posterior_weights = success_likelihood / success_likelihood.sum()

def weighted_quantile(values, quantiles, weights):
    order = np.argsort(values)
    sorted_values = values[order]
    sorted_weights = weights[order]
    cdf = np.cumsum(sorted_weights)
    cdf = cdf / cdf[-1]
    return np.interp(quantiles, cdf, sorted_values)

prior_camera_mean = float(camera_offset.mean())
posterior_camera_mean = float(np.sum(camera_offset * posterior_weights))
uniform_weights = np.full(n, 1.0 / n)
prior_camera_ci = weighted_quantile(
    camera_offset,
    np.array([0.025, 0.975]),
    uniform_weights,
).tolist()
posterior_camera_ci = weighted_quantile(
    camera_offset,
    np.array([0.025, 0.975]),
    posterior_weights,
).tolist()

prior_lighting_probs = np.bincount(lighting_category, minlength=3) / n
posterior_lighting_probs = np.array([
    posterior_weights[lighting_category == i].sum()
    for i in range(3)
])

# 对照组：如果 success 和 camera_offset 无关，posterior 应该基本等于 prior。
flat_likelihood = np.full(n, 0.45)
flat_weights = flat_likelihood / flat_likelihood.sum()
flat_camera_mean = float(np.sum(camera_offset * flat_weights))
flat_camera_ci = weighted_quantile(
    camera_offset,
    np.array([0.025, 0.975]),
    flat_weights,
).tolist()

mnpe_tests = [
    ("success_posterior_moves_camera_toward_reference", posterior_camera_mean < prior_camera_mean - 0.12),
    ("success_posterior_has_lower_upper_ci", posterior_camera_ci[1] < prior_camera_ci[1] - 0.05),
    ("normal_lighting_more_likely_given_success", posterior_lighting_probs[1] > prior_lighting_probs[1] + 0.12),
    ("uninformative_likelihood_keeps_prior_mean", abs(flat_camera_mean - prior_camera_mean) < 1e-12),
    ("uninformative_likelihood_keeps_prior_ci", np.max(np.abs(np.array(flat_camera_ci) - np.array(prior_camera_ci))) < 1e-12),
]

print(f"prior_camera_mean={prior_camera_mean:.3f}")
print(f"posterior_camera_mean_given_success={posterior_camera_mean:.3f}")
print(f"prior_camera_95ci={[round(x, 3) for x in prior_camera_ci]}")
print(f"posterior_camera_95ci_given_success={[round(x, 3) for x in posterior_camera_ci]}")
print(f"prior_lighting_probs={[round(float(x), 3) for x in prior_lighting_probs]}")
print(f"posterior_lighting_probs_given_success={[round(float(x), 3) for x in posterior_lighting_probs]}")
for name, ok in mnpe_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert all(ok for _, ok in mnpe_tests), mnpe_tests
write_status(
    "mnpe_lightweight_tests",
    {
        "all_passed": all(ok for _, ok in mnpe_tests),
        "prior_camera_mean": prior_camera_mean,
        "posterior_camera_mean_given_success": posterior_camera_mean,
        "prior_camera_95ci": prior_camera_ci,
        "posterior_camera_95ci_given_success": posterior_camera_ci,
        "prior_lighting_probs": prior_lighting_probs.tolist(),
        "posterior_lighting_probs_given_success": posterior_lighting_probs.tolist(),
        "flat_likelihood_camera_mean": flat_camera_mean,
        "flat_likelihood_camera_95ci": flat_camera_ci,
        "tests": [{"name": name, "passed": bool(ok)} for name, ok in mnpe_tests],
        "boundary": "This validates posterior-conditioning intuition on synthetic data; real RoboLab MNPE requires variation rollout CSV and sbi.",
    },
)

## 0.13 论文精讲：Baseline 场景生成方法

下面这节来自本目录的 [EXPLAIN_07_baseline_method.md](./EXPLAIN_07_baseline_method.md)，对应论文 Appendix C-C Baseline Method：baseline 是 scene generation 对照方法，不是策略 baseline。它用 LLM 选物体和网格布局，再把对象按 cell 顺序摆放并 jitter，一次 pass 后做物理 settle；它能保证基本分散，但缺少 `place-in`、`place-on`、`cluster-around` 和失败反馈修复。

# 精讲 7：Baseline 方法，论文里的对照场景生成怎么实现

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>颜色标识</strong>：绿色表示核心结论，蓝色表示源码/输入输出路径，橙色表示边界、风险和容易误解的点。
</div>

## 先说结论

论文 Appendix C-C 的 **Baseline Method** 指的是“场景生成 baseline”，不是 Pi05、GR00T、PaliGemma 这些策略 baseline。它是拿来和 RoboLab 自己的层级式 scene generation 方法对比的。

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>核心结论</strong>：baseline 是一种标准 domain randomization 风格的单次布局方法：LLM 先选一批对象，再给出一个桌面网格；算法把桌面切成 <code>rows x columns</code> 个 cell，把对象按顺序塞进 cell，并在 cell 内随机 jitter。它能保证基本间隔，但不会生成“放进容器”“堆叠在支撑物上”“围绕 anchor 聚类”这些有语义结构的真实场景。
</div>

一句话记住：

```text
baseline = 选物体 + 网格分格 + cell 内随机抖动 + 固定安全高度 + 一次物理 settle
RoboLab ours = 语义谓词 + 几何约束求解 + 物理约束求解 + 失败反馈修复
```

## 1. Baseline 在论文里解决什么问题

论文做 scene generation 时，需要证明自己的层级式方法不是“看起来复杂但没必要”。所以它设计了一个强但简单的对照组：

| 方法 | 作用 |
|---|---|
| Baseline | 代表常见 domain randomization / grid random placement：对象能分散摆在桌面上 |
| Ours | 代表 RoboLab 的层级式语义场景生成：对象有容器、支撑、聚类、相对位置和物理检查 |

这个 baseline 不是为了跑 policy 评测，而是为了回答：

```text
如果只用网格随机摆放，生成场景质量能不能接近 RoboLab 的层级式场景？
```

论文结论是不能。baseline 可以摆开物体，但在视觉真实感、功能性、布局正确性、完整性和 GPT preference 上都明显弱于层级式方法。

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>不要把 baseline 误读成策略 baseline</strong>：这里的 baseline 是 scene generation baseline。策略对比表里那些模型是 policy baseline/被评测策略；Appendix C-C 的 baseline 是场景生成方法对照。
</div>

## 2. Baseline 的输入输出

Baseline 输入：

| 输入 | 含义 |
|---|---|
| `theme` | 场景主题，例如 kitchen counter、bathroom counter、classroom supplies |
| `object catalog` | 可选对象和尺寸 |
| `target object count` | 希望场景里放多少物体 |
| `rows, columns` | LLM 建议的桌面网格尺寸 |
| `table bounds` | 桌面范围，例如 X/Y 边界 |
| `jitter range` | cell 内随机偏移范围 |

Baseline 输出：

| 输出 | 含义 |
|---|---|
| object list | LLM 选出来的对象 |
| `(x, y, z, yaw)` | 每个物体在桌面上的位置、高度和朝向 |
| settled scene | 物理仿真 settle 后的场景 |
| rendered images | 用于 VQA/GPT preference/quality 评分 |

代码层面可以抽象成：

```python
objects = llm_select_objects(theme, catalog, target_count)
rows, cols = llm_suggest_grid(objects, table_size)
cells = split_table_into_grid(table_bounds, rows, cols)

for object, cell in zip(objects, cells):
    x = cell.center_x + uniform(-jitter_x, jitter_x)
    y = cell.center_y + uniform(-jitter_y, jitter_y)
    z = safe_height(object)
    yaw = random_yaw()
    place_on_table(object, x, y, z, yaw)

simulate_under_gravity()
```

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>源码对照</strong>：当前 checkout 里没有看到单独的 <code>baseline.py</code> 主入口；可对照阅读的是主方法实现：<code>robolab/scene_gen/llm_scene_gen/predicates.py</code>、<code>spatial_solver.py</code>、<code>physical_solver.py</code>、<code>feedback_system.py</code>。Baseline 更像论文实验里的对照算法，而不是 RoboLab 用户日常运行的主路径。
</div>

## 3. Baseline 为什么合理

它不是一个“故意很差”的 strawman。它合理的地方在于：

1. **简单稳定**：网格保证对象基本分开，不容易初始就全撞在一起。
2. **实现成本低**：不需要复杂谓词库，也不需要多轮 LLM 修复。
3. **符合常见随机化习惯**：很多仿真 benchmark 会随机选物体、随机位置、随机 yaw。
4. **可批量生成**：一次 pass 就能产出很多场景。

所以它可以代表一类常见做法：

```text
只要让物体在桌面上随机且不重叠，就可以做大规模评测。
```

RoboLab 论文要反驳的正是这个假设：真实任务评测需要的不只是“物体不重叠”，还需要语义和功能结构。

## 4. Baseline 的核心局限

论文里 baseline 的关键限制是：对象只是被放到安全高度上，不做复杂堆叠或容器关系。

| 局限 | 具体表现 | 对机器人评测的影响 |
|---|---|---|
| 没有 containment | 水果不会自然在碗里，工具不会在托盘里 | 很多真实目标状态和初始状态不自然 |
| 没有 stacking/support | 物体不会在盘子、托盘、架子、其他物体上 | 过程化任务和堆叠任务缺少结构 |
| 没有 anchor/cluster | 物体均匀铺开，不像真实桌面上的“物品堆” | 视觉和空间关系不自然 |
| 没有语义修复 | 如果 LLM 选了不合适的大物体组合，只能物理 settle，不能回去改计划 | 高密度场景更容易失败或不完整 |
| 没有任务导向结构 | 场景可能能看，但不一定适合生成任务 | 后续 TaskGen 质量受限 |

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>物理 settle 不是万能修复</strong>：baseline 最后也会跑物理仿真，让小穿插或轻微不稳定通过重力 settle。但它没有能力修复“为什么一个碗里没有东西”“为什么所有物体网格排列得像展板”这种语义和结构问题。
</div>

## 5. RoboLab 主方法比 baseline 多了什么

论文 Appendix C 描述的主方法可以看成四层：

```text
Stage I: LLM 生成结构化 scene plan
  -> 选择对象
  -> 写出 place-in / place-on / cluster-around / place-on-base 等谓词

Stage II: 几何约束求解
  -> 把谓词变成 2D/3D 坐标
  -> 检查碰撞、边界、相对位置

Stage III: 物理仿真检查
  -> 在 Isaac/物理环境下 settle
  -> 检查掉落、位移、穿插

Stage IV: feedback repair
  -> 如果失败，把错误反馈给 LLM
  -> 让 LLM 改对象、改关系、改密度、改布局
```

这和 baseline 的差别不是“随机数用得更高级”，而是表达空间不同：

| 维度 | Baseline | RoboLab 主方法 |
|---|---|---|
| 布局单位 | 网格 cell | 语义谓词 |
| 空间关系 | 基本无，只靠 cell 顺序 | left/right/front/back/align/cluster 等 |
| 物理关系 | 全部 on table | place-on、place-in、place-anywhere |
| 失败处理 | 一次 pass | solver/physics feedback 后可修复 |
| 场景真实感 | 容易网格化 | 更像真实桌面结构 |
| 任务可用性 | 初始状态较浅 | 更适合生成多步骤/关系/过程化任务 |

## 6. 源码对应：主方法怎么落地

虽然 baseline 自身没有在当前 checkout 里作为主脚本暴露，但主方法的核心模块很清楚。

### `predicates.py`

这里定义 scene generation 的谓词类型：

```text
Spatial predicates:
  left-of / right-of / front-of / back-of
  place-on-base
  align-left / align-right / align-center
  facing-left / random-rot

Physical predicates:
  place-on
  place-in
  place-anywhere
```

说人话：这相当于给 LLM 一个“场景语法”。LLM 不直接输出乱七八糟的坐标，而是输出“苹果放进碗里”“勺子放在盘子上”“杯子在碗左边”这种可求解结构。

### `spatial_solver.py`

`SpatialSolver.solve()` 做 2D 空间求解：

```text
1. 根据场景复杂度调整 collision margin 和 iteration
2. 先处理 place-on-base
3. 再处理 relative position predicates
4. 再处理 orientation predicates
5. 检查 unsolved objects
6. 做 collision optimization
7. 如果失败，可尝试 relaxation
```

这比 baseline 的 grid 多了两个关键能力：

- 可以表达相对关系，而不是固定格子。
- 可以在碰撞失败时迭代优化，而不是一次摆完。

### `physical_solver.py`

`PhysicalSolver.solve()` 处理 3D/物理关系：

```text
place-on:
  把物体放在 support 上，检查 footprint 是否能放下

place-in:
  把多个物体按层塞到 container mouth 上方
  再让物理 settle 把它们落入容器

place-anywhere:
  在可行区域找一个不冲突的位置
```

这正是 baseline 缺少的能力：baseline 所有物体都只是桌面上的独立 item，主方法可以生成“容器里有东西”“支撑物上有东西”“密集物品堆”的场景。

### `feedback_system.py`

失败后会生成反馈，例如：

```text
SOLVER FAILURE:
  collisions detected
  objects out of bounds
  increase distance or reposition

PHYSICS VALIDATION FAILURE:
  object moved too much after simulation
  increase support_ratio
  place unstable objects inside containers
```

论文图 17 里的修复提示，就是把 solver/physics 的错误消息打包回 LLM，让它下一轮改场景计划。

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>怎么记</strong>：baseline 是“摆坐标”；RoboLab 主方法是“先写语义关系，再让 solver 把关系翻译成坐标，失败了还能反馈修复”。
</div>

## 7. 论文实验怎么看

论文 Appendix C-D 用生成场景指标比较 baseline 和 ours。核心指标包括：

| 指标 | 含义 |
|---|---|
| VQA score | 用视觉问答检查生成图像是否符合语义问题 |
| GPT preference | 给 GPT 看两张图，让它偏好哪种方法 |
| Real. | 视觉真实感 |
| Func. | 功能性，比如容器/支撑关系是否合理 |
| Lay. | 布局正确性 |
| Qual. | 总体质量 |
| Compl. | 场景完整性 |

论文报告的总体趋势是：RoboLab 主方法在所有这些维度上都超过 baseline，尤其是视觉真实感、语义功能性和场景完整性。表 VI 里 baseline 的 GPT preference 是 `18%`，ours 是 `82%`；这说明评估器更常偏好层级式方法生成的场景。

<div style="border:1px solid #fed7aa; border-left:6px solid #d97706; background:#fff7ed; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>指标不是机器人成功率</strong>：这些是 scene generation 质量指标，不是 Pi05 在任务里的 success rate。它们评价“生成场景像不像、合不合理、完不完整”，不是评价策略动作做得好不好。
</div>

## 8. 为什么 baseline 对 RoboLab-120 不够

RoboLab-120 的任务不是只看“桌上有哪些物体”，还看：

- 视觉属性：颜色、语义、大小。
- 关系推理：left/right、counting、and/or。
- 过程推理：affordance、reorientation、stacking。
- 多步骤：先 A 再 B。

如果场景只是网格化随机摆放，很多任务很难自然成立：

| 任务需求 | baseline 问题 |
|---|---|
| “把碗里的苹果拿出来” | baseline 初始状态没有 place-in |
| “把杯子放到托盘上” | baseline 不知道 support surface |
| “把左边的红物体放进箱子” | cell 位置可能有 left/right，但不是语义规划出来的场景关系 |
| “把三个 cube 叠起来” | baseline 不生成稳定堆叠结构 |
| “整理一堆杂乱柜台物品” | grid 排列不像真实杂乱场景 |

所以 baseline 可以作为对照，但不能作为高质量任务生成的主路径。

## 9. 本次 notebook 的轻量测试要验证什么

我们不需要在本机重跑 Isaac 渲染，也能验证 baseline 的结构局限。轻量测试做三件事：

1. 用 `grid_baseline_layout()` 复刻论文 baseline：对象按 grid cell 顺序摆放，cell 内 jitter。
2. 用一个 `hierarchical_layout_example()` 表示主方法能表达的结构：`place-in`、`place-on`、`cluster-around`。
3. 检查 baseline 能做到基本分离，但不能表达容器/支撑/聚类关系。

预期结论：

```text
baseline_min_distance_ok = True
baseline_semantic_relations_supported = False
hierarchical_relations_supported = True
baseline_top_level_slots > hierarchical_top_level_slots
```

这对应论文里的差异：baseline 不是不能摆东西，而是只能摆“平面散点”；RoboLab 主方法能生成“有功能结构的场景”。

## 10. 一句话总结

<div style="border:1px solid #bbf7d0; border-left:6px solid #16a34a; background:#f0fdf4; padding:10px 12px; border-radius:6px; margin:12px 0;">
<strong>核心结论</strong>：Appendix C-C 的 baseline 是“网格随机摆放 + 一次物理 settle”的场景生成对照。它能保证物体基本分开，却不能生成容器、支撑、堆叠、聚类和可修复的语义结构；RoboLab 主方法真正多出来的是谓词表达、几何/物理求解和失败反馈闭环。
</div>

In [ ]:
# ===== 精讲7：Baseline scene generation 轻量验证 =====
# 这个测试复刻论文 Appendix C-C 的核心差异：
# - baseline: grid cell + jitter + all objects on table
# - hierarchical/ours: explicit semantic relations such as place-in/place-on/cluster-around

import math
import numpy as np

def grid_baseline_layout(objects, rows, cols, table_bounds=(0.25, 0.85, -0.40, 0.40), jitter_ratio=0.18, seed=7):
    # 单次 grid+jitter baseline：每个 object 占一个桌面 cell，不表达容器/支撑关系。
    if len(objects) > rows * cols:
        raise ValueError("grid has fewer cells than objects")

    rng = np.random.default_rng(seed)
    x_min, x_max, y_min, y_max = table_bounds
    cell_w = (x_max - x_min) / cols
    cell_h = (y_max - y_min) / rows

    placements = []
    for index, obj in enumerate(objects):
        row = index // cols
        col = index % cols
        cx = x_min + (col + 0.5) * cell_w
        cy = y_min + (row + 0.5) * cell_h
        x = cx + rng.uniform(-jitter_ratio * cell_w, jitter_ratio * cell_w)
        y = cy + rng.uniform(-jitter_ratio * cell_h, jitter_ratio * cell_h)
        placements.append(
            {
                "object": obj,
                "type": "on-table",
                "x": float(x),
                "y": float(y),
                "z_policy": "safe_height",
                "yaw": float(rng.uniform(-math.pi, math.pi)),
            }
        )
    return placements

def hierarchical_layout_example():
    # 模拟 RoboLab 主方法能表达的语义谓词结构。
    return [
        {"object": "bowl", "type": "anchor", "x": 0.52, "y": 0.02},
        {"object": "plate", "type": "anchor", "x": 0.70, "y": -0.16},
        {"object": "banana", "type": "place-in", "container": "bowl"},
        {"object": "apple", "type": "place-in", "container": "bowl"},
        {"object": "spoon", "type": "place-on", "support": "plate"},
        {"object": "mug", "type": "cluster-around", "anchor": "bowl", "radius": 0.13},
    ]

objects = ["bowl", "plate", "banana", "apple", "spoon", "mug"]
baseline = grid_baseline_layout(objects, rows=2, cols=3)
hierarchical = hierarchical_layout_example()

def min_pair_distance(placements):
    coords = np.array([[p["x"], p["y"]] for p in placements], dtype=float)
    best = float("inf")
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            best = min(best, float(np.linalg.norm(coords[i] - coords[j])))
    return best

baseline_min_distance = min_pair_distance(baseline)
baseline_relation_types = {p["type"] for p in baseline}
hierarchical_relation_types = {p["type"] for p in hierarchical}

baseline_top_level_slots = sum(1 for p in baseline if p["type"] == "on-table")
hierarchical_top_level_slots = sum(1 for p in hierarchical if p["type"] in {"anchor", "cluster-around"})

baseline_tests = [
    ("baseline_keeps_basic_separation", baseline_min_distance > 0.12),
    ("baseline_all_objects_are_flat_table_items", baseline_relation_types == {"on-table"}),
    ("hierarchical_expresses_containment", "place-in" in hierarchical_relation_types),
    ("hierarchical_expresses_support", "place-on" in hierarchical_relation_types),
    ("hierarchical_expresses_cluster", "cluster-around" in hierarchical_relation_types),
    ("hierarchical_uses_fewer_top_level_table_slots", hierarchical_top_level_slots < baseline_top_level_slots),
]

print(f"baseline_min_distance={baseline_min_distance:.3f} m")
print(f"baseline_relation_types={sorted(baseline_relation_types)}")
print(f"hierarchical_relation_types={sorted(hierarchical_relation_types)}")
print(f"baseline_top_level_slots={baseline_top_level_slots}")
print(f"hierarchical_top_level_slots={hierarchical_top_level_slots}")
for name, ok in baseline_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert all(ok for _, ok in baseline_tests), baseline_tests
write_status(
    "baseline_method_lightweight_tests",
    {
        "all_passed": all(ok for _, ok in baseline_tests),
        "baseline_min_distance": baseline_min_distance,
        "baseline_relation_types": sorted(baseline_relation_types),
        "hierarchical_relation_types": sorted(hierarchical_relation_types),
        "baseline_top_level_slots": baseline_top_level_slots,
        "hierarchical_top_level_slots": hierarchical_top_level_slots,
        "baseline_example": baseline,
        "hierarchical_example": hierarchical,
        "tests": [{"name": name, "passed": bool(ok)} for name, ok in baseline_tests],
        "boundary": "This validates the structural difference between grid+jitter baseline and predicate-based scene generation; it does not render or physics-settle a USD scene.",
    },
)

## 0.14 论文精讲：论文实验总览与 Algorithm 1

下面这节来自本目录的 [EXPLAIN_08_paper_experiments.md](./EXPLAIN_08_paper_experiments.md)，对应论文中的实验体系：RoboLab-120 策略评测、细粒度能力分析、扰动敏感性、真实世界相关性、场景生成质量、任务生成质量，以及你截图里的 Algorithm 1 Spatial Constraint Solver。本次已加深为“实验地图 + 证据链”：补充 success/score/event/trajectory 的职责差异、主表能力画像读法、场景生成质量表和 task generation judge 六维、以及 4090 上可承受的小论文矩阵。

# 精讲 8：论文实验总览与 Algorithm 1 空间约束求解器

> **【绿色标识｜核心结论】** 论文里的“实验”不是单一跑分，而是一组互相支撑的验证：策略评测、细粒度分析、扰动敏感性、真实世界相关性、场景生成质量、任务生成质量。  
> **【蓝色标识｜源码路径】** 这些实验分别落在 `policies/pi0_family/`、`robolab/eval/`、`robolab/core/logging/`、`analysis/`、`robolab/scene_gen/`、`skills/robolab-taskgen/` 等模块。  
> **【橙色标识｜容易误解】** Algorithm 1 不是策略评测算法，也不是完整 3D 物理放置算法；它主要解决“桌面基础对象的 2D 布局约束”，后续的 `place-in`、`place-on` 等 3D 接触关系还要交给物理求解/settle。

## 先说结论

论文实验可以按 6 条主线理解：

| 实验主线 | 它回答的问题 | 主要输入 | 主要输出 | 代码/文件入口 |
|---|---|---|---|---|
| RoboLab-120 策略评测 | VLA/机器人策略在 120 个任务上到底能不能完成 | 任务集、策略服务、机器人环境 | success、score、subtask、episode log、视频、HDF5 | `policies/pi0_family/run.py`、`robolab/eval/runner.py` |
| 细粒度能力分析 | 失败到底来自视觉、空间关系、长时序还是程序操作 | episode 结果 + task metadata | 按属性/难度/对象数/子任务数分组的成功率 | `analysis/read_results.py`、`robolab/core/logging/results.py` |
| 扰动敏感性实验 | 光照、背景、桌面纹理、相机、物体初始位置变化会不会让策略崩 | variation rollout CSV / episode 结果 | 各扰动下成功率、MNPE 后验敏感性 | `policies/pi0_family/run_*_variation.py`、`analysis/sensitivity_analysis/` |
| 真实世界相关性验证 | RoboLab 仿真评测排名是否和真实机器人 arena 排名一致 | RoboLab score + RoboArena/Elo 类真实评测结果 | Spearman/Pearson 相关性 | 论文实验分析脚本口径，结果不等于单任务真实部署 |
| 场景生成质量实验 | 主方法生成的场景是否比 baseline 更真实、更完整、更可评测 | LLM 生成的场景、渲染图、VQA/GPT judge | VQA score、GPT preference、Real/Func/Lay/Compl/Qual | `robolab/scene_gen/llm_scene_gen/` |
| 任务生成质量实验 | 自动生成的任务代码和自然语言目标是否对齐、清晰、可行 | 任务代码、对象目录、谓词库、LLM judge prompt | alignment、clarity、feasibility、match、coverage | `skills/robolab-taskgen/`、任务验证脚本 |

一句话概括：

> RoboLab 不是只问“某个模型成功率多少”，而是把“场景是否可信、任务是否合理、策略是否泛化、扰动是否敏感、仿真排名是否有现实意义”拆开分别验证。

## 深挖 0：论文实验真正想证明的不是一个分数，而是一条证据链

这篇论文的实验体系可以按“从可用场景到可信结论”的证据链理解：

```text
场景生成质量
  -> 任务生成质量
  -> 策略 rollout 证据
  -> success / score / trajectory / event 统计
  -> 按能力轴和难度分组
  -> 扰动敏感性
  -> 真实世界相关性
```

每一层回答一个不同的问题：

| 层 | 核心问题 | 如果这一层不过关，会污染什么 |
|---|---|---|
| 场景生成 | 场景是否真实、合理、物理可用 | 后续策略失败可能只是场景垃圾 |
| 任务生成 | 指令和成功条件是否一致、可执行 | success/score 失去语义可信度 |
| rollout 记录 | 观测、动作、事件、视频是否可追溯 | 无法复查模型到底怎么失败 |
| 指标统计 | 成功率、score、轨迹质量是否区分清楚 | 平均成功率会掩盖部分完成和坏动作 |
| 分组分析 | 能力轴、难度、任务长度是否拆开看 | 无法定位视觉/关系/程序哪类能力弱 |
| 扰动敏感性 | 光照、背景、相机、位姿变化是否影响策略 | 无法判断模型是否只是碰巧适配某个设置 |
| 真实相关性 | 仿真排名是否和真实机器人评测同向 | 无法说明仿真 benchmark 对真实评估有用 |

**【绿色标识｜核心结论】**

RoboLab 的实验设计不是“跑 120 个任务得到一个排行榜”。更准确地说，它是在证明：这个 benchmark 产生的失败、趋势、敏感因素和真实世界排序有足够的信息量，能帮助分析 task-generalist policy 的泛化能力。

## 深挖 1：实验矩阵怎么读，别把所有结果混成一个平均数

论文里至少有四个维度会互相交叉：

| 维度 | 例子 | 读结果时要问 |
|---|---|---|
| policy | Pi0 / Pi0.5 / GR00T 等 | 哪个模型整体更强，是否只是某类任务强 |
| task attribute | color、semantics、spatial、stacking、reorientation | 强弱来自视觉、空间语言还是操作能力 |
| difficulty / horizon | simple、moderate、complex；子任务数量 | 长任务是否出现错误累积 |
| variation | lighting、background、camera、object pose | 模型是否对某个场景因素极敏感 |

所以一条实验记录至少应该包含：

```text
policy_id
task_name
task_attributes
difficulty
instruction_type
scene_variation_id
episode_id
success
score
event_log
trajectory_metrics
video / hdf5 path
```

这也是为什么本地复现不能只保存视频。视频能说明“看起来发生了什么”，但论文级分析需要 metadata 和结构化结果才能分组。

## 深挖 2：Success、Score、Event、Trajectory 四类指标各自回答什么

论文主表里有 success 和 score，后面还有 SPARC、speed、path length、event tracking 等指标。它们不是重复指标。

| 指标 | 说人话 | 典型解释 |
|---|---|---|
| `success` | 最终任务是否完全达成 | 适合排行榜，但太粗 |
| `score` | 子任务/事件完成程度 | 能区分完全失败和部分完成 |
| event log | 失败具体发生在哪里 | wrong object、drop、hit、tipped、out-of-scene |
| SPARC | 末端运动是否平滑 | 同样成功时，动作质量不同 |
| speed / path length | 动作是否高效、路径是否绕 | 反映控制质量和策略犹豫程度 |

举例：

| 现象 | success | score | event | 解释 |
|---|---|---|---|---|
| 抓对苹果但没放进碗 | False | 中等 | drop / incomplete | 感知可能对，放置或接触失败 |
| 抓了橙子放进碗 | False | 可能偏低 | wrong object | 目标绑定失败 |
| 完成任务但路径绕、抖动 | True | 1.0 | 可能无严重事件 | 任务成功但轨迹质量差 |
| 多步骤前两步成功，第三步失败 | False | 不为 0 | subtask incomplete | 长 horizon 错误累积 |

**【橙色标识｜容易误解】**

论文里的 score 不是 success 的同义词。`success=True` 通常意味着 score 达到最终完成口径；但 `success=False` 时，score 仍然能告诉你模型是不是完成了部分子任务。

## 深挖 3：主实验表应该按“能力画像”读

论文主表把 overall、difficulty、procedural、relational、visual 等维度放在一起。读法不是只看最高 overall，而是看画像：

| 画像 | 可能含义 |
|---|---|
| visual 高、relational 低 | 能识别物体属性，但空间语言/多对象关系弱 |
| simple 高、complex 低 | 单步或短 horizon 可以，长任务错误累积 |
| score 明显高于 success | 经常部分完成，但最后一个关键事件失败 |
| SPARC 更接近 0 但 success 不高 | 运动可能平滑，但任务语义/目标绑定失败 |
| speed/path length 很大 | 策略犹豫、绕路或反复修正 |

这类画像比“模型 A 比模型 B 高 3%”更有价值。RoboLab 的定位是分析 benchmark，不只是 leaderboard。

## 深挖 4：论文实验的六组验证如何互相支撑

| 实验 | 单独能说明什么 | 和其他实验如何互相支撑 |
|---|---|---|
| RoboLab-120 | 策略在任务集上的表现 | 给细粒度分析、真实相关性提供基础数据 |
| 细粒度分析 | 哪类能力弱 | 解释 RoboLab-120 平均分背后的失败结构 |
| 扰动敏感性 | 哪些场景因素影响策略 | 解释同一任务为什么在不同环境下表现不同 |
| 真实世界相关性 | 仿真趋势是否有现实意义 | 证明 RoboLab 不是只在仿真内部自洽 |
| 场景生成质量 | 自动场景是否可信 | 支撑 benchmark 可规模化生成 |
| 任务生成质量 | 自动任务是否语义可评测 | 支撑未来 benchmark 扩展不是纯人工瓶颈 |

这也是为什么精讲8要和精讲1、2、3、4、5、6、11、13、14 串起来看：

- 精讲1讲 real-to-sim 评估思想。
- 精讲2讲 scene/task/env 生成。
- 精讲3讲 task generation validation。
- 精讲4讲能力轴和难度公式。
- 精讲5/6讲 SPARC 和 MNPE。
- 精讲11讲 spatial/physical solver。
- 精讲13/14讲证据链和运行时代码。

精讲8在中间起“实验地图”的作用：告诉你每个机制最后服务于哪张表、哪类图、哪种结论。

## 实验 1：RoboLab-120 策略评测

最核心的主实验是 RoboLab-120 benchmark。它评测一组通用机器人策略在 120 个任务上的表现。

输入可以理解成四类：

| 输入 | 说人话解释 |
|---|---|
| 任务定义 | 每个任务是一个 Python task class，里面有语言指令、场景、对象、子任务、成功条件 |
| 策略服务 | 例如 OpenPI/pi0/pi05 一类 VLA policy server，RoboLab 通过 client 请求动作 |
| 仿真环境 | Isaac Sim / Isaac Lab 负责物理、渲染、机器人状态和相机观测 |
| 运行参数 | `--task`、`--num-envs`、`--num-runs`、`--instruction-type`、`--video-mode` 等 |

输出不是一个单独数字，而是一整套 episode 证据：

| 输出 | 用途 |
|---|---|
| `episode_results.jsonl` | 每条 episode 的 success、score、耗时、步数、任务名、策略名等 |
| HDF5 | 离线计算轨迹指标、回放状态和观测 |
| 视频 | 人眼检查策略到底做了什么 |
| event/subtask log | 定位失败发生在抓取、悬停、放置、完成判定中的哪一步 |

**【蓝色标识｜源码路径】**

- `policies/pi0_family/run.py`：Pi0/Pi05 family 的命令行评测入口。
- `robolab/eval/runner.py`：共享评测循环，负责按任务创建 env、运行 episode、写出结果。
- `robolab/core/environments/runtime.py`：根据任务和 policy 组装 Isaac/RoboLab 环境。
- `robolab/core/logging/results.py`：把 episode 结果汇总成分组统计。

**【橙色标识｜我们当前复现边界】**

我们已经完成的是 Pi05 在简单任务和复杂任务抽样上的 smoke/小样本复现。它证明链路通了，但还不是完整 RoboLab-120。完整实验需要对 120 个任务、固定 policy 集合、足够 episode 数、统一输出目录和统一分析脚本全部跑完。

## 实验 2：按能力轴、难度和任务结构做细粒度分析

论文不只报平均成功率，因为平均数会掩盖失败原因。RoboLab 会把任务按多个维度拆开：

| 分析维度 | 它想看什么 |
|---|---|
| 能力轴 | visual、procedural、relational 三类能力是否表现不同 |
| 属性标签 | color、semantics、spatial relation、counting、stacking、reorientation 等 |
| 难度 | easy / medium / hard，来自子任务数量和最高需求级别 |
| 场景规模 | 物体数量变多后，策略是否更容易拿错对象或漏执行 |
| 子任务数 | 多步骤任务越长，错误是否累积 |
| 指令类型 | default / vague / specific 指令是否影响成功率 |

这个分析的关键是：每个任务本身携带 metadata。评测后，分析脚本不是重新理解视频，而是读取 task metadata 和 episode result 做分组。

**【蓝色标识｜源码路径】**

- `robolab/core/task/task.py`：任务对象上有 `attributes`、`subtasks`、instruction 等字段。
- `robolab/core/task/subtask_utils.py`：计算子任务数和难度分数。
- `robolab/constants.py`：保存技能权重、难度阈值、benchmark 类别等配置。
- `analysis/read_results.py`：按 attributes、difficulty、scene、wrong objects、instruction type 等维度读结果。

**【绿色标识｜怎么读结果】**

如果一个模型在 easy 任务上成功，但在 hard 任务上大幅下降，通常说明不是“看不见物体”，而是长 horizon、对象保持、阶段切换或恢复能力不足。  
如果 visual 属性下降明显，重点看颜色/语义识别；如果 relational 下降明显，重点看左右、前后、容器、并列/或/计数等语言结构。

## 实验 3：扰动敏感性和 MNPE

扰动实验问的是：模型是不是只会在“刚好这张桌子、刚好这个相机、刚好这个光照”下成功。

论文里关心的扰动包括：

| 扰动类型 | 示例 |
|---|---|
| 光照 | 颜色、阴影、昏暗、过曝 |
| 视觉背景 | HDR/背景、桌面纹理 |
| 物体初始位置 | 10 cm / 20 cm / 30 cm 级别随机扰动 |
| 相机位姿 | 外部相机、腕部相机的位姿变化 |

普通做法是逐个扰动看成功率。MNPE 更进一步：把扰动参数当成变量，观察“成功 episode 的参数后验分布”和“原始采样分布”差多少。

说人话：

> 如果某个相机角度在原始采样里很多，但成功样本里几乎没有，它就是高风险区域；如果成功样本都集中在某个窄范围，说明策略对这个因素很敏感。

**【蓝色标识｜源码路径】**

- `policies/pi0_family/run_lighting.py`
- `policies/pi0_family/run_background_variation.py`
- `policies/pi0_family/run_table_variation.py`
- `policies/pi0_family/run_camera_pose_variation.py`
- `analysis/sensitivity_analysis/posterior_inference.py`

**【橙色标识｜复现注意】**

扰动实验很耗 GPU，因为它不是多跑几个 episode，而是系统性扫参数。4090 上建议先固定一个任务、一个策略、一个扰动维度做小样本，再扩到多任务和多扰动。

## 实验 4：真实世界相关性验证

论文还做了 real-world verification：把 RoboLab 中的策略表现和真实世界 benchmark/arena 中的策略排名比较。

这个实验的核心不是证明“仿真等于真实”，而是看：

| 问题 | 解释 |
|---|---|
| 排名是否一致 | RoboLab 里更强的 policy，在真实 arena 中是否也更强 |
| 相关性是否为正 | RoboLab 分数和真实世界 Elo/score 是否同向 |
| 仿真是否有评估价值 | 即使绝对成功率不同，排名和趋势是否仍可用 |

**【橙色标识｜容易误解】**

这个实验是 policy-level 的相关性，不是说某个 RoboLab 单任务成功就能直接迁移到真实机器人。它更像“这个仿真 benchmark 是否能给真实评测前的模型筛选提供参考”。

## 实验 5：场景生成质量实验

论文 Appendix C-D 对比了主方法和 baseline 的场景生成质量。baseline 是精讲7讲过的“LLM 选物体 + 网格布局 + jitter”。主方法则使用对象目录、谓词、空间约束、物理检查和反馈修复。

评估指标包括：

| 指标 | 含义 |
|---|---|
| VQA score | 用视觉问答检查场景是否满足描述 |
| GPT preference | 让 GPT 在两种生成结果之间做偏好判断 |
| Real. | 场景是否真实 |
| Func. | 是否能支持任务操作 |
| Lay. | 布局是否合理 |
| Compl. | 场景是否完整 |
| Qual. | 综合质量 |
| #Obj | 场景包含对象数量 |

论文总体结果里，主方法相对 baseline 明显更高：例如 overall VQA 从约 0.398 提到约 0.554，GPT preference 从 18% 提到 82%。这说明主方法不只是“摆得更密”，而是更能生成可评测、有关系结构、视觉和功能上更合理的场景。

### 场景生成质量表怎么读

Appendix C-D 的表不是在评估 policy，而是在评估“场景生成器”。它回答的是：主方法生成的桌面场景是否比 baseline 更适合作为 benchmark 场景。

| 读表维度 | 应该看什么 | 说明 |
|---|---|---|
| VQA | 渲染图是否满足文本问题 | 偏视觉一致性和可识别性 |
| Real. | 是否像真实桌面/工作台 | 对抗“网格摆拍感” |
| Func. | 是否能支持机器人操作 | 对抗“好看但不可操作” |
| Lay. | 布局是否自然、有组合关系 | 对抗均匀撒点 |
| Compl. | 场景是否完整 | 是否有足够对象和结构 |
| Qual. | 综合质量 | 人类/LLM judge 的整体观感 |
| #Obj | 对象数量 | 数量高不代表质量高，要和 Real/Func/Lay 一起看 |
| Pref. | GPT preference | 成对比较主方法和 baseline 的偏好 |

**【绿色标识｜核心结论】**

场景生成实验真正说明的是：谓词、solver 和反馈闭环让场景更像“可操作的真实桌面”，而不是只让 LLM 输出更多对象。

### 为什么 baseline 会输

baseline 的典型流程是：

```text
LLM 选对象 -> 网格位置 -> jitter -> 单次生成
```

它的问题是：

- 缺少 `place-in` / `place-on` 这样的语义关系，容器和支撑物不能形成真实组合。
- 网格布局天然有“摆拍感”，物体不容易形成真实的局部聚簇。
- 没有 feedback repair，失败样本很难在下一轮变好。
- 缺少物理/几何检查时，场景可能视觉上有对象，但作为机器人任务不可用。

主方法的优势来自：

```text
对象目录 + typed predicates + spatial solver + physical solver + validation + feedback
```

这就是为什么同样用 LLM，主方法和 baseline 的差异会很大：LLM 不是独立完成场景生成，而是被放进一个可验证的工程闭环。

**【蓝色标识｜源码路径】**

- `robolab/scene_gen/llm_scene_gen/predicates.py`：谓词库，描述 `place-in`、`place-on`、`cluster-around`、相对位置等。
- `robolab/scene_gen/llm_scene_gen/spatial_solver.py`：Algorithm 1 对应的 2D 空间约束求解器。
- `robolab/scene_gen/llm_scene_gen/physical_solver.py`：处理容器、支撑、碰撞、稳定性等更接近 3D 物理的问题。
- `robolab/scene_gen/llm_scene_gen/feedback_system.py`：失败后收集错误并反馈给 LLM 修复。

## Algorithm 1：Spatial Constraint Solver 是干什么的

你图里这段 Algorithm 1 是场景生成实验的核心之一。它的作用是：

> 给定一堆对象和空间谓词，先在桌面 2D 平面上找一个大体合理、无碰撞、满足相对关系的基础布局。

### 输入

| 符号 | 代码里对应什么 | 解释 |
|---|---|---|
| `B` Objects | object states / object dims | 场景对象以及每个对象的尺寸、半径、当前状态 |
| `P` Predicates | predicate list | LLM 生成的空间关系，例如放在桌面、围绕某物、在某物左边 |
| `Lmax` Table Bounds | table bounds | 桌面可用区域，限制 `(x, y)` 不能飞出桌子 |

### 输出

| 输出 | 解释 |
|---|---|
| `(x, y, theta)` | 每个基础对象在桌面平面上的位置和朝向 |

这里强调“基础对象”很重要。比如“香蕉在碗里”，碗是基础对象，香蕉的精确 3D 放置可能由后续物理求解处理。Algorithm 1 主要解决“碗、盘子、杯子、工具等基础对象先怎么摆开”。

## Algorithm 1 三阶段拆解

### Phase 1：初始化

论文伪代码：

```text
Randomize (x, y) for all loose objects inside Lmax
if predicate is place-on-base:
    put object at specified x, y, theta
elif predicate is cluster-around:
    polar place targets around anchor
```

说人话：

1. 先把没有硬约束的对象随机撒到桌面范围内。
2. 如果某个对象被指定放在基础位置，就直接设置它的 `(x, y, yaw)`。
3. 如果某些对象要围绕一个 anchor，就用极坐标把它们摆在 anchor 周围。

为什么要这么做？因为 LLM 给的是语义关系，不一定给每个对象的精确坐标。求解器先把“明显关系”落实成一个初始几何布局。

### Phase 2：相对关系约束

论文伪代码：

```text
while constraints not satisfied:
    ApplyRelativeConstraints(P)
ApplyOrientations(P)
```

说人话：

如果任务说“红杯在碗左边”“勺子朝前”“三个物体围着盘子”，这一阶段会反复调整位置和朝向，直到这些相对关系大体满足。

注意：这不是高精度优化器，更像一个规则驱动的约束传播器。它把语言里的空间词转换为可执行的几何关系。

### Phase 3：碰撞消解

论文伪代码：

```text
for k = 1 to Kmax:
    C <- FindCollisions(B, margin)
    if C is empty:
        return Success
    if collision count not decreasing:
        PerturbPositions(B)
    for each collision pair:
        ResolveOverlap(...)
        ClampToBounds(...)
return Failure
```

说人话：

1. 检查对象之间有没有重叠或距离太近。
2. 没有碰撞就成功。
3. 如果碰撞数量长时间不下降，就轻微扰动对象，避免卡死在坏局部解。
4. 对每一对碰撞对象，把它们沿相反方向推开。
5. 推开后再把坐标夹回桌面范围，防止对象被推到桌外。

### Margin `M = [mu, 1.25mu, 1.5mu, 2.0mu]`

margin 是对象之间的安全间距。伪代码会尝试一组 margin 候选。直观理解：

| margin | 效果 |
|---|---|
| 小 margin | 更容易塞进密集场景，但对象距离更近 |
| 大 margin | 更保守，要求对象之间留更多空间 |

当前源码实现里也会根据对象数量和大对象数量调整基础 collision margin。密集场景会有更长的优化迭代预算。

## Algorithm 1 和源码怎么对上

**【蓝色标识｜源码路径】** `robolab/scene_gen/llm_scene_gen/spatial_solver.py`

| 论文伪代码 | 源码职责 |
|---|---|
| `SpatialConstraintSolver` | `SpatialSolver` 类 |
| `Input: Objects B, Predicates P, Table Bounds Lmax` | `solve(object_states, object_dims, max_iterations, fixed_objects, allow_relaxation)` |
| `Randomize loose objects` | `solve()` 中对非固定对象初始化/重采样 |
| `place-on-base` | `_apply_place_on_base(...)` 一类函数 |
| `cluster-around` | 极坐标围绕 anchor 放置 target 的逻辑 |
| `ApplyRelativeConstraints(P)` | `_apply_relative_position(...)` 一类函数 |
| `ApplyOrientations(P)` | orientation/yaw 约束处理 |
| `FindCollisions` | 碰撞检测函数簇 |
| `ResolveOverlap` | overlap 消解函数簇 |
| `ClampToBounds` | 坐标边界 clamp |

**【橙色标识｜代码边界】**

`spatial_solver.py` 解决的是 2D 桌面平面布局。真正的 `place-in`、`place-on`、容器尺寸、支撑稳定性、物理 settle，需要和 `physical_solver.py`、Isaac Sim 物理模拟一起看。

## 实验 6：任务生成质量实验

任务生成实验验证的是：自动生成出来的 task 代码是不是“语言目标、成功条件、物理可行性”都靠谱。

论文里的流程可以拆成：

| 步骤 | 说人话解释 |
|---|---|
| 生成任务代码 | LLM 根据对象目录、模板、谓词库、难度约束写 task class |
| 语法验证 | 代码能不能 import，类和字段是否完整 |
| 资源验证 | 用到的对象是否存在，是否在禁用集合，容器尺寸是否放得下 |
| 语义验证 | 语言指令和程序化 success condition 是否一致 |
| 失败修复 | 把错误反馈给 LLM，要求它改代码再验证 |

论文报告的 task generation 评估包括 alignment、clarity、feasibility、match、object coverage、predicate coverage 等指标。总体上，自动生成任务已经能达到较高对齐度，但 predicate coverage 不高，说明任务生成仍需要人工审查和迭代，不应被当成完全自动、无需验收的 benchmark 生产线。

### 任务生成质量实验怎么读

任务生成的难点不是“让 LLM 写 Python”，而是让自然语言目标、场景对象、成功谓词和物理可行性同时成立。

| 维度 | 它检查什么 | 失败例子 |
|---|---|---|
| relation match | 空间/逻辑关系是否一致 | 指令说左边，代码检查右边 |
| target match | 目标状态是否一致 | 指令说放进碗，代码检查放到盘子 |
| object match | 对象引用是否正确 | 指令说红杯，代码用蓝杯 |
| quantifier match | all/any/count 是否一致 | 指令说两个，代码只检查一个 |
| clarity | 语言是否清楚 | 指令含糊，不知道目标对象 |
| feasibility | 机器人和场景是否可完成 | 容器太小、堆叠不稳、对象不可达 |

**【橙色标识｜边界】**

LLM-as-judge 可以提高扩展速度，但不能完全替代人工验收。尤其是 predicate coverage 和物理可行性，最终还需要静态检查、场景资源检查和实际仿真 smoke 共同兜底。

**【蓝色标识｜源码路径】**

- `skills/robolab-taskgen/SKILL.md`
- `skills/robolab-taskgen/references/examples.md`
- `skills/robolab-taskgen/references/conditionals.md`
- `robolab/core/task/task_utils.py`
- `robolab/core/scenes/utils.py`

## 这部分和我们后续复现怎么衔接

要按论文口径继续推进，建议把实验分成 4 个层级，不要一上来盲跑 120 个任务：

| 层级 | 目标 | 4090 上的建议 |
|---|---|---|
| L1 链路验证 | 单任务能不能通，视频/HDF5/JSON 是否完整 | 已完成 Pi05 BananaInBowlTask |
| L2 复杂任务抽样 | 多对象、空间关系、重定向、堆叠是否能跑起来 | 已跑复杂任务抽样，可继续补 5-10 个 |
| L3 小型 benchmark | 每个能力轴抽一批任务，按 difficulty 分组 | 建议先 15-30 个任务，`num_envs=1` |
| L4 论文级复现 | RoboLab-120 + 多策略 + 扰动 + 分析脚本 | 需要全量资产、长时间 GPU、统一结果目录 |

**【绿色标识｜复现优先级】**

下一步最有价值的不是立刻全量 120，而是：

1. 固定 Pi05，扩到每个能力轴至少 5 个任务。
2. 每个任务保存 `episode_results.jsonl`、HDF5、视频和子任务日志。
3. 用 `analysis/read_results.py` 按能力轴、难度、任务长度出表。
4. 再选一个成功率中等的任务做光照/背景/物体位置扰动。
5. 最后才把同一套任务换成 RoboChallenge pi 或 ReKep 做对照。

这样得到的结论会比“下载没齐就盲跑 RoboLab-120”更可信。

## 深挖 5：4090 上怎么设计一个“像论文但可承受”的实验矩阵

完整 RoboLab-120 + 多策略 + 多扰动很重。4090 上更现实的做法是先做一个“小论文矩阵”：

| 轴 | 最小设置 | 为什么 |
|---|---|---|
| task | 每个能力轴 5 个，共 15 个 | 覆盖 visual/procedural/relational |
| difficulty | easy/medium/hard 各至少 3 个 | 看难度斜率 |
| policy | Pi05 先跑，再加 RoboChallenge Pi / ReKep 对照 | 先稳定主链路，再做方法对比 |
| episode | 每任务 1-3 条 | 先出趋势，不冒充统计显著 |
| variation | 选 1 个任务做 lighting 或 object pose sweep | 先验证敏感性 pipeline |
| artifacts | JSONL + HDF5 + video + event log | 保证能复查 |

输出应该至少包含四张表：

1. `task_result_table.csv`：每条 episode 的 success、score、step、reason。
2. `axis_summary.csv`：按 visual/procedural/relational 分组。
3. `difficulty_summary.csv`：按 easy/medium/hard 分组。
4. `failure_reason_summary.csv`：wrong object、drop、collision、timeout、predicate incomplete 等。

**【绿色标识｜核心结论】**

这个小矩阵不是论文级全量复现，但它的结构是论文级的：任务选择、指标、证据、分组和边界都对齐论文，只是样本量更小。

## 最容易混淆的三件事

| 容易混淆 | 正确认知 |
|---|---|
| 场景生成实验 vs 策略评测实验 | 场景生成实验评估生成环境质量；策略评测实验评估模型执行任务能力 |
| Algorithm 1 vs 物理仿真 | Algorithm 1 做 2D 空间约束；Isaac/physical solver 做 3D 接触、稳定性和渲染 |
| 单任务成功 vs 论文级复现 | 单任务成功证明链路通；论文级复现需要固定任务集、策略集、采样次数和分析脚本 |

## 小结

论文实验的主线可以记成：

```text
生成可信场景 -> 生成可验证任务 -> 跑策略 -> 记录 episode -> 分组分析 -> 扰动敏感性 -> 和真实世界排名对照
```

Algorithm 1 位于第一步“生成可信场景”里。它不是最终评测指标，但它决定了场景能否在几何上成立。如果这一步失败，后面的任务、物理 settle、策略评测都会被污染。

In [ ]:
# ===== 精讲8：Algorithm 1 Spatial Constraint Solver 轻量验证 =====
# 这个 toy solver 只复刻论文 Algorithm 1 的结构：
# 输入：对象 B、谓词 P、桌面边界 Lmax。
# 输出：每个基础对象的 2D 位姿 (x, y, theta)。
# 边界：它不是官方 spatial_solver.py，也不做 USD/Isaac 物理 settle。

import json
import math
import numpy as np

def _clamp(value, lo, hi):
    # 把一个坐标限制在桌面边界内。
    return float(max(lo, min(hi, value)))

def _pose_array(poses, name):
    # 取对象的 2D 坐标，便于用向量方式算距离和方向。
    return np.array([poses[name]["x"], poses[name]["y"]], dtype=float)

def find_circle_collisions(poses, radii, margin):
    # 用圆近似对象 footprint；真实 RoboLab 会结合对象尺寸和更复杂的碰撞/物理检查。
    collisions = []
    names = list(poses)
    for i, a in enumerate(names):
        for b in names[i + 1 :]:
            pa = _pose_array(poses, a)
            pb = _pose_array(poses, b)
            distance = float(np.linalg.norm(pb - pa))
            required = float(radii[a] + radii[b] + margin)
            if distance < required:
                collisions.append(
                    {
                        "a": a,
                        "b": b,
                        "distance": distance,
                        "required": required,
                    }
                )
    return collisions

def solve_toy_spatial_constraints(objects, predicates, table_bounds, mu=0.02, max_iterations=200, seed=42):
    # 论文里的 margin schedule：先用基础安全距离，再尝试更保守的对象间距。
    margins = [mu, 1.25 * mu, 1.5 * mu, 2.0 * mu]
    rng = np.random.default_rng(seed)
    x_min, x_max, y_min, y_max = table_bounds
    radii = {name: float(spec["radius"]) for name, spec in objects.items()}
    last_error = "not started"

    def clamp_pose(poses, name):
        radius = radii[name]
        poses[name]["x"] = _clamp(poses[name]["x"], x_min + radius, x_max - radius)
        poses[name]["y"] = _clamp(poses[name]["y"], y_min + radius, y_max - radius)

    def initialize(margin):
        # Phase 1a：先把所有 loose objects 随机撒到桌面里。
        poses = {}
        for name, spec in objects.items():
            radius = radii[name]
            poses[name] = {
                "x": float(rng.uniform(x_min + radius + margin, x_max - radius - margin)),
                "y": float(rng.uniform(y_min + radius + margin, y_max - radius - margin)),
                "theta": 0.0,
            }

        fixed = set()
        for pred in predicates:
            # Phase 1b：place-on-base 是硬锚点，直接写入指定位置和朝向。
            if pred["type"] == "place-on-base":
                name = pred["object"]
                poses[name]["x"] = float(pred["x"])
                poses[name]["y"] = float(pred["y"])
                poses[name]["theta"] = float(pred.get("theta", 0.0))
                clamp_pose(poses, name)
                fixed.add(name)

        for pred in predicates:
            # Phase 1c：cluster-around 用极坐标把目标围绕 anchor 摆开。
            if pred["type"] == "cluster-around":
                anchor = pred["anchor"]
                targets = list(pred["targets"])
                radius = float(pred["radius"])
                start_angle = float(pred.get("start_angle", 0.0))
                for index, target in enumerate(targets):
                    angle = start_angle + 2.0 * math.pi * index / max(1, len(targets))
                    poses[target]["x"] = poses[anchor]["x"] + radius * math.cos(angle)
                    poses[target]["y"] = poses[anchor]["y"] + radius * math.sin(angle)
                    clamp_pose(poses, target)
        return poses, fixed

    def apply_relative_constraints(poses):
        # Phase 2：把语言里的 left/right/front/back 约束变成几何偏移。
        changed = False
        for pred in predicates:
            kind = pred["type"]
            if kind not in {"left-of", "right-of", "front-of", "behind"}:
                continue
            name = pred["object"]
            ref = pred["reference"]
            distance = float(pred.get("distance", 0.16))
            old = (poses[name]["x"], poses[name]["y"])
            if kind == "left-of":
                poses[name]["x"] = poses[ref]["x"]
                poses[name]["y"] = poses[ref]["y"] + distance
            elif kind == "right-of":
                poses[name]["x"] = poses[ref]["x"]
                poses[name]["y"] = poses[ref]["y"] - distance
            elif kind == "front-of":
                poses[name]["x"] = poses[ref]["x"] + distance
                poses[name]["y"] = poses[ref]["y"]
            elif kind == "behind":
                poses[name]["x"] = poses[ref]["x"] - distance
                poses[name]["y"] = poses[ref]["y"]
            clamp_pose(poses, name)
            changed = changed or old != (poses[name]["x"], poses[name]["y"])
        return changed

    def apply_orientations(poses):
        # Phase 2 的朝向部分：把 facing-* 约束写成 theta。
        theta_by_type = {
            "facing-front": 0.0,
            "facing-back": math.pi,
            "facing-left": math.pi / 2.0,
            "facing-right": -math.pi / 2.0,
        }
        for pred in predicates:
            if pred["type"] in theta_by_type:
                poses[pred["object"]]["theta"] = float(theta_by_type[pred["type"]])

    def resolve_overlap(poses, collision, fixed, margin):
        # Phase 3：把碰撞对象沿连线方向推开；固定锚点不移动。
        a = collision["a"]
        b = collision["b"]
        pa = _pose_array(poses, a)
        pb = _pose_array(poses, b)
        direction = pb - pa
        distance = float(np.linalg.norm(direction))
        if distance < 1e-9:
            direction = rng.normal(size=2)
            distance = float(np.linalg.norm(direction))
        direction = direction / distance
        required = radii[a] + radii[b] + margin
        push = float(required - distance + 1e-4)

        if a in fixed and b in fixed:
            return
        if a in fixed:
            poses[b]["x"] += float(direction[0] * push)
            poses[b]["y"] += float(direction[1] * push)
            clamp_pose(poses, b)
        elif b in fixed:
            poses[a]["x"] -= float(direction[0] * push)
            poses[a]["y"] -= float(direction[1] * push)
            clamp_pose(poses, a)
        else:
            poses[a]["x"] -= float(direction[0] * push / 2.0)
            poses[a]["y"] -= float(direction[1] * push / 2.0)
            poses[b]["x"] += float(direction[0] * push / 2.0)
            poses[b]["y"] += float(direction[1] * push / 2.0)
            clamp_pose(poses, a)
            clamp_pose(poses, b)

    for margin in margins:
        poses, fixed = initialize(margin)
        phase_log = [f"margin={margin:.3f}: initialized"]

        for _ in range(8):
            if not apply_relative_constraints(poses):
                break
        apply_orientations(poses)
        phase_log.append("relative constraints and orientations applied")

        collision_counts = []
        for iteration in range(max_iterations):
            collisions = find_circle_collisions(poses, radii, margin)
            if not collisions:
                return {
                    "success": True,
                    "margin": float(margin),
                    "iterations": iteration,
                    "poses": poses,
                    "phase_log": phase_log + ["collision resolution succeeded"],
                }

            collision_counts.append(len(collisions))
            if len(collision_counts) >= 10 and len(set(collision_counts[-10:])) == 1:
                # 如果碰撞数量长时间不下降，轻微扰动非固定对象，避免卡死。
                for name in poses:
                    if name not in fixed:
                        poses[name]["x"] += float(rng.normal(0.0, 0.01))
                        poses[name]["y"] += float(rng.normal(0.0, 0.01))
                        clamp_pose(poses, name)

            for collision in collisions:
                resolve_overlap(poses, collision, fixed, margin)

        last_error = f"failed at margin={margin:.3f} with {len(collisions)} collisions"

    return {
        "success": False,
        "margin": None,
        "iterations": max_iterations,
        "poses": poses,
        "phase_log": [last_error],
    }

toy_objects = {
    "bowl_anchor": {"radius": 0.055},
    "red_mug": {"radius": 0.045},
    "blue_mug": {"radius": 0.045},
    "cube": {"radius": 0.040},
    "spoon": {"radius": 0.035},
}
toy_predicates = [
    {"type": "place-on-base", "object": "bowl_anchor", "x": 0.55, "y": 0.00, "theta": 0.0},
    {"type": "cluster-around", "anchor": "bowl_anchor", "targets": ["red_mug", "blue_mug"], "radius": 0.13},
    {"type": "left-of", "object": "cube", "reference": "bowl_anchor", "distance": 0.18},
    {"type": "facing-front", "object": "spoon"},
]
toy_table_bounds = (0.25, 0.85, -0.40, 0.40)
toy_result = solve_toy_spatial_constraints(
    toy_objects,
    toy_predicates,
    toy_table_bounds,
    mu=0.02,
    max_iterations=200,
    seed=2604,
)

poses = toy_result["poses"]
radii = {name: spec["radius"] for name, spec in toy_objects.items()}
final_margin = toy_result["margin"] if toy_result["margin"] is not None else 0.02
final_collisions = find_circle_collisions(poses, radii, final_margin)

def in_bounds(name):
    x_min, x_max, y_min, y_max = toy_table_bounds
    radius = radii[name]
    return (
        x_min + radius <= poses[name]["x"] <= x_max - radius
        and y_min + radius <= poses[name]["y"] <= y_max - radius
    )

def xy_distance(a, b):
    return float(np.linalg.norm(_pose_array(poses, a) - _pose_array(poses, b)))

experiment_taxonomy = {
    "policy_benchmark": ["success", "score", "episode_results.jsonl"],
    "granular_analysis": ["attributes", "difficulty", "num_objects", "num_subtasks"],
    "sensitivity": ["lighting", "background", "table", "camera", "object_pose"],
    "real_world": ["RoboLab score", "RoboArena/Elo correlation"],
    "scene_generation": ["VQA", "GPT preference", "Real", "Func", "Lay", "Compl", "Qual"],
    "task_generation": ["alignment", "clarity", "feasibility", "match", "coverage"],
}

evidence_contract = {
    "success": "binary final completion; good for leaderboard but too coarse alone",
    "score": "graded subtask/event completion; separates partial completion from total failure",
    "event_log": "diagnostic failure evidence such as wrong object, drop, hit, tipped, out of scene",
    "trajectory": "SPARC, speed, and path length for motion quality",
    "artifacts": ["episode_results.jsonl", "run_<idx>.hdf5", "log_<run>_env<id>.json", "mp4"],
}

scene_generation_eval_schema = {
    "VQA": "render/text consistency and visual recognizability",
    "Real": "realism of the tabletop scene",
    "Func": "whether the scene supports manipulation",
    "Lay": "natural and structured layout",
    "Compl": "scene completeness",
    "Qual": "overall quality",
    "Pref": "paired GPT preference against baseline",
}

task_generation_judge_dimensions = {
    "relation_match": "spatial/logical relation is preserved",
    "target_match": "goal state matches the instruction",
    "object_match": "referenced objects are correct",
    "quantifier_match": "all/any/count semantics match",
    "clarity": "instruction is unambiguous",
    "feasibility": "task is physically and robotically achievable",
}

mini_paper_matrix = {
    "tasks": {"visual": 5, "procedural": 5, "relational": 5},
    "difficulty_minimum": {"easy": 3, "medium": 3, "hard": 3},
    "policies": ["pi05_first", "robochallenge_pi_optional", "rekep_optional"],
    "episodes_per_task": "1-3 for trend only; do not claim full statistical significance",
    "variation_probe": ["one lighting sweep or one object-pose sweep"],
    "required_outputs": ["task_result_table.csv", "axis_summary.csv", "difficulty_summary.csv", "failure_reason_summary.csv"],
}

toy_episode_results = [
    {"task": "visual_pick", "axis": "visual", "difficulty": "easy", "success": True, "score": 1.0, "reason": "success"},
    {"task": "spatial_left", "axis": "relational", "difficulty": "medium", "success": False, "score": 0.55, "reason": "wrong object"},
    {"task": "stack_two", "axis": "procedural", "difficulty": "hard", "success": False, "score": 0.35, "reason": "drop"},
]

def summarize_by_key(rows, key):
    groups = {}
    for row in rows:
        bucket = groups.setdefault(row[key], {"n": 0, "successes": 0, "score_sum": 0.0, "reasons": {}})
        bucket["n"] += 1
        bucket["successes"] += int(row["success"])
        bucket["score_sum"] += float(row["score"])
        bucket["reasons"][row["reason"]] = bucket["reasons"].get(row["reason"], 0) + 1
    return {
        name: {
            "n": values["n"],
            "success_rate": values["successes"] / values["n"],
            "mean_score": values["score_sum"] / values["n"],
            "reasons": values["reasons"],
        }
        for name, values in groups.items()
    }

axis_summary = summarize_by_key(toy_episode_results, "axis")
difficulty_summary = summarize_by_key(toy_episode_results, "difficulty")

paper_experiment_tests = [
    ("algorithm_returns_success", toy_result["success"]),
    ("all_objects_within_table_bounds", all(in_bounds(name) for name in poses)),
    ("no_final_circle_collisions", len(final_collisions) == 0),
    ("cluster_red_mug_near_anchor", abs(xy_distance("red_mug", "bowl_anchor") - 0.13) < 0.04),
    ("cluster_blue_mug_near_anchor", abs(xy_distance("blue_mug", "bowl_anchor") - 0.13) < 0.04),
    ("left_of_constraint_satisfied", poses["cube"]["y"] - poses["bowl_anchor"]["y"] >= 0.16),
    ("orientation_constraint_satisfied", abs(poses["spoon"]["theta"] - 0.0) < 1e-9),
    (
        "experiment_taxonomy_complete",
        {"policy_benchmark", "sensitivity", "real_world", "scene_generation", "task_generation"}.issubset(
            experiment_taxonomy
        ),
    ),
    (
        "evidence_contract_separates_success_score_events_trajectory",
        {"success", "score", "event_log", "trajectory", "artifacts"}.issubset(evidence_contract),
    ),
    (
        "scene_generation_eval_schema_has_quality_and_preference",
        {"VQA", "Real", "Func", "Lay", "Compl", "Qual", "Pref"}.issubset(scene_generation_eval_schema),
    ),
    (
        "task_generation_judge_has_six_dimensions",
        set(task_generation_judge_dimensions)
        == {"relation_match", "target_match", "object_match", "quantifier_match", "clarity", "feasibility"},
    ),
    (
        "mini_paper_matrix_covers_three_axes_and_outputs",
        set(mini_paper_matrix["tasks"]) == {"visual", "procedural", "relational"}
        and len(mini_paper_matrix["required_outputs"]) == 4,
    ),
    (
        "toy_axis_summary_preserves_partial_scores",
        axis_summary["relational"]["success_rate"] == 0.0
        and 0.0 < axis_summary["relational"]["mean_score"] < 1.0,
    ),
    (
        "toy_difficulty_summary_keeps_failure_reasons",
        difficulty_summary["hard"]["reasons"].get("drop") == 1,
    ),
]

print(json.dumps(toy_result, ensure_ascii=False, indent=2))
print("Evidence contract:")
print(json.dumps(evidence_contract, ensure_ascii=False, indent=2))
print("Mini paper matrix:")
print(json.dumps(mini_paper_matrix, ensure_ascii=False, indent=2))
print("Toy axis summary:")
print(json.dumps(axis_summary, ensure_ascii=False, indent=2))
for name, ok in paper_experiment_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert all(ok for _, ok in paper_experiment_tests), paper_experiment_tests
write_status(
    "paper_experiments_algorithm_lightweight_tests",
    {
        "all_passed": all(ok for _, ok in paper_experiment_tests),
        "toy_result": toy_result,
        "final_collisions": final_collisions,
        "experiment_taxonomy": experiment_taxonomy,
        "evidence_contract": evidence_contract,
        "scene_generation_eval_schema": scene_generation_eval_schema,
        "task_generation_judge_dimensions": task_generation_judge_dimensions,
        "mini_paper_matrix": mini_paper_matrix,
        "axis_summary": axis_summary,
        "difficulty_summary": difficulty_summary,
        "tests": [{"name": name, "passed": bool(ok)} for name, ok in paper_experiment_tests],
        "boundary": "This validates EXPLAIN_08's experiment map and Algorithm 1 structure on toy data; it is not a replacement for RoboLab-120, official analysis scripts, spatial_solver.py, or Isaac physics settle.",
    },
)

## 0.15 论文精讲：DTGE - 任务生成质量评估

下面这节来自本目录的 [EXPLAIN_09_dtge.md](./EXPLAIN_09_dtge.md)。这里的 DTGE 指论文 Appendix D 的 Details on Task Generation Evaluation：它不是策略跑分，而是评估 LLM 自动生成的任务代码是否“指令清楚、成功条件匹配、物理可行、对象/谓词覆盖合理”。

# 精讲 9：DTGE - Details on Task Generation Evaluation

> **【绿色标识｜核心结论】** DTGE 不是一个新模型，也不是策略执行评测；它是论文 Appendix D 的 “Details on Task Generation Evaluation”，核心问题是：自动生成出来的 RoboLab task，语言指令和程序化成功条件是否真的一致。  
> **【蓝色标识｜源码路径】** 它对应的代码表面主要在 `skills/robolab-taskgen/`、`robolab/core/task/task.py`、`robolab/core/task/conditionals.py`、`robolab/tasks/_utils/generate_task_metadata.py`。  
> **【橙色标识｜容易误解】** DTGE 评的是任务生成质量，不是机器人策略成功率。一个 task 被 DTGE 判为高质量，只说明“任务定义可信”，不说明 Pi05、RoboChallenge pi 或 ReKep 一定能完成它。

## 先说结论

DTGE 可以用一句话概括：

> 给 LLM 自动生成的任务代码做“审题”：自然语言说的目标，和 Python success condition 真正检查的目标，是否是同一件事。

它关注的是 task 本身有没有问题，而不是 policy 表现：

| 问题 | DTGE 看不看 | 解释 |
|---|---:|---|
| 指令是否清楚 | 看 | 例如“把苹果放进碗里”是否明确 |
| 成功条件是否匹配指令 | 看 | 例如代码是否真的检查 `object_in_container(apple, bowl)` |
| 物理上是否可实现 | 看 | 例如太大的物体不能塞进太小容器 |
| 任务覆盖了哪些对象 | 看 | 用 object coverage 统计 |
| 任务用了多少谓词种类 | 看 | 用 predicate coverage 统计 |
| 策略是否执行成功 | 不看 | 那是 RoboLab-120 policy benchmark |
| 轨迹是否平滑 | 不看 | 那是 SPARC/trajectory metrics |

## DTGE 在论文里的完整流程

论文 Appendix D 的流程可以拆成 5 步：

| 步骤 | 说人话解释 | 输入 | 输出 |
|---|---|---|---|
| 1. 生成任务 | 用 LLM 根据场景描述和类别模板生成 Python task | scene description、category templates | task Python code |
| 2. 静态抽取 | 不运行机器人，只读代码 | task class | instruction + termination success predicate |
| 3. LLM-as-judge | 让 LLM 评估语言和代码是否一致 | instruction、success condition | 6 个维度评分 |
| 4. 聚合评分 | 把多个维度合成 alignment / match / verdict | judge scores | aligned / partial / misaligned |
| 5. 覆盖率统计 | 看任务是否充分利用场景对象和谓词库 | generated task set | object coverage / predicate coverage |

这一步很关键：**DTGE 的静态抽取对象是 task 代码，不是视频、不是 HDF5、也不是 policy rollout。**

## 论文里的数据口径

论文使用 7 个类别，每个类别 116 个任务，总计 812 个自动生成任务。评估结果大致如下：

| Category | Count | Alignment | Clarity | Feasibility | Match | Aligned% | Partial% | Object Coverage | Predicate Coverage |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| color | 116 | 0.81 | 0.94 | 0.80 | 0.90 | 57 | 40 | 0.88 | 0.29 |
| conjunction | 116 | 0.97 | 0.98 | 1.00 | 0.98 | 91 | 9 | 0.88 | 0.29 |
| counting | 116 | 0.87 | 0.97 | 0.90 | 0.92 | 60 | 38 | 0.88 | 0.29 |
| recognition | 116 | 0.96 | 0.97 | 0.96 | 0.97 | 85 | 15 | 0.88 | 0.29 |
| semantics | 116 | 0.89 | 0.95 | 0.94 | 0.94 | 72 | 27 | 0.88 | 0.29 |
| sorting | 116 | 0.94 | 0.95 | 0.97 | 0.96 | 86 | 14 | 0.88 | 0.29 |
| spatial | 116 | 0.92 | 0.98 | 0.89 | 0.95 | 80 | 17 | 0.88 | 0.29 |
| Overall | 812 | 0.91 | 0.96 | 0.92 | 0.95 | 76 | 23 | 0.88 | 0.29 |

**【绿色标识｜读表结论】**

- Overall alignment 0.91：语言目标和代码成功条件整体比较一致。
- Clarity 0.96：生成的自然语言大多清楚。
- Match 0.95：relation / target / object / quantifier 这类语义匹配总体较好。
- Object coverage 0.88：多数可操作对象能被任务覆盖到。
- Predicate coverage 0.29：谓词覆盖很低，说明生成器偏向使用少数可靠谓词，而不是探索所有谓词。

**【橙色标识｜风险点】**

color 类 alignment 较低，说明“颜色词 -> 具体对象实例”的 grounding 容易出错。spatial 类 feasibility 较低，说明精确空间摆放可能超出机器人实际容忍度。

## 六个 judge 维度怎么理解

DTGE 的 judge 不是只给一个“好/坏”，而是拆成更具体的维度：

| 维度 | 问的问题 | 例子 |
|---|---|---|
| relation match | 空间/逻辑关系是否一致 | 指令说“放进碗”，代码却检查“放到盘子上”就是错 |
| target match | 目标状态是否一致 | 指令要最终在 bowl，代码却把 bowl 写成 plate |
| object match | 被操作对象是否一致 | 指令是 apple，代码检查 banana |
| quantifier match | 数量词是否一致 | 指令说 all，代码只要求 any |
| instruction clarity | 指令是否清晰 | “put it there” 缺少对象和目标 |
| physical feasibility | 物理上是否可完成 | 大盒子塞进小碗不可行 |

可以把它记成：

```text
关系对不对 -> 目标对不对 -> 对象对不对 -> 数量对不对 -> 人话清不清楚 -> 物理能不能做
```

## 代码里它评的到底是什么

RoboLab 的 task class 结构大致是：

```python
@configclass
class BananaInBowlTerminations:
    success = DoneTerm(
        func=object_in_container,
        params={"object": "banana", "container": "bowl"},
    )

@dataclass
class BananaInBowlTask(Task):
    instruction = {
        "default": "Pick up the banana and place it in the bowl",
        "vague": "Put the fruit in the bowl",
        "specific": "Grasp the yellow banana and place it inside the bowl",
    }
    terminations = BananaInBowlTerminations
    attributes = ["semantics"]
    subtasks = [
        pick_and_place(object=["banana"], container="bowl", logical="all", score=1.0)
    ]
```

DTGE 关心的核心是：

| 代码字段 | DTGE 怎么用 |
|---|---|
| `instruction` | 抽出自然语言目标 |
| `terminations.success.func` | 抽出成功谓词，例如 `object_in_container` |
| `terminations.success.params` | 抽出对象、容器、reference、surface、logical 等参数 |
| `attributes` | 对应 color、sorting、spatial 等类别分析 |
| `subtasks` | 辅助理解任务结构，但 DTGE 的主判断是 instruction 和 success condition 是否一致 |

## 和精讲3的区别

精讲3讲的是“怎么生成任务”：给 LLM 对象目录、谓词库、模板、约束，让它写 task，然后做语法/资源/尺寸检查和失败修复。

精讲9讲的是“怎么评价生成出来的任务质量”：

| 精讲3：TaskGen | 精讲9：DTGE |
|---|---|
| 关注生成流程 | 关注评估流程 |
| 目标是产出 task code | 目标是判断 task code 是否可信 |
| 检查语法、资源、容器尺寸 | 检查语言和 success condition 是否对齐 |
| 失败后给 LLM 修复 prompt | 输出 alignment、clarity、feasibility、coverage |

两者关系是：

```text
TaskGen 负责“造题”
DTGE 负责“审题”
Policy benchmark 负责“做题”
```

## 为什么要有 DTGE

自动生成任务最大的风险是“看起来像任务，但其实评测目标错了”。

例如：

```text
指令：把苹果放进碗里
代码：object_in_container(object="banana", container="bowl")
```

这个 task 如果拿去评测 policy，就会出现严重污染：

- 模型按指令拿苹果，代码判失败。
- 模型误拿香蕉，代码可能判成功。
- 最后你以为是模型失败，其实是 benchmark 自己标错了。

DTGE 的作用就是在 policy rollout 前尽量发现这种问题。

## Object Coverage 和 Predicate Coverage

### Object Coverage

object coverage 问的是：

> 一个场景里的可操作对象，有多少至少出现在一个生成任务里？

如果一个厨房场景有 20 个可操作对象，但生成任务只反复使用苹果和碗，那么 object coverage 就低。  
论文里 object coverage 约 0.88，说明任务生成能覆盖大多数对象。

### Predicate Coverage

predicate coverage 问的是：

> 可用成功谓词里，有多少被生成任务真正使用过？

论文里 predicate coverage 约 0.29。这个数低不一定是坏事，它说明生成器比较保守，更偏向可靠谓词，例如 `object_in_container`、`object_on_top`、`object_left_of`、`stacked`，而不是把所有边缘谓词都用一遍。

**【橙色标识｜取舍】**

高 predicate coverage 代表任务多样，但可能降低可靠性；低 predicate coverage 代表更稳，但 benchmark 的行为覆盖面可能不够广。

## 复现 DTGE 需要准备什么

如果我们要按论文复现 DTGE，需要准备：

| 资产 | 用途 |
|---|---|
| 场景集合 | 论文里是多个场景，每个场景生成多类任务 |
| Category templates | color、conjunction、counting、recognition、semantics、sorting、spatial |
| taskgen prompt | 生成 Python task code |
| 静态分析脚本 | 从 task code 中抽 instruction 和 termination condition |
| judge prompt | 让 LLM 评分 relation/target/object/quantifier/clarity/feasibility |
| 汇总脚本 | 统计 alignment、match、coverage、verdict |

**【橙色标识｜复现边界】**

论文里使用 o1 做生成和 judge，并使用 temperature 0 以增强复现性。但 LLM-as-judge 仍然可能随模型版本、系统提示、解析规则变化而漂移。因此我们复现时应该保存 prompt、输入 task code、judge 原始 JSON、聚合脚本版本，而不是只保存最终表格。

## 和我们当前 4090 复现的关系

当前我们已经跑通 Pi05 单任务和复杂任务抽样。DTGE 是下一层质量控制：

1. 如果我们使用官方 RoboLab-120 任务，DTGE 主要作为理解论文任务质量的背景。
2. 如果我们自己用 LLM 扩展任务，DTGE 必须变成生成流程的一部分。
3. 如果要对比 RoboChallenge pi、OpenPI pi05、ReKep，必须先确认 task 本身没有 instruction-code mismatch，否则模型对比会被污染。

最实用的执行顺序是：

```text
任务代码静态校验 -> DTGE 审题 -> no-policy 环境初始化 -> policy rollout -> result analysis
```

这样能把失败分层：

| 失败阶段 | 说明 |
|---|---|
| 静态校验失败 | task 文件格式、对象名或 predicate 参数有问题 |
| DTGE 失败 | 指令和 success condition 不一致 |
| no-policy 初始化失败 | 资产、场景、物理或注册有问题 |
| policy rollout 失败 | 策略本身没有完成任务 |

## 小结

DTGE 的价值不在于给 policy 打分，而在于保护 benchmark 的可信度：

```text
没有 DTGE：模型失败和任务标错混在一起
有 DTGE：先确认题目靠谱，再评价模型会不会做
```

对我们的复现来说，DTGE 是后续扩展任务、做模型对比、跑 RoboLab-120 子集时必须保留的一道质量门。

In [ ]:
# ===== 精讲9：DTGE 轻量验证 =====
# 这个 cell 模拟论文 Appendix D 的核心口径：
# 1. 从生成的 task Python 代码中静态抽取 instruction 和 terminations.success。
# 2. 用简化规则模拟 LLM-as-judge 的 6 个维度。
# 3. 统计 object coverage / predicate coverage。
# 边界：这里不是论文使用的 o1 judge，也不是官方评测脚本；它只验证 DTGE 的数据流和判定口径。

import ast
import json
import re
from collections import Counter

def ast_literal(node):
    # 安全地把 AST 常量、列表、字典还原成 Python 值；遇到函数名则返回名字字符串。
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.List):
        return [ast_literal(item) for item in node.elts]
    if isinstance(node, ast.Tuple):
        return [ast_literal(item) for item in node.elts]
    if isinstance(node, ast.Dict):
        return {ast_literal(k): ast_literal(v) for k, v in zip(node.keys, node.values)}
    if isinstance(node, ast.Name):
        return node.id
    if isinstance(node, ast.Attribute):
        return node.attr
    return None

def extract_task_features(source):
    # DTGE 的“静态抽取”：不跑 Isaac，不跑 policy，只读 task code 的结构。
    tree = ast.parse(source)
    features = {
        "task_class": None,
        "instruction": "",
        "attributes": [],
        "success_func": None,
        "success_params": {},
    }
    for node in tree.body:
        if not isinstance(node, ast.ClassDef):
            continue
        if node.name.endswith("Task"):
            features["task_class"] = node.name
            for stmt in node.body:
                target_names = []
                if isinstance(stmt, ast.Assign):
                    target_names = [target.id for target in stmt.targets if isinstance(target, ast.Name)]
                elif isinstance(stmt, ast.AnnAssign) and isinstance(stmt.target, ast.Name):
                    target_names = [stmt.target.id]
                if "instruction" in target_names:
                    value = ast_literal(stmt.value)
                    if isinstance(value, dict):
                        features["instruction"] = value.get("default", next(iter(value.values())))
                    else:
                        features["instruction"] = value
                if "attributes" in target_names:
                    features["attributes"] = ast_literal(stmt.value) or []
        if node.name.endswith("Terminations"):
            for stmt in node.body:
                if not isinstance(stmt, ast.Assign):
                    continue
                targets = [target.id for target in stmt.targets if isinstance(target, ast.Name)]
                if "success" not in targets or not isinstance(stmt.value, ast.Call):
                    continue
                for keyword in stmt.value.keywords:
                    if keyword.arg == "func":
                        features["success_func"] = ast_literal(keyword.value)
                    if keyword.arg == "params":
                        features["success_params"] = ast_literal(keyword.value) or {}
    return features

def flatten_values(value):
    # 把 params 中的 object/container/surface/reference 等值拍平，便于比较指令里出现的对象名。
    if value is None:
        return []
    if isinstance(value, str):
        return [value]
    if isinstance(value, (list, tuple, set)):
        out = []
        for item in value:
            out.extend(flatten_values(item))
        return out
    if isinstance(value, dict):
        out = []
        for item in value.values():
            out.extend(flatten_values(item))
        return out
    return []

def infer_relation_from_instruction(text):
    # 简化版 relation parser：真实论文用 LLM judge，这里只用关键词验证数据流。
    lower = text.lower()
    if "left of" in lower or "to the left" in lower:
        return "object_left_of"
    if "right of" in lower or "to the right" in lower:
        return "object_right_of"
    if "stack" in lower:
        return "stacked"
    if "sort" in lower:
        return "object_groups_in_containers"
    if " on " in f" {lower} " or "onto" in lower:
        return "object_on_top"
    if " in " in f" {lower} " or "inside" in lower or "into" in lower:
        return "object_in_container"
    return None

def infer_quantifier_from_instruction(text):
    lower = text.lower()
    if re.search(r"\ball\b", lower):
        return "all"
    if re.search(r"\bany\b", lower):
        return "any"
    if re.search(r"\b(two|three|four|2|3|4)\b", lower):
        return "count"
    return "single"

def judge_task_alignment(features, scene_objects, object_sizes=None, container_sizes=None):
    # 简化 judge：输出和论文对应的 relation/target/object/quantifier/clarity/feasibility 维度。
    instruction = features["instruction"] or ""
    lower = instruction.lower()
    expected_relation = infer_relation_from_instruction(instruction)
    actual_func = features["success_func"]
    params = features["success_params"]
    referenced_in_code = set(flatten_values(params))
    mentioned_in_text = {obj for obj in scene_objects if obj.replace("_", " ") in lower or obj in lower}

    relation_match = 1.0 if expected_relation == actual_func else 0.0
    if expected_relation is None and actual_func is None:
        relation_match = 0.0

    target_keys = ["container", "reference_object", "surface"]
    target_values = {params.get(key) for key in target_keys if params.get(key)}
    target_match = 1.0 if not target_values or target_values.issubset(mentioned_in_text) else 0.0

    object_match = 1.0
    if mentioned_in_text:
        object_match = len(mentioned_in_text & referenced_in_code) / len(mentioned_in_text)

    expected_quantifier = infer_quantifier_from_instruction(instruction)
    actual_logical = params.get("logical", "single")
    if expected_quantifier == "single":
        quantifier_match = 1.0 if actual_logical in {"single", "all"} else 0.5
    elif expected_quantifier == "count":
        quantifier_match = 1.0 if "K" in params or "count" in params else 0.5
    else:
        quantifier_match = 1.0 if actual_logical == expected_quantifier else 0.0

    clarity = 1.0
    if len(instruction.split()) < 4 or instruction.lower() in {"put it there", "move it"}:
        clarity = 0.3
    if not mentioned_in_text:
        clarity = min(clarity, 0.5)

    feasibility = 1.0
    if object_sizes and container_sizes and actual_func == "object_in_container":
        obj = params.get("object")
        container = params.get("container")
        if isinstance(obj, list):
            obj = obj[0] if obj else None
        if obj in object_sizes and container in container_sizes:
            feasibility = 1.0 if object_sizes[obj] <= container_sizes[container] else 0.0

    semantic_match = (relation_match + target_match + object_match + quantifier_match) / 4.0
    alignment = (
        0.20 * relation_match
        + 0.20 * target_match
        + 0.20 * object_match
        + 0.15 * quantifier_match
        + 0.10 * clarity
        + 0.15 * feasibility
    )
    if min(relation_match, target_match, object_match, quantifier_match, clarity, feasibility) >= 0.99:
        verdict = "aligned"
    elif semantic_match >= 0.5 and clarity >= 0.5 and feasibility >= 0.5:
        verdict = "partial"
    else:
        verdict = "misaligned"

    return {
        "relation_match": relation_match,
        "target_match": target_match,
        "object_match": object_match,
        "quantifier_match": quantifier_match,
        "clarity": clarity,
        "feasibility": feasibility,
        "semantic_match": semantic_match,
        "alignment": alignment,
        "verdict": verdict,
        "mentioned_in_text": sorted(mentioned_in_text),
        "referenced_in_code": sorted(referenced_in_code),
        "expected_relation": expected_relation,
        "actual_func": actual_func,
    }

def coverage_metrics(task_features, scene_objects, available_predicates):
    # DTGE 的 coverage：对象是否被任务覆盖，谓词库是否被充分使用。
    used_objects = set()
    used_predicates = set()
    for features in task_features:
        used_objects.update(obj for obj in flatten_values(features["success_params"]) if obj in scene_objects)
        if features["success_func"]:
            used_predicates.add(features["success_func"])
    return {
        "object_coverage": len(used_objects) / len(scene_objects),
        "predicate_coverage": len(used_predicates) / len(available_predicates),
        "used_objects": sorted(used_objects),
        "used_predicates": sorted(used_predicates),
    }

aligned_task_code = (
    "class AppleInBowlTerminations:\n"
    "    success = DoneTerm(func=object_in_container, params={\"object\": \"apple\", \"container\": \"bowl\", \"logical\": \"all\"})\n"
    "\n"
    "class AppleInBowlTask(Task):\n"
    "    instruction = {\"default\": \"Place the apple in the bowl\"}\n"
    "    attributes = [\"semantics\"]\n"
)

mismatched_task_code = (
    "class AppleInBowlTerminations:\n"
    "    success = DoneTerm(func=object_in_container, params={\"object\": \"banana\", \"container\": \"bowl\", \"logical\": \"all\"})\n"
    "\n"
    "class AppleInBowlTask(Task):\n"
    "    instruction = {\"default\": \"Place the apple in the bowl\"}\n"
    "    attributes = [\"semantics\"]\n"
)

infeasible_task_code = (
    "class BoxInSmallBowlTerminations:\n"
    "    success = DoneTerm(func=object_in_container, params={\"object\": \"large_box\", \"container\": \"small_bowl\", \"logical\": \"all\"})\n"
    "\n"
    "class BoxInSmallBowlTask(Task):\n"
    "    instruction = {\"default\": \"Place the large_box in the small_bowl\"}\n"
    "    attributes = [\"spatial\"]\n"
)

scene_objects = ["apple", "banana", "bowl", "plate", "large_box", "small_bowl"]
available_predicates = [
    "object_in_container",
    "object_on_top",
    "object_left_of",
    "object_right_of",
    "stacked",
    "object_groups_in_containers",
    "object_upright",
]
object_sizes = {"apple": 0.07, "banana": 0.12, "large_box": 0.30}
container_sizes = {"bowl": 0.18, "small_bowl": 0.12}

extracted = [extract_task_features(code) for code in [aligned_task_code, mismatched_task_code, infeasible_task_code]]
judged = [
    judge_task_alignment(features, scene_objects, object_sizes=object_sizes, container_sizes=container_sizes)
    for features in extracted
]
coverage = coverage_metrics(extracted, scene_objects, available_predicates)

dtge_table = {
    "color": {"count": 116, "alignment": 0.81, "clarity": 0.94, "feasibility": 0.80, "match": 0.90, "aligned_pct": 57, "partial_pct": 40, "object_coverage": 0.88, "predicate_coverage": 0.29},
    "conjunction": {"count": 116, "alignment": 0.97, "clarity": 0.98, "feasibility": 1.00, "match": 0.98, "aligned_pct": 91, "partial_pct": 9, "object_coverage": 0.88, "predicate_coverage": 0.29},
    "counting": {"count": 116, "alignment": 0.87, "clarity": 0.97, "feasibility": 0.90, "match": 0.92, "aligned_pct": 60, "partial_pct": 38, "object_coverage": 0.88, "predicate_coverage": 0.29},
    "recognition": {"count": 116, "alignment": 0.96, "clarity": 0.97, "feasibility": 0.96, "match": 0.97, "aligned_pct": 85, "partial_pct": 15, "object_coverage": 0.88, "predicate_coverage": 0.29},
    "semantics": {"count": 116, "alignment": 0.89, "clarity": 0.95, "feasibility": 0.94, "match": 0.94, "aligned_pct": 72, "partial_pct": 27, "object_coverage": 0.88, "predicate_coverage": 0.29},
    "sorting": {"count": 116, "alignment": 0.94, "clarity": 0.95, "feasibility": 0.97, "match": 0.96, "aligned_pct": 86, "partial_pct": 14, "object_coverage": 0.88, "predicate_coverage": 0.29},
    "spatial": {"count": 116, "alignment": 0.92, "clarity": 0.98, "feasibility": 0.89, "match": 0.95, "aligned_pct": 80, "partial_pct": 17, "object_coverage": 0.88, "predicate_coverage": 0.29},
    "overall": {"count": 812, "alignment": 0.91, "clarity": 0.96, "feasibility": 0.92, "match": 0.95, "aligned_pct": 76, "partial_pct": 23, "object_coverage": 0.88, "predicate_coverage": 0.29},
}

dtge_tests = [
    ("static_extraction_gets_instruction", extracted[0]["instruction"] == "Place the apple in the bowl"),
    ("static_extraction_gets_success_predicate", extracted[0]["success_func"] == "object_in_container"),
    ("aligned_task_verdict_aligned", judged[0]["verdict"] == "aligned" and judged[0]["alignment"] > 0.95),
    ("object_mismatch_detected", judged[1]["object_match"] < 1.0 and judged[1]["verdict"] != "aligned"),
    ("physical_infeasibility_detected", judged[2]["feasibility"] == 0.0 and judged[2]["verdict"] != "aligned"),
    ("coverage_metrics_in_range", 0.0 <= coverage["object_coverage"] <= 1.0 and 0.0 <= coverage["predicate_coverage"] <= 1.0),
    ("predicate_coverage_is_conservative", coverage["predicate_coverage"] < 0.5),
    ("paper_table_total_is_812", sum(v["count"] for k, v in dtge_table.items() if k != "overall") == dtge_table["overall"]["count"]),
    ("paper_overall_alignment_matches_appendix_d", dtge_table["overall"]["alignment"] == 0.91),
]

print("Extracted features:")
print(json.dumps(extracted, ensure_ascii=False, indent=2))
print("Judged examples:")
print(json.dumps(judged, ensure_ascii=False, indent=2))
print("Coverage:")
print(json.dumps(coverage, ensure_ascii=False, indent=2))
for name, ok in dtge_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert all(ok for _, ok in dtge_tests), dtge_tests
write_status(
    "dtge_lightweight_tests",
    {
        "all_passed": all(ok for _, ok in dtge_tests),
        "extracted": extracted,
        "judged": judged,
        "coverage": coverage,
        "paper_table": dtge_table,
        "tests": [{"name": name, "passed": bool(ok)} for name, ok in dtge_tests],
        "boundary": "This simulates DTGE static extraction and judge dimensions; real reproduction requires the paper's exact LLM prompts/model settings and generated task corpus.",
    },
)

## 0.16 论文精讲：Scene Generation Prompt 设计

下面这节来自本目录的 [EXPLAIN_10_prompt_design.md](./EXPLAIN_10_prompt_design.md)，对应论文 Appendix C 的 Stage I Semantic Planning prompt：为什么要写现实场景原则、坐标系统、placement types、JSON only、对象目录、尺寸限制和 medium scene strategy，以及这些约束如何对接 `predicates.py`、`spatial_solver.py`、`physical_solver.py` 和 `feedback_system.py`。

# 精讲 10：Scene Generation Prompt 设计分析

> **【绿色标识｜核心结论】** 这几段 prompt 不是为了“写得像提示词工程”，而是在把 LLM 输出压成 RoboLab 可解析、可求解、可验证的中间表示：对象列表 + typed predicates。  
> **【蓝色标识｜源码路径】** 输出会对接 `robolab/scene_gen/llm_scene_gen/predicates.py`、`spatial_solver.py`、`physical_solver.py` 和 `feedback_system.py`。  
> **【橙色标识｜容易误解】** prompt 里让 LLM 写坐标，但不是让 LLM 完成全部几何/物理求解。LLM 负责语义规划和初始关系，solver 负责碰撞、边界、支撑和容器内放置。

## 先说结论

这三张图可以理解成一个三层 prompt 合约：

| 层 | 作用 | 解决的问题 |
|---|---|---|
| System Prompt | 定义“现实机器人桌面场景”的先验和谓词语言 | 防止 LLM 输出网格、纯坐标、不可解析的故事 |
| Output Format + Critical Rules | 定义 JSON schema 和硬规则 | 防止输出 markdown、对象名幻觉、依赖顺序错误 |
| User Prompt Template | 注入当前主题、目标对象数、桌面尺寸、对象目录和策略 | 防止场景和资产目录/桌面容量不匹配 |

一句话：

> System prompt 负责“世界观和语法”，Output prompt 负责“接口契约”，User prompt 负责“本次实例的动态约束”。

## 为什么不让 LLM 直接生成 USD

如果让 LLM 直接写 `.usda`，会遇到几个问题：

| 问题 | 后果 |
|---|---|
| USD 语法细节复杂 | LLM 容易写出不可加载文件 |
| 资产路径必须精确 | 对象名、payload path、prim path 任何一个错都会失败 |
| 物理几何很难靠文本一次写准 | 容易碰撞、悬空、穿模、出界 |
| 关系难以检查 | “苹果在碗里”如果只是一串 transform，很难知道语义是否满足 |

所以 RoboLab 采用分层：

```text
LLM 生成 typed predicates
-> predicate parser 转成 ObjectState/Predicate
-> spatial solver 解 2D 桌面布局
-> physical solver 解 place-in/place-on
-> feedback system 把错误反馈给 LLM 修复
-> 最后再生成 USD scene
```

这就是为什么 prompt 强调 `place-on-base`、`place-in`、`place-on`、`cluster-around`，而不是直接要求 LLM 输出完整 USD。

## 第一张图：System Prompt 为什么这么写

### 1. “scene generation expert”

开头把角色限定为 scene generation expert，目的是让模型优先考虑“生成可用场景”，而不是写故事、写任务说明或写自然语言描述。

它的隐含要求是：

- 输出要服务于机器人 manipulation。
- 场景要能被后续 solver/Isaac 使用。
- 重点是 realistic 和 physically plausible。

### 2. Real-world scene principles

这一段最重要的是反 baseline：

| Prompt 规则 | 为什么写 |
|---|---|
| Objects form clusters, not grids | 避免退化成均匀网格布局；论文 baseline 正是 grid+jitter 思路 |
| Containers have objects inside | 用 3D 容器关系增加语义和空间密度 |
| Supports have objects on top | 让场景有支撑关系，而不是所有物体都摊在桌上 |
| Objects scatter around containers | 模拟真实桌面上的局部聚集 |
| Orientations vary | 避免所有对象 0/90 度，看起来像程序摆拍 |

**【绿色标识｜核心直觉】**

真实桌面不是棋盘。人类会把相关对象聚在一起，碗里有水果，盘子上有食物，工具在容器附近。这个 prompt 是在给 LLM 注入“非网格、有关联、可操作”的场景先验。

### 3. Coordinate system

这段非常工程化：

```text
Table bounds: X=[0.25 to 0.85], Y=[-0.40 to 0.40]
Table center: (0.55, 0.0)
Front=+X, Back=-X, Left=+Y, Right=-Y
```

它解决两个问题：

| 问题 | 为什么必须写清楚 |
|---|---|
| 坐标边界 | 防止 LLM 把对象放出桌子 |
| 左右前后定义 | 防止 LLM 的自然语言方向和 solver 的方向相反 |

源码里的 `spatial_solver.py` 也按这个方向解释相对位置：Front 是 `+X`，Left 是 `+Y`，Right 是 `-Y`。

### 4. Placement types

这其实是给 LLM 的小型 DSL：

| Predicate | 语义 | 后续谁处理 |
|---|---|---|
| `place-on-base` | 对象直接放在桌面上，通常是 anchor 或 loose object | `SpatialSolver` |
| `place-in` | 对象在容器里 | `PhysicalSolver` |
| `place-on` | 对象在支撑物上 | `PhysicalSolver` |
| `cluster-around` | 多个对象围绕 anchor 分布 | Stage I 语义规划 + 空间布局 |

为什么每个类型都配例子？因为 LLM 对“抽象规范”容易跑偏，给 JSON 样例能显著降低格式错误。

### 5. Scene structure

```text
1. Place 1-2 anchor objects
2. Put objects inside containers
3. Put objects on supports
4. Cluster objects around anchors
5. Add loose objects
```

这是一条依赖顺序：

```text
anchor 先存在 -> 才能 place-in / place-on / cluster-around
```

如果 LLM 先写 `place-in apple -> bowl`，但没有先把 `bowl` 放到桌上，solver 就缺 anchor。  
所以 prompt 明确要求 containers/supports must be placed first。

### 6. Realistic spacing

spacing 不是装饰，它直接影响 solver 成功率：

| 间距规则 | 作用 |
|---|---|
| Anchors 0.25-0.35m apart | 防止两个大容器/支撑物互相碰撞 |
| Clustered 0.08-0.15m from anchor | 既像真实聚集，又不完全重叠 |
| Loose objects fill space naturally | 避免所有对象堆在一个点 |

## 第二张图：Output Format + Critical Rules

### JSON only

这条最硬：

```text
OUTPUT FORMAT (JSON only, no markdown)
Return ONLY valid JSON, no markdown
```

原因很简单：后面的程序要 `json.loads()`。  
如果模型输出：

```text
Here is the JSON:
```json
...
```
```

就需要额外清洗，自动管线不稳定。

### objects 数组

```json
"objects": [{"name": "bowl_0"}, {"name": "plate_large"}]
```

这一步的作用是把“本场景要用哪些资产”显式列出来。后续会拿这些名字去对象目录里查 `usd_path`、尺寸、类别。

**【橙色标识｜关键风险】**

对象名必须和 catalog 精确匹配。`bowl`、`bowl_0`、`red_bowl` 可能是三个完全不同的名字；名字错了不是“语义相近”，而是资产加载失败。

### predicates 数组

`predicates` 是场景结构的核心。它不是描述文本，而是 solver 可执行的命令：

```json
{"type": "place-on-base", "object": "bowl_0", "x": 0.40, "y": 0.15, "yaw": 23}
{"type": "place-in", "objects": ["apple_01", "orange_01"], "container": "bowl_0"}
{"type": "place-on", "object": "banana", "support": "plate_large", "position": "center"}
{"type": "cluster-around", "objects": ["mug", "spoon"], "anchor": "bowl_0", "radius": 0.12}
```

这就是论文里 “semantic planning” 的中间层：LLM 不直接做物理，而是选择结构关系。

### Critical rules

| Rule | 为什么必要 |
|---|---|
| Object names must match catalog | 避免资产幻觉 |
| Containers/supports before children | 满足依赖关系 |
| containment + stacking + clustering | 生成比 grid baseline 更丰富的场景 |
| vary yaw | 避免机械网格感 |
| JSON only | 保证可机器解析 |

## 第三张图：User Prompt Template

这张图是运行时注入的动态约束。

### Scene request / target object count

```text
SCENE REQUEST: theme from dataset
TARGET: target object count objects
```

这告诉 LLM：

- 当前主题是什么，比如 kitchen cabinet、office desk、workshop bench。
- 大概要多少对象，不是越多越好。

### Table size

```text
TABLE SIZE: 0.7m x 1.0m = 0.70m2
```

这是给模型一个物理容量感。  
没有这句，LLM 很容易在一张小桌上塞 30 个大物体。

### Size limits

```text
max 1-2 large objects
prefer smaller for 8+ items
```

这条是为了提高 solver 成功率。大物体多了，碰撞和出界会急剧增加。  
如果目标对象数是 10-14，中小物体 + 容器/支撑关系比全是大物体更可行。

### Available objects by category

```text
CONTAINERS
SUPPORTS
FOOD
TOOLS
OTHER
```

这不是为了好看，而是给 LLM 做选择空间裁剪：

| 分类 | 作用 |
|---|---|
| CONTAINERS | 可用于 `place-in` |
| SUPPORTS | 可用于 `place-on` |
| FOOD | 常作为被操作对象 |
| TOOLS | 可作为桌面工具/cluster 对象 |
| OTHER | 补充主题多样性 |

如果只给一个长对象列表，LLM 很难知道哪个能装东西、哪个能当支撑。

### Medium scene strategy

```text
10-14 objects:
- Use 1-2 anchors
- Put 2-4 objects in containers
- Stack 1-2 items on supports
- Cluster 2-3 objects near anchors
- Vary yaw
```

这相当于把目标对象数转成布局配方。  
对 10-14 个对象，最容易失败的是桌面拥挤，所以 prompt 引导 LLM 使用：

- 容器：把多个物体放进一个 footprint。
- 支撑：利用竖直方向。
- cluster：让局部关系自然，但不要求每个对象都有精确坐标。

## 为什么这些 prompt 能提升场景质量

可以从论文 baseline 对比理解：

| Baseline grid+jitter | Predicate prompt 方法 |
|---|---|
| 均匀网格 | 局部聚集 |
| 所有对象基本在桌面上 | 有容器/支撑/围绕关系 |
| 一次 pass | 失败后有反馈修复 |
| 难表达语义结构 | typed predicates 可验证 |
| 容易“分散但无意义” | 更像真实可操作桌面 |

论文 Appendix C 的实验也显示，主方法在 VQA、realism、functionality、layout、completeness、quality 和 GPT preference 上整体优于 baseline。

## Prompt 和代码的对应关系

| Prompt 片段 | 代码落点 |
|---|---|
| `place-on-base`、`place-in`、`place-on` | `predicates.py` 中的 predicate class / enum |
| 坐标边界和 left/right/front/back | `spatial_solver.py` 的相对位置和边界逻辑 |
| 容器/支撑必须先放 | `PhysicalSolver` 需要已有 support/container anchor |
| grammar / solver / physics failure feedback | `feedback_system.py` |
| object catalog exact name | `assets/objects/object_catalog.json` 和 scene generation skill |
| size limits | prompt 阶段减少不可能布局，solver 阶段继续验证 |

## 几个用例如何判断好坏

### 用例 1：好的 medium kitchen prompt 输出

特征：

- JSON 可解析。
- 对象都来自 catalog。
- 1-2 个 anchor 先 `place-on-base`。
- 有 `place-in`，也有 `place-on` 或 `cluster-around`。
- yaw 不是全 0/90/180。
- 坐标在桌面范围内。
- 大物体不超过 1-2 个。

### 用例 2：对象名幻觉

坏输出：

```json
{"name": "beautiful_red_ceramic_bowl"}
```

如果 catalog 里没有这个精确名字，后续无法加载 USD。  
所以 prompt 用 “MUST match EXACTLY from catalog” 约束。

### 用例 3：依赖顺序错误

坏输出：

```json
{"type": "place-in", "objects": ["apple_01"], "container": "bowl_0"}
```

但没有先：

```json
{"type": "place-on-base", "object": "bowl_0", ...}
```

这会让容器没有基础位置。prompt 要求 containers/supports placed first，就是为了避免这个问题。

### 用例 4：网格化和 yaw 单调

坏输出：

- 物体均匀排成 3x4 网格。
- yaw 全是 0、90、180。

这类输出可能可解析，但不像真实桌面，也接近 baseline。  
所以 prompt 强调 clusters、around anchors、vary yaw。

### 用例 5：大物体过多

坏输出：

- 10 个对象里选了 5 个 footprint 很大的箱子/托盘。

即使 JSON 格式正确，也很可能 solver 碰撞或超出桌面。  
所以 user prompt 里动态加入 size warnings 和 “max 1-2 large objects”。

## 怎么把 prompt 写得更稳

如果后续要自己扩展 prompt，我建议保留这几个硬约束：

1. 永远让输出是 JSON only。
2. 永远给可用 object catalog 子集，不要让模型自由编对象名。
3. 永远把 coordinate frame 写清楚，尤其 left/right/front/back。
4. 永远把 dependency order 写清楚：anchor -> in/on -> cluster -> loose。
5. 永远把 table size 和 large object warning 写进去。
6. 对不同 target count 使用不同策略块：sparse、medium、dense。
7. 失败反馈要具体指出是 grammar、solver、physics 还是 intersection failure。

## 小结

这几段 prompt 的核心不是“说服 LLM 生成好看的文字”，而是把 LLM 变成 scene planner：

```text
主题 + 对象目录 + 桌面容量
-> 选择对象
-> 组织成容器/支撑/聚类关系
-> 输出 typed predicate JSON
-> solver/physics/feedback 接管
```

Prompt 写成这样，是为了让生成结果既有真实桌面结构，又能被程序稳定解析、求解和修复。

In [ ]:
# ===== 精讲10：Scene generation prompt 轻量验证 =====
# 这个 cell 不调用 LLM，而是模拟 prompt 期望得到的 JSON 输出，
# 并检查几类常见错误：对象名幻觉、依赖顺序错误、网格化 yaw、大物体过多、非 JSON 输出。

import json
import math
from collections import Counter

catalog = {
    "bowl_0": {"category": "container", "footprint": 0.045, "can_contain": True, "can_support": False},
    "plate_large": {"category": "support", "footprint": 0.060, "can_contain": False, "can_support": True},
    "apple_01": {"category": "food", "footprint": 0.010, "can_contain": False, "can_support": False},
    "orange_01": {"category": "food", "footprint": 0.012, "can_contain": False, "can_support": False},
    "banana": {"category": "food", "footprint": 0.018, "can_contain": False, "can_support": False},
    "mug": {"category": "other", "footprint": 0.015, "can_contain": True, "can_support": False},
    "spoon": {"category": "tool", "footprint": 0.006, "can_contain": False, "can_support": False},
    "tray": {"category": "support", "footprint": 0.095, "can_contain": False, "can_support": True},
    "storage_bin": {"category": "container", "footprint": 0.110, "can_contain": True, "can_support": False},
    "large_box": {"category": "other", "footprint": 0.120, "can_contain": True, "can_support": True},
}

good_output = {
    "objects": [
        {"name": "bowl_0"},
        {"name": "plate_large"},
        {"name": "apple_01"},
        {"name": "orange_01"},
        {"name": "banana"},
        {"name": "mug"},
        {"name": "spoon"},
    ],
    "predicates": [
        {"type": "place-on-base", "object": "bowl_0", "x": 0.40, "y": 0.15, "yaw": 23},
        {"type": "place-on-base", "object": "plate_large", "x": 0.65, "y": -0.10, "yaw": 156},
        {"type": "place-in", "objects": ["apple_01", "orange_01"], "container": "bowl_0"},
        {"type": "place-on", "object": "banana", "support": "plate_large", "position": "center"},
        {"type": "cluster-around", "objects": ["mug", "spoon"], "anchor": "bowl_0", "radius": 0.12},
    ],
}

hallucinated_object_output = {
    "objects": [{"name": "beautiful_red_ceramic_bowl"}, {"name": "apple_01"}],
    "predicates": [
        {"type": "place-on-base", "object": "beautiful_red_ceramic_bowl", "x": 0.55, "y": 0.0, "yaw": 30},
        {"type": "place-in", "objects": ["apple_01"], "container": "beautiful_red_ceramic_bowl"},
    ],
}

dependency_error_output = {
    "objects": [{"name": "bowl_0"}, {"name": "apple_01"}],
    "predicates": [
        {"type": "place-in", "objects": ["apple_01"], "container": "bowl_0"},
    ],
}

grid_like_output = {
    "objects": [{"name": name} for name in ["bowl_0", "plate_large", "apple_01", "orange_01"]],
    "predicates": [
        {"type": "place-on-base", "object": "bowl_0", "x": 0.35, "y": -0.20, "yaw": 0},
        {"type": "place-on-base", "object": "plate_large", "x": 0.55, "y": -0.20, "yaw": 90},
        {"type": "place-on-base", "object": "apple_01", "x": 0.35, "y": 0.00, "yaw": 180},
        {"type": "place-on-base", "object": "orange_01", "x": 0.55, "y": 0.00, "yaw": 270},
    ],
}

too_many_large_objects_output = {
    "objects": [{"name": name} for name in ["tray", "storage_bin", "large_box", "plate_large", "bowl_0"]],
    "predicates": [
        {"type": "place-on-base", "object": "tray", "x": 0.35, "y": -0.25, "yaw": 19},
        {"type": "place-on-base", "object": "storage_bin", "x": 0.50, "y": 0.00, "yaw": 71},
        {"type": "place-on-base", "object": "large_box", "x": 0.70, "y": 0.22, "yaw": 133},
        {"type": "place-on-base", "object": "plate_large", "x": 0.65, "y": -0.18, "yaw": 44},
        {"type": "place-on-base", "object": "bowl_0", "x": 0.43, "y": 0.20, "yaw": 28},
    ],
}

markdown_wrapped_output = "Here is the JSON:\\n```json\\n" + json.dumps(good_output) + "\\n```"

def parse_json_only(candidate):
    # Prompt 要求 JSON only：字符串必须直接是 JSON 对象，不能有 markdown 包裹。
    if isinstance(candidate, dict):
        return candidate, []
    if not isinstance(candidate, str):
        return None, ["output is neither dict nor JSON string"]
    stripped = candidate.strip()
    if not stripped.startswith("{") or not stripped.endswith("}"):
        return None, ["output is not JSON-only; likely contains prose or markdown"]
    try:
        return json.loads(stripped), []
    except json.JSONDecodeError as exc:
        return None, [f"invalid JSON: {exc}"]

def validate_scene_prompt_output(candidate, catalog, table_bounds=(0.25, 0.85, -0.40, 0.40)):
    parsed, issues = parse_json_only(candidate)
    if parsed is None:
        return {"valid": False, "issues": issues}

    names = [obj.get("name") for obj in parsed.get("objects", [])]
    name_set = set(names)
    predicates = parsed.get("predicates", [])
    x_min, x_max, y_min, y_max = table_bounds

    if not names:
        issues.append("objects list is empty")
    if not isinstance(predicates, list) or not predicates:
        issues.append("predicates list is empty")

    missing = sorted(name for name in name_set if name not in catalog)
    if missing:
        issues.append(f"catalog name mismatch: {missing}")

    placed_base = set()
    yaw_values = []
    predicate_types = Counter()
    for pred in predicates:
        pred_type = pred.get("type")
        predicate_types[pred_type] += 1
        if pred_type == "place-on-base":
            obj = pred.get("object")
            if obj not in name_set:
                issues.append(f"predicate references object not in objects list: {obj}")
            x = pred.get("x")
            y = pred.get("y")
            yaw = pred.get("yaw")
            if not isinstance(x, (int, float)) or not isinstance(y, (int, float)):
                issues.append(f"place-on-base missing numeric x/y for {obj}")
            elif not (x_min <= x <= x_max and y_min <= y <= y_max):
                issues.append(f"object out of table bounds: {obj}")
            if isinstance(yaw, (int, float)):
                yaw_values.append(float(yaw) % 360)
            else:
                issues.append(f"place-on-base missing numeric yaw for {obj}")
            placed_base.add(obj)
        elif pred_type == "place-in":
            container = pred.get("container")
            if container not in placed_base:
                issues.append(f"container/support must be placed first: {container}")
            if container in catalog and not catalog[container].get("can_contain"):
                issues.append(f"object is not a container: {container}")
            for obj in pred.get("objects", []):
                if obj not in name_set:
                    issues.append(f"place-in references missing object: {obj}")
        elif pred_type == "place-on":
            support = pred.get("support")
            if support not in placed_base:
                issues.append(f"container/support must be placed first: {support}")
            if support in catalog and not catalog[support].get("can_support"):
                issues.append(f"object is not a support: {support}")
            if pred.get("object") not in name_set:
                issues.append(f"place-on references missing object: {pred.get('object')}")
        elif pred_type == "cluster-around":
            anchor = pred.get("anchor")
            radius = pred.get("radius")
            if anchor not in placed_base:
                issues.append(f"cluster anchor must be placed first: {anchor}")
            if not isinstance(radius, (int, float)) or not (0.08 <= radius <= 0.20):
                issues.append(f"cluster radius outside prompt range: {radius}")
            for obj in pred.get("objects", []):
                if obj not in name_set:
                    issues.append(f"cluster-around references missing object: {obj}")
        else:
            issues.append(f"unknown predicate type: {pred_type}")

    large_objects = sorted(name for name in name_set if name in catalog and catalog[name]["footprint"] > 0.08)
    if len(large_objects) > 2:
        issues.append(f"too many large objects for medium scene: {large_objects}")

    if yaw_values:
        cardinal_count = sum(1 for yaw in yaw_values if min(abs(yaw - c) for c in [0, 90, 180, 270, 360]) < 1e-6)
        if cardinal_count == len(yaw_values):
            issues.append("all yaw angles are cardinal grid angles")

    has_semantic_structure = any(t in predicate_types for t in ["place-in", "place-on", "cluster-around"])
    if len(names) >= 5 and not has_semantic_structure:
        issues.append("medium scene lacks containment/support/clustering structure")

    return {
        "valid": len(issues) == 0,
        "issues": issues,
        "object_count": len(names),
        "predicate_types": dict(predicate_types),
        "large_objects": large_objects,
        "yaw_values": yaw_values,
    }

def render_medium_user_prompt(theme, target_count, catalog, suggested):
    # 模拟第三张图里的 runtime user prompt：把 catalog 子集和策略注入给 LLM。
    by_category = {}
    for name, meta in catalog.items():
        by_category.setdefault(meta["category"], []).append(name)
    large = [name for name, meta in catalog.items() if meta["footprint"] > 0.08]
    return {
        "scene_request": theme,
        "target_count": target_count,
        "table_size": "0.7m x 1.0m = 0.70m^2",
        "size_limits": {
            "large_objects": large,
            "rule": "max 1-2 large objects; prefer smaller for 8+ items",
        },
        "available_objects": {
            "containers": sorted([name for name, meta in catalog.items() if meta["can_contain"]]),
            "supports": sorted([name for name, meta in catalog.items() if meta["can_support"]]),
            "food": sorted(by_category.get("food", [])),
            "tools": sorted(by_category.get("tool", [])),
            "other": sorted(by_category.get("other", [])),
        },
        "medium_scene_strategy": [
            "Use 1-2 containers/supports as anchors",
            "Put 2-4 objects in containers",
            "Stack 1-2 items on supports",
            "Cluster 2-3 objects near anchors",
            "Vary yaw angles",
        ],
        "suggested_for_diversity": suggested,
    }

validations = {
    "good_output": validate_scene_prompt_output(good_output, catalog),
    "hallucinated_object_output": validate_scene_prompt_output(hallucinated_object_output, catalog),
    "dependency_error_output": validate_scene_prompt_output(dependency_error_output, catalog),
    "grid_like_output": validate_scene_prompt_output(grid_like_output, catalog),
    "too_many_large_objects_output": validate_scene_prompt_output(too_many_large_objects_output, catalog),
    "markdown_wrapped_output": validate_scene_prompt_output(markdown_wrapped_output, catalog),
}
rendered_prompt = render_medium_user_prompt(
    "kitchen counter with fruit and utensils",
    10,
    catalog,
    ["bowl_0", "plate_large", "banana", "spoon"],
)

prompt_design_tests = [
    ("good_output_is_valid", validations["good_output"]["valid"]),
    ("good_output_uses_semantic_predicates", {"place-in", "place-on", "cluster-around"}.issubset(validations["good_output"]["predicate_types"])),
    ("hallucinated_object_detected", any("catalog name mismatch" in issue for issue in validations["hallucinated_object_output"]["issues"])),
    ("dependency_error_detected", any("placed first" in issue for issue in validations["dependency_error_output"]["issues"])),
    ("grid_yaw_detected", any("cardinal grid angles" in issue for issue in validations["grid_like_output"]["issues"])),
    ("too_many_large_objects_detected", any("too many large objects" in issue for issue in validations["too_many_large_objects_output"]["issues"])),
    ("markdown_wrapper_rejected", any("JSON-only" in issue for issue in validations["markdown_wrapped_output"]["issues"])),
    ("runtime_prompt_has_catalog_categories", bool(rendered_prompt["available_objects"]["containers"]) and bool(rendered_prompt["available_objects"]["supports"])),
    ("runtime_prompt_has_size_warning", "max 1-2 large objects" in rendered_prompt["size_limits"]["rule"]),
]

print(json.dumps(validations, ensure_ascii=False, indent=2))
print(json.dumps(rendered_prompt, ensure_ascii=False, indent=2))
for name, ok in prompt_design_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert all(ok for _, ok in prompt_design_tests), prompt_design_tests
write_status(
    "prompt_design_lightweight_tests",
    {
        "all_passed": all(ok for _, ok in prompt_design_tests),
        "validations": validations,
        "rendered_prompt": rendered_prompt,
        "tests": [{"name": name, "passed": bool(ok)} for name, ok in prompt_design_tests],
        "boundary": "These tests validate prompt-output constraints and runtime prompt structure with synthetic examples; they do not call the paper's LLM or render USD scenes.",
    },
)

## 0.17 论文精讲：空间求解器、物理放置求解器与失败反馈

下面这节来自本目录的 [EXPLAIN_11_spatial_physical_solver_feedback.md](./EXPLAIN_11_spatial_physical_solver_feedback.md)，对应论文 Appendix C 的 Algorithm 1 Spatial Constraint Solver、你截图里的 Figure 17 feedback block，以及 Algorithm 2 Physical Placement Solver。重点是理解：空间求解器先解基础对象 2D 位姿，物理放置求解器再解 `place-on` / `place-in` 的 3D 坐标，失败反馈再回到下一轮 prompt。本次已加深 Spatial/Physical 部分：补充 typed predicate 中间表示、dense scene margin retry、相对关系坐标系、collision local repair、support 局部坐标旋转、container ellipse/layer packing、stability threshold 和反馈诊断压缩。

# 精讲 11：空间求解器、物理放置求解器与失败反馈

> **【绿色标识｜核心结论】** 这两张图讲的是 RoboLab 场景生成的“自动修复闭环”：LLM 先产出语义谓词，空间求解器解 2D 基础布局，物理放置求解器解 3D 支撑/容器关系；如果任一步失败，就把诊断反馈追加回下一轮 prompt。  
> **【蓝色标识｜源码路径】** 主要对应 `robolab/scene_gen/llm_scene_gen/spatial_solver.py`、`physical_solver.py`、`feedback_system.py`、`predicates.py`。  
> **【橙色标识｜容易误解】** Algorithm 1 和 Algorithm 2 不是互相替代。Algorithm 1 解决“桌面上基础对象怎么摆”，Algorithm 2 解决“对象怎么放到支撑物上或容器里”。

## 先说结论

RoboLab 场景生成可以理解成三段流水线：

```text
Stage I: LLM semantic planning
  输入：主题、对象目录、桌面尺寸、目标对象数
  输出：objects + predicates JSON

Stage II-A: Spatial Constraint Solver
  输入：基础对象 + 空间谓词 + 桌面边界
  输出：base objects 的 2D 位姿 (x, y, yaw)

Stage II-B: Physical Placement Solver
  输入：对象 + physical predicates + 已解出的 base poses
  输出：所有对象的 3D 位姿 (x, y, z)

Feedback Loop
  输入：语法错误、碰撞、出界、物理不稳定、容器/支撑失败
  输出：下一轮 prompt 的 PREVIOUS ATTEMPT FAILED block
```

一句话：

> LLM 负责“怎么组织场景”，solver 负责“几何和物理上能不能摆”，feedback 负责“失败后怎么让 LLM 改”。

## 图 1：Previous Attempt Failed 反馈块

你第一张图是 Figure 17：当 spatial solving、physical placement、grammar checks 或 intersection validation 失败时，系统把一段反馈追加到 user prompt。

图里的结构是：

```text
PREVIOUS ATTEMPT FAILED:
feedback string produced by spatial/physical solver or grammar checks

Please fix the issues. Common fixes:
- Use MORE containment (place-in) to reduce table crowding
- Use MORE stacking (place-on) to utilize vertical space
- Use clustering (cluster-around) instead of individual placements
- Select SMALLER objects if collisions persist
```

### 为什么这个反馈块很重要

没有这个块，LLM 每次失败后可能还是生成同一种坏布局。  
有了这个块，失败信息会变成下一轮生成的约束。

| 失败类型 | 反馈要告诉 LLM 什么 |
|---|---|
| collision | 哪些对象重叠，需要增加距离、换小物体、用容器/支撑减少桌面占用 |
| out of bounds | 哪些对象超出桌面，需要调整坐标或减少对象 |
| grammar issue | 哪些对象缺少 x/y/yaw，或者谓词不完整 |
| physical instability | 哪些对象掉落/移动，需要更稳定支撑或减少堆叠 |
| containment failure | 容器放不下，需要减少内部对象或换更大容器 |

**【绿色标识｜核心直觉】**

这个 feedback block 不是给人看的错误日志，而是给 LLM 的“修题提示”。它把 solver 的低层错误翻译成 LLM 能执行的修复策略。

### 为什么建议“更多 containment / stacking / clustering”

这几个建议都在解决同一个问题：桌面面积有限。

| 修复建议 | 解决什么 |
|---|---|
| MORE containment (`place-in`) | 多个小物体共享一个容器 footprint，减少桌面拥挤 |
| MORE stacking (`place-on`) | 利用 z 方向，把对象放到支撑物上 |
| clustering (`cluster-around`) | 让对象围绕 anchor 局部聚集，而不是每个对象都独立占点 |
| smaller objects | 直接降低碰撞概率和容器/支撑失败概率 |

它和精讲10的 prompt 是一对：精讲10是“第一轮怎么写好”，这张图是“失败后怎么改好”。

## Algorithm 1：Spatial Constraint Solver 回顾

Algorithm 1 的输入输出是：

| 项 | 内容 |
|---|---|
| 输入 | Objects `B`、Predicates `P`、Table Bounds `Lmax` |
| 输出 | base objects 的 2D 坐标 `(x, y, theta)` |

它处理的是基础对象，典型包括：

- bowl、bin、plate、tray、box、mug 等 anchor。
- loose object，如果没有 `place-in` / `place-on` physical relation，也需要直接放到桌面。

### 三个阶段

| 阶段 | 做什么 | 目的 |
|---|---|---|
| Phase 1 Initialization | 随机放 loose objects，处理 `place-on-base` 和 `cluster-around` | 得到初始布局 |
| Phase 2 Relative Constraints | 处理 left/right/front/back/align/facing 等关系 | 满足语言中的相对关系 |
| Phase 3 Collision Resolution | 找碰撞、推开、clamp 到桌面边界、必要时扰动 | 保证 2D footprint 不重叠且在桌面内 |

### 和源码的对应

| 论文概念 | 源码落点 |
|---|---|
| `place-on-base` | `SpatialSolver._apply_place_on_base` |
| relative constraints | `SpatialSolver._apply_relative_position` |
| orientation | `SpatialSolver._apply_orientation` |
| collision resolution | `SpatialSolver._optimize_placement` 及 overlap 相关函数 |
| margin retry | `solve()` 中的 `margins_to_try` |

**【橙色标识｜边界】**

Algorithm 1 不负责“苹果在碗里”的最终 z 坐标，也不负责“香蕉在盘子上”的高度。它只保证 bowl/plate 这些基础对象先有合理的桌面位置。

## 图 2：Algorithm 2 Physical Placement Solver

你第二张图是 Algorithm 2：Physical Placement Solver。它接在空间求解器之后。

输入输出：

| 项 | 内容 |
|---|---|
| 输入 | Objects `B`、Predicates `P`、Solved Base Poses |
| 输出 | 所有对象的 3D 坐标 `(x, y, z)` |

它主要处理两类 physical predicate：

| Predicate | 例子 | 需要解决的问题 |
|---|---|---|
| `place-on` | banana on plate | 在支撑物顶面找一个不重叠位置，并设置正确高度 |
| `place-in` | apple/orange in bowl | 在容器内部/口径范围内 pack 多个对象，并设置 z 高度 |

## Algorithm 2 上半部分：Solve Stacking

截图里的逻辑是：

```text
for all p where p.type == place-on:
    s <- p.support
    B_peers <- objects already on s
    (x, y) <- FindSpot(s, p.object, B_peers)
    p.object.z <- s.z + s.height + p.object.height / 2
    p.object.(x, y) <- (x, y)
```

说人话：

1. 找到支撑物 `s`，比如 plate/tray/shelf。
2. 看这个支撑物上已经放了哪些对象，避免新对象和它们重叠。
3. 在支撑物顶面找一个合适 slot。
4. `x/y` 用 slot 的位置。
5. `z` 设到支撑物顶面之上：支撑物 top + 物体半高 + 小 buffer。

### 源码如何做得更工程化

当前 `physical_solver.py` 里不是只处理一个对象，而是会把同一支撑物上的 `place-on` 分组：

| 源码函数 | 作用 |
|---|---|
| `_solve_place_on_group` | 把同一个 support 上的多个对象联合排布 |
| `_find_joint_support_slots` | 用回溯找多个 sibling 的非重叠 slot |
| `_candidate_support_offsets` | 生成 center、edge、环形候选位置 |
| `_fits_support_rectangle` | 检查 footprint 是否落在支撑物矩形内 |
| `_finish_place_on` | 设置 z/pitch/roll/is_placed |

**【绿色标识｜核心直觉】**

`place-on` 不是简单把对象中心放到 plate 中心。它要考虑对象 footprint、支撑物尺寸、支撑物 yaw、同层 sibling 是否重叠，以及最终 z 高度。

## Algorithm 2 下半部分：Solve Containment

截图里的逻辑是：

```text
for all p where p.type == place-in:
    c <- p.container
    if TotalArea(p.objects) > 0.8 * Area(c):
        p.objects <- SortAndFilter(p.objects, c.capacity)
    (R, C) <- ComputeGridDimensions(c.dims, |p.objects|)
    for i = 0 to |p.objects| - 1:
        (r, c) <- (i // C, i % C)
        (xloc, yloc) <- GridCellCenter(...)
        Jitter(...)
        p.objects[i].(x, y) <- container center + local offset
        p.objects[i].z <- container z + container height / 2 + buffer
```

说人话：

1. 找容器，比如 bowl/bin/box。
2. 看要放进去的对象总体面积是不是太大。
3. 太大就排序和过滤，优先放更合适的对象。
4. 根据容器大小和对象数量划分网格。
5. 每个对象放到一个网格 cell 中心附近，加一点 jitter，避免机械排列。
6. z 设到容器上方一点，后续物理 settle 可以让它落进去。

### 源码如何做得更工程化

当前 `physical_solver.py` 的 `place-in` 逻辑比伪代码更细：

| 源码函数/逻辑 | 作用 |
|---|---|
| `_solve_place_in` | 容器内对象 packing 主函数 |
| `_candidate_local_yaws` | 为对象尝试多个 yaw，降低 footprint 失败概率 |
| `_candidate_container_offsets` | 在容器口径内生成候选局部 offset |
| `_fits_container_ellipse` | 检查对象 footprint 是否放得进容器口径 |
| `_rect_overlaps_layer` | 检查同层对象是否重叠 |
| layers | 如果一层放不下，就向上分层，而不是横向穿出容器 |

**【绿色标识｜核心直觉】**

`place-in` 的本质是 packing，不是简单把所有对象坐标都设成容器中心。否则多个水果会完全重叠。

## 深挖 A：这两个 solver 共同操作的“中间表示”是什么

Spatial 和 Physical 不是直接操作自然语言，也不是直接操作 USD 场景。它们操作的是 LLM 生成后的 typed predicates 和对象状态。

可以把数据结构理解成四张表：

| 数据 | 说人话解释 | 主要由谁用 |
|---|---|---|
| `object_states` | 每个对象当前解到哪里了：`x/y/z/yaw`、是否已放置、挂了哪些 predicates | Spatial + Physical |
| `object_dims` | 每个对象的宽、深、高 | 碰撞、边界、支撑、容器 packing |
| `predicates` | `place-on-base`、`left-of`、`place-on`、`place-in` 等结构化约束 | 两个 solver 分阶段消费 |
| `object_metadata/object_paths` | USD 路径、类别、容器/支撑能力、物理属性等 | Physical/后续场景实例化 |

**【绿色标识｜核心结论】**

这一步的关键不是“LLM 直接生成坐标”，而是“LLM 生成可检查的约束”。坐标由 solver 根据约束、尺寸和桌面边界解出来。

一个典型对象会经历这样的状态变化：

```text
初始：ObjectState(name="apple_01", x=None, y=None, z=None, yaw=None)
LLM：给它 place-in(container="bowl_0") predicate
Spatial：发现它有 physical predicate，所以暂时不强行解桌面 2D pose
Physical：等 bowl_0 的 base pose 解好后，再把 apple_01 放进 bowl_0 内部
输出：ObjectState(x=..., y=..., z=..., yaw=..., is_placed=True)
```

**【橙色标识｜容易误解】**

`place-in` / `place-on` 对象通常不是 Algorithm 1 的主要求解对象。Algorithm 1 主要先把 container/support/loose base objects 放好；Algorithm 2 再处理依附在它们上面的对象。

## 深挖 B：Spatial Solver 不是“随机撒点”，而是带回退的约束满足

Algorithm 1 看起来短，但源码里有几层工程保护。

### B1. 它会根据场景密度调整模式

源码里会根据对象数量、平均 footprint、大对象数量进入不同模式：

| 模式 | 触发直觉 | 策略 |
|---|---|---|
| normal | 对象少，尺寸普通 | 使用默认 collision margin |
| hard/container | 对象较多，或大容器/大支撑多 | 缩小 margin、增加迭代数 |
| ultra-dense | 18 个以上对象 | 更小 margin、更高迭代次数 |

这件事很重要，因为桌面任务不是数学竞赛里的干净约束。真实 benchmark 场景会有 bowl、plate、bin、mug、tool、food 混在一起，大对象多时，默认 5cm 间距可能直接无解。

### B2. `margins_to_try` 不是“越放越松”，而是多次重启搜索

源码里的 `margins_to_try` 会在失败时调整 collision margin 并重新随机化未固定对象。它的目的不是保证一定变好，而是避免卡在某个坏初始化。

可以把它理解成：

```text
attempt 0: 用当前 margin 初始化并优化
失败 -> 换 margin + 重新随机未固定对象
attempt 1: 再解一次
失败 -> 再换 margin + 再随机
...
```

**【绿色标识｜核心直觉】**

这是 sampling + local repair，不是一次解析式求解。RoboLab 追求的是几分钟内批量生成“足够好、可评测”的场景，而不是求全局最优布局。

### B3. 相对关系在 RoboLab 坐标系里有明确方向

源码注释里有一个非常关键的约定：

```text
Front = +X
Back  = -X
Left  = +Y
Right = -Y
```

所以：

| 语言关系 | 坐标变化 |
|---|---|
| `left-of(A, B)` | `A.y = B.y + distance` |
| `right-of(A, B)` | `A.y = B.y - distance` |
| `front-of(A, B)` | `A.x = B.x + distance` |
| `back-of(A, B)` | `A.x = B.x - distance` |

这和很多人直觉里的屏幕坐标不同。看视频时“左/右”还会受到相机视角影响，所以 debug spatial relation 时应该优先看世界坐标，而不是只看截图。

### B4. collision resolution 是局部推开，不是全局重排

源码里的 `_check_collisions` 用 footprint 半径近似检测重叠，`_resolve_collision` 根据两物体中心连线把它们推开，再 clamp 回桌面边界。

这类方法的优点是快，缺点是可能局部卡住。于是又加了两个机制：

- 如果 collision 数量长期不下降，就随机扰动位置打破僵局。
- 如果碰到 fixed object，例如 rack fixture，只移动非固定对象。

### B5. 为什么 physical predicate 要被跳过

Spatial Solver 检查“未完全求解对象”时，会跳过带 `place-on` / `place-in` / `place-anywhere` 等 physical predicate 的对象。否则 apple/orange 这类要放进 bowl 的对象会被误认为“还没有桌面坐标，所以 spatial solve 失败”。

**【橙色标识｜常见错误】**

如果你自己写任务生成器，把所有物体都强行 `place-on-base`，再额外给 `place-in`，会造成语义冲突：对象既被要求在桌面固定位置，又被要求在容器内部。更合理的做法是让 container/support 有 base pose，让内部对象只带 physical predicate。

## 深挖 C：Physical Solver 的核心是“局部坐标系里的 packing”

Algorithm 2 解决的不是“z 加一下”这么简单。真正麻烦的是：支撑物有 yaw，容器有口径，多个 sibling 会互相重叠，对象本身也有 yaw。

### C1. `place-on` 为什么要按 support 分组

如果有三个对象都放在同一个 tray 上，不能逐个独立求解：

```text
place-on(spoon, tray)
place-on(fork, tray)
place-on(mug, tray)
```

如果每个对象都单独“优先放 center”，它们都会抢 tray 中心。源码会把同一 support 上的 `place-on` predicate 分组，再联合找多个 slot。

说人话：

> `place-on` 的难点不是把一个对象放上去，而是把同一层 sibling 放得下、分得开、还落在支撑物 footprint 内。

### C2. 支撑物局部坐标要随 yaw 旋转回世界坐标

`_candidate_support_offsets` 这类函数生成的是 support 局部坐标里的候选点，例如 center、边缘、环形点。真正写回对象 pose 时，要把局部 offset 按 support yaw 旋转到世界坐标。

否则会出现：

- plate 旋转了，但 banana 仍按世界 x/y 放，看起来偏到 plate 外。
- tray 斜着放，但上面的 spoon/fork 仍按未旋转矩形判断，实际渲染会穿出边界。

### C3. `place-in` 不是矩形网格就够，容器经常更像椭圆口径

bowl、cup、round bin 这类容器不能只用矩形 bounding box 判断。源码里会用类似 `_fits_container_ellipse` 的检查：对象 footprint 要落在容器口径的有效椭圆区域内。

这解释了为什么有时从 top view 看对象似乎在 bowl 的 bounding box 里，但仍被判定为不合格：它可能落在矩形角落，而那个角落实际上在圆形/椭圆口径之外。

### C4. 多层 packing 是最后手段，不是默认堆叠

当容器一层放不下时，solver 可以考虑 layers。这个设计的意义是：优先让容器内部对象在同一层分开；只有实在放不下时才向上分层。

但是 layers 也会引入风险：

- 上层对象更容易物理 settle 后滚动。
- 容器浅时，z 太高会导致对象看起来像浮在外面。
- 对于任务评测来说，层数过多会让抓取和可见性变差。

所以 prompt 里才会建议：如果碰撞持续，优先选更小对象或减少内部对象，而不是盲目把所有东西塞进容器。

### C5. stability threshold 是从“看起来放上去了”到“物理上稳定”的边界

`physical_solver.py` 的初始化参数里有 `stability_threshold`。这代表一个思想：场景初始位姿写出来后，还需要物理 settle。对象最大位移超过阈值，就说明这个放置不稳定。

**【绿色标识｜核心结论】**

RoboLab 的 physical placement 不是只做几何 packing。几何 packing 只是初始条件；最终还要考虑物理 settle 后对象是否还在合理位置。

## 深挖 D：失败反馈其实是“诊断信息压缩器”

Figure 17 的 feedback block 看起来简单，但它背后做了一个重要转译：把 solver 看得懂的失败，翻译成 LLM 能改的语言。

| solver 失败 | 原始含义 | 给 LLM 的修复策略 |
|---|---|---|
| collision unresolved | footprint 放不下或局部优化卡死 | 换小物体、减少 base objects、增加 `place-in` / `place-on` |
| out of bounds | 对象中心或 footprint 超出桌面 | 调整 anchor 坐标，减少边缘放置 |
| support slot not found | 支撑物太小或 sibling 太多 | 换更大 support，减少 `place-on` 数量 |
| container crowding | 容器口径/面积不足 | 换更大 container，减少内部对象，选小物体 |
| unstable after settle | 初始几何可行但物理不稳定 | 降低堆叠高度、换平面支撑、减少层数 |
| invalid object name | 不在 catalog | 只能回到对象目录重选 |

这也是为什么反馈里出现的建议是“MORE containment / MORE stacking / clustering / smaller objects”。它们不是随口建议，而是在压缩几类最常见失败：

- 桌面面积不够：用 containment/stacking 节省 base footprint。
- 对象太分散：用 clustering 生成局部自然组合。
- 物体过大：选 smaller objects 降低碰撞、容器和支撑失败概率。

**【橙色标识｜边界】**

反馈不能保证下一轮一定成功。它只是把失败信号加入 prompt，提高下一轮 LLM 生成可解 predicates 的概率。真正的成功仍然要经过 grammar check、spatial solve、physical solve、intersection validation 和物理 settle。

## Algorithm 1 和 Algorithm 2 如何串起来

用一个 bowl + plate + fruit 例子：

```json
{
  "objects": [
    {"name": "bowl_0"},
    {"name": "plate_large"},
    {"name": "apple_01"},
    {"name": "orange_01"},
    {"name": "banana"}
  ],
  "predicates": [
    {"type": "place-on-base", "object": "bowl_0", "x": 0.40, "y": 0.15, "yaw": 23},
    {"type": "place-on-base", "object": "plate_large", "x": 0.65, "y": -0.10, "yaw": 156},
    {"type": "place-in", "objects": ["apple_01", "orange_01"], "container": "bowl_0"},
    {"type": "place-on", "object": "banana", "support": "plate_large", "position": "center"}
  ]
}
```

执行顺序是：

1. Spatial solver 先把 `bowl_0` 和 `plate_large` 放到桌面上。
2. Physical solver 用 bowl 的 pose 计算 apple/orange 的 `(x, y, z)`。
3. Physical solver 用 plate 的 pose 计算 banana 的 `(x, y, z)`。
4. 如果碰撞、出界或放不下，feedback block 让 LLM 下一轮增加 containment/stacking/clustering 或换小物体。

## 失败反馈为什么要追加到 Prompt，而不是直接自动改

有些错误 solver 可以修：

- 小碰撞可以推开。
- yaw 可以重采样。
- 容器内局部 packing 可以换 slot。

但有些错误需要 LLM 重新规划：

| 错误 | 为什么不能只靠 solver |
|---|---|
| 选了太多大物体 | solver 无法凭空换对象 |
| 缺少 container/support anchor | 需要改 predicate 结构 |
| 所有对象都独立放桌面导致拥挤 | 需要语义上改成 `place-in` / `place-on` |
| 对象名不存在 | 需要回到 catalog 重新选 |
| 任务主题不匹配 | solver 不懂主题语义 |

所以 feedback 不是替代 solver，而是把低层失败转回上游规划。

## 与 Prompt 设计的关系

精讲10讲的是：第一轮 prompt 如何尽量避免坏输出。  
精讲11讲的是：坏输出已经发生后，solver 如何检测，以及怎么把失败变成下一轮 prompt 的修复建议。

可以合起来记：

```text
Prompt 先验 -> Predicate JSON -> Spatial Solver -> Physical Solver -> Feedback -> Prompt 修复
```

这就是 RoboLab 场景生成能规模化的关键：不要求 LLM 一次生成完美场景，而是把生成变成可检查、可反馈、可重试的闭环。

## 小结

两张图的核心含义是：

- Fig.17 是“失败怎么反馈给 LLM”。
- Algorithm 2 是“支撑/容器关系怎么变成 3D 坐标”。
- Algorithm 1 + Algorithm 2 合起来，才是从 typed predicates 到可用 3D scene 的求解过程。

最终目标不是生成看起来热闹的桌面，而是生成：

```text
语义上合理
几何上不碰撞
物理上可 settle
任务上可评测
失败后可自动修复
```

In [ ]:
# ===== 精讲11：Spatial + Physical solver + feedback 轻量验证 =====
# 这个 cell 用纯 Python 模拟 Algorithm 1/2 的核心数据流：
# - spatial: 先放 bowl/plate 这类 base objects
# - physical: 再把 banana 放到 plate 上，把 apple/orange 放进 bowl
# - feedback: 对碰撞/拥挤失败生成下一轮 prompt 修复建议
# 边界：这是教学用 toy solver，不替代 RoboLab 的 physical_solver.py/Isaac physics settle。

import json
import math

object_dims = {
    "bowl_0": (0.18, 0.18, 0.08),
    "plate_large": (0.26, 0.20, 0.025),
    "apple_01": (0.055, 0.055, 0.055),
    "orange_01": (0.060, 0.060, 0.060),
    "banana": (0.14, 0.045, 0.035),
    "spoon_0": (0.16, 0.035, 0.018),
    "large_box": (0.32, 0.24, 0.16),
    "storage_bin": (0.24, 0.18, 0.12),
}

def base_pose_from_place_on_base(predicates, dims):
    # Algorithm 1 的简化版：只处理 place-on-base，并把 z 设为对象半高。
    poses = {}
    for pred in predicates:
        if pred["type"] != "place-on-base":
            continue
        name = pred["object"]
        sx, sy, sz = dims[name]
        poses[name] = {
            "x": float(pred["x"]),
            "y": float(pred["y"]),
            "z": sz / 2.0,
            "yaw": float(pred.get("yaw", 0.0)),
            "source": "spatial/place-on-base",
        }
    return poses

def apply_relative_position(poses, obj_name, reference_name, relation, distance=0.10):
    # Algorithm 1 Phase 2 教学版：按 RoboLab 世界坐标约定应用相对关系。
    # Front=+X, Back=-X, Left=+Y, Right=-Y。
    ref = poses[reference_name]
    pose = poses.setdefault(obj_name, {"x": ref["x"], "y": ref["y"], "z": 0.0, "yaw": 0.0, "source": "spatial/relative"})
    if relation == "left-of":
        pose["x"] = ref["x"]
        pose["y"] = ref["y"] + distance
    elif relation == "right-of":
        pose["x"] = ref["x"]
        pose["y"] = ref["y"] - distance
    elif relation == "front-of":
        pose["x"] = ref["x"] + distance
        pose["y"] = ref["y"]
    elif relation == "back-of":
        pose["x"] = ref["x"] - distance
        pose["y"] = ref["y"]
    else:
        raise ValueError(f"unknown relation: {relation}")
    pose["source"] = f"spatial/{relation}/{reference_name}"
    return pose

def footprint_radius(name, dims, margin=0.02):
    return max(dims[name][0], dims[name][1]) / 2.0 + margin

def circle_collides(a, b, poses, dims, margin=0.02):
    dx = poses[a]["x"] - poses[b]["x"]
    dy = poses[a]["y"] - poses[b]["y"]
    return math.hypot(dx, dy) < footprint_radius(a, dims, margin) + footprint_radius(b, dims, margin)

def resolve_collision_pair(a, b, poses, dims, table_bounds=(0.25, 0.85, -0.40, 0.40), margin=0.02):
    # Algorithm 1 Phase 3 教学版：沿中心连线把两个对象推开，再 clamp 回桌面边界。
    dx = poses[a]["x"] - poses[b]["x"]
    dy = poses[a]["y"] - poses[b]["y"]
    dist = math.hypot(dx, dy)
    if dist < 1e-6:
        dx, dy, dist = 1.0, 0.0, 1.0
    dx, dy = dx / dist, dy / dist
    required = footprint_radius(a, dims, margin) + footprint_radius(b, dims, margin)
    push = max(0.0, required - dist) / 2.0 + 0.005
    poses[a]["x"] += dx * push
    poses[a]["y"] += dy * push
    poses[b]["x"] -= dx * push
    poses[b]["y"] -= dy * push
    min_x, max_x, min_y, max_y = table_bounds
    for name in [a, b]:
        radius = footprint_radius(name, dims, margin)
        poses[name]["x"] = min(max(poses[name]["x"], min_x + radius), max_x - radius)
        poses[name]["y"] = min(max(poses[name]["y"], min_y + radius), max_y - radius)
    return not circle_collides(a, b, poses, dims, margin)

def support_offsets(position):
    # Algorithm 2 的 FindSpot 教学版：center 优先，edge 给几个候选点。
    if position == "edge":
        return [(0.08, 0.0), (-0.08, 0.0), (0.0, 0.06), (0.0, -0.06), (0.0, 0.0)]
    return [(0.0, 0.0), (0.04, 0.0), (-0.04, 0.0), (0.0, 0.04), (0.0, -0.04)]

def rotate_offset(yaw_deg, local_x, local_y):
    # support/container 的局部候选 offset 需要按 yaw 旋转回世界坐标。
    theta = math.radians(yaw_deg)
    return (
        local_x * math.cos(theta) - local_y * math.sin(theta),
        local_x * math.sin(theta) + local_y * math.cos(theta),
    )

def rect_fits_support(local_x, local_y, obj_dim, support_dim):
    return (
        abs(local_x) + obj_dim[0] / 2.0 <= support_dim[0] / 2.0
        and abs(local_y) + obj_dim[1] / 2.0 <= support_dim[1] / 2.0
    )

def solve_place_on(pred, poses, dims, occupied_by_support):
    # Algorithm 2 上半部分：在 support 顶面找 slot，并设置 z。
    obj = pred["object"]
    support = pred["support"]
    support_pose = poses.get(support)
    if support_pose is None:
        return False, f"support {support} has no solved base pose"
    support_dim = dims[support]
    obj_dim = dims[obj]
    support_slots = occupied_by_support.setdefault(support, [])
    for local_x, local_y in support_offsets(pred.get("position", "center")):
        if not rect_fits_support(local_x, local_y, obj_dim, support_dim):
            continue
        # 简化的同层 overlap 检查：slot 中心不要太近。
        too_close = any(abs(local_x - sx) < (obj_dim[0] + sw) / 2.0 and abs(local_y - sy) < (obj_dim[1] + sh) / 2.0 for sx, sy, sw, sh in support_slots)
        if too_close:
            continue
        world_dx, world_dy = rotate_offset(support_pose.get("yaw", 0.0), local_x, local_y)
        support_slots.append((local_x, local_y, obj_dim[0], obj_dim[1]))
        poses[obj] = {
            "x": support_pose["x"] + world_dx,
            "y": support_pose["y"] + world_dy,
            "z": support_pose["z"] + support_dim[2] / 2.0 + obj_dim[2] / 2.0 + 0.001,
            "yaw": support_pose.get("yaw", 0.0),
            "source": f"physical/place-on/{support}",
        }
        return True, "placed on support"
    return False, f"no support slot found for {obj} on {support}"

def solve_place_in(pred, poses, dims, area_limit_ratio=0.80):
    # Algorithm 2 下半部分：在容器口径内做简单网格 packing，并设置 z。
    container = pred["container"]
    container_pose = poses.get(container)
    if container_pose is None:
        return False, f"container {container} has no solved base pose"
    container_dim = dims[container]
    objects = list(pred["objects"])
    total_area = sum(dims[obj][0] * dims[obj][1] for obj in objects)
    container_area = container_dim[0] * container_dim[1]
    if total_area > area_limit_ratio * container_area:
        return False, f"container crowding: total object area {total_area:.3f} > {area_limit_ratio:.1f} * container area {container_area:.3f}"

    count = len(objects)
    cols = max(1, math.ceil(math.sqrt(count)))
    rows = max(1, math.ceil(count / cols))
    usable_x = container_dim[0] * 0.65
    usable_y = container_dim[1] * 0.65
    for index, obj in enumerate(objects):
        row = index // cols
        col = index % cols
        local_x = (col + 0.5) * usable_x / cols - usable_x / 2.0
        local_y = (row + 0.5) * usable_y / rows - usable_y / 2.0
        # 小 jitter，模拟论文 Algorithm 2 的 Jitter(...)
        local_x += (index - (count - 1) / 2.0) * 0.004
        local_y += ((index % 2) - 0.5) * 0.004
        poses[obj] = {
            "x": container_pose["x"] + local_x,
            "y": container_pose["y"] + local_y,
            "z": container_pose["z"] + container_dim[2] / 2.0 + dims[obj][2] / 2.0 + 0.01,
            "yaw": container_pose.get("yaw", 0.0) + index * 23.0,
            "source": f"physical/place-in/{container}",
        }
    return True, "placed in container"

def compact_scene_feedback(failure_message, collisions=None, out_of_bounds=None):
    # 模拟 Figure 17：把底层失败转成下一轮 prompt 的修复建议。
    lines = [
        "PREVIOUS ATTEMPT FAILED:",
        failure_message,
        "",
        "Please fix the issues. Common fixes:",
        "- Use MORE containment (place-in) to reduce table crowding",
        "- Use MORE stacking (place-on) to utilize vertical space",
        "- Use clustering (cluster-around) instead of individual placements",
        "- Select SMALLER objects if collisions persist",
    ]
    if collisions:
        lines.insert(2, "Collisions: " + ", ".join(f"{a}/{b}" for a, b in collisions))
    if out_of_bounds:
        lines.insert(2, "Out of bounds: " + ", ".join(out_of_bounds))
    return "\n".join(lines)

predicates_ok = [
    {"type": "place-on-base", "object": "bowl_0", "x": 0.40, "y": 0.15, "yaw": 23},
    {"type": "place-on-base", "object": "plate_large", "x": 0.65, "y": -0.10, "yaw": 156},
    {"type": "place-in", "objects": ["apple_01", "orange_01"], "container": "bowl_0"},
    {"type": "place-on", "object": "banana", "support": "plate_large", "position": "edge"},
]
poses = base_pose_from_place_on_base(predicates_ok, object_dims)
relative_probe = dict(poses)
relative_pose = apply_relative_position(relative_probe, "spoon_0", "bowl_0", "left-of", distance=0.12)
collision_probe = {
    "bowl_0": {"x": 0.45, "y": 0.00, "z": object_dims["bowl_0"][2] / 2.0, "yaw": 0.0},
    "plate_large": {"x": 0.52, "y": 0.00, "z": object_dims["plate_large"][2] / 2.0, "yaw": 0.0},
}
collision_before = circle_collides("bowl_0", "plate_large", collision_probe, object_dims)
collision_resolved = resolve_collision_pair("bowl_0", "plate_large", collision_probe, object_dims)
occupied_by_support = {}
physical_messages = []
for pred in predicates_ok:
    if pred["type"] == "place-on":
        ok, msg = solve_place_on(pred, poses, object_dims, occupied_by_support)
        physical_messages.append((pred["type"], ok, msg))
    if pred["type"] == "place-in":
        ok, msg = solve_place_in(pred, poses, object_dims)
        physical_messages.append((pred["type"], ok, msg))

crowded_pred = {"type": "place-in", "objects": ["large_box", "storage_bin", "banana"], "container": "bowl_0"}
crowded_ok, crowded_msg = solve_place_in(crowded_pred, dict(poses), object_dims)
feedback = compact_scene_feedback(crowded_msg, collisions=[("large_box", "bowl_0")])
expected_edge_dx, expected_edge_dy = rotate_offset(poses["plate_large"]["yaw"], 0.0, 0.06)

spatial_physical_tests = [
    ("base_anchors_have_2d_and_z_pose", all(name in poses and poses[name]["z"] > 0 for name in ["bowl_0", "plate_large"])),
    ("relative_left_of_uses_positive_y", relative_pose["y"] > relative_probe["bowl_0"]["y"] and abs(relative_pose["x"] - relative_probe["bowl_0"]["x"]) < 1e-9),
    ("collision_resolution_pushes_objects_apart", collision_before and collision_resolved),
    ("place_on_sets_banana_above_plate", poses["banana"]["z"] > poses["plate_large"]["z"]),
    ("place_on_rotates_support_local_offset", abs((poses["banana"]["x"] - poses["plate_large"]["x"]) - expected_edge_dx) < 1e-9 and abs((poses["banana"]["y"] - poses["plate_large"]["y"]) - expected_edge_dy) < 1e-9),
    ("place_in_sets_fruit_above_bowl", poses["apple_01"]["z"] > poses["bowl_0"]["z"] and poses["orange_01"]["z"] > poses["bowl_0"]["z"]),
    ("container_objects_have_distinct_xy", (poses["apple_01"]["x"], poses["apple_01"]["y"]) != (poses["orange_01"]["x"], poses["orange_01"]["y"])),
    ("physical_messages_successful", all(ok for _kind, ok, _msg in physical_messages)),
    ("crowded_container_failure_detected", not crowded_ok and "container crowding" in crowded_msg),
    ("feedback_mentions_previous_attempt_failed", feedback.startswith("PREVIOUS ATTEMPT FAILED")),
    ("feedback_suggests_containment_stacking_clustering", all(term in feedback for term in ["place-in", "place-on", "cluster-around"])),
    ("feedback_suggests_smaller_objects", "SMALLER objects" in feedback),
]

print("Solved poses:")
print(json.dumps(poses, ensure_ascii=False, indent=2))
print("Physical messages:")
print(json.dumps(physical_messages, ensure_ascii=False, indent=2))
print("Relative probe:")
print(json.dumps(relative_probe, ensure_ascii=False, indent=2))
print("Collision probe after resolution:")
print(json.dumps(collision_probe, ensure_ascii=False, indent=2))
print("Crowded failure message:", crowded_msg)
print("Feedback block:")
print(feedback)
for name, ok in spatial_physical_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert all(ok for _, ok in spatial_physical_tests), spatial_physical_tests
write_status(
    "spatial_physical_feedback_lightweight_tests",
    {
        "all_passed": all(ok for _, ok in spatial_physical_tests),
        "poses": poses,
        "physical_messages": physical_messages,
        "relative_probe": relative_probe,
        "collision_probe": collision_probe,
        "crowded_ok": crowded_ok,
        "crowded_message": crowded_msg,
        "feedback": feedback,
        "tests": [{"name": name, "passed": bool(ok)} for name, ok in spatial_physical_tests],
        "boundary": "This is a toy implementation for explaining Algorithm 1/2 and Figure 17; real RoboLab placement uses spatial_solver.py, physical_solver.py, and Isaac/physics validation.",
    },
)

## 0.18 论文精讲：Gaussian 方法与 NVIDIA 2026 前沿路线

下面这节来自本目录的 [EXPLAIN_12_gaussian_sim_methods.md](./EXPLAIN_12_gaussian_sim_methods.md)，回答两个问题：

- RoboLab 本文到底用了哪些 “Gaussian” 方法，它们分别负责视觉、物理还是统计分析。
- 2026 年 NVIDIA 在 NuRec、3DGUT/3DGRT、Lyra、Physically Embodied Gaussians 等方向上，怎么把高斯重建推进到可交互仿真。

# 精讲 12：RoboLab 里的 Gaussian 方法与 NVIDIA 2026 前沿路线

> **【绿色标识｜核心结论】** RoboLab 本文不是“只靠高斯 splat 保证仿真”。它采用的是混合路线：Gaussian Splat 提供高保真视觉背景，collision mesh 提供物理碰撞，mesh foreground/SimReady 资产提供可交互对象，VoMP/质量摩擦等属性补足物理参数。  
> **【蓝色标识｜源码/论文路径】** 论文 Figure 13 明确提到 Gaussian Splat + Mesh scene、3DGRUT/3DGRT/3DGUT 相关 collision mesh、mesh foreground，以及 VoMP 估计质量/密度。  
> **【橙色标识｜容易误解】** 3DGS/高斯 splat 本身通常是视觉表示，不天然等价于可碰撞、可抓取、可物理交互的仿真世界。要进入机器人仿真，必须和 collider mesh、物理属性、USD/Isaac Sim 场景结构结合。

## 先说结论

本文和“高斯”相关的内容可以分成三类：

| 类别 | 是否是 RoboLab 本文直接用到 | 作用 | 注意边界 |
|---|---:|---|---|
| Gaussian Splat + Mesh scene | 是，Figure 13 | 用高斯 splat 做高保真背景视觉，用 mesh/collider 做几何和碰撞 | 高斯负责看起来真实，不直接负责物理 |
| 3DGRT / 3DGUT / 3DGRUT 相关技术 | Figure 13 引用，用于 splat/mesh 相关重建与渲染路线 | 支持高斯粒子 ray tracing、畸变相机、rolling shutter、secondary rays 等 | 这是视觉/重建渲染能力，不等于完整 manipulation physics |
| MNPE 里的 Gaussian KDE | 是，Appendix B | 做 posterior/importance sampling 的统计估计 | 这是统计上的高斯核密度，不是 3D Gaussian Splatting |

一句话：

```text
RoboLab 的仿真可信度 = 视觉高保真 + 几何碰撞 + 物理属性 + 任务/策略评测证据
不是 = 单独一个 Gaussian Splat
```

## 本文具体用了哪些 Gaussian 相关方法

### 1. Gaussian Splat 背景

论文 Figure 13 说 RoboLab 展示了一个 Gaussian Splat + Mesh 场景：背景是 Gaussian splat。

说人话：

> 真实环境里很复杂的背景、墙面、柜子、灯光细节，如果全部手工建模成 mesh 很慢；Gaussian Splat 可以从图像/视频重建出高保真外观，用来让机器人看到更接近真实世界的视觉输入。

它主要保证的是：

- 视觉纹理真实。
- 视角变化时外观连续。
- 对 VLA policy 的视觉感知更接近真实环境。

它不直接保证：

- 机器人能不能撞上墙。
- 物体能不能被抓起。
- 支撑面是否有摩擦/质量/稳定性。

### 2. Collision Mesh for Splat

Figure 13 还说 Gaussian splat background 有 collision mesh。  
这一步是把“能看见”的高斯场景补成“能碰撞”的仿真场景。

为什么必须有 collision mesh？

| 只有 Gaussian Splat | 加上 collision mesh |
|---|---|
| 视觉真实 | 视觉真实 + 机器人不会穿墙/穿桌 |
| 适合渲染 | 适合物理仿真 |
| 不天然提供稳定接触 | 可以被 PhysX/Isaac 等物理系统使用 |

**【绿色标识｜核心直觉】**

Gaussian Splat 是“眼睛看到的世界”，collision mesh 是“机器人身体碰到的世界”。二者必须对齐，仿真才有意义。

### 3. Mesh Foreground

Figure 13 提到前景是 mesh foreground。  
这很重要，因为机器人 manipulation 最关心的是近场可操作对象：

- 要被抓。
- 要碰撞。
- 要落到容器里。
- 要有质量、摩擦、惯性。

这些对象通常不能只用 splat 表示。RoboLab 的对象 catalog 里每个 object 都有 visual mesh、collision mesh、质量、摩擦、语言标签等信息。

### 4. VoMP / 空间变化密度与质量估计

论文还提到对象有 spatially varying density，质量由 VoMP 估计。  
说人话：

> 对同一个几何外形，不同材料/内部结构会导致不同质量和重心。VoMP 这类方法试图从体积机械属性场估计对象的物理属性，让模拟里“拿起来、掉落、碰撞”的行为更可信。

它补的是物理属性，不是视觉渲染。

### 5. MNPE 里的 Gaussian KDE

Appendix B 的 MNPE 敏感性分析里还有 Gaussian kernel density estimation，用于 importance sampling correction。  
这也是“高斯”，但不是 3DGS。

| 名称 | 语境 | 作用 |
|---|---|---|
| Gaussian Splatting | 视觉/重建/渲染 | 表示 3D 场景外观 |
| Gaussian KDE | 统计推断 | 估计采样分布/权重 |

**【橙色标识｜不要混淆】**

MNPE 的 Gaussian KDE 不会生成 3D 场景，也不参与 Isaac Sim 渲染；它只是分析扰动参数与成功/失败之间关系的统计工具。

## RoboLab 如何用这些方法“保证仿真”

更准确的说法不是“保证”，而是“提高可信度并暴露边界”：

```text
真实/生成场景视觉
  -> Gaussian Splat / mesh 提高外观保真

可碰撞几何
  -> collision mesh / foreground mesh 提供物理接触

可操作物体
  -> catalog objects + visual/collision mesh + mass/friction

环境扰动
  -> 光照、背景、纹理、相机、物体位姿变化

策略评测
  -> success / score / subtask / SPARC / MNPE / videos
```

也就是说，Gaussian 只是其中一层。真正让仿真可评测的是多层证据链。

## 为什么 Gaussian Splat 不能单独当仿真

原因很直接：

| 需求 | 3DGS 是否天然满足 |
|---|---:|
| RGB 渲染真实 | 是 |
| 新视角渲染 | 是 |
| 碰撞检测 | 通常不是 |
| 抓取接触 | 不是 |
| 摩擦/质量/惯性 | 不是 |
| 稳定堆叠 | 不是 |
| 任务成功条件 | 不是 |

所以真实工作流通常是：

```text
Gaussian visual volume
+ collider mesh
+ USD scene hierarchy
+ rigid body / articulation / PhysX or Newton
+ semantic labels / task predicates
= robot simulation scene
```

## 2026 NVIDIA 前沿路线：哪些和本文最相关

下面这些不都属于 RoboLab 本文“已经完整使用”的方法，但都是 NVIDIA 在 2026 前后围绕 Gaussian / real-to-sim / physical AI 推进的关键方向。

### 前沿来源速查表

> **【蓝色标识｜来源链接】** 这几项不是泛泛概念，下面给出可点击来源、核心内容和我们读它时应该抓住的点。

| 前沿方向 | 来源链接 | 原始内容核心 | 为什么和 RoboLab 有关 | 建议重点关注 |
|---|---|---|---|---|
| Omniverse NuRec | [NVIDIA Omniverse NuRec](https://developer.nvidia.com/omniverse/nurec) | 从 camera/lidar sensor data 重建环境，打包成 USD scene，并用 3D Gaussian splatting 在 Isaac Sim / AlpaSim / CARLA 中交互式渲染。 | 对应 RoboLab 的 real-to-sim 方向：真实传感器数据 -> 可交互仿真场景。 | NCore data standard、USD scene、gRPC rendering、Isaac Sim 集成。 |
| 3DGUT | [NVIDIA Research 3DGUT](https://research.nvidia.com/labs/toronto-ai/3DGUT/) | 用 Unscented Transform 扩展 3DGS，支持 fisheye、rolling shutter、secondary rays，如反射/折射。 | RoboLab 里真实相机视角、畸变和反光物体会影响 VLA 输入；3DGUT 说明高斯渲染如何更接近真实传感器。 | nonlinear projection、rolling shutter、secondary rays。 |
| Isaac Sim 6.0 EDR | [Isaac Sim 6.0 Early Developer Release](https://forums.developer.nvidia.com/t/announcement-isaac-sim-6-0-early-developer-release-for-gtc26/363709) | Kit 110 提到 NuRec 3D Gaussian splatting libraries、Fabric Scene Delegate、多物理后端、Warp-based Core API。 | 说明 NuRec/3DGS 正进入 Isaac Sim runtime，不再只是离线 viewer。 | NuRec + Fabric integration、PhysX/Newton、Warp-based Core API。 |
| Isaac Sim 主页 | [NVIDIA Isaac Sim](https://developer.nvidia.com/isaac/sim) | Isaac Sim 是基于 Omniverse/OpenUSD 的机器人仿真、测试、合成数据框架，可接收 CAD/URDF/NuRec/TeleOp 等输入。 | RoboLab 的仿真运行时和未来 NuRec/SimReady 场景都落在 Isaac Sim 生态里。 | OpenUSD、physics、sensor、synthetic data、Cosmos augment。 |
| Lyra | [NVIDIA Research Lyra](https://research.nvidia.com/labs/toronto-ai/lyra/) | 用 video diffusion self-distillation 生成 3D/4D Gaussian scenes，可从 text/image/video 得到 3DGS 世界，并演示导入 Isaac Sim。 | 指向未来：从 prompt/image/video 自动生成可探索场景，再交给 Isaac/NuRec 做 physical AI 测试。 | 3DGS decoder、text/image/video-to-3D、Isaac Sim import。 |
| Physically Embodied Gaussians | [NVIDIA Blog: Warp + Gaussian Splatting](https://developer.nvidia.com/blog/building-robotic-mental-models-with-nvidia-warp-and-gaussian-splatting/) | 把 particles 作为物理结构、3D Gaussians 作为视觉外观，用 differentiable rendering 让视觉误差反向修正物理状态。 | 这是 RoboLab 之外更激进的方向：机器人持续维护视觉-物理一致的内部世界模型。 | particles + Gaussians dual representation、NVIDIA Warp、gsplat、closed-loop correction。 |
| Marble + Isaac Sim + NuRec | [NVIDIA Blog: Isaac Sim + World Labs Marble](https://developer.nvidia.com/blog/simulate-robotic-environments-faster-with-nvidia-isaac-sim-and-world-labs-marble/) | 从 Marble 导出 Gaussian splat PLY 和 collider GLB，经 3DGRUT/NuRec 转成 USDZ，在 Isaac Sim 中对齐视觉高斯和碰撞网格。 | 最像 RoboLab Figure 13 的工程版本：photoreal splat 负责视觉，collider mesh 负责物理。 | PLY/GLB 导出、PLY-to-USDZ、Gaussian/collider 对齐、physics collider。 |

### 1. Omniverse NuRec：把真实传感器数据变成可交互仿真

NuRec 是 NVIDIA Omniverse 的神经重建和 3D Gaussian Splatting 库。它的核心路线是：

```text
camera / lidar data
-> reconstruction / NCore data standard
-> USD scene + trajectories/metadata
-> gsplat / NuRec rendering
-> Isaac Sim / AlpaSim / CARLA interactive simulation
```

它解决的是：

- 从真实传感器数据重建真实环境。
- 用 Gaussian splatting 在仿真器里交互式渲染。
- 通过 USD 和 Isaac Sim 对接机器人测试。

**【绿色标识｜和 RoboLab 的关系】**

RoboLab 论文展示的是 Gaussian Splat + Mesh 场景；NuRec 是 NVIDIA 更工程化的“真实数据 -> 高斯重建 -> OpenUSD/Isaac Sim 渲染”的产品/库路线。

**来源链接**：[NVIDIA Omniverse NuRec](https://developer.nvidia.com/omniverse/nurec)  
**重点阅读**：How Omniverse NuRec Works、NCore data standard、NuRec rendering 和 Isaac Sim integration。

### 2. 3DGRT / 3DGUT：让 Gaussian 不只适合理想 pinhole camera

传统 3DGS 很快，但有局限：

- 默认偏理想 pinhole camera。
- 畸变相机、fisheye、rolling shutter 难处理。
- secondary rays，如反射/折射/阴影，支持有限。

NVIDIA 的 3DGRT/3DGUT 路线分别解决：

| 方法 | 重点 |
|---|---|
| 3DGRT | 对 Gaussian particle 做 ray tracing，用 BVH/RT 硬件处理复杂 rays |
| 3DGUT | 用 Unscented Transform 替代线性化投影，支持非线性相机、rolling shutter，并和 secondary ray tracing 对齐 |

**【机器人意义】**

机器人和自动驾驶常用 wide-FOV、fisheye、rolling-shutter 相机。3DGUT 这类方法让 Gaussian 重建更接近真实传感器，而不是只在理想相机下好看。

**来源链接**：[NVIDIA Research 3DGUT](https://research.nvidia.com/labs/toronto-ai/3DGUT/)  
**重点阅读**：nonlinear camera projections、rolling shutter、reflections/refractions、3DGRT 对齐。

### 3. Isaac Sim 6.0 Early Developer Release：NuRec 进入仿真运行时

NVIDIA 2026 的 Isaac Sim 6.0 early developer release 提到 Kit 110 带来 NuRec 3D Gaussian splatting libraries 与 Fabric Scene Delegate integration。  
这意味着 Gaussian/NuRec 不只是离线 viewer，而是在 Isaac Sim 运行时中越来越一等公民。

相关趋势：

- NuRec 3DGS 库和 Isaac Sim 集成。
- Fabric 加速 runtime scene data。
- 多物理后端：PhysX / Newton。
- Warp-based Core API，更利于跨 physics backend。

**来源链接**：[Isaac Sim 6.0 Early Developer Release for GTC'26](https://forums.developer.nvidia.com/t/announcement-isaac-sim-6-0-early-developer-release-for-gtc26/363709)  
**补充入口**：[NVIDIA Isaac Sim](https://developer.nvidia.com/isaac/sim)

### 4. Lyra / Lyra 2.0：从生成式视频模型到 3DGS 世界

Lyra 是 NVIDIA 的 generative 3D scene reconstruction 路线。  
核心思想：

```text
text / single image / video
-> camera-controlled video diffusion
-> self-distillation
-> 3D Gaussian Splatting decoder
-> explicit 3D / 4D scene
```

Lyra 1.0 偏“单图/视频 -> 3D/4D Gaussian”。  
Lyra 2.0 进一步面向 explorable generative 3D worlds，强调长 horizon、3D consistent generation。

**【和 RoboLab 的关系】**

RoboLab 主要是在已有场景/资产/任务上做 benchmark。Lyra 这种路线更像未来：直接从 prompt 或图像生成可探索 3D 世界，再导入 Isaac Sim/NuRec 做物理 AI 测试。

**来源链接**：[NVIDIA Research Lyra](https://research.nvidia.com/labs/toronto-ai/lyra/)  
**重点阅读**：3DGS decoder、text/image/video-to-3D、Humanoid Robot Simulation in Generated 3D Scenes。

### 5. Physically Embodied Gaussians：视觉高斯 + 物理粒子的实时闭环

NVIDIA 技术博客里还有一个前沿方向：Physically Embodied Gaussians。  
它把世界拆成双表示：

| 表示 | 作用 |
|---|---|
| particles | 物理结构，被 physics engine 驱动 |
| 3D Gaussians | 视觉外观，用 Gaussian splatting 渲染 |

视觉误差通过 differentiable rendering 反过来修正物理状态。  
这条路线更像“机器人脑内持续更新的世界模型”，不是 RoboLab 本文的 benchmark pipeline，但方向非常接近 physical AI 的未来。

**来源链接**：[Building Robotic Mental Models with NVIDIA Warp and Gaussian Splatting](https://developer.nvidia.com/blog/building-robotic-mental-models-with-nvidia-warp-and-gaussian-splatting/)  
**重点阅读**：dual representation、differentiable rendering、Warp + gsplat、real-time correction。

### 6. Marble + Isaac Sim + NuRec 工作流

NVIDIA 也演示了把 World Labs Marble 生成的 Gaussian splat PLY 和 collider GLB 导入 Isaac Sim：

```text
Marble scene
-> Gaussian splat PLY + collider GLB
-> NuRec / 3DGRUT conversion to USDZ
-> align Gaussian volume and collider mesh
-> Isaac Sim physics + lighting + robot
```

这条路线说明一个核心工程事实：

> 高斯体负责 photoreal visual，collider mesh 负责 physics。两者必须对齐。

**来源链接**：[Simulate Robotic Environments Faster with NVIDIA Isaac Sim and World Labs Marble](https://developer.nvidia.com/blog/simulate-robotic-environments-faster-with-nvidia-isaac-sim-and-world-labs-marble/)  
**重点阅读**：Gaussian splat PLY、collider GLB、PLY-to-USDZ、Gaussian volume 与 collider mesh 对齐。

## 和本文 RoboLab 的关系表

| 方法 | RoboLab 本文状态 | 对仿真可信度的贡献 |
|---|---|---|
| Gaussian Splat + Mesh scene | 本文 Figure 13 展示 | 提高背景/真实场景视觉一致性 |
| collision mesh for splat | 本文 Figure 13 提到 | 让高斯背景具有可碰撞几何代理 |
| mesh foreground | 本文 Figure 13 提到 | 提供可交互/可操作前景对象 |
| VoMP mass/density | 本文 Figure 13 / Appendix A 引用 | 补物体机械属性 |
| 3DGRT / 3DGUT | 本文 Figure 13 引用相关方法 | 改善高斯渲染对复杂相机/secondary rays 的支持 |
| NuRec | NVIDIA 当前前沿工程路线 | 把 camera/lidar 重建结果封装成 OpenUSD/Isaac 可交互仿真 |
| Lyra / Lyra 2.0 | NVIDIA 2026 generative world route | 从 text/image/video 生成 3DGS 世界 |
| Physically Embodied Gaussians | 相邻前沿研究方向 | 视觉高斯和物理粒子实时闭环 |

## 我们实际复现时该怎么用

在 4090 上继续推进 RoboLab 时，建议分三层：

### L1：RoboLab 官方 benchmark

先把官方 mesh/USD 场景和任务跑稳。  
不要一上来改成 Gaussian/NuRec 场景，否则失败原因会混杂。

### L2：Gaussian scene smoke

如果后续拿到 Gaussian Splat + collider mesh：

1. 检查 Gaussian volume 和 collider mesh 是否对齐。
2. 检查 scale 是否接近真实米制单位。
3. 用简单机器人/box 碰撞验证不会穿模。
4. 再接 RoboLab task/policy。

### L3：NuRec / Lyra 前沿路线

如果想复现 2026 NVIDIA 前沿：

1. NuRec：真实 camera/lidar -> OpenUSD/Isaac interactive simulation。
2. 3DGUT/3DGRT：复杂相机、rolling shutter、secondary rays 的高斯渲染。
3. Lyra 2.0：从单图/文本生成可探索 3DGS world。
4. Physically Embodied Gaussians：视觉和物理状态实时闭环。

## 小结

本文使用 Gaussian 的关键不是“高斯替代仿真”，而是：

```text
Gaussian Splat 解决视觉真实
Collision Mesh 解决物理碰撞
Mesh Foreground 解决可交互对象
VoMP/物理属性 解决质量/密度/摩擦
Isaac Sim 解决动力学与传感器
MNPE 解决扰动敏感性统计分析
```

2026 NVIDIA 的前沿趋势则是把这条链路做得更自动、更实时、更生成式：

```text
NuRec: 真实数据 -> OpenUSD/Isaac 高斯仿真
3DGUT/3DGRT: 高斯渲染支持复杂相机和 secondary rays
Lyra 2.0: 生成式 3DGS 世界
Embodied Gaussians: 视觉高斯 + 物理粒子实时闭环
```

In [ ]:
# ===== 精讲12：Gaussian 仿真路线轻量验证 =====
# 这个 cell 不跑 3DGS/NuRec 重建，而是把论文和 NVIDIA 2026 前沿路线
# 拆成可检查的数据结构，避免把“视觉高斯”“物理碰撞”“统计高斯核”混在一起。

import json

robolab_gaussian_stack = {
    "gaussian_splat_background": {
        "direct_in_paper": True,
        "input": ["images_or_reconstructed_scene"],
        "output": ["photorealistic_background_rendering"],
        "role": "visual_rendering",
        "physics_contact": False,
        "plain_cn": "负责让背景看起来真实，但本身不是机器人碰撞体。",
    },
    "collision_mesh_for_splat": {
        "direct_in_paper": True,
        "input": ["reconstructed_or_estimated_geometry"],
        "output": ["collider_proxy_for_physics"],
        "role": "physical_collision",
        "physics_contact": True,
        "plain_cn": "负责让机器人和背景几何发生碰撞，避免只看得到却穿过去。",
    },
    "mesh_foreground_objects": {
        "direct_in_paper": True,
        "input": ["SimReady_or_catalog_assets"],
        "output": ["interactive_visual_mesh_and_collision_mesh"],
        "role": "manipulable_objects",
        "physics_contact": True,
        "plain_cn": "负责桌面上的可抓取、可放置、可堆叠前景对象。",
    },
    "vomp_mass_density": {
        "direct_in_paper": True,
        "input": ["object_geometry_and_mechanical_property_estimate"],
        "output": ["mass_density_or_mass_proxy"],
        "role": "mechanical_properties",
        "physics_contact": "supports_dynamics",
        "plain_cn": "补质量/密度这类物理参数，不是渲染方法。",
    },
    "mnpe_gaussian_kde": {
        "direct_in_paper": True,
        "input": ["sampled_variation_parameters"],
        "output": ["importance_sampling_density_weight"],
        "role": "statistical_density_estimation",
        "physics_contact": False,
        "plain_cn": "这是 MNPE 统计分析里的高斯核密度估计，不是 3D Gaussian Splatting。",
    },
}

nvidia_frontiers_2026 = {
    "omniverse_nurec": {
        "status": "frontier_engineering",
        "url": "https://developer.nvidia.com/omniverse/nurec",
        "source_title": "NVIDIA Omniverse NuRec",
        "input": ["camera_data", "lidar_data"],
        "output": ["OpenUSD_scene", "interactive_gaussian_rendering"],
        "why_relevant": "把真实数据重建成 Isaac/Omniverse 可用的神经重建场景。",
        "read_focus": ["NCore data standard", "USD scene", "gRPC rendering", "Isaac Sim integration"],
    },
    "3dgut_3dgrt": {
        "status": "research_to_tooling",
        "url": "https://research.nvidia.com/labs/toronto-ai/3DGUT/",
        "source_title": "NVIDIA Research 3DGUT",
        "input": ["3D_Gaussians"],
        "output": ["nonlinear_camera_rendering", "rolling_shutter", "secondary_rays"],
        "why_relevant": "让高斯渲染支持复杂相机和反射/折射等 secondary rays。",
        "read_focus": ["nonlinear camera projections", "rolling shutter", "reflections/refractions", "3DGRT alignment"],
    },
    "isaac_sim_6_nurec": {
        "status": "2026_early_developer_release",
        "url": "https://forums.developer.nvidia.com/t/announcement-isaac-sim-6-0-early-developer-release-for-gtc26/363709",
        "source_title": "Isaac Sim 6.0 Early Developer Release for GTC'26",
        "input": ["NuRec_3DGS_scene"],
        "output": ["Fabric_Scene_Delegate_integration"],
        "why_relevant": "把 NuRec/3DGS 更直接地放进 Isaac Sim 工程栈。",
        "read_focus": ["NuRec 3DGS libraries", "Fabric Scene Delegate", "PhysX/Newton", "Warp-based Core API"],
    },
    "lyra_2": {
        "status": "2026_generative_world_model",
        "url": "https://research.nvidia.com/labs/toronto-ai/lyra/",
        "source_title": "NVIDIA Research Lyra",
        "input": ["text", "single_image", "video"],
        "output": ["3DGS_world", "explorable_long_horizon_world"],
        "why_relevant": "从生成式输入产生 3DGS 世界，适合未来快速构造仿真场景。",
        "read_focus": ["3DGS decoder", "text/image/video-to-3D", "Isaac Sim import", "dynamic 3D/4D scenes"],
    },
    "physically_embodied_gaussians": {
        "status": "adjacent_frontier_research",
        "url": "https://developer.nvidia.com/blog/building-robotic-mental-models-with-nvidia-warp-and-gaussian-splatting/",
        "source_title": "NVIDIA Blog: Building Robotic Mental Models with NVIDIA Warp and Gaussian Splatting",
        "input": ["few_camera_views", "robot_priors", "interaction_feedback"],
        "output": ["particles_plus_3D_Gaussians_live_world_model"],
        "why_relevant": "把视觉高斯和物理粒子绑定，让机器人边看边更新物理世界模型。",
        "read_focus": ["particles + Gaussians dual representation", "differentiable rendering", "NVIDIA Warp", "gsplat"],
    },
    "marble_isaac_nurec_workflow": {
        "status": "integration_workflow",
        "url": "https://developer.nvidia.com/blog/simulate-robotic-environments-faster-with-nvidia-isaac-sim-and-world-labs-marble/",
        "source_title": "NVIDIA Blog: Simulate Robotic Environments Faster with NVIDIA Isaac Sim and World Labs Marble",
        "input": ["Gaussian_splat_PLY", "collider_GLB"],
        "output": ["USDZ_or_Isaac_Sim_scene"],
        "why_relevant": "工程上展示了 photoreal splat 与 collider mesh 必须对齐。",
        "read_focus": ["Gaussian splat PLY", "collider GLB", "PLY-to-USDZ", "Gaussian/collider alignment"],
    },
}

nvidia_frontier_link_map = {
    name: {
        "source_title": record["source_title"],
        "url": record["url"],
        "read_focus": record["read_focus"],
    }
    for name, record in nvidia_frontiers_2026.items()
}

def classify_gaussian_method(name, record):
    # 把“高斯”按职责拆开，防止把视觉表示、碰撞几何、统计 KDE 混为一谈。
    if record["role"] == "visual_rendering":
        return "视觉层"
    if record["role"] in {"physical_collision", "manipulable_objects", "mechanical_properties"}:
        return "物理/几何层"
    if record["role"] == "statistical_density_estimation":
        return "统计分析层"
    return "其他"

layer_map = {
    name: classify_gaussian_method(name, record)
    for name, record in robolab_gaussian_stack.items()
}

gaussian_method_tests = [
    (
        "paper_has_visual_splat_and_physics_mesh",
        robolab_gaussian_stack["gaussian_splat_background"]["direct_in_paper"]
        and robolab_gaussian_stack["collision_mesh_for_splat"]["direct_in_paper"],
    ),
    (
        "gaussian_splat_not_claimed_as_physics",
        robolab_gaussian_stack["gaussian_splat_background"]["physics_contact"] is False,
    ),
    (
        "collision_mesh_is_contact_layer",
        robolab_gaussian_stack["collision_mesh_for_splat"]["physics_contact"] is True,
    ),
    (
        "foreground_mesh_is_manipulation_layer",
        robolab_gaussian_stack["mesh_foreground_objects"]["role"] == "manipulable_objects",
    ),
    (
        "vomp_is_mechanical_not_rendering",
        robolab_gaussian_stack["vomp_mass_density"]["role"] == "mechanical_properties",
    ),
    (
        "mnpe_kde_is_statistical_not_3dgs",
        layer_map["mnpe_gaussian_kde"] == "统计分析层",
    ),
    (
        "frontiers_cover_nurec_3dgut_lyra",
        all(key in nvidia_frontiers_2026 for key in ["omniverse_nurec", "3dgut_3dgrt", "lyra_2"]),
    ),
    (
        "each_frontier_has_url",
        all(record["url"].startswith("https://") for record in nvidia_frontiers_2026.values()),
    ),
    (
        "each_frontier_has_read_focus",
        all(record["read_focus"] for record in nvidia_frontiers_2026.values()),
    ),
    (
        "frontier_link_map_includes_marble_workflow",
        "marble_isaac_nurec_workflow" in nvidia_frontier_link_map,
    ),
    (
        "each_frontier_has_input_and_output",
        all(record["input"] and record["output"] for record in nvidia_frontiers_2026.values()),
    ),
    (
        "isaac_sim_6_marked_as_2026_integration",
        nvidia_frontiers_2026["isaac_sim_6_nurec"]["status"] == "2026_early_developer_release",
    ),
    (
        "not_all_frontiers_claimed_as_robolab_direct_use",
        nvidia_frontiers_2026["physically_embodied_gaussians"]["status"] == "adjacent_frontier_research",
    ),
]

report = {
    "robolab_gaussian_stack": robolab_gaussian_stack,
    "layer_map": layer_map,
    "nvidia_frontiers_2026": nvidia_frontiers_2026,
    "nvidia_frontier_link_map": nvidia_frontier_link_map,
    "tests": [{"name": name, "passed": bool(ok)} for name, ok in gaussian_method_tests],
    "all_passed": all(ok for _, ok in gaussian_method_tests),
    "boundary": "This is a source-grounded conceptual test. It does not run NuRec/3DGS rendering or Isaac Sim physics.",
}

print(json.dumps(report, ensure_ascii=False, indent=2))
for name, ok in gaussian_method_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert report["all_passed"], gaussian_method_tests
write_status("gaussian_sim_methods_lightweight_tests", report)

## 0.19 论文精讲：剩余核心内容与评测证据链

下面这节来自本目录的 [EXPLAIN_13_remaining_core_topics.md](./EXPLAIN_13_remaining_core_topics.md)。它不是重复前面的 scene/task/solver/MNPE/Gaussian，而是补论文里还没系统讲透的评测侧核心：实验协议、`success` 与 `score` 的区别、语言变体、复杂度 sweep、事件追踪、真实世界相关性、统计置信和限制边界。

# 精讲 13：对照论文后还没讲透的核心内容

> **【绿色标识｜核心结论】** 前 12 讲已经覆盖了 RoboLab 的生成链路、任务结构、能力轴、SPARC、MNPE、baseline、DTGE、prompt、solver、Gaussian 路线。剩下最值得补的一块不是“再讲一个算法”，而是论文如何把一次机器人 rollout 变成可信评测证据：实验协议、严格成功率与部分分数、语言变体、错误事件、真实世界相关性、统计置信和限制边界。
>
> **【蓝色标识｜源码/输入输出路径】** 这部分主要对应论文 III-C Metrics、IV Experiments、IV-D Real-World Verification、V Limitations，以及 RoboLab docs 里的 `analysis/read_results.py`、`docs/data.md`、`docs/event_tracking.md`、`docs/dashboard.md`、`docs/subtask.md`。
>
> **【橙色标识｜容易误解】** 视频好看不等于复现成功；单任务成功不等于 RoboLab-120；`success=True` 不等于模型完全懂任务；`score>0` 才能告诉我们模型在哪些中间步骤已经会了、在哪里失败。

## 先做覆盖差分

| 论文核心点 | 前面是否已讲 | 还缺什么 |
|---|---:|---|
| scene/task/env generation | 已讲，精讲 1-3、10-11 | 不重复 |
| competency axes/difficulty | 已讲，精讲 4 | 不重复 |
| SPARC trajectory metrics | 已讲，精讲 5 | 不重复 |
| MNPE sensitivity | 已讲，精讲 6 | 不重复 |
| scene generation baseline / DTGE / prompt | 已讲，精讲 7、9、10 | 不重复 |
| Gaussian Splat + Mesh / NuRec | 已讲，精讲 12 | 不重复 |
| evaluation protocol | 弱讲 | 需要明确输入、输出、episode 数、policy/robot/action/camera 绑定 |
| success vs score | 弱讲 | 需要解释为什么论文强调 score/success gap |
| language variations | 弱讲 | 需要解释 vague/default/specific 为什么是泛化测试 |
| event tracking / failure taxonomy | 弱讲 | 需要解释 wrong object、drop、hit table 等事件怎么帮助诊断 |
| real-world verification | 弱讲 | 需要解释 RoboArena 相关性是什么、不是什么 |
| statistical confidence / dashboard | 弱讲 | 需要解释 95% CI、Beta interval、Student-t、结果聚合 |
| limitations | 弱讲 | 需要讲清楚哪些任务 RoboLab 暂时不擅长评估 |

## 1. 实验协议：RoboLab 真正测的是什么

论文的实验设置不是“打开一个仿真，看看机器人能不能动”。它更像一个标准化测评表：

```text
task + scene + instruction variant
  + robot embodiment
  + policy client/server
  + camera/action/observation config
  + controlled variations
  + repeated episodes
  -> episode_results + videos + HDF5 + event logs + metrics
```

论文 IV-A 的关键信息：

- 策略是现实数据训练出来的 task-generalist policy，而不是在 RoboLab 任务里专门训练的 policy。
- action space 是 Franka 7-DOF joint positions + 1-DOF binary gripper command。
- 环境用 office-like background 和 natural lighting，腕部相机与外部相机尽量贴近 DROID 真实机器人设置。
- 每个任务重复运行多个 episode，论文里每个任务是 10 episodes。

说人话：

> RoboLab 的重点是把真实机器人策略拿到一个高保真、可控、可重复的仿真考试场里，看它是否真的泛化，而不是让模型提前适应这个仿真器。

### 输入输出

| 层 | 输入 | 输出 | 为什么重要 |
|---|---|---|---|
| task | scene、instruction、termination、subtasks、attributes | env config | 决定要考什么 |
| policy | observation images、proprio、instruction | robot action | 决定模型如何行动 |
| robot/env | action、physics、camera、contact | next observation、events | 决定 rollout 过程 |
| logging | per-step state、events、subtask status | HDF5、JSONL、videos | 决定可解释证据 |
| analysis | episode_results、HDF5、event logs | success、score、CI、wrong object、SPARC | 决定论文表格 |

## 2. Success 和 Score 的区别：论文最容易被低估的一点

论文 IV-B 明确强调：成功率是最终是否完成任务，score 表示过程中是否完成了部分里程碑。这个差异非常关键。

例如一个任务：

```text
Put the cube and the banana in the bowl
```

可以拆成：

```text
cube: grab -> hover/above bowl -> drop -> in bowl
banana: grab -> hover/above bowl -> drop -> in bowl
```

如果模型只把 banana 放进 bowl，但 cube 没完成：

```text
success = False
score   > 0
```

这不是“失败就没价值”。它说明模型可能已经掌握了：

- 语言里的一部分目标。
- 某个对象的识别。
- 抓取/搬运/放置中的部分动作。

但它还没掌握：

- 多对象组合。
- 顺序规划。
- 全部目标的终止条件。

**【绿色标识｜核心直觉】**

`success` 回答“最后有没有做完”；`score` 回答“做到了哪一步”。RoboLab 的分析价值主要来自第二个问题。

## 3. Language Variations：为什么 vague/default/specific 很重要

论文 III-C 和 IV-B 都强调 instruction variation。RoboLab 同一个底层任务可以有不同语言版本：

| 类型 | 例子 | 考什么 |
|---|---|---|
| default | Put the cube and the banana in the bowl | 标准任务理解 |
| vague | Put everything in the bowl | 抽象语言到目标集合 |
| specific | Pick up the rubiks cube and the yellow banana and place both inside the bowl | 细粒度对象绑定 |

论文观察到：指令越抽象，很多模型成功率会明显下降。  
这说明模型可能不是在理解“目标状态”，而是在依赖训练中常见的语言模板。

说人话：

> 真懂任务的机器人，不应该因为“把香蕉放进碗里”和“把黄色水果放进碗里”的说法不同，就完全换一种行为。

我们后续复现如果要靠近论文，不能只跑 default instruction，至少要跑：

```bash
python analysis/read_results.py <run> --by-instruction-type
```

否则看不到语言泛化这一层。

## 4. 三条复杂度 sweep：不是能力轴，而是诊断轴

前面精讲 4 已经讲了 visual/procedural/relational 能力轴。论文 IV-B 还有另一组实验轴：

| 诊断轴 | 怎么变 | 观察什么 |
|---|---|---|
| instruction specificity | vague/default/specific | 语言抽象程度影响 |
| scene complexity | 增加桌面可见物体数量 | 视觉 clutter 和目标识别稳定性 |
| task horizon | 增加 subtasks/多步长度 | 多步骤规划和长期执行稳定性 |

这三条轴不是重新定义任务类别，而是把同一类任务放进不同压力测试里。

例如：

```text
同一个 pick-place 能力
  -> 指令变 vague
  -> 桌面多几个干扰物
  -> 从一个物体变成多个物体
```

如果只看总成功率，很难知道模型是“看错了对象”“没听懂语言”还是“长时序执行崩了”。  
这三条 sweep 就是把失败原因拆开。

## 5. Event Tracking：为什么 Figure 3 比成功率更有用

论文 Figure 3 展示了几类失败：

- 抓了错误对象。
- 太早放手，目标没进容器。
- 先重定向了无关对象。
- 多次尝试错误对象。

RoboLab docs 的 event tracking 也对应这些诊断事件，例如：

- `WRONG_OBJECT_GRABBED`
- `TARGET_OBJECT_DROPPED`
- `GRIPPER_HIT_TABLE`
- `OBJECT_MOVED`
- `OBJECT_OUT_OF_SCENE`
- `OBJECT_TIPPED_OVER`
- `MULTIPLE_OBJECTS_GRABBED`

说人话：

> 成功率告诉你“考了多少分”；事件日志告诉你“错题错在哪里”。

### 输入输出

```text
input:
  per-step world state
  gripper contact state
  target object list
  object poses and movement thresholds

output:
  event log JSON
  wrong-object counts
  dropped-object counts
  object-moved/tipped/out-of-scene counts
```

这也解释了为什么我们复现时要保存：

- `log_0_env0.json`
- `episode_results.jsonl`
- `run_0.hdf5`
- 视频

视频是给人看，event/HDF5/JSON 是给统计和论文表格用。

## 6. Real-World Verification：RoboLab 能不能代表真实世界

论文 IV-D 做了一件重要但容易被误读的事：它把 RoboLab-120 上的策略成功率和 RoboArena 真实机器人 benchmark 的 Elo 排名做相关性比较。

这里重点是：

- RoboLab 输出 success rate。
- RoboArena 输出 Elo score。
- 两者单位不一样。
- 因此 Spearman rank correlation 更合适，因为它看的是策略排名是否一致。

说人话：

> RoboLab 不是声称“仿真分数等于真实分数”，而是说“在这些策略上，仿真排序和真实排序有一致性，所以它可以作为真实评测的有用 proxy”。

边界也要讲清楚：

- 这不是任务级别逐项一致性证明。
- 这不是 motion-level 行为一致性证明。
- 这不是说所有真实世界 failure 都能在仿真里复现。
- 样本策略数量有限，论文也把更深的相关性分析留给 future work。

## 7. 统计置信和 Dashboard：为什么不能只看一条视频

论文和项目 docs 都强调结果分析：

- 每个任务有多 episode。
- `episode_results.jsonl` 是聚合主数据。
- HDF5 存轨迹、动作、状态、subtask 等细节。
- dashboard 和 analysis 脚本按任务、属性、难度、场景、wrong objects、instruction type 聚合。
- dashboard 中 success rate 使用 Beta credible interval，score 使用 Student-t 区间。

我们复现时如果只给一条视频，只能说明：

```text
这条 episode 发生过
```

还不能说明：

```text
这个策略在这个任务族上稳定有效
```

**【橙色标识｜复现边界】**

我们当前已经有 Pi05 单任务成功闭环和 3 个复杂任务抽样，但这仍然不是完整 RoboLab-120，也不是论文级统计结果。

## 8. Appendix A 里还有两个细节值得记

### 8.1 Statistical significance

论文 Appendix A-A 讨论统计显著性。这里的要点不是某个单独公式，而是评测必须有 episode 数和置信区间。  
对我们来说，最实用的原则是：

```text
不要用 1 episode 的 success=True 推导整体能力。
至少按任务重复，再按属性/难度/语言变体聚合。
```

### 8.2 Anomalous long-horizon success

Appendix A-D 还解释了一个看似反常的现象：某些长 horizon 任务反而表现更好。  
论文的解读是，这可能不是模型突然会了长任务，而是特定任务的结构、对象分布或目标状态让它更容易触发部分成功。

说人话：

> 不要看到“长任务成功率更高”就立刻下结论说模型更擅长长时序。要回头看具体任务、对象和 subtask 结构。

## 9. Limitations：论文自己承认哪些事还没解决

论文 V Limitations 说得很直接，RoboLab 目前主要是：

- rigid-body tabletop scenes。
- 桌面操作。
- 可用谓词描述的目标状态。

还不充分覆盖：

- cloth、cables、bags 等 deformable objects。
- 需要精细力控的 contact-rich skills。
- 复杂摩擦、顺应性接触、低层控制任务。
- open-ended ambiguous long-horizon tasks，比如 “clean up all the laundry”。
- 残余视觉分布偏移。

这对我们复现很重要。因为当一个任务失败时，先判断它属于哪类：

| 失败类型 | 该不该怪 RoboLab |
|---|---|
| 资产缺失、环境启动失败 | 复现环境问题 |
| 模型抓错对象 | policy/generalization 问题 |
| 模型做了一半但未完成 | score/subtask 诊断问题 |
| cloth/cable/force-control 失败 | RoboLab 当前能力边界 |
| 单任务视频成功但统计不足 | 证据规模问题 |

## 10. 和我们当前复现记录怎么接起来

我们现在已经完成：

- `BananaInBowlTask` Pi05 成功闭环。
- 三个复杂任务抽样。
- no-policy 初始化 smoke。
- 视频、HDF5、JSON、event log 同步。
- notebook 里对生成、solver、prompt、metrics、MNPE、Gaussian 的解释。

如果要更接近论文级复现，下一步应该按这个顺序：

1. 同一个任务跑 `default/vague/specific`，看语言变体。
2. 同一个任务跑 object pose / camera pose / lighting / background sweep，接 MNPE。
3. 复杂任务至少按属性分组，不只挑几个视频。
4. 用 `analysis/read_results.py --by-attributes --by-difficulty --by-instruction-type --by-wrong-objects` 输出表。
5. 如果接 RoboChallenge/ReKep/OpenPI 多策略，再做 per-policy rank 对比。

## 小结

前面 12 讲回答了：

```text
RoboLab 怎么生成任务、怎么建场景、怎么判成功、怎么分析轨迹/敏感性。
```

精讲 13 补的是：

```text
RoboLab 怎么把 rollout 变成可信实验结论。
```

最重要的记法：

```text
success: 最终是否做完
score: 做到了哪一步
language variation: 是否真懂任务语义
event tracking: 错在什么行为
complexity sweep: 哪种压力导致失败
CI/dashboard: 结果是否有统计可信度
real-world verification: 仿真排序是否接近真实排序
limitations: 哪些问题不是这个 benchmark 当前擅长回答的
```

In [ ]:
# ===== 精讲13：剩余核心内容覆盖轻量验证 =====
# 这个 cell 把“还有哪些论文核心内容没讲透”变成一份可检查 coverage map。
# 它不替代论文实验，只验证 notebook 新增章节确实覆盖评测证据链的关键节点。

import json
from math import sqrt

remaining_core_topics = {
    "evaluation_protocol": {
        "paper_section": "IV-A Experiment Setup",
        "input": ["task", "scene", "instruction", "robot", "policy", "camera_config", "variation_seed"],
        "output": ["episode_results", "HDF5", "videos", "event_logs"],
        "why": "定义什么才算论文级 rollout，而不是单个好看的仿真视频。",
        "covered_in_explain13": True,
    },
    "success_vs_score_gap": {
        "paper_section": "III-C Metrics / IV-B Task Results",
        "input": ["subtask_state", "termination_state"],
        "output": ["strict_success", "partial_score"],
        "why": "success 看最终完成，score 看中间能力；二者差距暴露模型部分理解。",
        "covered_in_explain13": True,
    },
    "language_variations": {
        "paper_section": "III-C Language Variations / IV-B instruction specificity",
        "input": ["default_instruction", "vague_instruction", "specific_instruction"],
        "output": ["success_by_instruction_type"],
        "why": "验证策略是否理解目标状态，而不是只记住固定语言模板。",
        "covered_in_explain13": True,
    },
    "complexity_sweeps": {
        "paper_section": "IV-B instruction specificity / scene complexity / task horizon",
        "input": ["instruction_specificity", "visible_object_count", "subtask_count"],
        "output": ["performance_degradation_curve"],
        "why": "定位失败来自语言抽象、视觉干扰还是长时序规划。",
        "covered_in_explain13": True,
    },
    "event_tracking_failures": {
        "paper_section": "Figure 3 and docs/event_tracking.md",
        "input": ["world_state", "contact_state", "target_object_list"],
        "output": ["wrong_object", "dropped_target", "hit_table", "object_moved"],
        "why": "把失败从一个 False 拆成可诊断的行为错误。",
        "covered_in_explain13": True,
    },
    "real_world_verification": {
        "paper_section": "IV-D Real-World Verification",
        "input": ["robolab_success_rate_by_policy", "roboarena_elo_by_policy"],
        "output": ["rank_correlation", "proxy_interpretation"],
        "why": "解释 RoboLab 为什么可以作为真实评测 proxy，但不是逐任务等价证明。",
        "covered_in_explain13": True,
    },
    "statistical_confidence_dashboard": {
        "paper_section": "Appendix A-A and dashboard docs",
        "input": ["episode_results_jsonl", "score_samples", "success_samples"],
        "output": ["CI", "dashboard_tables", "analysis_csv"],
        "why": "防止用一条 episode 视频替代多 episode 统计结论。",
        "covered_in_explain13": True,
    },
    "limitations_boundary": {
        "paper_section": "V Limitations",
        "input": ["task_type", "object_physics", "instruction_ambiguity"],
        "output": ["benchmark_scope_judgment"],
        "why": "区分策略失败、环境失败和 benchmark 当前不擅长回答的问题。",
        "covered_in_explain13": True,
    },
}

def strict_success(final_state):
    # 论文里的 success 是最终状态是否满足 termination，不关心中途做到多少。
    return bool(final_state.get("termination_success", False))

def partial_score(group_progress):
    # 教学版 score：每个对象有 4 步，返回平均完成比例。
    return sum(done / total for done, total in group_progress.values()) / len(group_progress)

def spearman_rank(xs, ys):
    # 简化 Spearman：输入无并列名次时，计算两个排序的 Pearson。
    x_rank = {item: rank for rank, item in enumerate(sorted(xs, key=xs.get), start=1)}
    y_rank = {item: rank for rank, item in enumerate(sorted(ys, key=ys.get), start=1)}
    common = list(x_rank)
    xr = [x_rank[k] for k in common]
    yr = [y_rank[k] for k in common]
    mean_x = sum(xr) / len(xr)
    mean_y = sum(yr) / len(yr)
    cov = sum((a - mean_x) * (b - mean_y) for a, b in zip(xr, yr))
    var_x = sum((a - mean_x) ** 2 for a in xr)
    var_y = sum((b - mean_y) ** 2 for b in yr)
    return cov / sqrt(var_x * var_y)

def route_failure_case(case):
    # 把失败归因到复现问题、策略问题、证据规模问题或 benchmark 边界。
    if case.get("asset_missing") or case.get("env_crash"):
        return "复现环境问题"
    if case.get("deformable") or case.get("force_control"):
        return "RoboLab 当前边界"
    if case.get("single_episode_only"):
        return "证据规模不足"
    if case.get("wrong_object") or case.get("dropped_target"):
        return "策略泛化/执行问题"
    return "需要查看 event/HDF5/video"

score_example = {
    "banana": (4, 4),
    "rubiks_cube": (1, 4),
}
strict_example = strict_success({"termination_success": False})
score_value = partial_score(score_example)

robolab_policy_rank = {"pi05": 0.28, "pi0": 0.155, "groot": 0.072, "paligemma": 0.034}
roboarena_policy_rank = {"pi05": 1300, "pi0": 1200, "groot": 1100, "paligemma": 1000}
rank_corr = spearman_rank(robolab_policy_rank, roboarena_policy_rank)

explain13_tests = [
    ("all_remaining_topics_marked_covered", all(topic["covered_in_explain13"] for topic in remaining_core_topics.values())),
    ("each_topic_has_input_output_why", all(topic["input"] and topic["output"] and topic["why"] for topic in remaining_core_topics.values())),
    ("success_can_be_false_while_score_positive", strict_example is False and 0.0 < score_value < 1.0),
    ("language_variations_include_three_types", set(remaining_core_topics["language_variations"]["input"]) == {"default_instruction", "vague_instruction", "specific_instruction"}),
    ("complexity_sweeps_cover_three_axes", set(remaining_core_topics["complexity_sweeps"]["input"]) == {"instruction_specificity", "visible_object_count", "subtask_count"}),
    ("event_tracking_covers_wrong_and_dropped", {"wrong_object", "dropped_target"}.issubset(set(remaining_core_topics["event_tracking_failures"]["output"]))),
    ("real_world_proxy_uses_rank_correlation", remaining_core_topics["real_world_verification"]["output"][0] == "rank_correlation" and rank_corr > 0.99),
    ("limitation_router_detects_deformable_boundary", route_failure_case({"deformable": True}) == "RoboLab 当前边界"),
    ("limitation_router_detects_single_episode_gap", route_failure_case({"single_episode_only": True}) == "证据规模不足"),
    ("environment_failure_not_confused_with_policy_failure", route_failure_case({"asset_missing": True}) == "复现环境问题"),
]

report = {
    "remaining_core_topics": remaining_core_topics,
    "score_example": {
        "group_progress": score_example,
        "strict_success": strict_example,
        "partial_score": score_value,
        "interpretation": "模型做完 banana 但 cube 只完成抓取，所以最终失败但有部分分数。",
    },
    "rank_correlation_example": {
        "robolab_policy_rank": robolab_policy_rank,
        "roboarena_policy_rank": roboarena_policy_rank,
        "spearman": rank_corr,
        "boundary": "Toy ranking example for explaining proxy interpretation, not a fresh RoboArena result.",
    },
    "failure_routing_examples": {
        "asset_missing": route_failure_case({"asset_missing": True}),
        "deformable": route_failure_case({"deformable": True}),
        "single_episode": route_failure_case({"single_episode_only": True}),
        "wrong_object": route_failure_case({"wrong_object": True}),
    },
    "tests": [{"name": name, "passed": bool(ok)} for name, ok in explain13_tests],
    "all_passed": all(ok for _, ok in explain13_tests),
    "boundary": "This coverage check validates the explanation map. It does not rerun RoboLab-120 or compute new policy statistics.",
}

print(json.dumps(report, ensure_ascii=False, indent=2))
for name, ok in explain13_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert report["all_passed"], explain13_tests
write_status("remaining_core_topics_lightweight_tests", report)

## 0.19b 论文精讲：精讲13补充，评测证据链深挖

下面这节来自本目录的 [EXPLAIN_13_deep_evaluation_evidence_chain.md](./EXPLAIN_13_deep_evaluation_evidence_chain.md)。它把精讲13从“覆盖目录”补成“证据链深挖”：论文级 episode 样本单位、`success`/`score` 的数学直觉、event tracking 的输入输出、语言变体与复杂度 sweep 的交叉诊断、统计置信、dashboard、RoboArena 真实世界相关性，以及 4090 上更接近论文口径的小矩阵应该怎么设计。

# 精讲 13-补充：评测证据链深挖，从单条 rollout 到论文结论

> **【绿色标识｜核心结论】** 原来的精讲 13 讲到了“论文还剩哪些评测侧内容”，但讲得像目录。真正需要讲深的是：RoboLab 不是用一条视频证明模型会做任务，而是用一套证据链把 `task/policy/episode/event/HDF5/dashboard/statistics` 连起来，回答“策略到底会什么、错在哪里、结论有多稳、和真实世界排序有没有关系”。

> **【蓝色标识｜源头范围】** 这章主要对应 RoboLab 论文 III-C Metrics、IV Experiments、IV-D Real-World Verification、V Limitations、Appendix A-A/A-C/A-D，以及项目文档 `docs/data.md`、`docs/event_tracking.md`、`docs/subtask.md`、`docs/dashboard.md`。论文 HTML 版本为 arXiv:2604.09860v3，日期 2026-05-14。

> **【橙色标识｜最容易踩坑】** “我看到视频成功了”只能证明那条 episode 发生过；“某任务成功率 1/1”不能证明该任务稳定；“success=False”也不能证明模型完全不会，因为 `score` 和 event log 可能显示它完成了重要子目标。

---

## 0. 这章到底补什么

前面 0-12 讲已经覆盖了生成链路、能力轴、SPARC、MNPE、prompt、solver、Gaussian。精讲 13 的深层内容不是再补一个算法，而是补一条实验逻辑：

```text
一次 policy rollout
  -> 逐步记录世界状态、动作、事件、子任务进度
  -> 得到 success / score / event counts / trajectory metrics
  -> 按任务、能力轴、难度、语言变体、扰动维度聚合
  -> 给出置信区间、对比表、复杂度曲线、真实世界排序相关性
  -> 才能说“这个策略在 RoboLab 上有什么泛化能力和失败模式”
```

如果把这条链断开，就会出现几种误读：

| 误读 | 实际应该怎么判断 |
|---|---|
| 视频里放进去了，所以模型成功 | 还要看 `episode_results.jsonl` 的 `success` 和 termination 条件 |
| `success=False`，所以完全失败 | 还要看 normalized `score`、subtask 状态和失败事件 |
| 某任务 1/1 成功，所以这个任务会了 | 单任务单 episode 只是 smoke，不是统计评测 |
| 仿真成功率高，所以真实机器人一定强 | 论文看的是与真实 benchmark 的策略排序相关性，不是逐任务等价 |
| 复杂任务失败，所以模型不会长程 | 需要区分对象识别、空间关系、可供性、动作执行、事件中断和任务结构偏差 |

---

## 1. 论文级样本单位：到底什么叫一个 episode

RoboLab 里的最小评测单位不是“任务名”，而是一条绑定了完整上下文的 episode：

```text
episode_id =
  task_id
  + scene_id
  + instruction_variant
  + policy_id
  + robot_id
  + camera_config
  + action_space_config
  + variation_seed
  + run_seed
```

说人话：  
同一个 `BananaInBowlTask`，如果换了指令说法、换了相机、换了光照、换了对象初始位姿，或者换了策略，就不是同一条评测样本。它们应该被分开记录，再按维度聚合。

### 输入输出

| 阶段 | 输入 | 输出 | 如果缺失会怎样 |
|---|---|---|---|
| 任务定义 | scene、objects、instruction、subtasks、termination | env/task config | 不知道模型被要求做什么 |
| 策略调用 | RGB、robot state、语言指令、历史动作 | action chunk / joint command / gripper command | 无法复盘策略实际看到了什么 |
| 仿真步进 | action、物理状态、相机、碰撞 | next observation、object poses、contacts | 无法判断错误来自策略还是物理 |
| 事件追踪 | world state、目标对象、接触状态 | wrong object、drop、hit table 等事件 | `False` 变成黑盒失败 |
| 结果聚合 | success、score、events、trajectory | dashboard / CSV / JSON summary | 无法形成论文表格 |

所以我们复现时保存这些文件不是“仪式感”，而是证据分工：

| 文件 | 给谁看 | 主要回答 |
|---|---|---|
| `video.mp4` / viewport 视频 | 人 | 直观看模型行为 |
| `episode_results.jsonl` | 统计脚本 | 成功率、分数、任务属性聚合 |
| `run_*.hdf5` | 深度复盘 | 图像、动作、状态、subtask 轨迹 |
| `log_*.json` / event log | 失败诊断 | 错拿、掉落、碰撞、移动、越界 |
| dashboard | 研究者/汇报 | 按能力轴、难度、语言、扰动汇总 |

---

## 2. `success` 和 `score`：不是两个名字，而是两种科学问题

论文 III-C 里的 normalized score 和 IV-B / Appendix A-C 的 score 曲线，是 RoboLab 最关键的评测设计之一。

### 2.1 `success` 问的是最终状态

```python
success = task.termination_condition(world_state)
```

它只回答：

```text
最后目标状态是否满足？
```

例如：

```text
把苹果和橙子放进碗里
```

如果苹果进了碗，橙子没进：

```text
success = False
```

这符合严格任务定义，但信息量很低。

### 2.2 `score` 问的是过程里学会了多少

一个 pick-place 子任务可以拆成并行事件或里程碑：

```text
target object identified
  -> grasped
  -> lifted / moved toward goal
  -> released near goal
  -> final relation satisfied
```

教学版可以写成：

```python
score = mean(completed_milestones(object_i) / total_milestones(object_i))
```

如果香蕉 4/4 完成，方块 1/4 完成：

```text
success = False
score = (4/4 + 1/4) / 2 = 0.625
```

这说明模型不是完全不会，它至少完成了部分对象的识别、抓取或搬运。

### 2.3 为什么 complex 任务里 `success-score gap` 特别重要

论文 Appendix A-C 强调：复杂任务 strict success 掉得很厉害，但 score 仍保留了不少部分进展。  
这句话的含义是：

```text
复杂任务失败，并不等于每个子能力都失败。
```

要把复杂任务失败拆成：

| 可能原因 | success | score | event log |
|---|---:|---:|---|
| 完全不动 | False | 接近 0 | 无有效抓取 |
| 抓对但没放准 | False | 中等 | dropped / misplaced |
| 完成一半对象 | False | 中高 | missing remaining targets |
| 做错对象 | False | 可能中等 | wrong object |
| 最终全部完成 | True | 高 | 可能仍有中途错误 |

> **【绿色标识｜核心记法】** `success` 是终局判分，`score` 是过程剖面。RoboLab 真正有分析价值的是把二者一起看。

---

## 3. Event tracking：把失败从 “False” 变成错误类型

论文 Figure 3 很重要，因为它没有只展示成功/失败，而是展示模型执行中的具体错误：

- 过早松手，目标没进容器。
- 抓了错误对象。
- 先重定向了无关对象。
- 多次尝试错误对象。

RoboLab 的 event tracking 逻辑就是把这些可观察行为变成可统计事件。

### 3.1 事件检测需要什么输入

```text
world_state:
  object poses
  object velocities
  object semantic labels
  gripper pose
  gripper contact
  target object set
  workspace bounds
```

事件不是凭视频主观判断，而是从状态变化里推出来：

```text
gripper contacts non-target object
  -> WRONG_OBJECT_GRABBED

target object z suddenly decreases after grasp
  -> TARGET_OBJECT_DROPPED

gripper collision with table while moving
  -> GRIPPER_HIT_TABLE

object pose changes beyond threshold without being intended
  -> OBJECT_MOVED

object leaves workspace bounds
  -> OBJECT_OUT_OF_SCENE
```

### 3.2 事件日志怎么用于诊断

| 事件模式 | 更可能说明什么 | 后续该看什么 |
|---|---|---|
| wrong object 很多 | 视觉/语义/语言绑定失败 | object labels、camera view、instruction specificity |
| dropped target 很多 | 抓取、gripper 后处理、释放时机失败 | action chunk、gripper threshold、EEF trajectory |
| hit table 很多 | 运动轨迹不合理或相机/动作尺度错 | SPARC、speed、joint limits、控制频率 |
| object moved/tipped 很多 | 物理接触或场景拥挤 | asset collision、mass/friction、initial placement |
| out of scene | 策略失控或物理异常 | safety bounds、action clipping、policy server output |

### 3.3 视频和 event log 的分工

视频适合做定性解释，比如“模型看起来犹豫”。  
event log 适合做统计判断，比如“错误对象抓取率在 vague instruction 下上升”。

真正的评测汇报应该同时有：

```text
视频截图/片段
  + event counts
  + success/score
  + 轨迹质量
  + 任务属性
```

---

## 4. Instruction variation：语言泛化不是换一句话那么简单

RoboLab 的 language variation 不只是 prompt 多样化，它是在测试模型是否把语言映射到目标状态。

### 4.1 三种指令的含义

| 指令类型 | 例子 | 模型需要做什么 |
|---|---|---|
| specific | Put the yellow banana into the bowl | 绑定显式属性和对象 |
| default | Put the banana into the bowl | 按常规对象名执行 |
| vague | Put the fruit into the bowl | 从类别/上下文推断目标集合 |

vague 难在哪里？

```text
语言没有直接给对象实例名
  -> 模型要识别类别
  -> 在场景中绑定正确对象
  -> 排除无关但相似对象
  -> 执行目标关系
```

### 4.2 为什么语言变体要和事件日志一起看

只看 success：

```text
vague success rate 下降
```

你不知道原因。  
结合 event log 后，可以拆成：

| 下降原因 | 可观察信号 |
|---|---|
| 不知道 “fruit” 指哪些对象 | wrong object / no grasp |
| 目标集合不完整 | partial score 高但 success 低 |
| 抓对但放错 | target dropped / wrong relation |
| 指令过长导致动作延迟 | timeout / low progress |

这就是为什么 notebook 后续如果要做论文级复现，不能只固定 default instruction。

---

## 5. Complexity sweep：三条压力轴怎么读

论文 IV-B 里的 complexity sweep 包含：

```text
instruction specificity
scene complexity
task horizon
```

它们不是新的能力轴，而是压力测试轴。

### 5.1 instruction specificity

考的是语言绑定：

```text
specific -> default -> vague
```

如果 vague 下降明显，说明模型可能依赖模板语言，而不是真正形成目标状态。

### 5.2 scene complexity

考的是视觉 clutter 和目标选择：

```text
visible objects 少
  -> 目标更显眼
visible objects 多
  -> 干扰物更多
  -> 错拿概率上升
```

这里最好配合：

```text
wrong_object count
target_object_visibility
camera pose
object semantic category
```

否则“复杂场景失败”仍然太粗。

### 5.3 task horizon

考的是多步骤持续执行：

```text
subtask_count 增加
  -> 每一步都可能失败
  -> 累积失败概率上升
```

如果每个子任务成功概率是 `p`，粗略地：

```text
full_success ≈ p^N
```

当 `p=0.8`：

| 子任务数 | 理想化 full success |
|---:|---:|
| 1 | 0.80 |
| 3 | 0.512 |
| 5 | 0.328 |
| 7 | 0.210 |

这说明长任务成功率低不一定意味着“长程推理全错”，也可能是每一步都有小概率执行误差，最后连乘放大。

### 5.4 为什么 Appendix A-D 要提醒 “subtask=7 异常”

论文指出某个 7-subtask bin 有成功率回升，但原因可能是该 bin 被结构简单、重复 pick-place 的任务主导。  
这提醒我们：

```text
task horizon 数字本身不等于因果长程推理难度。
```

真正难的是：

- 后一步依赖前一步结果。
- 目标状态会改变可供性。
- 对象之间有空间/容器/堆叠约束。
- 错一步会导致后面无解。

---

## 6. 统计置信：为什么 10 episode 仍然很粗

论文 Appendix A-A 说得很明确：因为仿真和策略都有随机性，每个 task/policy 要跑多 episode。论文 benchmark 每个任务是 `N=10`，但作者也提醒，单任务层面的置信区间仍然很宽。

### 6.1 二项成功率的直觉

如果某任务 10 次里成功 5 次：

```text
success_rate = 50%
```

但这不代表真实成功概率精确等于 50%。因为样本太少，95% 区间会很宽。

论文给出的直觉是：

```text
N=10 时，p≈0.5 附近单任务 95% CI 半宽约 30%
N=100 时，p≈0.5 附近半宽约 10%
```

说人话：

> 10 次评测适合看总体趋势，不适合对单个任务做很细的强结论。1 次评测只适合 smoke。

### 6.2 为什么 aggregate 更可信

RoboLab-120 聚合时，有很多任务和 episode：

```text
effective_samples = tasks * episodes_per_task
```

总体 success rate / score 会比单任务更稳定。  
但这也有代价：聚合会掩盖特定类别的失败。

所以论文同时看：

```text
overall
  + difficulty
  + competency axis
  + task category
  + instruction specificity
  + scene complexity
  + task horizon
```

---

## 7. Dashboard：不是展示页，而是评测数据库视图

RoboLab dashboard 的价值不是“好看”，而是把底层证据按研究问题切开。

### 7.1 Dashboard 应该服务的问题

| 问题 | 应该聚合什么 |
|---|---|
| 哪个策略总体最强 | policy -> success/score |
| 哪类任务最难 | task category / competency axis |
| 是否只会简单任务 | difficulty level |
| 是否语言泛化差 | instruction type |
| 是否看错对象 | wrong object counts |
| 是否动作轨迹差 | SPARC / speed |
| 是否对扰动敏感 | camera / lighting / object pose variation |

### 7.2 为什么 dashboard 要接 HDF5 和 JSON

`episode_results.jsonl` 适合聚合。  
HDF5 适合深挖单条轨迹。

理想 workflow：

```text
dashboard 发现：某策略 spatial tasks score 低
  -> 筛出失败 episode
  -> 看 event counts：wrong object 还是 dropped target
  -> 打开 HDF5/video：看失败发生在哪一步
  -> 回到 task metadata：看是不是对象/关系/视角导致
```

这才是 RoboLab 比“只给 success table”更有价值的地方。

---

## 8. Real-world verification：相关性到底说明什么

论文 IV-D 把 RoboLab 结果和 RoboArena 真实世界评测做比较。这里非常容易误读。

### 8.1 它不是说仿真分数等于真实分数

两个 benchmark 的输出不是同一种东西：

```text
RoboLab: success rate / score in simulation
RoboArena: real-world pairwise ranking / Elo
```

所以比较重点不是数值相等，而是排序是否一致：

```text
policy A > policy B > policy C
```

这就是为什么 rank correlation 比直接比较分数更合理。

### 8.2 它能支持什么结论

可以支持：

```text
RoboLab 在一定策略集合上能给出和真实评测相近的排序信号。
```

不能支持：

```text
RoboLab 每个任务都等价于真实世界。
RoboLab success rate 可以直接换算成真实成功率。
RoboLab 覆盖所有真实机器人失败模式。
```

### 8.3 我们复现时怎么用这个结论

我们当前单任务 Pi05 成功不能用来证明真实相关性。  
但它可以作为第一步：

```text
先证明 pipeline 能跑通
  -> 再跑多个任务/策略
  -> 再比较策略排序
  -> 最后才谈 proxy validity
```

---

## 9. Limitations：哪些失败不应该强行归因给策略

论文 V Limitations 说明 RoboLab 当前主要面向 rigid-body tabletop manipulation。这个边界必须写进复现报告。

### 9.1 当前擅长的问题

- 桌面刚体对象。
- pick-place、containment、stacking、reorientation。
- 视觉属性：颜色、语义、尺寸。
- 空间关系：left/right/inside/on/near 等。
- 程序能力：可供性、重定向、堆叠。

### 9.2 当前不充分覆盖的问题

- deformable objects：布料、线缆、袋子。
- contact-rich skills：插拔、旋拧、精细力控。
- open-ended household chores：例如 “整理房间”。
- 强因果长程任务：后一步依赖前一步创造的状态。
- 真实机器人中的硬件误差、传感器噪声、网络延迟、控制器差异。

### 9.3 失败归因表

| 现象 | 首先归因 | 下一步检查 |
|---|---|---|
| Isaac 启动失败 | 环境/依赖 | driver、Isaac Sim、assets、headless |
| LFS 指针导致找不到 USD | 资产下载 | git-lfs、asset path、文件大小 |
| 空动作也能初始化 | 环境 smoke 成功 | 不能算策略成功 |
| Pi05 抓错对象 | 策略视觉/语义泛化 | event wrong object、camera view |
| Pi05 抓对但掉落 | 控制/抓取/动作后处理 | gripper、action chunk、release timing |
| 任务需要布料/线缆 | benchmark 边界 | 不应拿 RoboLab 当前结果过度外推 |
| 只有 1 条成功视频 | 证据规模不足 | 增加 episode、做 CI |

---

## 10. 给 4090 复现的实际实验矩阵

4090 不适合一上来跑完整 RoboLab-120，但可以做一个低成本、论文口径更接近的小矩阵。

### 10.1 第一版：12-task subset

| 类别 | 任务数 | 每任务 episode | 目标 |
|---|---:|---:|---|
| visual | 4 | 3 | 颜色/语义/尺寸 |
| relational | 4 | 3 | 空间关系/计数/连接词 |
| procedural | 4 | 3 | 可供性/重定向/堆叠 |

总量：

```text
12 tasks * 3 episodes = 36 episodes
```

这比 1 条视频可靠很多，但还不是论文完整结果。

### 10.2 第二版：语言变体矩阵

选 3 个任务：

```text
default / vague / specific
```

每种 3 episodes：

```text
3 tasks * 3 variants * 3 episodes = 27 episodes
```

看：

- success_by_instruction_type
- score_by_instruction_type
- wrong_object_by_instruction_type

### 10.3 第三版：扰动小矩阵

选 2 个任务，做：

```text
camera pose: baseline / mild / strong
lighting: baseline / dark / bright
object pose: baseline / shifted / cluttered
```

这里重点不是跑很多，而是给 MNPE / sensitivity analysis 准备结构化输入。

---

## 11. 复现报告应该怎么写才不虚

一个合格的 RoboLab 复现报告应该避免写：

```text
模型成功完成了 RoboLab。
```

应该写成：

```text
在 RTX 4090 上，我们完成了 Pi05 + RoboLab 的单任务闭环。
BananaInBowlTask: success=True, score=1.0, step=198。
另抽样 3 个复杂任务，1 成功 2 失败。
这些结果证明 policy server、Isaac runtime、task loading、video/HDF5/event logging 链路可用。
它仍然不是 RoboLab-120 完整复现，也不足以做论文级统计结论。
```

如果要写“模型哪里弱”，至少要补：

```text
任务属性
instruction type
success
score
event counts
失败视频
HDF5/JSON 证据路径
```

---

## 12. 一句话总图

```mermaid
flowchart LR
    A["Task / Scene / Instruction"] --> B["Policy Rollout"]
    B --> C["Video"]
    B --> D["HDF5 State + Action"]
    B --> E["Event Log"]
    B --> F["Episode Result"]
    F --> G["Success"]
    F --> H["Normalized Score"]
    E --> I["Failure Taxonomy"]
    D --> J["Trajectory Metrics"]
    G --> K["CI / Dashboard"]
    H --> K
    I --> K
    J --> K
    K --> L["Policy Comparison"]
    L --> M["Real-world Rank Correlation"]
```

记住这条链：

```text
视频是证据入口，不是结论本身。
success 是终局，score 是过程，event 是错因，CI 是可信度，rank correlation 是和真实世界的连接。
```

---

## 13. 这章对应到我们已经产出的文件

| 复现文件 | 对应这章哪一块 |
|---|---|
| `COMPLETE_REPRO_pi05_banana_20260620.md` | 单任务闭环证据，但不是统计结论 |
| `COMPLEX_TASKS_pi05_20260620.md` | 复杂任务抽样，适合接 score/event 解释 |
| `robolab_repro_artifacts/repro_status.json` | 复现门禁状态 |
| `robolab_repro_artifacts/pi05_policy_smoke_summary.json` | policy smoke 摘要 |
| `robolab_repro_artifacts/task_summary.csv` | 任务结果表 |
| `RoboLab_4090_repro_learning_record.ipynb` | 把论文、代码和复现证据串起来 |

后续如果要把这章真正落地，应该新增一个自动报告：

```text
episode_results.jsonl
  + event logs
  + task metadata
  + video paths
  -> failure_report.md
```

每条失败自动输出：

```text
任务名
能力轴
语言变体
success / score
失败事件 Top-K
对应视频
可能归因
下一步实验
```

这会把 RoboLab 从“跑出来了”推进到“可以分析策略能力”。

In [ ]:
# ===== 精讲13补充：评测证据链深挖轻量验证 =====
# 这个 cell 不跑 RoboLab，只把补充章里的关键概念做成可检查 schema。

import json
from math import sqrt

episode_identity_fields = [
    "task_id",
    "scene_id",
    "instruction_variant",
    "policy_id",
    "robot_id",
    "camera_config",
    "action_space_config",
    "variation_seed",
    "run_seed",
]

evidence_chain = {
    "video": "human_inspection",
    "episode_results_jsonl": "aggregate_success_score",
    "hdf5": "state_action_image_replay",
    "event_log": "failure_taxonomy",
    "dashboard": "policy_task_axis_comparison",
}

def normalized_score(progress_by_object):
    return sum(done / total for done, total in progress_by_object.values()) / len(progress_by_object)

def binomial_ci_half_width(p, n):
    # 教学用近似：1.96 * sqrt(p(1-p)/n)。论文里强调 N=10 单任务 CI 仍很宽。
    return 1.96 * sqrt(p * (1 - p) / n)

def full_success_from_step_success(p_step, n_subtasks):
    return p_step ** n_subtasks

def classify_failure_signal(events, task_meta):
    if task_meta.get("asset_missing") or task_meta.get("env_crash"):
        return "repro_environment"
    if task_meta.get("deformable") or task_meta.get("force_control"):
        return "benchmark_boundary"
    if events.get("wrong_object", 0) > 0:
        return "visual_language_binding"
    if events.get("dropped_target", 0) > 0:
        return "grasp_or_release_control"
    if events.get("hit_table", 0) > 0:
        return "trajectory_or_action_scale"
    return "needs_video_hdf5_review"

reproduction_matrices = {
    "subset_12_tasks": {
        "tasks": 12,
        "episodes_per_task": 3,
        "purpose": "balanced visual/procedural/relational smoke beyond a single video",
    },
    "language_variation": {
        "tasks": 3,
        "variants": ["default", "vague", "specific"],
        "episodes_per_variant": 3,
        "purpose": "separate language binding failure from motion failure",
    },
    "small_sensitivity": {
        "tasks": 2,
        "axes": ["camera_pose", "lighting", "object_pose"],
        "levels": ["baseline", "mild", "strong"],
        "purpose": "prepare structured data for MNPE/sensitivity analysis",
    },
}

score_case = {"banana": (4, 4), "cube": (1, 4)}
score_value = normalized_score(score_case)
ci_n10 = binomial_ci_half_width(0.5, 10)
ci_n100 = binomial_ci_half_width(0.5, 100)
long_horizon_curve = {
    n: full_success_from_step_success(0.8, n)
    for n in [1, 3, 5, 7]
}

deep_tests = [
    ("episode_identity_has_full_context", len(episode_identity_fields) >= 9 and "instruction_variant" in episode_identity_fields),
    ("evidence_chain_has_video_hdf5_event_dashboard", {"video", "hdf5", "event_log", "dashboard"}.issubset(evidence_chain)),
    ("score_can_capture_partial_progress", 0 < score_value < 1),
    ("ci_gets_tighter_with_more_episodes", ci_n100 < ci_n10),
    ("long_horizon_success_decays_with_subtasks", long_horizon_curve[7] < long_horizon_curve[3] < long_horizon_curve[1]),
    ("wrong_object_routes_to_visual_language_binding", classify_failure_signal({"wrong_object": 2}, {}) == "visual_language_binding"),
    ("dropped_target_routes_to_control", classify_failure_signal({"dropped_target": 1}, {}) == "grasp_or_release_control"),
    ("deformable_routes_to_benchmark_boundary", classify_failure_signal({}, {"deformable": True}) == "benchmark_boundary"),
    ("language_matrix_has_three_variants", set(reproduction_matrices["language_variation"]["variants"]) == {"default", "vague", "specific"}),
    ("sensitivity_matrix_has_three_axes", set(reproduction_matrices["small_sensitivity"]["axes"]) == {"camera_pose", "lighting", "object_pose"}),
]

report = {
    "episode_identity_fields": episode_identity_fields,
    "evidence_chain": evidence_chain,
    "score_case": {
        "progress_by_object": score_case,
        "normalized_score": score_value,
        "interpretation": "success=False can still have meaningful partial score.",
    },
    "ci_half_width_demo": {
        "n10_at_p05": ci_n10,
        "n100_at_p05": ci_n100,
        "interpretation": "single-task N=10 is coarse; N=100 is materially tighter.",
    },
    "long_horizon_curve_demo": long_horizon_curve,
    "failure_routing": {
        "wrong_object": classify_failure_signal({"wrong_object": 2}, {}),
        "dropped_target": classify_failure_signal({"dropped_target": 1}, {}),
        "deformable": classify_failure_signal({}, {"deformable": True}),
    },
    "reproduction_matrices": reproduction_matrices,
    "tests": [{"name": name, "passed": bool(ok)} for name, ok in deep_tests],
    "all_passed": all(ok for _, ok in deep_tests),
    "boundary": "Conceptual validation only. It does not rerun RoboLab-120, RoboArena, or MNPE.",
}

print(json.dumps(report, ensure_ascii=False, indent=2))
for name, ok in deep_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert report["all_passed"], deep_tests
write_status("remaining_core_topics_deep_lightweight_tests", report)

## 0.20 代码精讲：policy rollout 到证据链

下面这节来自本目录的 [EXPLAIN_14_core_code_runtime_chain.md](./EXPLAIN_14_core_code_runtime_chain.md)。它补的是前面还没系统讲透的源码主干：`runner.py` 如何调度任务，`episode.py` 如何逐步执行 policy，`InferenceClient` / `Pi0DroidJointposClient` 如何把观测转成模型请求，`WorldState` / `EventTracker` / `RecorderManager` / `summarize_run` / `dashboard loader` 如何把 rollout 变成可分析证据。

# 精讲 14：还没讲透的核心代码 - policy rollout 到证据链

> **【绿色标识｜核心结论】** 前面讲了很多“任务怎么生成、成功怎么定义、论文怎么评测”。但真正复现时最常出问题的是另一条链路：`runner.py` 选任务和创建输出目录，`episode.py` 每一步调用 policy，再 `env.step`，`InferenceClient` 把观测转成模型请求，`WorldState/conditionals/EventTracker` 判断发生了什么，`RecorderManager/summarize/results/dashboard` 把过程写成 HDF5、JSON、视频和表格。
>
> **【蓝色标识｜源码路径】** 这节基于远端 4090 源码快照 `7d45d74904eade3b578a8eb1f2f9f89bc3d40326`：`robolab/eval/runner.py`、`robolab/eval/episode.py`、`robolab/eval/base_client.py`、`policies/pi0_family/client.py`、`robolab/eval/summarize.py`、`robolab/core/world/world_state.py`、`robolab/core/task/event_tracker.py`、`robolab/core/logging/recorder_manager.py`、`robolab/core/logging/results.py`、`dashboard/loaders/local.py`。
>
> **【橙色标识｜容易误解】** `policies/pi0_family/run.py` 不是“模型本体”。它只是 RoboLab 侧的 runner/client；OpenPI 模型通常在另一个 server 进程里，RoboLab 通过 websocket 请求动作。

## 先说还缺哪块代码

前面精讲已经覆盖：

- Task 定义：scene、instruction、termination、subtasks。
- Scene/Task generation：prompt、solver、physical placement、validation。
- Metrics：SPARC、MNPE、difficulty、DTGE。
- Evidence interpretation：success/score、event、CI、真实世界相关性。

但还没系统讲透：

```text
一个 policy-controlled episode 到底怎么跑起来？
观测怎么进模型？
动作怎么回到仿真？
多 env 并行时怎么避免串动作？
视频/HDF5/JSON/event log 是谁写的？
dashboard 为什么能读到结果？
```

这就是本节要补的核心代码。

## 1. 总调用链

把真实 policy eval 压成一张图：

```mermaid
flowchart TD
    A["policies/pi0_family/run.py<br/>解析 pi05 参数，启动 Isaac App"] --> B["robolab/eval/runner.py<br/>选择 task/tag，创建 output 目录"]
    B --> C["create_env(..., policy=pi05)<br/>绑定 task + robot + camera + action + recorder"]
    B --> D["client_factory(args)<br/>创建 Pi0DroidJointposClient"]
    C --> E["robolab/eval/episode.py::run_episode"]
    D --> E
    E --> F["obs -> InferenceClient.infer(obs, instruction, env_id)"]
    F --> G["Pi05 client<br/>extract obs / pack request / websocket / unpack actions"]
    G --> H["env.step(actions)"]
    H --> I["RobolabEnv + Recorder + WorldState"]
    I --> J["conditionals/subtask/event_tracker"]
    I --> K["HDF5 / videos / per-env event JSON"]
    E --> L["summarize_run"]
    L --> M["episode_results.jsonl"]
    M --> N["analysis/read_results.py & dashboard/loaders/local.py"]
```

说人话：

> `runner.py` 管“跑哪些题”，`episode.py` 管“每一步怎么推进”，`client.py` 管“怎么问模型要动作”，`WorldState/conditionals/event_tracker` 管“发生了什么”，`recorder/summarize/results/dashboard` 管“证据怎么落盘和展示”。

## 2. `runner.py`：不是跑一步，而是跑整场考试

核心函数：

```text
add_common_eval_args(parser)
run_evaluation(args, policy, client_factory)
```

### 输入

- `--task` / `--tag` / 默认全任务。
- `--num-envs`：并行环境数。
- `--num-runs`：每个 task 顺序跑几批。
- `--instruction-type`：`default/vague/specific` 等语言变体。
- `--enable-subtask`：是否记录 subtask 进度。
- `--video-mode`：保存 sensor/viewport 视频。
- `--output-folder-name`：指定旧目录可续跑。
- `--num-episodes-adaptive` 与 `--ci-pp-width`：按成功率置信区间自适应补 episode。

### 输出

```text
output/<timestamp>_<policy>/
  episode_results.jsonl
  <TaskName>/
    env_cfg.json
    run_0.hdf5
    log_0_env0.json
    *.mp4
```

### 代码关键点

`run_evaluation()` 做这几件事：

1. 根据 `policy` 和时间戳创建输出目录。
2. 根据 `task/tag` 找任务列表。
3. 调 `init_experiment()` 创建或加载 `episode_results.jsonl`。
4. 每个 task 调 `create_env()`。
5. 每个 task 创建一次 policy client。
6. 每个 run 调 `run_episode()`。
7. 调 `summarize_run()` 把这批结果追加到 JSONL。
8. 最后 `summarize_experiment_results()` 打印总表。

**【绿色标识｜调试直觉】**

如果你发现某个任务没跑，先看 `runner.py` 这一层：是不是 `--task/--tag` 筛选错了？是不是 `episode_results.jsonl` 里已经有记录导致续跑跳过？是不是 `num_envs` 过大 OOM？

## 3. `episode.py`：真正的闭环在这里

核心函数：

```text
run_episode(env, env_cfg, episode, client, headless, save_videos, video_mode)
```

### 每一步做什么

```text
reset env
setup HDF5 file and video writers
for step in max_steps:
    for each active env:
        action = client.infer(obs, instruction, env_id)
    obs, reward, term, trunc, info = env.step(actions)
    subtask_status.append(get_all_env_subtask_infos(env))
    write video frames
    if env.all_terminated:
        break
release videos
client.reset()
return env_results, subtask_status, timing
```

### 关键设计

| 设计 | 作用 |
|---|---|
| `clients = [client] * env.num_envs` | 同一个连接服务多个并行环境 |
| `env.active_env_ids` | 只给未终止 env 请求动作 |
| frozen env action 为 0 | 已结束的环境不再继续污染轨迹 |
| `set_hdf5_file(f"run_{episode}.hdf5")` | 每个 run 一个 HDF5 |
| `set_episode_index(env_id)` | HDF5 里用 `demo_<env_id>` 对齐并行 env |
| `TimingStats` | 分开记录 policy/env/video 的耗时 |
| `client.reset()` | 清空 action chunk，避免下一 episode 执行旧动作 |

**【橙色标识｜常见坑】**

如果 OpenPI 一次返回 15 步 action chunk，`client.infer()` 不是每个 sim step 都真的请求 server。它会先消费缓存动作，缓存耗尽后才重新发 websocket 请求。所以平均 inference time 和 sim step 数不是一回事。

## 4. `InferenceClient`：所有策略接入的最小合同

抽象类在：

```text
robolab/eval/base_client.py
```

它要求策略实现 4 个 hook：

| hook | 输入 | 输出 | 职责 |
|---|---|---|---|
| `_extract_observation(raw_obs, env_id)` | RoboLab 原始 obs | flat numpy dict | 从 batched tensor 里取一个 env 的图像/关节 |
| `_pack_request(extracted_obs, instruction)` | flat obs + 语言 | server request | 改成模型服务端需要的 key/schema |
| `_query_server(request)` | request | response | 请求模型服务 |
| `_unpack_response(response)` | response | action chunk | 拿出动作数组 |

父类 `infer()` 把它们串起来：

```text
extract -> needs_refresh? -> pack -> query -> unpack -> postprocess -> cache -> next_action
```

### 为什么这层很关键

RoboLab 支持不同 policy，不可能把 Pi0/Pi05/GR00T/ReKep 全写死在 evaluator 里。  
所以它只要求每个 policy 提供同一个接口：

```text
obs + instruction -> action
```

这也是我们以后接 RoboChallenge 或 ReKep 的入口。

## 5. `Pi0DroidJointposClient`：OpenPI/Pi05 的适配层

文件：

```text
policies/pi0_family/client.py
```

它做的事情很具体：

### 输入

来自 RoboLab 的 observation：

```text
raw_obs["image_obs"]["over_shoulder_left_camera"][env_id]
raw_obs["image_obs"]["wrist_cam"][env_id]
raw_obs["proprio_obs"]["arm_joint_pos"][env_id]
raw_obs["proprio_obs"]["gripper_pos"][env_id]
```

### 发给 OpenPI 的 request

```text
observation/exterior_image_1_left
observation/wrist_image_left
observation/joint_position
observation/gripper_position
prompt
```

注意这些 key 不是随便起的，它们要匹配 DROID checkpoint 的训练 schema。

### 输出

OpenPI server 返回：

```text
response["actions"] -> numpy action chunk
```

Pi05 默认 `open_loop_horizon = 15`。  
最后一维 gripper 会二值化：

```text
chunk[..., -1] = chunk[..., -1] > 0.5
```

说人话：

> 这层是“翻译官”：RoboLab 的观察格式翻译成 OpenPI 听得懂的请求，再把 OpenPI 的动作翻译回 RoboLab action space。

## 6. `RobolabEnv.step` 与 frozen env：为什么并行评测不会乱

文件：

```text
robolab/core/environments/env.py
```

前面精讲没有深入这块。它的关键是：评测和 RL 训练不一样。

训练时一个 env 结束可以立刻 reset。  
评测时不能这么做，因为我们还要：

- 知道它在哪一步成功/失败。
- 导出这一条 episode 的 HDF5。
- 写事件日志。
- 不让已结束 env 继续接收动作。

所以 RoboLab 用：

```text
active_env_ids
_frozen_envs
all_terminated
reset_eval_state()
```

来管理并行环境。  
这就是为什么 `episode.py` 每步只对 `active_env_ids` 请求动作。

## 7. `WorldState`：谓词函数不直接到处翻 Isaac 数据

文件：

```text
robolab/core/world/world_state.py
```

它是仿真查询统一入口，提供：

- entities / objects / articulations / extras。
- pose、velocity、frame pose。
- dimensions、AABB、bbox、centroid。
- robot joint state。
- contact force / in contact。

### 为什么要有这一层

如果 `object_in_container()`、`object_left_of()`、`object_grabbed()` 每个函数都自己读 Isaac/Usd/ContactSensor，代码会非常乱，也很难批量处理 `num_envs`。

`WorldState` 把这些统一起来：

```text
conditionals.py -> get_world(env) -> geometry/contact/pose query
```

### 两个重要缓存

| 缓存 | 用途 |
|---|---|
| `_local_geometry_cache` | 本地 bbox/dimensions 静态不变，避免每 step 重算 USD 几何 |
| `_predicate_state` | 顺序谓词需要跨 step 记忆，例如先后关系、是否曾经放入过 |

**【绿色标识｜调试直觉】**

如果视频里对象看起来已经进碗，但 `success=False`，重点查 `WorldState` + `conditionals.py`：bbox、contact、tolerance、gripper detached、container relation 是否和肉眼判断一致。

## 8. `EventTracker`：失败不是一个 False，而是一串行为事件

文件：

```text
robolab/core/task/event_tracker.py
```

它追踪：

- wrong object grabbed。
- gripper hit table。
- gripper fully closed but no grasp。
- non-target object moved/bumped。
- object out of scene。
- object tipped over。
- target object dropped。
- gripper hit non-target object。
- multiple objects grabbed。

### 输入

```text
env
per_env_intended target object set
frozen_mask
ignore_objects
upright_objects
```

### 输出

```text
[(info_string, StatusCode, env_mask), ...]
```

`env_mask` 很重要：它告诉你这个事件发生在哪些并行 env 上。

说人话：

> `EventTracker` 是失败诊断器。它让我们知道模型是没动、抓错、撞桌、提前放手，还是把无关物体撞飞。

## 9. `RobolabRecorderManager`：HDF5 为什么能撑长 episode

文件：

```text
robolab/core/logging/recorder_manager.py
```

它扩展 Isaac Lab 的 recorder，重点是：

- 每个 env 有独立 `EpisodeData`。
- 每个 run 可以切换到 `run_<idx>.hdf5`。
- 每个 env 可以写到 `demo_<env_id>`。
- 长 episode 支持 streaming flush，降低内存压力。
- 已结束 env 可以单独导出。

这解释了为什么结果结构通常是：

```text
TaskName/
  run_0.hdf5
    /data/demo_0/...
    /data/demo_1/...
  log_0_env0.json
  log_0_env1.json
```

**【橙色标识｜常见坑】**

如果 HDF5 为空，不一定是 policy 没跑；也可能是 recorder 没初始化、没有调用 `set_hdf5_file`、env 没成功 export、或者 episode 在导出前异常退出。

## 10. `summarize_run` 与 `results.py`：把过程折叠成一行 JSONL

文件：

```text
robolab/eval/summarize.py
robolab/core/logging/results.py
```

`run_episode()` 返回的是过程数据；论文表格需要的是 episode 级摘要。  
`summarize_run()` 做折叠：

1. 读取 final subtask info。
2. 找到 `run_<idx>.hdf5`。
3. 从 recorder 取出 per-env events。
4. 写 `log_<run>_env<id>.json`。
5. 从 HDF5 读 `demo_<env_id>`。
6. 计算 trajectory metrics。
7. 读取 final score。
8. build `run_summary`。
9. `update_experiment_results()` append 到 `episode_results.jsonl`。

`episode_results.jsonl` 是后续 analysis/dashboard 的主账本。  
没有它，视频和 HDF5 仍然有价值，但论文级聚合会断。

## 11. `dashboard/loaders/local.py`：结果为什么能在网页里看

Dashboard 不重新跑 Isaac，也不 import 任务代码。它读文件：

```text
episode_results.jsonl
task folder
HDF5
videos
event logs
```

核心设计：

- `episode_results.jsonl` 是 canonical source。
- HDF5 是否有 episode 用轻量检查。
- loader 用 mtime 做 cache key，文件变了就刷新。
- success rate 用 Beta credible interval。
- score 用 Student-t CI。

说人话：

> Dashboard 是离线证据查看器，不是仿真运行器。它的可靠性取决于前面 runner/summarize 是否把证据写完整。

## 12. 按故障反查代码

| 现象 | 优先看哪里 | 原因 |
---|---|---|
| 命令启动后没跑目标任务 | `runner.py` task/tag 筛选、续跑跳过 | 任务列表或 JSONL 已有记录 |
| OpenPI server 已开但动作不对 | `Pi0DroidJointposClient._pack_request` | key/schema/image resize/语言 prompt 不匹配 |
| 多 env 行为串了 | `InferenceClient._chunks/_counters` 是否按 `env_id` 分开 | action chunk 缓存污染 |
| 视频里成功但 `success=False` | `conditionals.py` + `WorldState` | contact/tolerance/detached/bbox 判断和肉眼不同 |
| 错误原因为空 | `EventTracker` + recorder event export | event 没触发或没写入 per-env log |
| HDF5 空 | `RobolabRecorderManager` | file handler、episode index、export 时机 |
| JSONL 没新增 | `summarize_run` + `update_experiment_results` | episode summary 没 build 或 append |
| Dashboard 看不到结果 | `dashboard/loaders/local.py` | 路径、mtime cache、缺 JSONL/HDF5/videos |

## 13. 复现时最重要的代码心智模型

记住下面这个分层：

```text
runner:        选任务、建输出目录、续跑、批量调度
episode:       每一步 obs -> client -> action -> env.step
client:        把 RoboLab obs 翻译成 policy server 请求
env/world:     仿真状态、几何、接触、成功条件
event:         把失败拆成可解释事件
recorder:      写 HDF5 和视频
summarize:     折叠成 episode_results.jsonl
dashboard:     离线读取证据并展示统计
```

如果后续你要接 RoboChallenge 的 pi 或 ReKep，最该改的是：

- policy client：观测 schema、动作 schema、连接方式。
- runner 参数：policy 名称、模型服务地址、任务子集。
- 结果分析：同一套 `episode_results.jsonl` 聚合，不要改掉主证据格式。

## 小结

精讲14补的是代码运行主干：

```text
任务不是直接被模型解决的。
任务先被 runner 选中，
被 create_env 装成可运行环境，
被 episode loop 按 step 推进，
被 client 转成模型请求，
被 WorldState/conditionals/event 判断行为，
被 recorder/summarize 写成证据，
最后才被 analysis/dashboard 变成论文表格。
```

这条链路比单个模型调用更重要。因为复现失败时，问题可能在模型、观测、动作、仿真、谓词、记录、汇总、展示任意一层。

In [ ]:
# ===== 精讲14：核心代码运行链路轻量验证 =====
# 这个 cell 不启动 Isaac，也不连 OpenPI server。
# 它把源码主干拆成可检查的节点与接口，验证 runner -> episode -> client -> env/world
# -> event -> recorder -> summarize -> dashboard 这条证据链完整闭合。

import json

runtime_code_chain = [
    {
        "stage": "runner",
        "files": ["robolab/eval/runner.py", "policies/pi0_family/run.py"],
        "core_functions": ["add_common_eval_args", "run_evaluation", "client_factory"],
        "input": ["task_or_tag", "num_envs", "num_runs", "instruction_type", "video_mode", "policy_name"],
        "output": ["output_dir", "task_envs", "episode_results_jsonl", "env_cfg"],
        "failure_hint": "任务没跑、续跑跳过、num_envs OOM、输出目录不对，优先查这里。",
    },
    {
        "stage": "episode_loop",
        "files": ["robolab/eval/episode.py"],
        "core_functions": ["run_episode", "TimingStats"],
        "input": ["env", "env_cfg", "episode_index", "InferenceClient", "video_mode"],
        "output": ["env_results", "subtask_status", "timing", "videos", "run_hdf5"],
        "failure_hint": "动作没进 env、视频不写、已终止 env 还在动，优先查这里。",
    },
    {
        "stage": "inference_client_contract",
        "files": ["robolab/eval/base_client.py"],
        "core_functions": ["infer", "_extract_observation", "_pack_request", "_query_server", "_unpack_response", "_postprocess_chunk"],
        "input": ["raw_obs", "instruction", "env_id"],
        "output": ["action", "viz"],
        "failure_hint": "接新模型/新机器人时，先确认这五个 hook 的输入输出。",
    },
    {
        "stage": "pi05_client_adapter",
        "files": ["policies/pi0_family/client.py"],
        "core_functions": ["Pi0DroidJointposClient", "_extract_observation", "_pack_request", "_infer_with_retry", "_postprocess_chunk"],
        "input": ["over_shoulder_left_camera", "wrist_cam", "arm_joint_pos", "gripper_pos", "prompt"],
        "output": ["openpi_request", "action_chunk", "binary_gripper_action"],
        "failure_hint": "server 连上但动作怪，重点查 OpenPI request key、图像 resize、action horizon、gripper 二值化。",
    },
    {
        "stage": "env_world_conditionals",
        "files": ["robolab/core/environments/env.py", "robolab/core/world/world_state.py", "robolab/core/task/conditionals.py"],
        "core_functions": ["RobolabEnv.step", "get_world", "WorldState", "object_in_container", "object_grabbed"],
        "input": ["actions", "scene_entities", "contact_sensors", "object_poses"],
        "output": ["termination_state", "subtask_state", "spatial_relations", "contact_relations"],
        "failure_hint": "视频看着成功但 success=False，重点查 WorldState、contact、bbox、tolerance、detached 条件。",
    },
    {
        "stage": "event_tracking",
        "files": ["robolab/core/task/event_tracker.py"],
        "core_functions": ["EventTracker", "check_events", "reset_envs"],
        "input": ["per_env_intended_targets", "frozen_mask", "world_state"],
        "output": ["wrong_object", "target_dropped", "hit_table", "object_moved", "env_mask"],
        "failure_hint": "失败原因空或不准，查 event tracker 是否触发和 recorder 是否导出事件。",
    },
    {
        "stage": "recording",
        "files": ["robolab/core/logging/recorder_manager.py", "robolab/core/logging/streaming_hdf5_handler.py"],
        "core_functions": ["RobolabRecorderManager", "set_hdf5_file", "set_episode_index", "flush_buffer"],
        "input": ["per_step_state", "actions", "obs", "env_ids"],
        "output": ["run_0.hdf5", "demo_0", "demo_1"],
        "failure_hint": "HDF5 空、demo 缺失、长 episode 内存涨，优先查 recorder manager。",
    },
    {
        "stage": "summarize_results",
        "files": ["robolab/eval/summarize.py", "robolab/core/logging/results.py"],
        "core_functions": ["summarize_run", "build_run_summary", "update_experiment_results", "summarize_experiment_results"],
        "input": ["env_results", "events", "HDF5", "env_cfg", "timing"],
        "output": ["log_0_env0.json", "episode_results.jsonl", "trajectory_metrics", "score"],
        "failure_hint": "视频/HDF5 有了但 JSONL 没新增，查 summarize_run 和 update_experiment_results。",
    },
    {
        "stage": "dashboard_loader",
        "files": ["dashboard/loaders/local.py"],
        "core_functions": ["_cached_load_episodes", "_sr_beta_ci", "_score_ci", "EpisodeRow", "TaskSummary"],
        "input": ["episode_results.jsonl", "task_folders", "HDF5", "videos", "logs"],
        "output": ["dashboard_rows", "success_rate_CI", "score_CI", "video_links"],
        "failure_hint": "Dashboard 看不到结果，查路径、mtime cache、JSONL/HDF5/video 是否齐全。",
    },
]

stage_order = [item["stage"] for item in runtime_code_chain]

def route_runtime_issue(symptom):
    # 教学版故障路由，用来把复现问题快速定位到代码层。
    text = symptom.lower()
    if "task" in text or "skip" in text or "oom" in text:
        return "runner"
    if "server" in text or "request" in text or "action horizon" in text or "gripper" in text:
        return "pi05_client_adapter"
    if "success=false" in text or "condition" in text or "contact" in text:
        return "env_world_conditionals"
    if "wrong object" in text or "dropped" in text or "hit table" in text:
        return "event_tracking"
    if "hdf5" in text or "demo" in text:
        return "recording"
    if "jsonl" in text or "summary" in text:
        return "summarize_results"
    if "dashboard" in text or "ci" in text:
        return "dashboard_loader"
    return "episode_loop"

expected_contract_hooks = {
    "_extract_observation",
    "_pack_request",
    "_query_server",
    "_unpack_response",
    "_postprocess_chunk",
}
client_hooks = set(runtime_code_chain[2]["core_functions"])

code_chain_tests = [
    ("chain_starts_with_runner", stage_order[0] == "runner"),
    ("chain_ends_with_dashboard_loader", stage_order[-1] == "dashboard_loader"),
    ("episode_loop_between_runner_and_client", stage_order.index("runner") < stage_order.index("episode_loop") < stage_order.index("inference_client_contract")),
    ("client_contract_has_required_hooks", expected_contract_hooks.issubset(client_hooks)),
    ("pi05_client_exposes_droid_observation_keys", {"over_shoulder_left_camera", "wrist_cam", "arm_joint_pos", "gripper_pos", "prompt"}.issubset(set(runtime_code_chain[3]["input"]))),
    ("world_layer_outputs_conditions", {"termination_state", "subtask_state", "contact_relations"}.issubset(set(runtime_code_chain[4]["output"]))),
    ("event_layer_outputs_failure_taxonomy", {"wrong_object", "target_dropped", "hit_table"}.issubset(set(runtime_code_chain[5]["output"]))),
    ("recording_layer_outputs_hdf5_demos", {"run_0.hdf5", "demo_0"}.issubset(set(runtime_code_chain[6]["output"]))),
    ("summarize_layer_outputs_jsonl", "episode_results.jsonl" in runtime_code_chain[7]["output"]),
    ("issue_router_maps_hdf5_to_recording", route_runtime_issue("HDF5 demo_0 is empty") == "recording"),
    ("issue_router_maps_success_false_to_world", route_runtime_issue("video success but success=False contact mismatch") == "env_world_conditionals"),
    ("issue_router_maps_server_to_client", route_runtime_issue("OpenPI server request action horizon wrong") == "pi05_client_adapter"),
]

report = {
    "runtime_code_chain": runtime_code_chain,
    "stage_order": stage_order,
    "issue_routing_examples": {
        "hdf5_empty": route_runtime_issue("HDF5 demo_0 is empty"),
        "success_false": route_runtime_issue("video success but success=False contact mismatch"),
        "server_request": route_runtime_issue("OpenPI server request action horizon wrong"),
        "dashboard_missing": route_runtime_issue("dashboard cannot find CI rows"),
    },
    "tests": [{"name": name, "passed": bool(ok)} for name, ok in code_chain_tests],
    "all_passed": all(ok for _, ok in code_chain_tests),
    "boundary": "This validates the source-code mental model only. It does not import Isaac Sim or call a policy server.",
}

print(json.dumps(report, ensure_ascii=False, indent=2))
for name, ok in code_chain_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert report["all_passed"], code_chain_tests
write_status("core_code_runtime_chain_lightweight_tests", report)

## 0.20b 代码精讲：精讲14补充，核心运行时代码深挖

下面这节来自本目录的 [EXPLAIN_14_deep_runtime_code_chain.md](./EXPLAIN_14_deep_runtime_code_chain.md)。它把精讲 14 从“文件导览”继续推进到“输入、处理、输出、故障路由和复现证据链”：为什么 `runner.py` 要延迟导入 Isaac 相关模块，`episode.py` 怎样处理 active/frozen env，`InferenceClient` 为什么按 `env_id` 缓存 action chunk，Pi05 client 如何把 RoboLab 观测转成 OpenPI/Pi05 请求，`WorldState/EventTracker` 如何支撑谓词与失败事件，HDF5、event log、`episode_results.jsonl` 和 dashboard 分别承担什么证据角色。

# 精讲 14-补充：核心运行时代码深挖，policy rollout 如何变成证据

> <span style="color:#166534"><strong>[绿色 核心结论]</strong></span>：RoboLab 的核心代码不是一个“跑模型”的脚本，而是一条可审计的评测流水线。`runner.py` 负责选择任务和组织实验，`episode.py` 负责把策略放进仿真闭环，`InferenceClient` 和 Pi05 client 负责把观测转成模型请求，`WorldState` 和 `EventTracker` 负责解释场景中发生了什么，`RecorderManager`、`summarize.py`、`results.py` 和 dashboard 负责把过程落成 HDF5、JSONL、事件日志、视频和统计表。

> <span style="color:#1d4ed8"><strong>[蓝色 源码路径]</strong></span>：本节基于官方 RoboLab GitHub 主干源码讲解，重点对应 `robolab/eval/runner.py`、`robolab/eval/episode.py`、`robolab/eval/base_client.py`、`policies/pi0_family/client.py`、`robolab/core/world/world_state.py`、`robolab/core/task/event_tracker.py`、`robolab/eval/summarize.py`、`robolab/core/logging/results.py`、`robolab/core/logging/recorder_manager.py`、`dashboard/loaders/local.py`。

> <span style="color:#b45309"><strong>[橙色 易错点]</strong></span>：不要把“有视频”误认为“复现成功”。视频只是观测证据之一。真正的评测结果要看 `episode_results.jsonl`、每个 env 的事件日志、HDF5 里的 subtask score、成功标志、失败原因和统计置信区间。

## 0. 为什么原来的精讲 14 还不够

原来的精讲 14 已经把主文件列出来了，但还停留在“这个文件负责什么”的层面。实际复现时，问题通常不会问得这么抽象，而是会变成这些具体问题：

- 为什么我看到机械臂动了，但是 `success=false`？
- 为什么 Pi05 server 明明在跑，但 RoboLab 一直等待或动作不对？
- 为什么多 env 并行时一个 env 成功了，另一个 env 的视频和日志像是串了？
- 为什么 dashboard 里没有视频，或者有视频却没有 score？
- 为什么同一个任务跑过一次后，再跑会被跳过？
- 4090 显存不够时，到底该降 `num_envs` 还是降任务数量？

这些问题都要沿着运行时代码链去查。下面按“输入、处理、输出、常见故障”的方式拆。

## 1. 总体心智模型：三层闭环

```mermaid
flowchart TD
    A["调度层<br/>run.py + runner.py"] --> B["闭环层<br/>episode.py + InferenceClient + env.step"]
    B --> C["语义判定层<br/>WorldState + conditionals + EventTracker"]
    C --> D["证据层<br/>RecorderManager + HDF5 + event log + summarize.py"]
    D --> E["分析层<br/>results.py + dashboard"]
```

说人话：

- 调度层决定“考哪几道题、跑几遍、结果放哪”。
- 闭环层决定“每一个仿真 step 里，模型看什么、输出什么动作、环境怎么更新”。
- 语义判定层决定“苹果是否真的进碗、是否抓错物体、是否撞桌、是否把无关物体撞飞”。
- 证据层决定“这些过程证据怎么保存，最终成功率、score 和失败原因怎么生成”。
- 分析层决定“复现报告、dashboard、表格、对比图怎么读这些结果”。

关键是：RoboLab 的评测不是只保存最后一帧，而是把一条 episode 的动作、观测、子任务状态、失败事件和统计结果都拆开保存。

## 2. `runner.py`：考试组织者，不直接控制机械臂

### 2.1 输入是什么

`runner.py` 接收的不是传感器观测，而是评测参数：

| 参数 | 作用 | 复现时的判断 |
|---|---|---|
| `--task` | 指定一个或多个任务名 | 最适合单任务调试 |
| `--tag` | 按能力标签筛任务 | 适合视觉、关系、程序能力子集评测 |
| `--num-envs` | 并行环境数 | 4090 上通常先从 1 开始 |
| `--num-runs` | 每个 task 顺序跑几批 | 每批包含 `num_envs` 个 episode |
| `--instruction-type` | 默认、模糊、具体等指令变体 | 用来测语言鲁棒性 |
| `--video-mode` | 保存 sensor、viewport、all 或 none | 调试阶段建议 all，批量评测可关视频 |
| `--output-folder-name` | 指向已有输出目录 | 用于续跑或复查旧结果 |
| `--num-episodes-adaptive` | 自适应采样上限 | 用成功率置信区间控制样本量 |

### 2.2 它为什么要延迟导入 Isaac 相关模块

源码里有一个很实际的设计：公共参数解析必须在 Isaac AppLauncher 启动前完成，但很多 Isaac Lab、environment、episode 相关模块只能在仿真 app 初始化后安全导入。所以 `add_common_eval_args()` 很轻，`run_evaluation()` 里面才导入重模块。

<span style="color:#166534"><strong>[绿色 核心结论]</strong></span>：这解释了为什么 `runner.py` 看起来像普通 Python 脚本，但不能像普通库一样随便导入全部依赖。Isaac Sim 的生命周期会影响 import 顺序。

### 2.3 它的主要处理流程

简化成伪代码：

```python
output_dir = make_output_dir(timestamp, policy, instruction_type)
task_envs = get_envs(task=args.task or tag=args.tag or all)
episode_results = init_or_load_episode_results(output_dir)

for task_env in task_envs:
    if task_already_complete(task_env):
        continue
    env, env_cfg = create_env(task_env, policy=policy, num_envs=args.num_envs)
    client = client_factory(args)
    for run_idx in runs_or_adaptive_loop:
        if run_already_complete(task_env, run_idx):
            continue
        env_results, events, timing = run_episode(env, env_cfg, run_idx, client)
        episode_results = summarize_run(...)
        env.reset_eval_state()
    env.close()
summarize_experiment_results(episode_results)
```

### 2.4 输出是什么

`runner.py` 本身不产出动作，它产出一个实验目录结构：

```text
output/<timestamp>_<policy>[_instruction_type]/
  episode_results.jsonl
  <TaskName>/
    run_0.hdf5
    log_0_env0.json
    <instruction>_0.mp4
    <instruction>_0_viewport.mp4
    env_cfg.json
```

### 2.5 常见问题怎么定位

| 现象 | 优先查 |
|---|---|
| 任务没有跑 | `--task` 名称、`--tag` 过滤、任务注册是否存在 |
| 直接显示 skipped | `episode_results.jsonl` 里可能已有同 task/episode 记录 |
| 输出目录不是预期目录 | `--output-folder-name`、policy 名、instruction type |
| 4090 OOM | 先把 `--num-envs` 降到 1，不要先盲目改模型 |
| 批量评测太慢 | 关视频、缩小 task 子集、只跑目标能力轴 |

## 3. `episode.py`：真正的 step 闭环在这里

`episode.py::run_episode()` 是最核心的动态执行函数。它把 policy、env、recorder、video writer 串起来。

### 3.1 输入是什么

| 输入 | 含义 |
|---|---|
| `env` | RoboLab/Isaac Lab 环境，里面有机器人、物体、相机、action space |
| `env_cfg` | 任务配置，包含 instruction、sim dt、render interval、decimation 等 |
| `episode` | run index，用来命名 `run_<idx>.hdf5` |
| `client` | 一个 `InferenceClient` 子类，比如 Pi05 client |
| `headless` | 是否显示窗口 |
| `save_videos` 和 `video_mode` | 是否保存 sensor/viewport 视频 |

### 3.2 单步循环到底做了什么

```text
reset env
设置 run_<idx>.hdf5
设置每个 env 的 demo_<env_id>
设置视频 writer

for step in max_steps:
    等 Isaac timeline 播放
    给 active env 请求动作
    拼成 actions tensor
    env.step(actions)
    读取 subtask info
    写视频帧
    如果所有 env 都结束，break

释放视频 writer
client.reset()
return env_results, subtask_status, timing
```

### 3.3 为什么 action 要先建全零 tensor

多 env 并行时，不是每个 env 都还活着。已经结束的 env 会 frozen。`episode.py` 先建一个形状为 `(num_envs, action_dim)` 的零 action tensor，只给 `env.active_env_ids` 填模型动作。

这有两个好处：

- 已结束的 env 不再继续污染轨迹。
- `env.step(actions)` 仍然能拿到固定形状的 batched action。

<span style="color:#b45309"><strong>[橙色 易错点]</strong></span>：如果你看到一个 env 成功后视频停止更新，这是正常的 frozen 行为。不要把“某个 env 不再写视频帧”误判成 video writer 崩了。

### 3.4 `action_dim` 为什么从 action manager 取

Isaac Lab 环境里 action space 可能经过 action manager 管理，最终控制维度不一定只看 Gym action space。代码优先读 `env.action_manager.total_action_dim`，没有才 fallback 到 `env.action_space.shape[-1]`。

这对接新策略很重要：

- 如果策略输出维度少了，`actions[env_id] = ...` 会报 shape 错。
- 如果策略输出维度多了，可能被截断前就失败。
- 如果 gripper 维度约定不一致，机械臂会出现“轨迹对但夹爪反”的现象。

### 3.5 `client.reset()` 为什么必须在 finally 里

Pi05 这类策略通常返回 action chunk，例如一次返回 15 步。`InferenceClient` 会缓存 chunk 并逐步消费。如果 episode 结束后不 reset，下一个 episode 可能从旧 chunk 中继续拿动作。

<span style="color:#166534"><strong>[绿色 核心结论]</strong></span>：episode 边界不仅是仿真环境 reset，也是 policy client 的状态边界。

## 4. `InferenceClient`：所有策略接入的最小合同

`InferenceClient` 是 RoboLab policy adapter 的关键抽象。它不是模型本体，而是“仿真观测和策略服务之间的合同”。

### 4.1 必须实现的四个 hook

| hook | 输入 | 输出 | 负责什么 |
|---|---|---|---|
| `_extract_observation(raw_obs, env_id)` | RoboLab 原始 batched obs | 单 env flat dict | 从仿真观测里拿图像、关节、夹爪等 |
| `_pack_request(extracted_obs, instruction)` | flat obs 和语言指令 | server request | 转成模型服务要求的 wire format |
| `_query_server(request)` | request | raw response | 负责 websocket、HTTP 或本地推理调用 |
| `_unpack_response(response)` | raw response | action chunk | 从模型响应里拿动作数组 |

### 4.2 可选 hook

| hook | 作用 |
|---|---|
| `_postprocess_chunk(chunk)` | gripper 二值化、维度 padding、坐标符号修正 |
| `_build_visualization(extracted_obs)` | 拼接调试图像，给视频或窗口显示 |
| `reset()` | 清 chunk cache，也可扩展为通知 server 清 session |
| `close()` | 关闭 websocket 或释放资源 |

### 4.3 action chunk 机制

默认 `infer()` 的逻辑是：

```python
extracted = _extract_observation(obs, env_id)
if chunk_cache_empty_or_used_up(env_id):
    request = _pack_request(extracted, instruction)
    response = _query_server(request)
    chunk = _postprocess_chunk(_unpack_response(response))
    cache[env_id] = chunk
action = next_action(cache[env_id])
return {"action": action, "viz": visualization}
```

这说明两件事：

- 仿真每一步都会调用 `infer()`。
- 但不是每一步都会访问模型 server。只有当前 env 的 chunk 用完，才会重新 query。

### 4.4 为什么 cache 必须按 `env_id` 分开

多 env 并行时，每个 env 的图像、状态、任务进度都不同。即使共用同一个 websocket client，也不能共用同一个 action chunk。`InferenceClient` 用 `_chunks[env_id]` 和 `_counters[env_id]` 隔离状态。

<span style="color:#b45309"><strong>[橙色 易错点]</strong></span>：如果自定义 client 把 chunk cache 写成单个变量，多 env 时会出现“env0 的动作被 env1 消费”的问题。视频看起来会很诡异，日志也会难以解释。

## 5. `Pi0DroidJointposClient`：Pi05 在 RoboLab 里的接口层

Pi05 client 做的是“RoboLab 观测 schema 到 OpenPI/Pi05 server schema”的转换。

### 5.1 输入观测一般包含什么

Pi05/DROID joint-position 风格的输入通常包括：

- over-shoulder camera 图像。
- wrist camera 图像。
- robot arm joint positions。
- gripper position。
- language instruction prompt。

这些会被转成 Pi server 期望的 request key，例如外部视角图像、腕部图像、关节位置、夹爪位置和 prompt。

### 5.2 输出动作是什么

server 返回的是 action chunk，形状大致是：

```text
(open_loop_horizon, action_dim)
```

其中 `open_loop_horizon` 由策略变体决定。Pi0 系列和 Pi05 系列的默认 horizon 不完全一样，Pi05 常见默认值是 15。

### 5.3 为什么 gripper 要后处理

很多 VLA policy 输出的 gripper 值是连续数，但仿真控制器可能希望它接近开/关二值动作。Pi05 client 会对最后一维 gripper action 做阈值化。

这解释了一个常见现象：轨迹看上去对，但抓取失败。可能不是视觉识别错，而是 gripper 后处理、动作维度或控制约定没对齐。

### 5.4 4090 上最常见的 Pi05 接入故障

| 现象 | 可能原因 | 优先检查 |
|---|---|---|
| RoboLab 一直等待动作 | Pi05 server 没启动或端口不对 | websocket URL、server 日志 |
| 图像能保存但模型动作乱 | camera key 或图像 resize 不匹配 | `_extract_observation` 和 `_pack_request` |
| 动作维度报错 | action_dim 与 env 控制器不一致 | env action manager、client 输出 shape |
| 每隔若干步卡一下 | action chunk 用完后重新请求 server | 这是正常推理节奏，除非延迟过大 |
| 下一条 episode 一开始动作异常 | client cache 没 reset 或 server session 未清 | `client.reset()` / server reset |

## 6. `WorldState`：谓词和事件的统一读模型

RoboLab 不是用像素直接判断成功，而是通过仿真世界状态判断对象关系。`WorldState` 提供统一查询：

- body 和 articulation 查询。
- 物体 pose、velocity、frame pose。
- AABB、OBB、centroid、dimensions。
- robot joint positions。
- contact、support、gripper/object 接触。
- batched env 查询。

### 6.1 `env_id=None` 和 `env_id=int` 的区别

`WorldState` 很多 getter 支持两种模式：

| 模式 | 输出 | 用途 |
|---|---|---|
| `env_id=None` | 返回所有 env 的 batched tensor | EventTracker 批量判断 |
| `env_id=0` | 返回单个 env 的结果 | 单任务调试、可视化、兼容旧代码 |

这就是 RoboLab 能把多个 parallel env 一起评估的基础。

### 6.2 local geometry cache 为什么重要

物体的本地几何尺寸和局部 bounding box 通常不随 env 和 timestep 改变。`WorldState` 会缓存 local geometry，避免每步都从 USD prim 重新算。

<span style="color:#166534"><strong>[绿色 核心结论]</strong></span>：RoboLab 的 success 判断依赖几何和接触，不只是语言和图像。几何缓存是让谓词检查可扩展的工程细节。

### 6.3 predicate state reset 为什么重要

某些谓词不是单帧判断，而是带历史状态。例如“对象曾经被抓起后又掉落”。`WorldState` 为这些 stateful predicate 提供存储，并在 env reset 时清理对应 env 的状态。

如果 reset 不干净，会出现：

- 新 episode 继承旧 episode 的“已经抓过”状态。
- frozen env 的状态被误清，导致结束后结果漂移。
- 多 env 中某个 env 的历史状态污染另一个 env。

## 7. `EventTracker`：把连续仿真变成可读失败事件

`EventTracker` 不是成功谓词本身，它更像“事故记录器”。它观察一系列异常事件：

| 事件类型 | 说明 |
|---|---|
| wrong object grabbed | 抓了非目标物体 |
| wrong object detached | 抓错物体后来又脱离 |
| gripper hit table | 夹爪撞桌 |
| gripper fully closed | 夹爪闭合状态变化 |
| object bumped/moved | 非目标物体被轻碰或明显移动 |
| object out of scene | 物体离开 workspace |
| object tipped over | 要求直立的物体倾倒 |
| target object dropped | 目标运输中掉落 |
| gripper hit object | 夹爪碰到非目标物体 |
| multiple objects grabbed | 同时抓住多个物体 |

### 7.1 为什么它要区分 active 和 frozen env

已经终止的 env 不应该继续产生失败事件。否则一个 episode 成功后，物体因为物理残余晃动产生事件，可能污染最终 reason。

所以 `EventTracker` 使用 active/frozen mask，只对活跃 env 继续记录事件。

### 7.2 事件不是逐帧大日志，而是稀疏变化

新版 event log 更像：

```json
{
  "schema_version": 2,
  "task": "SomeTask",
  "env_id": 0,
  "run": 0,
  "success": false,
  "final_step": 142,
  "events": [
    {"code": 201, "info": "Wrong object grabbed: 'mug_0'", "step": 37}
  ]
}
```

这比每一步都写完整状态更轻，也更适合做失败统计。

## 8. `RecorderManager` 和 HDF5：真正的轨迹证据

视频适合人看，但机器分析需要结构化数据。RoboLab 用 HDF5 保存每个 run 的轨迹。

典型结构可以理解为：

```text
run_0.hdf5
  data/
    demo_0/
      obs/
      actions/
      states/
      subtask/
        score
    demo_1/
      ...
```

### 8.1 为什么一个 run 一个 HDF5

`run_idx` 是顺序批次，`env_id` 是并行环境编号。所以：

- `run_0.hdf5` 包含这一批所有并行 env。
- `demo_0` 对应 env0。
- `demo_1` 对应 env1。

最终 episode id 通常按这个公式生成：

```text
episode_id = run_idx * num_envs + env_id
```

### 8.2 HDF5 和 JSONL 谁更权威

两者用途不同：

- HDF5 是轨迹和 subtask score 的底层证据。
- `episode_results.jsonl` 是每个 episode 的汇总索引。
- `log_<run>_env<id>.json` 是失败和事件解释。
- mp4 是人工检查画面。

<span style="color:#166534"><strong>[绿色 核心结论]</strong></span>：要复查一条结果，应该先用 `episode_results.jsonl` 定位 episode，再打开对应 HDF5、event log 和视频，而不是只看视频文件名。

## 9. `summarize.py`：把原始输出折叠成 episode 记录

`summarize_run()` 消费的是 `run_episode()` 的返回：

```text
env_results, subtask_status, timing
```

然后它做这些事：

1. 从 recorder 取每个 env 的事件列表。
2. 写 `log_<run_idx>_env<eid>.json`。
3. 读取 `run_<run_idx>.hdf5` 中 `demo_<env_id>` 的轨迹。
4. 计算轨迹 metrics。
5. 从 HDF5 的 `subtask/score` 读取最终 score。
6. 组装 `run_summary`。
7. append 到 `episode_results.jsonl`。

### 9.1 `score` 为什么优先从 HDF5 读

源码里把 HDF5 的 final-step subtask score 当作 canonical score。事件日志里也可能带 score，但 summary 优先用 HDF5 读到的 final score。

这能避免一个问题：event log 是稀疏事件流，最后一个 event 不一定等于最终状态。

### 9.2 `reason` 怎么来

失败时，reason 会优先来自最终 subtask info 或事件流中的最后关键事件。成功时，summary 会尽量回溯到成功相关事件，而不是被成功后短暂物理漂移污染。

<span style="color:#b45309"><strong>[橙色 易错点]</strong></span>：如果看到 `success=false` 但视频最后像成功了，要查 final score、终止谓词和最后 reason。可能是放置位置没过阈值、目标接触关系没稳定、或任务要求的后续子任务没完成。

## 10. `results.py`：统计、续跑和失败聚合

`results.py` 是评测结果的工具箱。它不是 step loop，但对复现实验很关键。

### 10.1 它做什么

| 能力 | 用途 |
|---|---|
| load episode results | 读取 JSONL 或旧 JSON |
| check run complete | 续跑时判断是否跳过 |
| update experiment results | 追加一条 episode summary |
| beta confidence interval | 自适应采样和统计置信区间 |
| summarize error reasons | 聚合失败原因 |
| wrong object stats | 统计常抓错的物体 |
| filter results | 按 task、tag、policy、attribute 分析 |

### 10.2 Beta credible interval 的意义

评测成功率不是只有一个点估计。比如 2/3 成功和 20/30 成功都是 66.7%，但可信度完全不同。RoboLab 里用 Beta 分布估计二项成功率区间，给自适应采样和 dashboard 更稳的统计依据。

说人话：样本少时，成功率数字不要太当真；样本变多，区间变窄，结论才更稳定。

## 11. dashboard loader：它是结果视图，不是结果来源

Dashboard 读取本地输出目录，找到：

- `episode_results.jsonl`
- 每个 task 的 HDF5。
- per-env event log。
- sensor/viewport 视频。
- 统计缓存和置信区间。

它负责展示，不负责重新判定成功。

<span style="color:#b45309"><strong>[橙色 易错点]</strong></span>：dashboard 里看不到视频，不一定是实验没跑。可能是 `video-mode=none`、文件名匹配失败、路径没同步、或只保存了 sensor 没保存 viewport。先查输出目录，再查 dashboard loader。

## 12. 从“一条复现”看完整数据流

```mermaid
sequenceDiagram
    participant R as runner.py
    participant E as episode.py
    participant C as Pi05 client
    participant S as Pi05 server
    participant Env as RoboLab env
    participant W as WorldState/EventTracker
    participant H as HDF5/log/video
    participant Sum as summarize.py/results.py

    R->>Env: create_env(task, num_envs, policy)
    R->>C: client_factory(args)
    R->>E: run_episode(env, cfg, run_idx, client)
    E->>Env: reset()
    E->>H: set run_idx HDF5 and video writers
    loop each sim step
        E->>C: infer(obs, instruction, env_id)
        C->>S: query only when chunk refresh is needed
        S-->>C: action chunk
        C-->>E: next action
        E->>Env: env.step(actions)
        Env->>W: update/query world predicates and events
        Env->>H: record obs/action/state/subtask score
        E->>H: write video frame
    end
    E-->>R: env_results, events, timing
    R->>Sum: summarize_run(...)
    Sum->>H: read HDF5, write event logs
    Sum->>Sum: build run_summary
    Sum->>H: append episode_results.jsonl
```

## 13. 新接一个 policy 应该改哪里

如果以后要把 RoboChallenge 的 Pi、ReKep、OpenVLA 或自研策略接进 RoboLab，不要一上来改 `episode.py`。优先按这个顺序：

1. 新建或复用一个 `InferenceClient` 子类。
2. 实现 `_extract_observation()`，确认相机、关节、夹爪从 RoboLab obs 中取对。
3. 实现 `_pack_request()`，确认服务端需要的 key、图像大小、dtype、范围。
4. 实现 `_query_server()`，确认 websocket/HTTP/本地函数调用。
5. 实现 `_unpack_response()`，确认输出 shape 是 `(horizon, action_dim)`。
6. 在 `_postprocess_chunk()` 里处理 gripper、单位、坐标系和 padding。
7. 写一个 `policies/<policy>/run.py`，只负责解析策略特有参数和传 `client_factory`。
8. 先 `num_envs=1` 跑单个简单任务，再跑复杂任务和多 env。

<span style="color:#166534"><strong>[绿色 核心结论]</strong></span>：RoboLab 希望你把策略差异封装在 client 里，而不是把每个策略的特殊逻辑塞进通用 episode loop。

## 14. 新接一个 robot 或 action space 应该查哪里

如果换机器人或控制器，风险主要在 action schema：

| 检查点 | 为什么重要 |
|---|---|
| action manager 的 `total_action_dim` | 决定 `actions` tensor 宽度 |
| env action term | 决定 action 每一维怎么解释 |
| gripper convention | 开/关方向、连续/离散、阈值 |
| joint order | 模型输出顺序必须和机器人控制顺序一致 |
| camera preset | 模型训练时的视角和评测视角要尽量对齐 |
| observation normalization | 图像范围、关节单位、夹爪单位 |

如果只换 robot，不处理这些约定，很容易出现“模型能看到目标，但动作完全不合理”。

## 15. 复现时的故障路由表

| 现象 | 最可能层 | 先看文件 |
|---|---|---|
| 没有生成输出目录 | 调度层 | `runner.py`、启动命令 |
| 任务直接跳过 | 调度层/结果层 | `episode_results.jsonl`、`check_run_complete` |
| Isaac 能启动但没有动作 | client 层 | Pi05 server 日志、`base_client.py`、Pi05 client |
| 有动作但抓不到 | action schema | action_dim、gripper 后处理、joint order |
| 视频显示成功但结果失败 | 谓词/summary 层 | event log、HDF5 score、termination predicate |
| 结果里没有 score | recorder/summary 层 | `run_<idx>.hdf5`、`subtask/score` |
| dashboard 不显示视频 | 展示层 | 文件路径、video mode、dashboard loader |
| 多 env 结果串扰 | episode/client 层 | `env_id` cache、active env、demo index |
| 4090 OOM | runtime 配置 | `--num-envs`、视频、并行任务数量 |

## 16. 这节和论文能力评估的关系

论文讲视觉、程序、关系能力，表面看是 benchmark 设计；源码落地时对应这些 runtime 点：

- 视觉能力失败，常落在观测图像、camera key、物体语义、颜色/大小识别和 wrong object event。
- 程序能力失败，常落在多子任务顺序、抓取释放、重定向、堆叠稳定性和 subtask score。
- 关系能力失败，常落在 WorldState 的空间谓词、对象集合、计数、连接词解析和 termination condition。

所以看结果时不要只说“模型不行”。要问它失败在哪个层：

```text
感知错了？
语言目标绑定错了？
动作 chunk 执行错了？
物理接触没稳定？
终止谓词没满足？
记录和汇总读错了？
```

这才是 RoboLab 作为评测框架的价值。

## 17. 最小可复现检查清单

一条完整复现至少要能回答：

- 用的是哪个 policy、哪个 task、哪个 instruction type？
- `num_envs`、`num_runs`、`video-mode` 是什么？
- Pi05/OpenPI server 是否有对应日志？
- 输出目录在哪里？
- 是否有 `run_0.hdf5`？
- 是否有 `log_0_env0.json`？
- `episode_results.jsonl` 里 success、score、reason 是什么？
- 视频和 HDF5 的 episode 是否能用 `run_idx/env_id` 对齐？
- 如果失败，失败原因是否来自事件日志或 final subtask info？
- 如果做对比，是否使用同一 task、同一初始扰动、同一视频/score 采集规则？

## 18. 本节来源记录

- RoboLab paper: https://arxiv.org/html/2604.09860
- RoboLab project page: https://research.nvidia.com/labs/srl/projects/robolab/
- RoboLab GitHub: https://github.com/NVlabs/RoboLab
- `runner.py`: https://github.com/NVlabs/RoboLab/blob/main/robolab/eval/runner.py
- `episode.py`: https://github.com/NVlabs/RoboLab/blob/main/robolab/eval/episode.py
- `base_client.py`: https://github.com/NVlabs/RoboLab/blob/main/robolab/eval/base_client.py
- Pi0/Pi05 client: https://github.com/NVlabs/RoboLab/blob/main/policies/pi0_family/client.py
- `world_state.py`: https://github.com/NVlabs/RoboLab/blob/main/robolab/core/world/world_state.py
- `event_tracker.py`: https://github.com/NVlabs/RoboLab/blob/main/robolab/core/task/event_tracker.py
- `summarize.py`: https://github.com/NVlabs/RoboLab/blob/main/robolab/eval/summarize.py
- `results.py`: https://github.com/NVlabs/RoboLab/blob/main/robolab/core/logging/results.py

In [ ]:
# ===== 精讲14-补充：核心运行时代码深挖轻量验证 =====
# 这个 cell 不启动 Isaac Sim，也不连接 Pi05 server。
# 它把精讲 14-补充中的运行时代码链路转成结构化检查，
# 用于确认 notebook 已覆盖从 runner 到 dashboard 的核心证据流。
import json

deep_runtime_layers = [
    {
        "id": "runner",
        "source": "robolab/eval/runner.py",
        "inputs": ["task", "tag", "num_envs", "num_runs", "instruction_type", "video_mode", "output_folder_name"],
        "responsibilities": [
            "parse_shared_eval_args",
            "defer_heavy_isaac_imports_until_run_evaluation",
            "select_task_envs",
            "create_output_dir",
            "resume_or_skip_completed_runs",
            "create_env",
            "create_policy_client",
            "call_run_episode",
            "call_summarize_run",
        ],
        "outputs": ["output_dir", "episode_results.jsonl", "per_task_output_folder"],
    },
    {
        "id": "episode_loop",
        "source": "robolab/eval/episode.py",
        "inputs": ["env", "env_cfg", "episode", "client", "headless", "save_videos", "video_mode"],
        "responsibilities": [
            "reset_env",
            "set_run_hdf5_file",
            "set_per_env_demo_index",
            "create_video_writers",
            "infer_actions_for_active_envs_only",
            "step_env",
            "collect_subtask_infos",
            "skip_video_for_frozen_envs",
            "release_video_writers",
            "reset_client_chunk_cache",
        ],
        "outputs": ["env_results", "subtask_status", "timing"],
    },
    {
        "id": "inference_client_contract",
        "source": "robolab/eval/base_client.py",
        "inputs": ["raw_obs", "instruction", "env_id"],
        "responsibilities": [
            "_extract_observation",
            "_pack_request",
            "_query_server",
            "_unpack_response",
            "_postprocess_chunk",
            "_build_visualization",
            "per_env_action_chunk_cache",
        ],
        "outputs": ["action", "viz"],
    },
    {
        "id": "pi05_adapter",
        "source": "policies/pi0_family/client.py",
        "inputs": ["over_shoulder_camera", "wrist_camera", "arm_joint_pos", "gripper_pos", "prompt"],
        "responsibilities": [
            "resize_and_pack_images",
            "pack_joint_and_gripper_state",
            "send_websocket_request",
            "unpack_action_chunk",
            "binarize_gripper_action",
            "flush_chunk_cache_after_reconnect",
        ],
        "outputs": ["open_loop_action_chunk"],
    },
    {
        "id": "world_state",
        "source": "robolab/core/world/world_state.py",
        "inputs": ["env", "body_name", "env_id"],
        "responsibilities": [
            "query_pose_velocity_geometry_contact",
            "support_batched_env_id_none_queries",
            "cache_static_local_geometry",
            "reset_stateful_predicate_latches_per_env",
        ],
        "outputs": ["pose", "velocity", "bbox", "dimensions", "contact", "predicate_state"],
    },
    {
        "id": "event_tracker",
        "source": "robolab/core/task/event_tracker.py",
        "inputs": ["env", "per_env_intended_objects", "frozen_mask"],
        "responsibilities": [
            "record_wrong_object_grabbed",
            "record_gripper_hit_table",
            "record_object_moved_or_bumped",
            "record_object_out_of_scene",
            "record_tipped_objects",
            "record_target_dropped",
            "respect_active_env_mask",
        ],
        "outputs": ["sparse_events_with_status_codes"],
    },
    {
        "id": "recorder_hdf5",
        "source": "robolab/core/logging/recorder_manager.py",
        "inputs": ["obs", "actions", "states", "subtask_score", "env_id", "run_idx"],
        "responsibilities": [
            "stream_episode_data",
            "write_run_level_hdf5",
            "store_demo_per_env",
            "persist_subtask_score",
        ],
        "outputs": ["run_<idx>.hdf5", "data/demo_<env_id>/subtask/score"],
    },
    {
        "id": "summarize_results",
        "source": "robolab/eval/summarize.py + robolab/core/logging/results.py",
        "inputs": ["env_results", "events", "hdf5", "timing", "env_cfg"],
        "responsibilities": [
            "write_per_env_event_log",
            "load_trajectory_metrics_from_hdf5",
            "read_canonical_final_score_from_hdf5",
            "build_run_summary",
            "append_episode_results_jsonl",
            "compute_beta_confidence_interval",
        ],
        "outputs": ["log_<run>_env<id>.json", "episode_results.jsonl", "confidence_intervals"],
    },
    {
        "id": "dashboard_view",
        "source": "dashboard/loaders/local.py",
        "inputs": ["episode_results.jsonl", "event_logs", "hdf5", "videos"],
        "responsibilities": [
            "load_offline_result_tree",
            "match_results_to_artifacts",
            "display_success_rate_score_ci_and_videos",
        ],
        "outputs": ["human_readable_dashboard"],
    },
]

client_hook_contract = {
    "_extract_observation": "把 RoboLab 原始 batched obs 转成单 env flat dict",
    "_pack_request": "把 flat obs 和 instruction 转成策略服务请求",
    "_query_server": "负责 websocket、HTTP 或本地推理调用",
    "_unpack_response": "把服务响应转成 (horizon, action_dim) action chunk",
    "_postprocess_chunk": "处理 gripper、padding、符号和单位等动作约定",
    "_build_visualization": "生成调试可视化图像",
}

def episode_id(run_idx: int, num_envs: int, env_id: int) -> int:
    return run_idx * num_envs + env_id

def needs_refresh(counter: int, horizon: int, has_chunk: bool = True) -> bool:
    return (not has_chunk) or counter >= horizon

def estimate_server_queries(num_steps: int, horizon: int) -> int:
    return (num_steps + horizon - 1) // horizon

def route_runtime_issue(symptom: str) -> str:
    symptom = symptom.lower()
    if "skip" in symptom or "output dir" in symptom:
        return "runner/results"
    if "server" in symptom or "websocket" in symptom or "timeout" in symptom:
        return "inference_client/pi05_adapter"
    if "shape" in symptom or "gripper" in symptom or "joint" in symptom:
        return "action_schema"
    if "success=false" in symptom or "score" in symptom or "predicate" in symptom:
        return "world_state/event_tracker/summarize"
    if "video" in symptom or "dashboard" in symptom:
        return "dashboard/artifact_matching"
    return "manual_trace_required"

required_layers = {
    "runner",
    "episode_loop",
    "inference_client_contract",
    "pi05_adapter",
    "world_state",
    "event_tracker",
    "recorder_hdf5",
    "summarize_results",
    "dashboard_view",
}
layer_ids = {layer["id"] for layer in deep_runtime_layers}

tests = [
    ("covers_all_runtime_layers", required_layers.issubset(layer_ids)),
    (
        "runner_keeps_import_lifecycle_explicit",
        "defer_heavy_isaac_imports_until_run_evaluation"
        in next(layer for layer in deep_runtime_layers if layer["id"] == "runner")["responsibilities"],
    ),
    (
        "episode_handles_active_and_frozen_envs",
        {"infer_actions_for_active_envs_only", "skip_video_for_frozen_envs"}.issubset(
            set(next(layer for layer in deep_runtime_layers if layer["id"] == "episode_loop")["responsibilities"])
        ),
    ),
    (
        "client_contract_has_six_hooks",
        set(client_hook_contract) == {
            "_extract_observation",
            "_pack_request",
            "_query_server",
            "_unpack_response",
            "_postprocess_chunk",
            "_build_visualization",
        },
    ),
    (
        "pi05_adapter_mentions_images_state_prompt_and_gripper",
        {"over_shoulder_camera", "wrist_camera", "arm_joint_pos", "gripper_pos", "prompt"}.issubset(
            set(next(layer for layer in deep_runtime_layers if layer["id"] == "pi05_adapter")["inputs"])
        )
        and "binarize_gripper_action"
        in next(layer for layer in deep_runtime_layers if layer["id"] == "pi05_adapter")["responsibilities"],
    ),
    ("episode_id_formula_matches_parallel_env_indexing", episode_id(3, 4, 2) == 14),
    ("chunk_refresh_logic_detects_empty_or_exhausted_chunk", needs_refresh(0, 15, False) and needs_refresh(15, 15, True)),
    ("chunk_refresh_logic_reuses_non_exhausted_chunk", not needs_refresh(7, 15, True)),
    ("server_query_estimate_respects_horizon", estimate_server_queries(31, 15) == 3),
    (
        "world_state_supports_batched_queries_and_predicate_reset",
        {"support_batched_env_id_none_queries", "reset_stateful_predicate_latches_per_env"}.issubset(
            set(next(layer for layer in deep_runtime_layers if layer["id"] == "world_state")["responsibilities"])
        ),
    ),
    (
        "event_tracker_records_failure_events_and_respects_masks",
        "respect_active_env_mask"
        in next(layer for layer in deep_runtime_layers if layer["id"] == "event_tracker")["responsibilities"],
    ),
    (
        "hdf5_is_registered_as_score_source",
        "data/demo_<env_id>/subtask/score"
        in next(layer for layer in deep_runtime_layers if layer["id"] == "recorder_hdf5")["outputs"],
    ),
    (
        "summarize_uses_hdf5_and_jsonl",
        {"read_canonical_final_score_from_hdf5", "append_episode_results_jsonl"}.issubset(
            set(next(layer for layer in deep_runtime_layers if layer["id"] == "summarize_results")["responsibilities"])
        ),
    ),
    ("routes_server_issue_to_client", route_runtime_issue("websocket server timeout") == "inference_client/pi05_adapter"),
    ("routes_success_false_to_semantic_chain", route_runtime_issue("success=false but video looks ok score missing") == "world_state/event_tracker/summarize"),
    ("routes_dashboard_issue_to_artifact_matching", route_runtime_issue("dashboard cannot show video") == "dashboard/artifact_matching"),
]

report = {
    "deep_runtime_layers": deep_runtime_layers,
    "client_hook_contract": client_hook_contract,
    "diagnostic_examples": {
        "episode_id_run3_env2_numenv4": episode_id(3, 4, 2),
        "queries_for_31_steps_horizon15": estimate_server_queries(31, 15),
        "success_false_route": route_runtime_issue("success=false but video looks ok score missing"),
    },
    "tests": [{"name": name, "passed": bool(ok)} for name, ok in tests],
    "all_passed": all(ok for _, ok in tests),
    "boundary": "Coverage check for EXPLAIN_14_deep_runtime_code_chain only; it does not execute Isaac Sim or connect to Pi05/OpenPI.",
}

print(json.dumps(report, ensure_ascii=False, indent=2))
for name, ok in tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert report["all_passed"], tests
write_status("core_code_runtime_deep_lightweight_tests", report)

## 0.21 全文精讲：审稿人视角评价与未来创新方向

下面这节来自本目录的 [EXPLAIN_15_reviewer_synthesis.md](./EXPLAIN_15_reviewer_synthesis.md)。它把论文从头到尾重新串起来，并换成“审稿人视角”看它到底强在哪里、主要风险在哪里、复现侧应该优先优化什么，以及未来可以往哪些创新方向继续做。

# 精讲 15：全文总梳理 + 审稿人视角评价 + 未来优化创新方向

> **【绿色标识｜核心结论】** 如果以审稿人的身份看，RoboLab 的主要价值不在于提出一个单点算法，而在于把“高保真仿真、任务生成、策略接入、细粒度诊断、扰动敏感性、真实世界相关性”打成一个可运行的评测框架。它更像一篇 benchmark/system paper，而不是纯算法 paper。
>
> **【蓝色标识｜来源路径】** 本节综合论文 Abstract、Introduction、III RoboLab、IV Experiments、V Limitations、VI Conclusion，以及 GitHub README 里的 benchmark、server-client policy、multi-env evaluation、dashboard、installation 和 hardware 信息。
>
> **【橙色标识｜审稿边界】** 下面的“审稿意见”是学习和复现视角的技术评价，不代表真实会议评审结论。它的目的不是给论文打标签，而是帮我们判断：这篇论文强在哪里、弱在哪里、复现时还要补哪些证据，以及未来可以从哪里做创新。

## 1. 从全文角度，一句话讲清楚这篇论文

RoboLab 想解决的问题是：

```text
现有机器人 benchmark 要么真实世界太贵，
要么仿真太低保真、太容易被训练集/环境重叠“刷满”，
导致我们不知道通用机器人策略到底有没有真正泛化。
```

它的回答是：

```text
用 Isaac Lab/Isaac Sim 搭一个高保真、可扩展、可自动评分的仿真评测平台；
用 RoboLab-120 覆盖视觉、过程化、关系三类能力；
用 success/score/subtask/event/SPARC/MNPE/dashboard 把失败拆细；
再和真实世界 RoboArena 做排序相关性验证。
```

说人话：

> RoboLab 不是教机器人怎么做任务，而是给通用机器人模型出一套更难、更真实、更能解释失败原因的考试。

## 2. 全文结构总图

```mermaid
flowchart TD
    A["问题动机<br/>真实评测贵，旧仿真易饱和"] --> B["RoboLab 框架<br/>scene + task + env + policy"]
    B --> C["RoboLab-120<br/>visual / procedural / relational"]
    B --> D["场景与任务生成<br/>LLM + predicates + solvers + validation"]
    C --> E["评测指标<br/>success / score / subtask / events / SPARC"]
    E --> F["敏感性分析<br/>MNPE: camera / lighting / object pose"]
    E --> G["实验结论<br/>当前 SOTA 泛化仍弱"]
    G --> H["真实世界验证<br/>RoboArena 排序相关"]
    H --> I["限制<br/>刚体桌面、接触丰富/变形物体不足、真实差距仍在"]
```

## 3. 审稿人视角：论文定位

### 论文类型

| 维度 | 判断 |
|---|---|
| 类型 | Benchmark / system / evaluation paper |
| 主要贡献 | 评测平台、任务集、诊断工具、扰动分析、真实相关性验证 |
| 不是重点 | 新 policy architecture、新训练算法、新控制器 |
| 适合读者 | 机器人评测、VLA 模型、仿真平台、Isaac/Omniverse、robot generalization |

### 我会给的总体评价

```text
倾向：Weak Accept / Accept 边界
理由：贡献完整、工程量大、问题重要、实验能暴露当前模型短板；
但真实世界相关性样本仍有限，任务覆盖仍偏桌面刚体，部分生成/仿真保真度的外部有效性还需要更强证明。
```

这里的判断比较务实：  
作为“提出新算法”的论文，它不强；作为“给领域提供评测基础设施和诊断方法”的论文，它很有价值。

## 4. 主要贡献拆解

| 贡献 | 审稿人会怎么看 | 强度 |
|---|---|---|
| RoboLab 框架 | 把任务、场景、机器人、策略和扰动绑定成可复现实验 | 强 |
| RoboLab-120 | 120 个新任务，按能力轴和难度组织 | 中强 |
| 细粒度指标 | success 之外有 score、subtask、event、trajectory metrics | 强 |
| MNPE 敏感性分析 | 用 posterior 看哪些扰动最影响成功 | 中强 |
| LLM scene/task generation | 提供可扩展任务生成路线和 validation/repair | 中强 |
| real-world verification | 和 RoboArena 排名相关，证明 proxy 价值 | 中 |
| GitHub/工具链 | 有 repo、dashboard、policy client、multi-env | 强，但复现成本高 |

## 5. 论文最强的地方

### 5.1 它把“泛化失败”拆细了

很多 benchmark 只给：

```text
success rate = 多少任务成功
```

RoboLab 给的是：

```text
success: 最后完成了吗
score: 做到哪一步了
subtask: 哪个中间条件过了
event: 抓错、掉落、碰撞、撞飞了吗
SPARC/path/speed: 动作质量如何
MNPE: 哪些扰动最影响结果
```

这对研究很重要。因为模型失败不是一个单一类别：

- 可能看错对象。
- 可能理解了对象但没抓稳。
- 可能会单步，不会组合。
- 可能 default instruction 会，vague instruction 不会。
- 可能外部相机能忍，腕部相机偏一点就崩。

**【绿色标识｜审稿赞点】** RoboLab 的真正强点是“诊断粒度”，不是“任务数量”本身。

### 5.2 它把 benchmark 饱和问题重新拉高难度

论文指出旧 benchmark 常有训练/测试域重叠，导致成功率容易饱和。RoboLab 的策略是：

- 使用真实世界训练策略。
- 在高保真仿真中做新任务。
- 任务按视觉、过程化、关系能力组合。
- 增加语言抽象、场景 clutter、长时序、扰动。

这让 benchmark 更像泛化考试，而不是训练集复述。

### 5.3 工程闭环比较完整

GitHub README 显示它不是只有论文图：

- RoboLab-120 任务。
- 自动成功/失败检测。
- server-client policy architecture。
- multi-environment parallel evaluation。
- dashboard。
- scene/task generation skills。
- Isaac Sim 5.0 / Isaac Lab 2.2.0 / uv sync 工程链路。

从复现角度，这种可运行工具链比只发 PDF 更有价值。

## 6. 主要问题和审稿质疑

### Major Concern 1：真实世界相关性还不够细

论文做了 RoboLab success rate 和 RoboArena Elo 的相关性比较。这个方向是对的，但审稿人会追问：

- 参与比较的策略数量是否足够？
- 相关性是 policy-level，不是 task-level。
- 是否有 motion-level 行为一致性？
- 仿真里成功/失败的原因和真实世界失败原因是否一致？

当前论文已经承认 deeper task-level and motion-level correlation 留给 future work。

**建议优化**：

```text
RoboLab score / event taxonomy / trajectory metrics
vs
真实机器人 event / human-labeled failure / trajectory smoothness
```

做成多层相关性，而不是只比较最终排名。

### Major Concern 2：任务覆盖仍偏刚体桌面

论文 limitations 明确说目前主要是 rigid-body tabletop scenes。  
这意味着它暂时不充分覆盖：

- cloth、cables、bags。
- force-control。
- compliant interaction。
- fine-grained friction。
- open-ended household tasks。

对 VLA 模型来说，这些恰好是现实世界很难的一部分。

**建议优化**：

增加分层 benchmark：

```text
RoboLab-Rigid
RoboLab-Deformable
RoboLab-ContactRich
RoboLab-LongHorizon
RoboLab-MobileManipulation
```

不要强行把所有任务塞进同一套谓词和评分系统。

### Major Concern 3：自动生成任务的有效性还需要长期治理

LLM 生成 scene/task 很有吸引力，但审稿人会关心：

- 生成任务是否有隐性偏差？
- 是否覆盖真实家庭任务分布？
- 是否会产生大量“看似合理但不重要”的任务？
- task validity 不等于 task usefulness。
- benchmark 是否会随着 LLM prompt 改动而漂移？

**建议优化**：

建立 task governance：

- 任务版本号。
- 任务生成 prompt 版本。
- object/scene coverage report。
- task usefulness human audit。
- adversarial task split。
- holdout scene/object templates。

### Major Concern 4：仿真保真度和可交互真实性仍需更强证据

RoboLab 的高保真很强，但机器人 manipulation 的关键不只是视觉。

审稿人会问：

- collision mesh 是否和 visual mesh 对齐？
- 物体质量/摩擦是否真实？
- 接触事件和真实接触是否一致？
- 容器、堆叠、重定向任务的物理可靠性如何？
- Gaussian/NuRec 场景进入仿真后，视觉和 collider 的误差如何量化？

**建议优化**：

加入 sim fidelity audit：

```text
visual alignment score
collision alignment score
physics parameter uncertainty
contact event agreement
robot trajectory agreement
```

### Major Concern 5：评测成本仍然偏高

README 里写完整任务需要大量 GPU 时间，官方建议 48GB+ VRAM。  
这对普通实验室或 4090 用户来说是门槛。

**建议优化**：

提供分层评测协议：

| 协议 | 用途 |
|---|---|
| RoboLab-Sanity-5 | 安装验证 |
| RoboLab-Smoke-20 | 模型接入验证 |
| RoboLab-Dev-40 | 快速迭代 |
| RoboLab-120 | 论文级主 benchmark |
| RoboLab-Stress | 长时序/扰动/高难专测 |

这样可以避免所有人一上来就跑完整 RoboLab-120。

## 7. 次要问题

| 问题 | 影响 | 建议 |
---|---|---|
| README 与当前 HEAD 的 `tests/` 路径不一致 | 新用户安装验证会困惑 | 加 diagnostic script 或修 README |
| 资产下载/LFS 速度慢 | 复现门槛高 | 发布分包 manifest 和断点校验工具 |
| 4090 只能低并行 | 用户误以为不能跑 | 明确 24GB 配置的推荐 task subset |
| dashboard 强依赖结果格式 | 自定义 policy 容易写错字段 | 提供 schema validator |
| LLM 生成任务 prompt 复杂 | 二次开发不易维护 | prompt version + validation report |

## 8. 从我们复现角度看到的优化点

我们这次 4090 复现暴露了几个实际问题：

1. 全量资产下载慢，且失败原因容易混在一起。
2. no-policy smoke 和真实 policy success 很容易被混淆。
3. 单任务成功视频容易让人误以为“复现了论文”。
4. Pi05 server、client、RoboLab runner 三层进程关系需要讲清楚。
5. 复杂任务失败时，必须同时看视频、event、HDF5、JSONL，不能只看最终 success。

所以，面向复现用户，RoboLab 最该补的是：

```text
一键分级验证脚本 + 资产完整性诊断 + 小型官方 subset + 结果 schema validator + 故障路由文档
```

## 9. 未来优化创新方向

下面按“可发论文/可做项目”的方向分层。

### 方向 1：RoboLab-RealCorrelation

目标：把仿真和真实世界的相关性从 policy-level 提升到 task-level、event-level、trajectory-level。

可以做：

- 同一任务在 RoboLab 和真实机器人上跑。
- 标注真实失败原因。
- 对齐 event taxonomy。
- 比较 subtask progression。
- 比较末端轨迹与接触时序。

创新点：

```text
不是只问“哪个模型排名更高”，而是问“仿真失败原因是否预测真实失败原因”。
```

### 方向 2：RoboLab-Adversarial Task Generation

目标：不是随机生成更多任务，而是自动生成“最能暴露模型弱点”的任务。

可以做：

- 从历史失败中学习弱点。
- LLM 生成针对性任务。
- solver 保证物理可行。
- 用 active learning 选择最有信息量的任务。

创新点：

```text
benchmark 从静态题库变成自适应考官。
```

### 方向 3：RoboLab-Causal MNPE

MNPE 当前更偏 posterior sensitivity。下一步可以做 causal analysis：

- camera pose 影响成功，是因为视觉输入变差，还是因为动作分布偏了？
- object distance 影响成功，是 reachability，还是训练数据偏置？
- lighting 不敏感是否真的稳健，还是任务太简单？

可以加入：

- causal graph。
- intervention design。
- counterfactual rollout。
- failure attribution。

创新点：

```text
从“哪些变量相关”推进到“哪些变量导致失败”。
```

### 方向 4：RoboLab-Deformable / Contact-Rich

扩展当前刚体桌面 benchmark：

- cloth folding。
- cable routing。
- bag opening。
- drawer/door/fridge contact-rich manipulation。
- force/torque-aware placement。

挑战：

- 成功谓词更难写。
- 物理仿真更难稳定。
- 单纯视觉成功不够，需要力/接触指标。

创新点：

```text
把通用机器人评测从 pick-place 推向真实家庭操作。
```

### 方向 5：RoboLab-NeuralScene / NuRec Integration

把精讲12里的 NuRec/Gaussian scene 路线接进 RoboLab：

```text
真实 camera/lidar
-> NuRec / Gaussian reconstruction
-> collider mesh alignment
-> RoboLab task insertion
-> policy evaluation
```

关键评估：

- visual realism。
- collider alignment。
- robot contact stability。
- policy transfer correlation。

创新点：

```text
从程序化高保真场景推进到真实场景级数字孪生评测。
```

### 方向 6：Low-Cost Evaluation Protocol for 24GB GPUs

这是很实用的方向。为 4090/3090 用户设计官方低成本协议：

- 自动测 VRAM。
- 自动选择 `num_envs`。
- 先跑 5 个 install tasks。
- 再跑 20 个 diagnostic tasks。
- 最后按需要扩展到 120。

创新点：

```text
让 benchmark 不只属于 48GB/80GB 用户。
```

### 方向 7：Unified Policy Adapter Benchmark

目前不同 VLA 模型接入成本高。可以做统一 adapter：

```text
RoboLab obs schema
-> adapter registry
-> policy request schema
-> action schema normalization
-> latency/action-horizon logging
```

覆盖：

- OpenPI/Pi05。
- RoboChallenge policy。
- GR00T。
- ReKep-style planner。
- future VLA APIs。

创新点：

```text
把“模型接入成本”从 benchmark 变量里剥离出去。
```

### 方向 8：Failure Report Generator

把 event、video、HDF5、score 自动变成 failure report：

```text
Episode failed because:
1. wrong object grabbed at step 52
2. target dropped at step 117
3. object never reached container relation
4. wrist camera was offset by 4cm
```

可以结合 VLM 自动看视频，但最终仍以 event/HDF5 为证据主线。

创新点：

```text
从 benchmark 分数推进到自动实验诊断助手。
```

## 10. 审稿意见模板

### Summary

本文提出 RoboLab，一个面向 task-generalist robot policies 的高保真仿真 benchmark。它包含 RoboLab-120 任务集，支持视觉、过程化、关系能力评估，提供自动成功检测、subtask scoring、event tracking、trajectory metrics、MNPE sensitivity analysis 和 dashboard，并通过与 RoboArena 排名相关性验证其真实世界 proxy 价值。

### Strengths

- 问题重要，直指当前 VLA/通用机器人评测瓶颈。
- benchmark 设计不只看 success rate，诊断粒度好。
- 工具链完整，有 repo、policy client、multi-env、dashboard。
- 任务生成和验证流程有扩展性。
- 敏感性分析能给出可操作的模型弱点。
- 与真实世界 benchmark 做了相关性验证。

### Weaknesses

- 真实世界验证仍偏 policy-level，缺少 task/event/motion-level 对齐。
- 任务覆盖仍偏刚体桌面，接触丰富和变形物体不足。
- LLM 生成任务的长期治理、偏差和版本稳定性还需要更多机制。
- 仿真保真度与真实接触/物理参数的一致性还可更系统量化。
- 复现成本高，普通 24GB GPU 用户需要官方分级协议。

### Questions for Authors

1. RoboLab 和 RoboArena 的相关性是否能按任务族、能力轴、事件类型进一步拆分？
2. 对自动生成任务，如何防止 benchmark 随 prompt/LLM 版本漂移？
3. 对 collision mesh、物体质量、摩擦和接触事件，有没有真实测量校准？
4. 是否计划发布小型标准 subset，支持 24GB GPU 上的快速比较？
5. 是否能提供 failure report 自动生成，连接 event log、video 和 HDF5？

### Overall

这是一篇实用价值很高的 benchmark/system paper。它的贡献不在单个算法，而在评测生态和诊断能力。作为领域工具，它值得关注；作为长期 benchmark，它还需要更强的真实世界对齐、更广任务物理覆盖、更稳定的任务治理和更低成本的复现协议。

## 11. 我们自己的后续路线建议

围绕这篇论文继续做，建议分三条线：

### 线 A：复现线

```text
完整稳定跑 Pi05 subset
-> 按 instruction type 分组
-> 跑 camera/object/background perturbation
-> 输出 dashboard/analysis 表
-> 再扩到更多任务
```

### 线 B：对比线

```text
Pi05 vs RoboChallenge pi vs ReKep
-> 同一任务、同一场景、同一评测输出
-> 比较 success/score/event/SPARC
```

### 线 C：创新线

```text
自动失败报告
-> 弱点驱动任务生成
-> 24GB GPU 低成本评测协议
-> NuRec/Gaussian real-scene integration
```

## 小结

全文视角下，RoboLab 的价值链是：

```text
高保真仿真
-> 可扩展任务/场景
-> 多策略接入
-> 自动成功与过程评分
-> 事件和轨迹诊断
-> 扰动敏感性分析
-> 真实世界排序相关
```

审稿人视角下，它的主要优点是：

```text
问题重要 + 工程完整 + 诊断粒度高 + 可扩展
```

主要短板是：

```text
真实相关性还不够细 + 任务物理覆盖有限 + 复现成本高 + 生成任务治理仍需加强
```

未来最有价值的创新方向是：

```text
真实-仿真多层相关性
自适应/对抗任务生成
因果敏感性分析
接触丰富/变形物体 benchmark
NuRec/Gaussian 真实场景接入
24GB GPU 低成本评测协议
统一 policy adapter
自动 failure report
```

In [ ]:
# ===== 精讲15：全文审稿视角轻量验证 =====
# 这个 cell 不代表正式审稿意见，也不新增论文实验。
# 它的作用是把“全文总梳理 + 审稿人评价 + 未来创新方向”变成可检查的结构化记录，
# 防止精讲只停留在长文本，后续复盘时可以直接检查覆盖面。

import json

reviewer_synthesis = {
    "paper_type": "benchmark_system_evaluation",
    "provisional_verdict": "weak_accept_to_accept_boundary_for_learning_review",
    "summary": (
        "RoboLab 最强的贡献不是单点策略算法，而是把高保真仿真、任务/场景生成、"
        "策略接入、细粒度诊断、扰动敏感性和真实世界相关性组织成一个可运行的评测框架。"
    ),
    "contributions": {
        "framework": "提供从 scene/task/env 到 policy rollout、日志、视频、指标和 dashboard 的完整评测闭环。",
        "benchmark": "构建 RoboLab-120，用 visual、procedural、relational 能力轴组织任务。",
        "diagnostics": "不只报告 success rate，还记录 score、事件失败、任务属性、对象数和子任务数。",
        "sensitivity": "用 MNPE/NPE 分析相机、光照、背景、纹理等扰动对策略表现的影响。",
        "generation": "用 LLM 生成语义场景和任务，再通过语法、资源、空间和物理检查做修复。",
        "real_world_proxy": "用 RoboArena 等真实机器人结果检查仿真 benchmark 的外部相关性。",
        "toolchain": "公开 GitHub、运行入口、dashboard、策略客户端和复现所需工程结构。",
    },
    "strengths": [
        "诊断粒度比只看二值成功率更高，能解释失败类型和能力短板。",
        "任务覆盖更接近真实桌面操作中的多对象、多属性、多步骤组合。",
        "generation + validation + feedback loop 让 benchmark 可扩展，而不是手写少量固定场景。",
        "同一框架能比较策略、扰动敏感性、任务难度和真实世界 proxy 排名。",
        "工程链路较完整，复现时可以追踪到视频、HDF5、JSONL、event log 和 dashboard。",
    ],
    "major_concerns": {
        "real_world_correlation_depth": (
            "真实世界相关性是重要亮点，但还需要更细粒度地说明哪些任务、哪些能力轴、"
            "哪些失败类型能稳定预测真实机器人表现。"
        ),
        "rigid_tabletop_scope": (
            "当前主要是刚体桌面操作，离软体、铰链、液体、工具使用和长程导航还有距离。"
        ),
        "task_generation_governance": (
            "LLM 生成任务很有扩展性，但需要更系统的版本控制、去重、难度校准和无效任务审计。"
        ),
        "sim_fidelity_audit": (
            "论文强调视觉/几何保真，但需要更多关于接触、摩擦、遮挡、质量、反光和 collision mesh 的误差分析。"
        ),
        "evaluation_cost": (
            "完整 RoboLab-120 对 24GB 4090 用户成本较高，需要官方级 subset、early-stop 和低成本统计协议。"
        ),
    },
    "minor_concerns": [
        "README 的测试命令和当前仓库文件面不完全一致，需要记录版本差异。",
        "资产 LFS 下载慢会影响复现体验，建议提供按任务解析依赖的 manifest。",
        "4090 复现需要明确 num_envs、视频开关、headless 和缓存策略。",
        "prompt、solver、validator 的版本最好和任务元数据一起固化。",
    ],
    "future_directions": {
        "real_correlation": "建立 RoboLab-RealCorrelation：按任务属性/失败类型预测真实机器人表现，而不是只做总体排名相关。",
        "adversarial_task_generation": "让 LLM/搜索器主动生成能区分策略能力边界的 hard cases，并用 validator 控制物理可行性。",
        "causal_mnpe": "把 MNPE 从相关性敏感分析推进到因果扰动设计，回答哪个场景因素真正导致失败。",
        "deformable_contact_rich": "加入软体、铰链、工具、液体、推拉旋拧等接触密集任务，突破刚体 tabletop 边界。",
        "neural_scene_nurec": "结合 NuRec、3DGS/3DGUT/3DGRT 等 neural scene 方法，让真实场景外观进入可交互评测闭环。",
        "low_cost_24gb_protocol": "设计 24GB GPU 友好的分层评测：single-env smoke、能力轴 subset、adaptive sampling、失败优先重跑。",
        "unified_policy_adapter": "统一 Pi0/Pi05、RoboChallenge、ReKep、RT-2/ACT 等策略的观测/action adapter 和结果 schema。",
        "failure_report_generator": "自动把 video、event、HDF5、trajectory metric 和 task metadata 汇总成可读 failure report。",
    },
    "our_next_routes": {
        "reproduction": [
            "保持 BananaInBowlTask 完整成功闭环作为 sanity baseline。",
            "继续按能力轴挑复杂任务补资产、跑视频、记录 failure reason。",
            "不要在资产未齐、策略 server 未稳、并行度未校准前盲跑完整 RoboLab-120。",
        ],
        "comparison": [
            "把 OpenPI pi05、RoboChallenge pi、ReKep 统一成同一张任务/输入/输出/失败类型表。",
            "先比较少量代表任务，再扩展到能力轴 subset。",
        ],
        "innovation": [
            "优先做 4090 低成本评测协议和自动 failure report。",
            "其次做 adversarial task generation 和真实世界相关性细分分析。",
        ],
    },
    "boundary": (
        "This is a learning-oriented reviewer synthesis, not an official peer-review decision "
        "and not a new experiment."
    ),
}

required_major_concerns = {
    "real_world_correlation_depth",
    "rigid_tabletop_scope",
    "task_generation_governance",
    "sim_fidelity_audit",
    "evaluation_cost",
}
required_future_routes = {
    "real_correlation",
    "adversarial_task_generation",
    "causal_mnpe",
    "deformable_contact_rich",
    "neural_scene_nurec",
    "low_cost_24gb_protocol",
    "unified_policy_adapter",
    "failure_report_generator",
}

review_tests = [
    ("classified_as_benchmark_system_paper", reviewer_synthesis["paper_type"] == "benchmark_system_evaluation"),
    ("has_seven_contribution_buckets", len(reviewer_synthesis["contributions"]) >= 7),
    (
        "strengths_cover_diagnostics_and_toolchain",
        any("诊断" in item for item in reviewer_synthesis["strengths"])
        and "toolchain" in reviewer_synthesis["contributions"],
    ),
    (
        "major_concerns_cover_five_review_axes",
        required_major_concerns.issubset(set(reviewer_synthesis["major_concerns"])),
    ),
    (
        "future_directions_cover_eight_routes",
        required_future_routes.issubset(set(reviewer_synthesis["future_directions"])),
    ),
    ("future_includes_24gb_protocol", "low_cost_24gb_protocol" in reviewer_synthesis["future_directions"]),
    ("future_includes_nurec_or_gaussian", "neural_scene_nurec" in reviewer_synthesis["future_directions"]),
    (
        "roadmap_has_reproduction_comparison_innovation",
        set(reviewer_synthesis["our_next_routes"]) == {"reproduction", "comparison", "innovation"},
    ),
    ("boundary_not_official_review", "not an official" in reviewer_synthesis["boundary"]),
    ("boundary_not_new_experiment", "not a new experiment" in reviewer_synthesis["boundary"]),
]

report = {
    "reviewer_synthesis": reviewer_synthesis,
    "tests": [{"name": name, "passed": bool(ok)} for name, ok in review_tests],
    "all_passed": all(ok for _, ok in review_tests),
    "boundary": "Coverage check for EXPLAIN_15 only; it does not execute Isaac Sim, policies, or RoboLab-120.",
}

print(json.dumps(report, ensure_ascii=False, indent=2))
for name, ok in review_tests:
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

assert report["all_passed"], review_tests
write_status("reviewer_synthesis_lightweight_tests", report)

## 0.22 Current RoboLab-120 Formal Run Status

This section is generated from [CURRENT_ROBOLAB120_STATUS.md](./CURRENT_ROBOLAB120_STATUS.md). It records the live Pi05 full-120 checkpoint on the RTX 4090, including manifest progress, error rows, test10 preflight, GPU state, and next analysis steps.

This is an in-progress checkpoint, not the final 120/120 score table.

# Current RoboLab-120 Reproduction Status (Pi05 / RTX 4090)

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
This is an in-progress checkpoint from 2026-06-20, not the final 120/120 result table. Final success-rate tables must be regenerated after the manifest reaches 120 rows.
</div>

## Progress

| Item | Value |
|---|---|
| Updated at | `2026-06-21T05:22:16+08:00` |
| Host | `y12` |
| Policy | `Pi05 / OpenPI pi05_droid_jointpos` |
| Run prefix | `robolab120_pi05_full_assetsfixed_20260620_170411` |
| State | `running` |
| Manifest progress | `110/120` |
| Rows with run+verify returncode 0 | `110/110` |
| Failed/error rows so far | `0` |
| Latest log line | ` 22%|██▏       | 199/900 [00:56<03:18,  3.53it/s]` |
| GPU used/total/util | `18894, 24564, 56` |

## Completed Preflight

- Required 120-task LFS assets and backgrounds were checked out before this clean run; `missing_count=0` was verified before launch.
- Preflight test10 run: `robolab120_pi05_assetsfixed_test10_20260620_162403`, `10/10` rows had `run_returncode=0` and `verify_returncode=0`.
- The formal run uses `NUM_ENVS=1`, `NUM_RUNS=1`, `VIDEO_MODE=all`, `STOP_ON_FAILURE=0`, which is the conservative RTX 4090 24GB setting.

## Recent Manifest Rows

| Task | run_returncode | verify_returncode | output_root |
|---|---:|---:|---|
| `TakeMeasuringSpoonOutTask` | `0` | `0` | `/home/yjl/codex_robolab_4090_20260619/RoboLab/output/robolab120_pi05_full_assetsfixed_20260620_170411_TakeMeasuringSpoonOutTask` |
| `TakeMugsOffOfShelfTask` | `0` | `0` | `/home/yjl/codex_robolab_4090_20260619/RoboLab/output/robolab120_pi05_full_assetsfixed_20260620_170411_TakeMugsOffOfShelfTask` |
| `TakeSpatulaOffShelfTask` | `0` | `0` | `/home/yjl/codex_robolab_4090_20260619/RoboLab/output/robolab120_pi05_full_assetsfixed_20260620_170411_TakeSpatulaOffShelfTask` |
| `ThrowAwayAppleTask` | `0` | `0` | `/home/yjl/codex_robolab_4090_20260619/RoboLab/output/robolab120_pi05_full_assetsfixed_20260620_170411_ThrowAwayAppleTask` |
| `ThrowAwaySnacksTask` | `0` | `0` | `/home/yjl/codex_robolab_4090_20260619/RoboLab/output/robolab120_pi05_full_assetsfixed_20260620_170411_ThrowAwaySnacksTask` |
| `ToolOrganizationBothTask` | `0` | `0` | `/home/yjl/codex_robolab_4090_20260619/RoboLab/output/robolab120_pi05_full_assetsfixed_20260620_170411_ToolOrganizationBothTask` |
| `ToolOrganizationTask` | `0` | `0` | `/home/yjl/codex_robolab_4090_20260619/RoboLab/output/robolab120_pi05_full_assetsfixed_20260620_170411_ToolOrganizationTask` |
| `ToolsPickingAllHammersTask` | `0` | `0` | `/home/yjl/codex_robolab_4090_20260619/RoboLab/output/robolab120_pi05_full_assetsfixed_20260620_170411_ToolsPickingAllHammersTask` |

## Current Errors

None so far.

## Evidence Files

- Manifest: `/home/yjl/roboplay/robolab_2604_09860_repro_jupyter/robolab_repro_artifacts/robolab120_pi05_full_assetsfixed_20260620_170411_task_run_manifest.jsonl`
- Status JSON: `robolab_repro_artifacts/current_robolab120_status.json`
- Output prefix: `/home/yjl/codex_robolab_4090_20260619/RoboLab/output/robolab120_pi05_full_assetsfixed_20260620_170411_<TaskName>`
- Per-task output should include `episode_results.jsonl`, HDF5, video, event/subtask logs, and artifact-check JSON.

## Next Steps

1. Wait for Pi05 full-120 to reach `120/120`.
2. Regenerate final tables with `analysis/read_results.py`, `scripts/summarize_ablation_outputs.py`, and `scripts/compare_policy_matrix_results.py`.
3. Pick a medium-success task for lighting/background/object-position perturbations.
4. Only then run the same task matrix through RoboChallenge pi or ReKep adapters; adapter-pending rows are not real zero scores.

# RoboLab Sample Videos

<div style="border:1px solid #bfdbfe; border-left:6px solid #2563eb; background:#eff6ff; padding:10px 12px; border-radius:6px; margin:12px 0;">
This folder intentionally contains only a few small MP4 samples for GitHub browsing. Full videos, HDF5 files, and raw episode logs stay on the RTX 4090 output disk.
</div>

| Video | Task | Size | Link | Note |
|---|---|---:|---|---|
| RoboLab-120 Pi05 full run sample: AnimalsInBinTask viewport | `AnimalsInBinTask` | 6.91 MB | [mp4](sample_videos/robolab120_animals_in_bin_viewport.mp4) | Viewport sample from the clean Pi05 full-120 run. Small enough for GitHub preview. |
| RoboLab-120 Pi05 full run sample: FruitsOnPlate3Task viewport | `FruitsOnPlate3Task` | 7.32 MB | [mp4](sample_videos/robolab120_fruits_on_plate3_viewport.mp4) | Viewport sample from the clean Pi05 full-120 run for a multi-object counting task. |
| Pi05 complex sample: Stack3RubiksCubeTask | `Stack3RubiksCubeTask` | 7.61 MB | [mp4](sample_videos/pi05_complex_stack3_rubiks_cube.mp4) | Complex stacking sample from the earlier Pi05 complex-task probe. |

## Evidence Boundary

- These videos are visual samples from the reproduction workflow, not the final RoboLab-120 score table.
- The final full-120 report should be based on `episode_results.jsonl`, artifact checks, HDF5 logs, and regenerated analysis tables after the manifest reaches 120/120.
- Keep only a small number of videos in Git. Complete video archives should use release artifacts or object storage.

## 0.23 已完成复现：简单任务闭环与复杂任务抽样

下面两份记录是当前最重要的实测证据：

- `COMPLETE_REPRO_pi05_banana_20260620.md`：一条完整成功闭环，证明 OpenPI pi05 -> RoboLab -> Isaac Sim -> 视频/HDF5/JSON 输出链路是通的。
- `COMPLEX_TASKS_pi05_20260620.md`：三个更复杂任务的抽样，证明模型在多物体、长时序、集合式目标上会明显变难。

读这两节时不要只看 success。更重要的是看输入指令、输出文件、步数、墙钟耗时、policy inference、视频规格和失败类型。

# RoboLab 单任务完整复现记录：Pi0.5 / BananaInBowlTask

## 结论

本次在 4090 远程主机上完成一条可交付的 RoboLab 复现闭环：使用 OpenPI `pi05_droid_jointpos` 策略，在 Isaac Sim / RoboLab 中运行 `BananaInBowlTask`，任务成功，并产出视频、HDF5 轨迹、事件日志和汇总结果。

## 运行环境

- 远程主机：`y12`
- GPU：NVIDIA GeForce RTX 4090，24GB VRAM
- 系统：Ubuntu 22.04.4
- RoboLab 仓库：`/home/yjl/codex_robolab_4090_20260619/RoboLab`
- OpenPI checkpoint：`/home/yjl/codex_robolab_4090_20260619/openpi_cache/openpi-assets-simeval/pi05_droid_jointpos`
- OpenPI server：`localhost:8000`

## 运行命令

```bash
cd /home/yjl/codex_robolab_4090_20260619/RoboLab

/home/yjl/.local/bin/uv run python policies/pi0_family/run.py \
  --policy pi05 \
  --remote-host localhost \
  --remote-port 8000 \
  --task BananaInBowlTask \
  --num-envs 1 \
  --num-runs 1 \
  --video-mode all \
  --output-folder-name pi05_banana_full_20260620_015206 \
  --headless \
  --device cuda:0
```

## 输出位置

远程输出目录：

```text
/home/yjl/codex_robolab_4090_20260619/RoboLab/output/pi05_banana_full_20260620_015206
```

本地同步目录：

```text
C:\Users\robot\Documents\成长学习库\robolab_2604_09860_repro_jupyter\remote_outputs\pi05_banana_full_20260620_015206
```

关键文件：

| 类型 | 文件 |
|---|---|
| 汇总结果 | `episode_results.jsonl` |
| 每环境事件日志 | `log_0_env0.json` |
| 轨迹数据 | `run_0.hdf5` |
| 策略相机视频 | `Pick_up_the_banana_and_place_it_in_the_bowl_0.mp4` |
| 第三人称 viewport 视频 | `Pick_up_the_banana_and_place_it_in_the_bowl_0_viewport.mp4` |
| 视频截图 | `viewport_mid.png` |

## 任务结果

| 指标 | 数值 |
|---|---:|
| 任务 | `BananaInBowlTask` |
| 指令 | `Pick up the banana and place it in the bowl` |
| 策略 | `pi05` |
| 成功 | `true` |
| episode step | `198` |
| 仿真时长 | `13.2 s` |
| wall time | `46.086 s` |
| policy inference avg | `13.6 ms` |
| env step avg | `214.2 ms` |
| video write avg | `4.9 ms` |
| FPS | `15` |

## 视频验证

策略相机视频：

- 分辨率：`1280x360`
- 帧数：`197`
- 时长：`13.134 s`
- 编码：`h264`
- 大小：`4,302,288 bytes`

第三人称 viewport 视频：

- 分辨率：`864x480`
- 帧数：`197`
- 时长：`13.134 s`
- 编码：`h264`
- 大小：`881,359 bytes`

## HDF5 轨迹内容

`run_0.hdf5` 包含：

- `actions`: `(198, 8)`，机器人动作序列
- `states/articulation/robot`: 机器人关节位置、速度、根位姿、根速度
- `states/rigid_object/banana`: 香蕉位姿和速度
- `states/rigid_object/bowl`: 碗位姿和速度
- `bbox`: 香蕉、碗、桌子的 3D bbox 和中心点
- `ee_pose`: 末端位置、姿态、线速度、角速度
- `initial_state`: 初始机器人、物体、相机状态

## 复现状态

这条复现已经完成：

- 策略服务可用
- Isaac Sim 环境成功启动
- RoboLab task 成功注册
- Pi0.5 策略完成推理闭环
- 任务成功
- 视频可读且非空白
- HDF5 轨迹和日志已保存

下一步对比实验建议以这条记录为模板，分别跑：

- RoboLab-120 全任务或分批任务
- RoboChallenge Pi baseline 的适配 smoke
- ReKep 方法的可行 smoke

# RoboLab 复杂任务复现实测记录 - OpenPI pi05 / RTX 4090

时间：2026-06-20  
远端机器：`y12` / RTX 4090 24GB  
RoboLab 路径：`/home/yjl/codex_robolab_4090_20260619/RoboLab`  
输出目录：`/home/yjl/codex_robolab_4090_20260619/RoboLab/output/pi05_complex_assets_ok_20260620_020721`

## 这次测了什么

前一个香蕉放碗任务太简单，所以这次改测多物体、多目标或更长时序的任务：

1. `ReorientAllMugsTask`：把多个杯子全部翻正，杯口朝上。
2. `Stack3RubiksCubeTask`：把 3 个魔方堆成塔。
3. `RedItemsInBinTask`：识别所有红色物体，并全部放入灰色收纳盒。

第一次尝试还包含 `BlockStackingSpecifiedOrderTask`，但该任务在本 checkout 中触发 contact sensor / asset 初始化失败：

```text
Sensor at path '/World/envs/env_.*/scene/red_block' could not find any bodies with contact reporter API.
```

这属于环境/资产配置失败，不是策略执行失败，所以没有计入下面的策略成功率。

## 总体结果

| 任务 | 能力轴 | 指令输入 | 成功 | 步数 | 仿真时长 | 墙钟耗时 | 平均策略推理 |
|---|---|---|---:|---:|---:|---:|---:|
| `ReorientAllMugsTask` | reorientation / complex | Reorient all the mugs upright so that the opening is facing upwards. | 否 | 1350 | 90.0s | 367.913s | 13.3ms |
| `Stack3RubiksCubeTask` | stacking / moderate | Stack the rubiks cubes in a tower | 是 | 353 | 23.53s | 84.072s | 13.4ms |
| `RedItemsInBinTask` | color / sorting / moderate | Put all the red things in the grey bin | 否 | 900 | 60.0s | 220.808s | 13.3ms |

结论：这 3 个较复杂任务中，pi05 成功 1 个，失败 2 个。失败不是程序崩溃，而是策略没有在最大步数内完成任务。

## 视频核验

每个任务都保存了主视频和 viewport 视频。`ffprobe` 已确认 mp4 可读：

| 任务 | 主视频 | 主视频规格 | viewport 规格 |
|---|---|---|---|
| `ReorientAllMugsTask` | `Reorient_all_the_mugs_upright_so_that_the_opening_is_facing_upwards_0.mp4` | 1280x360, 15 FPS, 89.93s, 1349 frames | 864x480, 15 FPS, 89.93s |
| `Stack3RubiksCubeTask` | `Stack_the_rubiks_cubes_in_a_tower_0.mp4` | 1280x360, 15 FPS, 23.47s, 352 frames | 864x480, 15 FPS, 23.47s |
| `RedItemsInBinTask` | `Put_all_the_red_things_in_the_grey_bin_0.mp4` | 1280x360, 15 FPS, 59.93s, 899 frames | 864x480, 15 FPS, 59.93s |

本地同步目录：

```text
C:\Users\robot\Documents\成长学习库\robolab_2604_09860_repro_jupyter\remote_outputs\pi05_complex_assets_ok_20260620_020721
```

远端原始目录：

```text
/home/yjl/codex_robolab_4090_20260619/RoboLab/output/pi05_complex_assets_ok_20260620_020721
```

## 关键观察

`Stack3RubiksCubeTask` 成功，说明 pi05 对短时序堆叠任务有一定泛化能力；它在 353 步内完成，没有跑满上限。

`ReorientAllMugsTask` 更难，因为它要求多个杯子的姿态都满足条件，不是只抓起一个物体。它跑满 1350 步仍未成功，是这次最能暴露策略长时序/多目标控制短板的任务。

`RedItemsInBinTask` 失败点在“所有红色物体”这个集合式目标上：需要视觉颜色识别、选择多个目标、逐个搬运和确认完成。它跑满 900 步没有达成成功条件，说明这个策略在多物体排序任务上还不稳定。

## 4090 运行情况

运行过程中显存接近 4090 上限，观测到约 `22551 MiB / 24564 MiB`。这解释了为什么不能盲目把并行环境数拉高；当前复现使用 `--num-envs 1` 是合理的。

这次完整命令：

```bash
cd /home/yjl/codex_robolab_4090_20260619/RoboLab
/home/yjl/.local/bin/uv run python policies/pi0_family/run.py \
  --policy pi05 \
  --remote-host localhost \
  --remote-port 8000 \
  --task ReorientAllMugsTask Stack3RubiksCubeTask RedItemsInBinTask \
  --num-envs 1 \
  --num-runs 1 \
  --video-mode all \
  --output-folder-name pi05_complex_assets_ok_20260620_020721 \
  --headless \
  --device cuda:0
```

## 结果文件

核心结果 JSONL：

```text
C:\Users\robot\Documents\成长学习库\robolab_2604_09860_repro_jupyter\remote_outputs\pi05_complex_assets_ok_20260620_020721\episode_results.jsonl
```

远端运行日志：

```text
C:\Users\robot\Documents\成长学习库\robolab_2604_09860_repro_jupyter\remote_outputs\pi05_complex_assets_ok_20260620_020721.log
```

主视频：

```text
C:\Users\robot\Documents\成长学习库\robolab_2604_09860_repro_jupyter\remote_outputs\pi05_complex_assets_ok_20260620_020721\ReorientAllMugsTask\Reorient_all_the_mugs_upright_so_that_the_opening_is_facing_upwards_0.mp4
C:\Users\robot\Documents\成长学习库\robolab_2604_09860_repro_jupyter\remote_outputs\pi05_complex_assets_ok_20260620_020721\Stack3RubiksCubeTask\Stack_the_rubiks_cubes_in_a_tower_0.mp4
C:\Users\robot\Documents\成长学习库\robolab_2604_09860_repro_jupyter\remote_outputs\pi05_complex_assets_ok_20260620_020721\RedItemsInBinTask\Put_all_the_red_things_in_the_grey_bin_0.mp4
```

## 下一步建议

如果继续做论文级对比，下一步不要直接跑完整 120 个任务。建议先按能力轴各抽 5-10 个任务，建立 `pi05`、RoboChallenge 的 pi、ReKep 三套方法的同一任务子集对比表。等确认每类任务资产都完整、不会卡在初始化错误后，再扩大到 RoboLab-120。

## 1. 阶段一：4090 机器前置检查

目标不是一次性证明能跑完整 benchmark，而是先确认系统符合 RoboLab 当前 README 的基础条件。

记录点：

- Ubuntu 版本、内核、Python、CUDA driver。
- RTX 4090 是否被 `nvidia-smi` 识别，空闲显存是多少。
- 磁盘空间。官方资产约 7GB，但视频、HDF5、缓存和模型权重会快速增长；建议仍预留 100GB+，完整实验最好 300GB 级别。
- `uv`、`ffmpeg`、`git-lfs`、`gcc-11` / `g++-11` 是否可用。

In [ ]:
# ===== 1. 定义前置检查命令 =====
# 每个 tuple 的第 1 项是日志标签，第 2 项是真正执行的 shell 命令。
preflight_commands = [
    ("os_release", "cat /etc/os-release"),  # 查看 Ubuntu 版本，RoboLab 要求 Ubuntu 22.04+。
    ("kernel", "uname -a"),  # 记录内核版本，方便排查驱动/Isaac 兼容性问题。
    ("disk", "df -h ."),  # 查看当前磁盘剩余空间，资产和模型权重会占用大量空间。
    ("gpu", "nvidia-smi"),  # 查看 GPU、驱动、显存占用和已有进程。
    ("driver_query", "nvidia-smi --query-gpu=name,driver_version,memory.total,memory.free --format=csv"),  # 提取关键 GPU 字段，便于写入报告。
    ("python", "python3 --version"),  # RoboLab 当前要求 Python 3.11。
    ("uv", "uv --version"),  # uv 是官方推荐的依赖同步工具。
    ("ffmpeg", "ffmpeg -version | head -n 2"),  # ffmpeg 用于视频/渲染输出相关流程。
    ("git_lfs", "git lfs version"),  # RoboLab 大量资产走 Git LFS。
    ("gcc", "gcc-11 --version | head -n 1"),  # 记录 gcc-11，便于排查编译型依赖。
    ("gpp", "g++-11 --version | head -n 1"),  # 记录 g++-11，便于排查 C++ 扩展编译问题。
]

# ===== 2. 逐条执行或 dry-run 前置检查 =====
preflight_results = []
for label, cmd in preflight_commands:
    preflight_results.append(
        run(cmd, execute=EXECUTE_PREFLIGHT, timeout=60, label=f"preflight:{label}")
    )

# ===== 3. 写出 preflight 状态文件 =====
# 即使命令是 dry-run，也会记录“计划检查哪些项目”，方便复现流程审计。
write_status(
    "preflight_status",
    {
        "execute_preflight": EXECUTE_PREFLIGHT,
        "is_linux": IS_LINUX,
        "commands": [{"label": label, "cmd": cmd} for label, cmd in preflight_commands],
    },
)

## 2. 阶段二：安装 RoboLab

当前推荐路线：

```bash
sudo apt install ffmpeg
git clone https://github.com/NVlabs/RoboLab
cd RoboLab
uv venv --python 3.11
source .venv/bin/activate
uv sync
uv run pytest tests/
```

注意：

- `uv sync` 会自动处理 Isaac Sim 5.0 + Isaac Lab 2.2.0。
- 首次运行非测试入口时设置 `OMNI_KIT_ACCEPT_EULA=Y`，避免 GUI/EULA 提示卡住。
- 4090 只有 24GB VRAM，不要直接照官方示例从 `--num-envs 10` 起步；先 `1`，确认输出完整后再试 `2/4`。

In [ ]:
# 把安装步骤集中在一个列表里，保证 dry-run 展示和真实执行使用同一套命令。
install_steps = [
    ("make_work_root", f"mkdir -p {shlex.quote(str(WORK_ROOT))}"),  # 创建 RoboLab 工作区父目录。
    ("clone_robolab", "git clone https://github.com/NVlabs/RoboLab"),  # 拉取 RoboLab 官方仓库。
    ("git_lfs_install", "git lfs install"),  # 启用 Git LFS 支持。
    ("git_lfs_pull", "git lfs pull"),  # 拉取 USD 场景、对象、机器人等大资产。
    ("uv_venv", "uv venv --python 3.11"),  # 创建 Python 3.11 虚拟环境。
    ("uv_sync", "uv sync"),  # 按 pyproject/lock 同步依赖，包括 Isaac Sim/Lab。
    ("freeze", "uv pip freeze"),  # 导出实际安装版本，作为复现证据。
]

if EXECUTE_INSTALL:
    # 真实安装路径：clone/更新仓库 -> 拉 LFS 资产 -> 用 uv 安装 Isaac/RoboLab 依赖。
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    if not ROBOLAB_DIR.exists():
        run("git clone https://github.com/NVlabs/RoboLab", cwd=WORK_ROOT, execute=True, check=True, timeout=1800)
    else:
        # 已存在的仓库只做 fast-forward 更新，避免悄悄覆盖本地修改。
        run("git rev-parse HEAD", cwd=ROBOLAB_DIR, execute=True, timeout=60, label="repo:current_head")
        run("git pull --ff-only", cwd=ROBOLAB_DIR, execute=True, check=False, timeout=600, label="repo:pull")
    run("git lfs install", cwd=ROBOLAB_DIR, execute=True, timeout=120, label="install:git_lfs_install")
    run("git lfs pull", cwd=ROBOLAB_DIR, execute=True, timeout=1800, label="install:git_lfs_pull")
    run("uv venv --python 3.11", cwd=ROBOLAB_DIR, execute=True, check=True, timeout=600, label="install:uv_venv")
    run("uv sync", cwd=ROBOLAB_DIR, env=OMNI_ENV, execute=True, check=True, timeout=7200, label="install:uv_sync")
    freeze = run("uv pip freeze", cwd=ROBOLAB_DIR, execute=True, timeout=300, label="install:freeze")
    (ARTIFACT_DIR / "uv_freeze.txt").write_text(freeze.get("stdout", ""), encoding="utf-8")
else:
    # dry-run 路径只记录命令，不修改本机文件和环境。
    for label, cmd in install_steps:
        target = WORK_ROOT if label in {"make_work_root", "clone_robolab"} else ROBOLAB_DIR
        run(cmd, cwd=target, execute=False, label=f"install:{label}")

write_status(
    "install_plan",
    {
        "execute_install": EXECUTE_INSTALL,
        "robolab_dir": str(ROBOLAB_DIR),
        "notes": [
            "Main path uses uv sync; do not build Isaac Sim from source unless debugging or developing Isaac Sim itself.",
            "Record uv sync duration and uv_freeze.txt after a real run.",
        ],
    },
)

## 3. 阶段三：安装验证

验证目标：

- Isaac Lab 可以 import。
- 任务定义有效。
- env factory 可用。
- 至少一个完整 episode 可以 headless 跑完。

通过标准：`uv run pytest tests/` 返回 0，并把完整 stdout/stderr 写入 `command_log.jsonl`。

In [ ]:
# 测试默认关闭，因为导入 Isaac/Kit 很慢且依赖 GPU/驱动环境。
run(
    "uv run pytest tests/",
    cwd=ROBOLAB_DIR,
    env=OMNI_ENV,
    execute=EXECUTE_TESTS,
    check=EXECUTE_TESTS,
    timeout=7200,
    label="verify:pytest",
)

## 4. 阶段四：无策略 smoke run

在接入 VLA / Pi0.5 / 自定义策略之前，先跑不依赖策略服务端的 sanity check。

记录点：

- 是否能 headless 启动。
- 是否生成 `output/` 目录、HDF5、subtask log、视频。
- GPU 显存峰值和每秒迭代数。

In [ ]:
# 这些示例先验证仿真、任务注册、日志导出链路，不涉及 VLA 策略。
no_policy_commands = [
    ("empty_episode", "python examples/run_empty.py --headless"),  # 最基础的环境初始化和空动作 episode。
    ("recorded_playback", "python examples/run_recorded.py --headless"),  # 检查录制轨迹回放入口。
    ("gripper_toggle", f"python examples/run_gripper_toggle.py --task {SMOKE_TASK} --headless"),  # 检查指定任务和夹爪动作入口。
]

for label, cmd in no_policy_commands:
    # check=False 表示即使某条 smoke 失败，也保留日志并继续看其它探针结果。
    run(
        cmd,
        cwd=ROBOLAB_DIR,
        env=OMNI_ENV,
        execute=EXECUTE_NO_POLICY_SMOKE,
        check=False,
        timeout=3600,
        label=f"smoke_no_policy:{label}",
    )

## 5. 阶段五：接入策略并运行单任务

RoboLab 使用 server-client 架构：模型/策略作为独立服务运行，RoboLab 通过轻量 inference client 调用它。

官方 quick run 示例使用 `--num-envs 10`。在 RTX 4090 上先用 `--num-envs 1`：

- 先确认策略服务端能启动。
- 再确认 RoboLab client 能拿到动作。
- 最后再逐步增加 `num_envs`。

建议第一任务：`BananaInBowlTask`。它在官方 VRAM guide 中属于较低压力任务，适合作为 4090 的第一条闭环验证。

In [ ]:
# RoboLab 这里是 client；Pi0/Pi05 模型本体需要单独的 OpenPI server。
policy_smoke_cmd = (
    f"uv run python policies/pi0_family/run.py "  # 使用 RoboLab 自带的 Pi0/Pi05 family runner。
    f"--policy {POLICY_NAME} "  # 选择策略名，例如 pi05。
    f"--task {SMOKE_TASK} "  # 指定第一条 smoke 任务。
    f"--num-envs {NUM_ENVS_4090_SMOKE} "  # 4090 首轮只开 1 个并行环境，降低 OOM 风险。
    f"--num-runs {NUM_RUNS} "  # 每个任务先跑 1 次，验证链路而不是做统计。
    f"--enable-subtask "  # 开启 subtask 进度记录，便于分析失败发生在哪一步。
    f"--headless"  # 无 GUI 运行，适合远端服务器。
)

# 在 OpenPI 权重下载完成、8000 端口 ready 前，这一步保持关闭。
run(
    policy_smoke_cmd,
    cwd=ROBOLAB_DIR,
    env=OMNI_ENV,
    execute=EXECUTE_POLICY_SMOKE,
    check=False,
    timeout=7200,
    label="policy_smoke:single_task",
)

## 6. 阶段六：4090 小子集评测计划

不要把完整 RoboLab-120 当成第一天目标。建议先做 5 个任务，覆盖三个能力轴：

| 任务 | 主要观察点 | 为什么放入第一批 |
|---|---|---|
| `BananaInBowlTask` | 语义识别 + pick/place | 低压力 smoke task |
| `RubiksCubeAndBananaTask` | conjunction / 多目标 | 检查语言组合是否稳定 |
| `RubiksCubeLeftOfBowlTask` | spatial relation | 观察左右空间关系推理 |
| `ReorientWhiteMugsTask` | procedural / reorientation | 检查重定向操作 |
| `Stack3RubiksCubeTask` | stacking / 多步骤 | 检查长 horizon 与失败模式 |

评测顺序：

1. `num_envs=1`，每个任务 1 run，确认不 OOM、输出完整。
2. 仍是 `num_envs=1`，每个任务 3-5 runs，观察方差。
3. 按显存余量尝试 `num_envs=2/4`。
4. 扩展到 visual / procedural / relational 各 10 个任务。

In [ ]:
# RoboLab 的 --task 接收多个任务名；这里拼成命令行参数。
subset_task_args = " ".join(SUBSET_TASKS)
subset_cmd = (
    f"uv run python policies/pi0_family/run.py "  # 使用同一个策略 runner，便于和单任务 smoke 对比。
    f"--policy {POLICY_NAME} "  # 使用同一个 policy，避免策略差异影响观察。
    f"--task {subset_task_args} "  # 一次传入多个任务名，形成小子集评测。
    f"--num-envs {NUM_ENVS_4090_CAUTIOUS} "  # 小子集可尝试 2 个并行环境，但仍保持保守。
    f"--num-runs {NUM_RUNS} "  # 先每个任务 1 run，确认稳定后再提高 run 数。
    f"--enable-subtask "  # 保存子任务进度，方便看 score 和失败阶段。
    f"--headless"  # 远端无桌面/节省资源时使用 headless。
)

print("Subset command:")
print(subset_cmd)
# 首次加载 Isaac、资产和模型可能很久，所以 timeout 留得比较长。
run(
    subset_cmd,
    cwd=ROBOLAB_DIR,
    env=OMNI_ENV,
    execute=EXECUTE_SUBSET_EVAL,
    check=False,
    timeout=24 * 3600,
    label="eval:subset_4090",
)

## 7. 结果读取与可视化

RoboLab 主要结果文件是 `episode_results.jsonl`。每行对应一个 episode，包含任务、策略、instruction、成功状态、score、耗时、轨迹指标和事件计数。

下面的 cell 会：

- 搜索 `ROBOLAB_DIR/output/**/episode_results.jsonl`。
- 没有真实结果时使用样例数据，确保 notebook 的图表流程可检查。
- 汇总任务成功率、能力轴/属性、难度、失败事件。

In [ ]:
def find_episode_results(root: Path):
    # RoboLab 可能把结果写到不同 output 子目录，所以递归查找。
    if not root.exists():
        return []
    return sorted(root.glob("output/**/episode_results.jsonl"))

def load_jsonl(path: Path):
    # 合并多个结果文件时保留来源路径，方便追溯每条记录来自哪里。
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row["_source_file"] = str(path)
            rows.append(row)
    return rows

result_files = find_episode_results(ROBOLAB_DIR)
rows = []
for path in result_files:
    rows.extend(load_jsonl(path))

if not rows:
    # 没有真实 policy 结果时，用样例数据保证表格/绘图 cell 可执行。
    # 注意：这些样例只验证 notebook 数据流，不代表真实 RoboLab 成绩。
    print("No real RoboLab results found yet. Using sample rows for visualization checks.")
    rows = [
        {
            "task_name": "BananaInBowlTask",
            "env_name": "BananaInBowlTask",
            "policy": POLICY_NAME,
            "instruction_type": "default",
            "attributes": ["visual", "semantics", "simple"],
            "success": True,
            "score": 1.0,
            "duration": 14.8,
            "episode_step": 222,
            "metrics": {"ee_sparc": -5.2, "ee_path_length": 1.1, "ee_speed_mean": 0.08},
            "events": {},
        },
        {
            "task_name": "RubiksCubeLeftOfBowlTask",
            "env_name": "RubiksCubeLeftOfBowlTask",
            "policy": POLICY_NAME,
            "instruction_type": "default",
            "attributes": ["relational", "spatial", "moderate"],
            "success": False,
            "score": 0.4,
            "duration": 50.0,
            "episode_step": 750,
            "metrics": {"ee_sparc": -8.9, "ee_path_length": 2.3, "ee_speed_mean": 0.05},
            "events": {"WRONG_OBJECT_GRABBED": 1},
        },
        {
            "task_name": "ReorientWhiteMugsTask",
            "env_name": "ReorientWhiteMugsTask",
            "policy": POLICY_NAME,
            "instruction_type": "specific",
            "attributes": ["procedural", "reorientation", "complex"],
            "success": False,
            "score": 0.2,
            "duration": 60.0,
            "episode_step": 900,
            "metrics": {"ee_sparc": -9.4, "ee_path_length": 3.0, "ee_speed_mean": 0.04},
            "events": {"TARGET_OBJECT_DROPPED": 2},
        },
    ]

def normalize_rows(rows):
    # 把嵌套的 metrics/events 拉平成普通列，便于 pandas 分组和绘图。
    flat = []
    for row in rows:
        metrics = row.get("metrics") or {}
        events = row.get("events") or {}
        attrs = row.get("attributes") or []
        flat.append(
            {
                "task_name": row.get("task_name") or row.get("env_name"),  # 任务名，用于按任务聚合成功率。
                "env_name": row.get("env_name"),  # RoboLab 注册环境名，通常和任务名一致。
                "policy": row.get("policy"),  # 本轮使用的策略名，例如 pi05。
                "instruction_type": row.get("instruction_type"),  # 指令类型：default/vague/specific 等。
                "attributes": ",".join(attrs) if isinstance(attrs, list) else str(attrs),  # 任务属性，如 spatial/color/stacking。
                "difficulty": next((x for x in attrs if x in {"simple", "moderate", "complex"}), "unknown") if isinstance(attrs, list) else "unknown",  # 从属性里抽取难度标签。
                "success": bool(row.get("success")),  # episode 是否整体成功。
                "score": row.get("score"),  # 子任务或进度分数，失败时也能反映部分完成度。
                "duration": row.get("duration"),  # episode 耗时。
                "episode_step": row.get("episode_step"),  # episode 实际步数。
                "ee_sparc": metrics.get("ee_sparc"),  # 末端执行器轨迹平滑度指标。
                "ee_path_length": metrics.get("ee_path_length"),  # 末端执行器轨迹长度。
                "ee_speed_mean": metrics.get("ee_speed_mean"),  # 末端执行器平均速度。
                "wrong_object_grabbed": events.get("WRONG_OBJECT_GRABBED", 0),  # 抓错物体次数。
                "target_object_dropped": events.get("TARGET_OBJECT_DROPPED", 0),  # 目标物体掉落次数。
            }
        )
    return flat

flat_rows = normalize_rows(rows)
if pd is None:
    print(json.dumps(flat_rows[:5], ensure_ascii=False, indent=2))
else:
    df = pd.DataFrame(flat_rows)
    display(df.head())
    # 先按任务聚合；后续可以再按属性、难度或能力轴继续聚合。
    summary = (
        df.groupby("task_name", dropna=False)
        .agg(
            episodes=("success", "size"),  # 每个任务累计 episode 数。
            success_rate=("success", "mean"),  # bool 均值就是成功率。
            mean_score=("score", "mean"),  # 平均进度/子任务分数。
            mean_duration_s=("duration", "mean"),  # 平均耗时。
            wrong_grabs=("wrong_object_grabbed", "sum"),  # 抓错物体总次数。
            dropped=("target_object_dropped", "sum"),  # 掉落总次数。
        )
        .reset_index()
        .sort_values("success_rate")
    )
    display(summary)
    summary.to_csv(ARTIFACT_DIR / "task_summary.csv", index=False)
    print("wrote", ARTIFACT_DIR / "task_summary.csv")

In [ ]:
if pd is not None and plt is not None:
    # 横向柱状图在任务数从几个扩展到几十个时仍然比较可读。
    plot_df = summary.copy()
    fig, ax = plt.subplots(figsize=(9, max(3, 0.45 * len(plot_df))))
    ax.barh(plot_df["task_name"], plot_df["success_rate"] * 100)
    ax.set_xlabel("Success rate (%)")
    ax.set_xlim(0, 100)
    ax.set_title("RoboLab subset success rate")
    for i, value in enumerate(plot_df["success_rate"] * 100):
        ax.text(min(value + 1, 98), i, f"{value:.1f}%", va="center")
    fig.tight_layout()
    out = ARTIFACT_DIR / "success_rate_by_task.png"
    fig.savefig(out, dpi=160)
    print("wrote", out)
else:
    # 没装绘图库时跳过绘图，保证 notebook 仍能执行完。
    print("Install pandas and matplotlib to render plots in this notebook.")

## 8. 与论文结果对照

对照时不要只看一个总成功率。建议至少拆成四层：

- overall：整体成功率、平均 score。
- ability axis：visual / procedural / relational。
- difficulty：simple / moderate / complex。
- failure mode：wrong object、drop、table hit、object out of scene、timeout。

论文表格给的是大规模基准上的整体趋势。你的 4090 小子集只是 smoke/subset，不应直接宣称复现了完整论文结论；只有完整任务覆盖、足够 episodes、相同/等价策略配置后，才做严格对比。

In [ ]:
# 把论文中的数值保存为参照锚点；小规模 smoke 不能直接对标完整 RoboLab-120。
paper_reference = [
    {
        "source": "arXiv 2604.09860v3 Table I",
        "label": "best reported policy row in table",
        "overall_success_pct": 28.0,
        "overall_score": 0.43,
        "note": "Use as a coarse orientation only; verify exact model identity and setup before formal comparison.",
    },
    {
        "source": "arXiv 2604.09860v3 Table I",
        "label": "GR00T N1.6",
        "overall_success_pct": 7.2,
        "overall_score": 0.17,
        "note": "Paper-scale baseline, not a target for a 5-task smoke subset.",
    },
]
ref_path = ARTIFACT_DIR / "paper_reference_orientation.json"
ref_path.write_text(json.dumps(paper_reference, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote", ref_path)

if pd is not None and "summary" in globals():
    # 这里的加权成功率只覆盖当前加载到的子集，不代表完整 benchmark。
    observed = {
        "subset_tasks": int(summary.shape[0]),
        "subset_episodes": int(summary["episodes"].sum()),
        "subset_success_pct": float(summary.eval("success_rate * episodes").sum() / summary["episodes"].sum() * 100),
        "warning": "This is a subset/smoke metric, not full RoboLab-120 reproduction.",
    }
    write_status("observed_vs_paper_orientation", observed)
    observed

## 9. 每日学习记录模板

每次执行后，把真实命令、报错、截图、指标写下来。复现类任务的关键不是“跑过”，而是能回放证据链。

In [ ]:
def append_learning_log(
    stage: str,
    goals=None,
    commands=None,
    problems=None,
    outputs=None,
    insights=None,
):
    # 每天/每阶段追加一段学习记录；marker 防止重复执行 notebook 时重复写入。
    goals = goals or []
    commands = commands or []
    problems = problems or []
    outputs = outputs or []
    insights = insights or []
    date = _dt.datetime.now().strftime("%Y-%m-%d")
    path = ARTIFACT_DIR / "learning_log.md"
    marker = f"## {date} - {stage}"
    if path.exists() and marker in path.read_text(encoding="utf-8"):
        print("learning log already contains", marker)
        return path
    block = []
    # 固定日志格式，后续可以直接粘贴到复现报告或做阶段对比。
    block.append(f"\n{marker}\n")
    block.append("### 今日目标\n")
    block.extend([f"- [ ] {x}\n" for x in goals] or ["- [ ] \n"])
    block.append("\n### 执行记录\n")
    if commands:
        block.append("```bash\n" + "\n".join(commands) + "\n```\n")
    else:
        block.append("```bash\n# command here\n```\n")
    block.append("\n### 遇到的问题及解决方案\n")
    block.append("| 问题 | 解决方案 |\n|---|---|\n")
    block.extend([f"| {p.get('problem', '')} | {p.get('solution', '')} |\n" for p in problems] or ["|  |  |\n"])
    block.append("\n### 关键输出/截图\n")
    block.extend([f"- {x}\n" for x in outputs] or ["- \n"])
    block.append("\n### 学习心得\n")
    block.extend([f"- {x}\n" for x in insights] or ["- \n"])
    path.open("a", encoding="utf-8").write("".join(block))
    print("appended", path)
    return path

append_learning_log(
    "环境配置准备",
    goals=["确认 Ubuntu / 驱动 / 4090 / Python 3.11 / uv / ffmpeg / git-lfs", "记录 RoboLab 当前 commit 与依赖版本"],
    commands=["nvidia-smi", "uv venv --python 3.11", "uv sync", "uv run pytest tests/"],
    insights=["4090 第一轮只跑 num_envs=1；完整 RoboLab-120 不作为安装当天目标。"],
)

## 10. 常见问题预案

| 问题 | 判断方式 | 处理 |
|---|---|---|
| `uv sync` 失败 | 看 `command_log.jsonl` 里的 stderr | 确认 Python 3.11、网络、磁盘、`uv` 版本；重试前保留日志 |
| 首次启动卡住 | 没有新日志、GPU 有占用 | 设置 `OMNI_KIT_ACCEPT_EULA=Y`，首次资产加载耐心等待 |
| OOM | `nvidia-smi` 显存接近 24GB，日志出现 CUDA/Vulkan OOM | `--num-envs 1`，关闭桌面重负载，减少视频/图像记录 |
| 渲染空白 | HDF5 有数据但视频空白 | 检查驱动、headless/EGL、Isaac Sim Vulkan 日志 |
| 任务失败率很高 | success 低但 score 有进展 | 拆看 subtask、wrong object、drop、timeout，不只看二元成功 |
| 小子集结果看似优于论文 | 任务太少或任务偏简单 | 标注为 subset smoke，不和论文完整 benchmark 直接比较 |

In [ ]:
# 最终 gate 同时检查本地 artifact 和远端 4090 证据。
# 判断标准故意保守：只有文件存在不算通过，还要看返回码、traceback 和导出结果。
remote_summary_for_status = {}
if REMOTE_SUMMARY_PATH.exists():
    remote_summary_for_status = json.loads(REMOTE_SUMMARY_PATH.read_text(encoding="utf-8"))
remote_install_ok = remote_summary_for_status.get("installation", {}).get("uv_sync_status") == "ok"
remote_smoke = remote_summary_for_status.get("smoke", {})
remote_smoke_ok = bool(
    # 无策略 smoke 只有在环境 setup、episode 导出都完成且没有 traceback 时才算通过。
    remote_smoke.get("completed_setup")
    and remote_smoke.get("exported_episode")
    and not remote_smoke.get("traceback_present")
)
remote_subset3_summary_for_status = {}
if REMOTE_SUBSET3_SUMMARY_PATH.exists():
    remote_subset3_summary_for_status = json.loads(REMOTE_SUBSET3_SUMMARY_PATH.read_text(encoding="utf-8"))
remote_subset3_ok = bool(
    # subset3 要求进程返回码干净，并至少导出 3 条 episode 记录。
    remote_subset3_summary_for_status.get("status_file_rc") == "0"
    and not remote_subset3_summary_for_status.get("traceback_present")
    and remote_subset3_summary_for_status.get("episodes_exported_count", 0) >= 3
)
remote_policy_summary_for_status = {}
if REMOTE_POLICY_SUMMARY_PATH.exists():
    remote_policy_summary_for_status = json.loads(REMOTE_POLICY_SUMMARY_PATH.read_text(encoding="utf-8"))
remote_no_policy = remote_policy_summary_for_status.get("no_policy_smoke", {})
remote_openpi = remote_policy_summary_for_status.get("openpi_policy_readiness", {})
remote_subset21_ok = bool(
    # subset21 的 summary 是累计文件，应该包含早先 Banana/subset3/subset10/subset19 的记录。
    remote_no_policy.get("subset10_status") == "0"
    and remote_no_policy.get("subset_more_status") == "0"
    and remote_no_policy.get("num_records_cumulative", 0) >= 21
    and {"PickDrillTask", "Stack3RubiksCubeTask"}.issubset(
        set(remote_no_policy.get("candidate_probe_successful_tasks", []))
    )
)
remote_openpi_env_ok = bool(
    # 这个 gate 只表示 OpenPI 环境、导入和 CLI 可用，不表示策略已经跑过。
    remote_openpi.get("openpi_client_install_status") == "0"
    and remote_openpi.get("openpi_uv_sync_status") == "0"
    and remote_openpi.get("openpi_verify_status") == "0"
)
pi05_policy_smoke_summary_for_status = {}
if (ARTIFACT_DIR / "pi05_policy_smoke_summary.json").exists():
    pi05_policy_smoke_summary_for_status = json.loads(
        (ARTIFACT_DIR / "pi05_policy_smoke_summary.json").read_text(encoding="utf-8")
    )
# 和 G9 分开：环境装好后，Pi05 checkpoint 下载/加载仍可能需要很多小时。
pi05_server_ready = bool(
    remote_openpi.get("server_port_8000_ready")
    or pi05_policy_smoke_summary_for_status.get("server_ready")
)
pi05_policy_smoke_ok = bool(
    pi05_policy_smoke_summary_for_status.get("checkpoint_verify_ok")
    and pi05_policy_smoke_summary_for_status.get("checkpoint_restored")
    and pi05_policy_smoke_summary_for_status.get("server_ready")
    and pi05_policy_smoke_summary_for_status.get("smoke_status") == "0"
    and any(row.get("success") is True for row in pi05_policy_smoke_summary_for_status.get("episode_records", []))
)

final_checklist = {
    # 本地 notebook 和记录文件 gate。
    "G0_preflight_saved": (ARTIFACT_DIR / "preflight_status.json").exists(),
    "G1_command_log_exists": COMMAND_LOG.exists(),
    "G2_local_install_freeze_saved_after_real_run": (ARTIFACT_DIR / "uv_freeze.txt").exists(),
    # 来自远端 RTX 4090 证据包的运行 gate。
    "G2_remote_4090_install_verified": remote_install_ok,
    "G3_task_summary_exists": (ARTIFACT_DIR / "task_summary.csv").exists(),
    "G4_learning_log_exists": (ARTIFACT_DIR / "learning_log.md").exists(),
    "G5_paper_reference_saved": (ARTIFACT_DIR / "paper_reference_orientation.json").exists(),
    "G6_remote_4090_no_policy_smoke_completed": remote_smoke_ok,
    "G7_remote_4090_subset3_no_policy_smoke_completed": remote_subset3_ok,
    "G8_remote_4090_subset21_no_policy_smoke_completed": remote_subset21_ok,
    # 策略准备 gate；只有 8000 端口真实监听后，G10 才能变为 true。
    "G9_openpi_environment_verified": remote_openpi_env_ok,
    "G10_pi05_policy_server_ready": pi05_server_ready,
    "G11_core_code_explanation_saved": (ARTIFACT_DIR / "core_code_reading_map.json").exists(),
    "G12_remote_4090_pi05_policy_smoke_success": pi05_policy_smoke_ok,
    "G13_real_to_sim_explain_saved": (NOTEBOOK_ROOT / "EXPLAIN_01_real_to_sim_eval.md").exists(),
    "G14_complete_pi05_banana_report_saved": (NOTEBOOK_ROOT / "COMPLETE_REPRO_pi05_banana_20260620.md").exists(),
    "G15_complex_task_report_saved": (NOTEBOOK_ROOT / "COMPLEX_TASKS_pi05_20260620.md").exists(),
    "G16_scene_task_env_explain_saved": (NOTEBOOK_ROOT / "EXPLAIN_02_scene_task_env_generation.md").exists(),
    "G17_task_generation_validation_explain_saved": (NOTEBOOK_ROOT / "EXPLAIN_03_task_generation_validation.md").exists(),
    "G18_taskgen_lightweight_tests_passed": (
        ARTIFACT_DIR / "taskgen_lightweight_validation_tests.json"
    ).exists(),
    "G19_competency_axes_difficulty_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_04_competency_axes_difficulty.md"
    ).exists(),
    "G20_competency_difficulty_tests_passed": (
        ARTIFACT_DIR / "competency_difficulty_lightweight_tests.json"
    ).exists(),
    "G21_sparc_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_05_sparc_trajectory_metric.md"
    ).exists(),
    "G22_sparc_lightweight_tests_passed": (
        ARTIFACT_DIR / "sparc_lightweight_tests.json"
    ).exists(),
    "G23_mnpe_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_06_mnpe_sensitivity_analysis.md"
    ).exists(),
    "G24_mnpe_lightweight_tests_passed": (
        ARTIFACT_DIR / "mnpe_lightweight_tests.json"
    ).exists(),
    "G25_global_overview_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_00_global_overview.md"
    ).exists(),
    "G25a_global_overview_structure_tests_passed": (
        ARTIFACT_DIR / "global_overview_structure_tests.json"
    ).exists(),
    "G26_baseline_method_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_07_baseline_method.md"
    ).exists(),
    "G27_baseline_method_lightweight_tests_passed": (
        ARTIFACT_DIR / "baseline_method_lightweight_tests.json"
    ).exists(),
    "G28_paper_experiments_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_08_paper_experiments.md"
    ).exists(),
    "G29_paper_experiments_algorithm_tests_passed": (
        ARTIFACT_DIR / "paper_experiments_algorithm_lightweight_tests.json"
    ).exists(),
    "G30_dtge_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_09_dtge.md"
    ).exists(),
    "G31_dtge_lightweight_tests_passed": (
        ARTIFACT_DIR / "dtge_lightweight_tests.json"
    ).exists(),
    "G32_prompt_design_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_10_prompt_design.md"
    ).exists(),
    "G33_prompt_design_lightweight_tests_passed": (
        ARTIFACT_DIR / "prompt_design_lightweight_tests.json"
    ).exists(),
    "G34_spatial_physical_feedback_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_11_spatial_physical_solver_feedback.md"
    ).exists(),
    "G35_spatial_physical_feedback_tests_passed": (
        ARTIFACT_DIR / "spatial_physical_feedback_lightweight_tests.json"
    ).exists(),
    "G36_gaussian_sim_methods_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_12_gaussian_sim_methods.md"
    ).exists(),
    "G37_gaussian_sim_methods_tests_passed": (
        ARTIFACT_DIR / "gaussian_sim_methods_lightweight_tests.json"
    ).exists(),
    "G38_remaining_core_topics_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_13_remaining_core_topics.md"
    ).exists(),
    "G39_remaining_core_topics_tests_passed": (
        ARTIFACT_DIR / "remaining_core_topics_lightweight_tests.json"
    ).exists(),
    "G39a_remaining_core_topics_deep_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_13_deep_evaluation_evidence_chain.md"
    ).exists(),
    "G39b_remaining_core_topics_deep_tests_passed": (
        ARTIFACT_DIR / "remaining_core_topics_deep_lightweight_tests.json"
    ).exists(),
    "G40_core_code_runtime_chain_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_14_core_code_runtime_chain.md"
    ).exists(),
    "G41_core_code_runtime_chain_tests_passed": (
        ARTIFACT_DIR / "core_code_runtime_chain_lightweight_tests.json"
    ).exists(),
    "G41a_core_code_runtime_deep_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_14_deep_runtime_code_chain.md"
    ).exists(),
    "G41b_core_code_runtime_deep_tests_passed": (
        ARTIFACT_DIR / "core_code_runtime_deep_lightweight_tests.json"
    ).exists(),
    "G42_reviewer_synthesis_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_15_reviewer_synthesis.md"
    ).exists(),
    "G43_reviewer_synthesis_tests_passed": (
        ARTIFACT_DIR / "reviewer_synthesis_lightweight_tests.json"
    ).exists(),
    "G44_recommended_reading_explain_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_16_recommended_reading.md"
    ).exists(),
    "G45_recommended_reading_tests_passed": (
        ARTIFACT_DIR / "recommended_reading_lightweight_tests.json"
    ).exists(),
    "G46_explain_source_question_index_saved": (
        NOTEBOOK_ROOT / "EXPLAIN_SOURCE_QUESTION_INDEX.md"
    ).exists(),
    "G47_explain_source_question_index_tests_passed": (
        ARTIFACT_DIR / "explain_source_question_index_tests.json"
    ).exists(),
    "note": "Local install gates depend on EXECUTE_INSTALL. Remote 4090 evidence is read from remote_logs/. G12 is a single-task Pi05 policy smoke, not a full RoboLab-120 benchmark.",
}
write_status("repro_status", final_checklist)
final_checklist